# Human–AI Loan Collaboration: Audited and Improved Analysis

## Public repository copy

This notebook contains the analytical workflow for the human–computer loan-decision study. It expects authorised, pseudonymised source files to be supplied locally; raw research data are not bundled with the public repository.

Executed outputs, local filesystem paths, execution timestamps, personal coder names, and participant-level quotation records have been removed from this copy. Aggregate outputs can be regenerated only in an approved local data environment. Do not commit a newly executed notebook or generated data outputs without completing the applicable data-governance review.

The notebook covers applicant-factor relationships, human–computer reliance, collaboration accuracy, Orange/Purple observed disparities, participant and round patterns, and transcript–behaviour relationships.

## Material corrections retained in this repository copy

- It is now **self-contained** and does not require unavailable project-specific modules or configuration files.
- Timeline parsing is corrected to **6,410 timestamped events**: 6,124 speech and 286 screen events.
- Human, computer and final accuracy are compared on the **same 1,920 cases**.
- The final-vs-initial accuracy gain is reported with participant-bootstrap uncertainty rather than asserted as definite synergy.
- Logistic models use **two-way cluster-robust uncertainty** for both participant and repeated applicant profile.
- Fairness interpretation compares signed and absolute group gaps. Final collaboration sharply reduces the computer gap but does not reduce the absolute gap relative to initial humans.
- AI-following rates in text analyses are calculated only for genuine initial human–computer disagreement cases.
- **Disagreement-clarification update:** this version explicitly distinguishes **initial** human–AI disagreement (the first human decision vs. the computer recommendation) from **final** human–AI disagreement (the final human decision, after reconsideration, vs. the computer recommendation). The four reliance categories (appropriate acceptance, appropriate resistance, overreliance, underreliance) remain defined only within the initial-disagreement subset, exactly as before — see Section 4 and `disagreement_clarification_change_log.md`.

# Part 1 — Study Overview

**Research purpose.** This notebook analyses how human loan decisions relate to an AI (computer) recommendation across an initial decision, an explicit or effective final decision, and the actual loan outcome, in order to characterise Human–AI reliance, agreement/disagreement patterns, collaboration accuracy, qualitative reasoning themes, and observed Orange/Purple background disparities.

**Study design.** 22 participants each completed 10 rounds, and each round presented the same 10 repeated applicant profiles (100 unique applicant profiles in total), giving a complete 22 × 10 × 10 grid of **2,200 applicant decision rows**. For every row, a participant recorded an **initial human decision**, was shown a **computer (AI) recommendation**, and then recorded a **final decision**. Each applicant profile carries a fixed **actual loan outcome** (Good/Bad) and a fixed **Orange/Purple background label**, both assigned by the study design rather than by the participant.

**Decision sequence per row.**
1. **Initial human decision** — the participant's own Approve/Reject judgement, before seeing the AI recommendation.
2. **AI (computer) recommendation** — the model's Approve/Reject prediction, shown to the participant next.
3. **Explicit final decision** — the participant's own Approve/Reject decision after seeing the AI recommendation, where one was recorded directly.
4. **Effective final decision** — the explicit final decision where recorded; where no explicit final decision was recorded, the initial human decision is carried forward, so that every row has a usable final decision. This carry-forward assumption is stated wherever the effective final decision is used, and is separately checked with an AI-versus-explicit-final sensitivity analysis that excludes carried-forward rows entirely (Part 6).
5. **Actual loan outcome** — the fixed Good/Bad ground truth for that applicant profile, used to score decision accuracy.

**Orange and Purple groups.** Every applicant profile carries a background label of Orange or Purple. These are observed group labels, not a validated protected characteristic with a documented assignment mechanism — see Part 9 and Part 14 for the fairness-reporting safeguards this implies.

**Key terminology used throughout this notebook.**
- *Initial Human–AI Agreement / Disagreement* — whether the **initial** human decision matches the AI recommendation.
- *Final Human–AI Agreement / Disagreement* — whether the **final** decision (explicit or effective, as stated) matches the AI recommendation.
- *AI–effective-final matched sample* — rows with both a valid AI recommendation and a valid effective final decision (Part 6, Branch 1).
- *AI–explicit-final matched sample* — the sensitivity-analysis subset restricted to rows with an explicitly recorded final decision, i.e. no carry-forward (Part 6, Branch 2).
- *Complete three-stage matched sample* — rows with a valid initial decision, AI recommendation and effective final decision, all three present (Part 7, Branch 3).

**Research questions.**
1. How often does the final human decision agree or disagree with the AI recommendation, and how sensitive is that answer to the carry-forward assumption?
2. How does an initial Human–AI disagreement resolve by the final decision — appropriate acceptance, appropriate resistance, overreliance or underreliance?
3. How accurate are the initial human decision, the AI recommendation and the final decision against the actual loan outcome, and what predicts final approval?
4. What Orange–Purple disparities are observed at each decision stage, and what are the limits on interpreting them causally?
5. What reasoning themes do participants express about AI and about their own judgement, and how do these relate to the observed decision patterns?

**Roadmap.** Part 2 covers reproducibility and setup; Part 3 loads and validates the source data; Parts 4–5 prepare and describe the decision records; Parts 6–7 analyse Human–AI agreement and the three-stage decision journey; Part 8 covers accuracy and behavioural modelling; Part 9 gathers the Orange–Purple observed-disparity analyses; Parts 10–13 cover the qualitative themes, their relations, and their integration with the quantitative findings; Parts 14–16 cover limitations, conclusions, and the reproducibility manifest.

*A note on notebook order: this notebook grew incrementally, and a small number of topics are analysed in more than one place at different levels of rigour (for example, an early descriptive treatment of Human–AI disagreement in Part 4/5, followed by the fully validated, denominator-checked treatment in Parts 6–7). Rather than physically re-ordering cells with deep, hard-to-verify variable dependencies -- which risks silently changing a validated result -- every such case is explicitly cross-referenced at the point where it recurs, and the later, more rigorous treatment is always the authoritative one.*

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
This study followed 22 participants across 10 rounds of 10 repeated applicant profiles, producing 2,200 decision rows with an initial human decision, an AI recommendation, a final decision and a fixed Orange/Purple background label per row. The key terms distinguishing initial, explicit-final and effective-final decisions are defined here and used precisely throughout. The next Part covers reproducibility and project setup.
</div>

# Part 2 — Reproducibility and Project Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, platform, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.sandwich_covariance import cov_cluster_2groups

SEED=20260725
rng=np.random.default_rng(SEED)
pd.set_option('display.max_columns',80)
pd.set_option('display.width',180)

def find_project_root(start=None, marker_globs=('combined_all_round_applicant_data*.csv',)):
    """Walk upward from `start` (default: current working directory) until a directory is
    found that recursively contains at least one file matching any of `marker_globs`."""
    start = Path(start) if start is not None else Path.cwd()
    for root in [start, *start.parents]:
        for pat in marker_globs:
            if next(root.rglob(pat), None) is not None:
                return root
    raise FileNotFoundError(
        f'Could not locate the project root: no file matching {marker_globs} was found '
        f'searching upward and recursively from {start}.')

def public_path(path):
    """Return a repository-relative display path without exposing a user's home path."""
    p = Path(path).resolve()
    try:
        return p.relative_to(PROJECT_ROOT.resolve()).as_posix()
    except (NameError, ValueError):
        return p.name

def _is_ignorable_path(path):
    """Excludes backup files, temporary files, notebook-generated output copies, and any
    file living inside an output folder (this notebook's own OUT tree, or any directory
    whose name signals it holds generated outputs rather than source data)."""
    name = path.name
    if re.search(r'(?i)(^~\$|\.bak$|\.tmp$|\.swp$|^\.|backup|checkpoint|autosave|temporary)', name):
        return True
    parts_lower = [p.lower() for p in path.parts]
    output_dir_markers = ('human_ai_loan_analysis_outputs', 'human_ai_loan_distinction_outputs',
                           '_outputs', 'cherry_on_top', 'enhanced', '.ipynb_checkpoints')
    if any(any(marker in part for marker in output_dir_markers) for part in parts_lower):
        return True
    return False

def find_best_source_file(root, glob_pattern, label, expected_subfolder=None, exact_name=None):
    """Recursively search `root` for files matching `glob_pattern`, then select among the
    candidates using this priority:
      1. file inside `expected_subfolder` (a path-substring, e.g. '01_Base_Data/01_Structured_Data')
      2. exact approved filename (`exact_name`, no numbered-variant suffix)
      3. most recently modified non-backup file
      4. otherwise stop and raise, displaying every remaining candidate for explicit review
    Backup/temporary/output-folder files are excluded before any of the above is applied.
    Never selects a file merely because its filename is shorter."""
    candidates = sorted(set(root.rglob(glob_pattern)))
    candidates = [c for c in candidates if not _is_ignorable_path(c)]
    if not candidates:
        raise FileNotFoundError(f'No source file found for {label} (pattern "{glob_pattern}") under {root}.')
    if len(candidates) == 1:
        print(f'Selected {label}: {public_path(candidates[0])}')
        return candidates[0]

    # Priority 1: inside the expected organised-project subfolder.
    pool = candidates
    if expected_subfolder is not None:
        in_subfolder = [c for c in pool if expected_subfolder.replace('\\', '/') in str(c).replace('\\', '/')]
        if len(in_subfolder) == 1:
            print(f'Selected {label} (in expected subfolder "{expected_subfolder}"): {public_path(in_subfolder[0])}')
            return in_subfolder[0]
        if in_subfolder:
            pool = in_subfolder

    # Priority 2: exact approved filename (no numbered-variant suffix).
    if exact_name is not None:
        exact = [c for c in pool if c.name == exact_name]
        if len(exact) == 1:
            print(f'Selected {label} (exact approved filename): {public_path(exact[0])}')
            return exact[0]
        if exact:
            pool = exact

    # Priority 3: latest modified.
    mtimes = {c: c.stat().st_mtime for c in pool}
    max_mtime = max(mtimes.values())
    latest = [c for c, t in mtimes.items() if t == max_mtime]
    if len(latest) == 1:
        print(f'Selected {label} (most recently modified of {len(pool)} candidates): {public_path(latest[0])}')
        return latest[0]

    # Priority 4: still ambiguous -- stop and display every candidate rather than guessing.
    raise RuntimeError(
        f'Could not unambiguously select a source file for {label}: {len(latest)} equally valid '
        f'candidates remain after applying subfolder / exact-name / latest-modified priority. '
        f'Candidates:\n' + '\n'.join(f'  - {c}' for c in latest))

def find_all_transcripts(root, glob_pattern='*_timeline_clean*.txt', expected_subfolder='01_Base_Data/02_Cleaned_Transcripts'):
    """Recursively find every transcript file under `root`, excluding backup/temporary/output
    files, preferring copies inside the expected organised-project subfolder when duplicates
    (e.g. identical session, different location) exist."""
    files = sorted(set(root.rglob(glob_pattern)))
    files = [f for f in files if not _is_ignorable_path(f)]
    if not files:
        raise FileNotFoundError(f'No transcript files found (pattern "{glob_pattern}") under {root}.')
    by_session = {}
    for f in files:
        session = session_name_from_transcript_path(f)
        by_session.setdefault(session, []).append(f)
    chosen = []
    for session, group in by_session.items():
        if len(group) == 1:
            chosen.append(group[0])
            continue
        in_subfolder = [g for g in group if expected_subfolder in str(g).replace('\\', '/')]
        pool = in_subfolder if len(in_subfolder) == 1 else group
        if len(pool) > 1:
            pool = sorted(pool, key=lambda p: p.stat().st_mtime, reverse=True)
            print(f'WARNING: {len(pool)} candidate transcripts found for session "{session}": '
                  f'{[public_path(p) for p in pool]} -- selecting the most recently modified.')
        chosen.append(pool[0])
    return sorted(chosen)

def session_name_from_transcript_path(path):
    """Strip the '_timeline_clean' suffix and any trailing numbered-variant marker (e.g. '(1)')
    before the .txt extension, recovering the original session name used as the key into the
    participant map (matches the CSV's 'source_folder' column)."""
    stem = path.stem
    return re.sub(r'_timeline_clean(\(\d+\))?$', '', stem)

PROJECT_ROOT = find_project_root()
print('Project root identified.')
DATA = PROJECT_ROOT
OUT = DATA / 'human_ai_loan_analysis_outputs'
for d in [OUT, OUT / 'data', OUT / 'tables', OUT / 'figures', OUT / 'reports']:
    d.mkdir(parents=True, exist_ok=True)

def encode(s): return s.map({'Approve':1.0,'Reject':0.0}).astype(float)
def correct(decision,outcome):
    truth=outcome.map({'Good':1.0,'Bad':0.0})
    z=pd.Series(np.nan,index=decision.index,dtype=float)
    m=decision.notna()&truth.notna(); z.loc[m]=(decision.loc[m]==truth.loc[m]).astype(float)
    return z

def bh(frame):
    x=frame.copy(); x['p_value_fdr']=np.nan
    m=x['p_value'].notna()&(x['term']!='Intercept')
    x.loc[m,'p_value_fdr']=multipletests(x.loc[m,'p_value'],method='fdr_bh')[1]
    return x

def fit_2way(formula,data):
    fit=smf.glm(formula,data=data,family=sm.families.Binomial()).fit()
    g1=pd.Categorical(data['participant_id']).codes
    g2=pd.Categorical(data['id']).codes
    cov,_,_=cov_cluster_2groups(fit,g1,g2)
    diag=np.diag(cov)
    if (diag < -1e-10).any(): raise ValueError('Invalid clustered covariance')
    se=np.sqrt(np.clip(diag,0,None)); z=fit.params.to_numpy()/se
    p=2*st.norm.sf(np.abs(z)); lo=fit.params.to_numpy()-1.96*se; hi=fit.params.to_numpy()+1.96*se
    return bh(pd.DataFrame({'term':fit.params.index,'coef':fit.params.to_numpy(),'std_error':se,
        'odds_ratio':np.exp(fit.params.to_numpy()),'ci_low':np.exp(lo),'ci_high':np.exp(hi),'p_value':p}))

def pboot(data,stat,n=500,seed=SEED):
    r=np.random.default_rng(seed); groups={k:v for k,v in data.groupby('participant_id',sort=False)}; keys=np.array(list(groups))
    vals=[]
    for _ in range(n):
        sample=r.choice(keys,len(keys),replace=True)
        vals.append(float(stat(pd.concat([groups[k] for k in sample],ignore_index=True))))
    return float(stat(data)),*map(float,np.quantile(vals,[.025,.975]))

def group_metrics(data,col):
    rows=[]
    for bg,g in data.groupby('background',observed=True):
        c=g[g[col].notna()]; pred=c[col].eq(1); truth=c.loan_performance.eq('Good')
        tp=(pred&truth).sum(); fp=(pred&~truth).sum(); fn=(~pred&truth).sum(); tn=(~pred&~truth).sum()
        rows.append({'background':bg,'n_total':len(g),'n_completed':len(c),'completion_rate':len(c)/len(g),
            'approval_rate':pred.mean(),'true_positive_rate':tp/(tp+fn),'false_positive_rate':fp/(fp+tn),
            'false_negative_rate':fn/(tp+fn),'accuracy':(pred==truth).mean(),
            'positive_predictive_value':tp/(tp+fp) if tp+fp else np.nan,'mean_earnings':c.earnings.mean()})
    return pd.DataFrame(rows)

def gap(metrics,metric):
    x=metrics.set_index('background'); return float(x.loc['Purple',metric]-x.loc['Orange',metric])

def savefig(fig,name):
    fig.tight_layout(); fig.savefig(OUT/'figures'/f'{name}.png',dpi=180,bbox_inches='tight'); fig.savefig(OUT/'figures'/f'{name}.svg',bbox_inches='tight'); plt.show()


def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1<<20),b''): h.update(chunk)
    return h.hexdigest()

print('Data directory:', public_path(DATA))
print('Output directory:', public_path(OUT))

## 2.1 Notebook-Wide Visual Style System

**What this is:** A single, central set of colour constants and helper functions, defined once here and reused by every chart and table in this notebook instead of being redefined ad hoc in individual cells.

**Reserved-colour rule:** Orange and purple are reserved exclusively for visuals that explicitly compare the Orange applicant group and the Purple applicant group (Part 9). They are never used decoratively elsewhere -- not for agreement/disagreement, AI-versus-human comparisons, accuracy, themes, validation, flow diagrams or models. Setting the neutral palette as matplotlib's *default* colour cycle below means any chart that does not explicitly request the Orange/Purple group colours cannot accidentally render them.

In [ ]:
from cycler import cycler

# --- Neutral, non-group palette -----------------------------------------------------------
# PALETTE_PRIMARY_BLUE reuses this notebook's own pre-existing neutral blue (already used
# consistently across dozens of figures below, e.g. the Human-AI Decision Analysis tree) rather
# than introducing a second, near-identical blue, so every chart -- old and new -- stays visually
# consistent. Every other neutral colour follows the requested palette exactly.
PALETTE_PRIMARY_BLUE = '#2a78d6'
PALETTE_DARK_NAVY    = '#1E3A5F'
PALETTE_TEAL         = '#0F766E'
PALETTE_GREEN        = '#15803D'
PALETTE_MUTED_RED    = '#B91C1C'
PALETTE_LIGHT_BLUE_GREY = '#E8EEF5'
PALETTE_GREY         = '#64748B'
PALETTE_LIGHT_GREY   = '#F1F5F9'
PALETTE_DARK_TEXT    = '#1F2937'
PALETTE_WHITE        = '#FFFFFF'

# --- RESERVED: Orange/Purple-group comparison visuals ONLY (Part 9) ------------------------
# Do not use GROUP_ORANGE / GROUP_PURPLE for any other chart, table or diagram in this notebook.
GROUP_ORANGE = '#EB6834'   # this notebook's pre-existing "orange" hue, now scoped to Orange-group visuals only
GROUP_PURPLE = '#7A4FB5'   # this notebook's pre-existing "purple" hue, now scoped to Purple-group visuals only

VALIDATION_COLORS = {'PASS': PALETTE_GREEN, 'FAIL': PALETTE_MUTED_RED, 'N/A': PALETTE_GREY, 'Not applicable': PALETTE_GREY}
AGREEMENT_COLOR, DISAGREEMENT_COLOR = PALETTE_PRIMARY_BLUE, PALETTE_MUTED_RED
INITIAL_COLOR, FINAL_COLOR = PALETTE_PRIMARY_BLUE, PALETTE_TEAL

# Installing the neutral colours as matplotlib's default cycle means any chart below that does
# not explicitly pass color=GROUP_ORANGE / color=GROUP_PURPLE can never accidentally render them.
plt.rcParams['axes.prop_cycle'] = cycler(color=[
    PALETTE_PRIMARY_BLUE, PALETTE_TEAL, PALETTE_GREEN, PALETTE_GREY, PALETTE_DARK_NAVY, PALETTE_MUTED_RED])
plt.rcParams.update({
    'font.size': 10.5, 'axes.titlesize': 13, 'axes.titleweight': 'bold', 'axes.labelsize': 10.5,
    'axes.edgecolor': PALETTE_GREY, 'axes.linewidth': 0.9,
    'axes.grid': True, 'grid.color': PALETTE_LIGHT_GREY, 'grid.linewidth': 0.8, 'grid.alpha': 0.9,
    'axes.axisbelow': True, 'legend.frameon': False, 'legend.fontsize': 9.5,
    'xtick.labelsize': 9.5, 'ytick.labelsize': 9.5, 'figure.dpi': 110, 'savefig.dpi': 200,
    'text.color': PALETTE_DARK_TEXT, 'axes.labelcolor': PALETTE_DARK_TEXT,
    'xtick.color': PALETTE_DARK_TEXT, 'ytick.color': PALETTE_DARK_TEXT,
})

def style_axes(ax, subtitle=None, footnote=None):
    """Apply the notebook-wide chart style to an existing axes: restrained spines, a small
    italic subtitle above the plot and/or an italic footnote below it. Titles/axis labels are
    left to the calling cell (set_title/set_xlabel/set_ylabel) since only that cell knows the
    right wording; this only standardises the surrounding presentation."""
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    for spine in ('left', 'bottom'):
        ax.spines[spine].set_color(PALETTE_GREY)
    if subtitle:
        ax.text(0.0, 1.03, subtitle, transform=ax.transAxes, fontsize=9.6, color=PALETTE_GREY, style='italic', ha='left', va='bottom')
    if footnote:
        ax.text(0.0, -0.18, footnote, transform=ax.transAxes, fontsize=8.2, color=PALETTE_GREY, style='italic', ha='left', va='top')
    return ax

def style_table(df, header_color=PALETTE_DARK_NAVY, pct_cols=None, int_cols=None):
    """Return a pandas Styler applying the notebook-wide table style: bold header in
    `header_color` (neutral navy by default; pass GROUP_ORANGE/GROUP_PURPLE only for tables
    that are themselves split into Orange-group / Purple-group columns or rows), visible outer
    and cell borders, alternating row shading, right-aligned numbers, percentages to two
    decimal places, counts as integers, and 'Missing' instead of a raw NaN."""
    d = df.copy()
    fmt = {}
    for c in (pct_cols or []):
        if c in d.columns:
            fmt[c] = (lambda v: 'Missing' if pd.isna(v) else f'{v:.2f}%')
    for c in (int_cols or []):
        if c in d.columns:
            fmt[c] = (lambda v: 'Missing' if pd.isna(v) else f'{int(v):,}')
    return (d.style
            .set_table_styles([
                {'selector': 'th', 'props': [('background-color', header_color), ('color', PALETTE_WHITE),
                                              ('font-weight', 'bold'), ('border', f'1px solid {PALETTE_GREY}'),
                                              ('padding', '6px 10px'), ('text-align', 'center')]},
                {'selector': 'td', 'props': [('border', f'1px solid {PALETTE_LIGHT_GREY}'), ('padding', '5px 10px')]},
                {'selector': 'table', 'props': [('border-collapse', 'collapse'), ('border', f'1.5px solid {PALETTE_GREY}')]},
                {'selector': 'tr:nth-child(even)', 'props': [('background-color', PALETTE_LIGHT_GREY)]},
            ])
            .set_properties(**{'text-align': 'right'})
            .format(fmt, na_rep='Missing'))

def section_summary_html(points, header_color=PALETTE_LIGHT_BLUE_GREY, border_color=PALETTE_PRIMARY_BLUE):
    """Render a neutral blue-grey 'Section Summary' box (never orange/purple) from a short list
    of point strings, for display via IPython.display.HTML/Markdown at the end of each Part."""
    items = ''.join(f'<li style="margin-bottom:4px;">{p}</li>' for p in points)
    return (f'<div style="background-color:{header_color}; border-left:5px solid {border_color}; '
            f'padding:12px 16px; border-radius:4px; margin:10px 0;">'
            f'<div style="font-weight:bold; color:{PALETTE_DARK_TEXT}; margin-bottom:6px;">Section Summary</div>'
            f'<ul style="margin:0; padding-left:18px; color:{PALETTE_DARK_TEXT};">{items}</ul></div>')

print('Central visual style system loaded: neutral palette installed as the default colour cycle;')
print('GROUP_ORANGE / GROUP_PURPLE reserved exclusively for Orange-Purple group-comparison visuals (Part 9).')

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
The notebook's imports, random seed, portable project-root/source-file detection and the central visual style system (neutral palette, with orange and purple reserved for Part 9) are established here. Source-file SHA-256 hashes are captured before any analysis touches them. The next Part loads and validates the source data itself.
</div>

# Part 3 — Data Loading and Validation

In [ ]:
csv = find_best_source_file(PROJECT_ROOT, 'combined_all_round_applicant_data*.csv', 'source CSV',
                             expected_subfolder='01_Base_Data/01_Structured_Data', exact_name='combined_all_round_applicant_data.csv')
book = find_best_source_file(PROJECT_ROOT, 'all_round_summary_table_data_restructured*.xlsx', 'source Excel workbook',
                              expected_subfolder='01_Base_Data/01_Structured_Data', exact_name='all_round_summary_table_data_restructured.xlsx')

# Real SHA-256 integrity check: hash every source file now, before any analysis touches it,
# so the same hash can be recalculated after all analysis/exports and compared.
source_file_hashes_before = {}
source_file_hashes_before[Path(csv)] = sha(csv)
source_file_hashes_before[Path(book)] = sha(book)
raw=pd.read_csv(csv); round_book=pd.read_excel(book,sheet_name='Round Results')
sessions=sorted(raw.source_folder.unique()); pmap={s:f'P{i+1:02d}' for i,s in enumerate(sessions)}
df=raw.copy(); df['participant_id']=df.source_folder.map(pmap); df=df.drop(columns='source_folder')
checks={
 '22 participants':df.participant_id.nunique()==22,'2,200 rows':len(df)==2200,'220 round-workbook rows':len(round_book)==220,
 '100 applicant profiles':df.id.nunique()==100,'10 rounds':set(df['round'])==set(range(1,11)),
 'complete 22x10x10 grid':(df.groupby(['participant_id','round']).size()==10).all(),
 'unique participant-round-position':not df.duplicated(['participant_id','round','row_in_round']).any(),
 'valid decisions':df.initial_decision.dropna().isin(['Approve','Reject']).all() and df.computer_prediction.isin(['Approve','Reject']).all(),
 'valid background/outcome':df.background.isin(['Orange','Purple']).all() and df.loan_performance.isin(['Good','Bad']).all()}
validation=pd.DataFrame({'check':checks.keys(),'passed':checks.values()}); display(style_table(validation)); assert validation.passed.all()
missing=df[['initial_decision','final_decision_displayed','effective_final_decision']].isna().sum().rename('n_missing').reset_index().rename(columns={'index':'field'})
display(style_table(missing))

In [ ]:
T=re.compile(r'^\[(\d{2}):(\d{2}):(\d{2})\]\s+\[(SPEECH|SCREEN)\]\s*(.*)$')
def parse(path,pid):
 rows=[]; cur=None; lines=path.read_text(encoding='utf-8').splitlines()
 for n,line in enumerate(lines,1):
  m=T.match(line)
  if m:
   if cur: cur['text']='\n'.join(cur.pop('lines')).strip(); cur['line_end']=n-1; rows.append(cur)
   h,mi,s,typ,txt=m.groups(); cur={'participant_id':pid,'timestamp_raw':f'{h}:{mi}:{s}','timestamp_seconds':int(h)*3600+int(mi)*60+int(s),'event_type':typ,'lines':[txt],'line_start':n}
  elif cur: cur['lines'].append(line)
 if cur: cur['text']='\n'.join(cur.pop('lines')).strip(); cur['line_end']=len(lines); rows.append(cur)
 for i,r in enumerate(rows,1): r['event_id']=f'{pid}_E{i:04d}'; r['unclear_count']=r['text'].lower().count('[unclear]')
 return rows
transcript_paths = find_all_transcripts(PROJECT_ROOT)
print(f'Transcripts found recursively under project root: {len(transcript_paths)}')
for _p in transcript_paths:
    source_file_hashes_before[Path(_p)] = sha(_p)
rows=[]
for path in sorted(transcript_paths):
 session=session_name_from_transcript_path(path); rows.extend(parse(path,pmap[session]))
events=pd.DataFrame(rows); assert len(events)==6410 and (events.event_type=='SPEECH').sum()==6124 and (events.event_type=='SCREEN').sum()==286
res=events[(events.event_type=='SCREEN')&events.text.str.contains('round_results_page',case=False,na=False)].copy(); res['round']=pd.to_numeric(res.text.str.extract(r'Round\s+(\d+)')[0],errors='coerce'); numbered=res.dropna(subset=['round']).copy(); corrupt=res[res['round'].isna()]
assert len(numbered)==220 and len(corrupt)==1
print('Events:',len(events),'Speech:',(events.event_type=='SPEECH').sum(),'Screen:',(events.event_type=='SCREEN').sum(),'[unclear]:',events.unclear_count.sum(),'Numbered results:',len(numbered),'Corrupt extra marker:',len(corrupt))

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
The source CSV, Excel workbook and all 22 transcript files were located and validated: 2,200 applicant rows across a complete 22×10×10 participant/round/applicant grid, and 6,410 parsed transcript events (6,124 speech, 286 screen). No missing-value or duplicate-key issues were found beyond those already accounted for. The next Part prepares the core decision-record variables.
</div>

# Part 4 — Decision Record Preparation

In [ ]:
df['initial_approve']=encode(df.initial_decision); df['computer_approve']=encode(df.computer_prediction); df['final_approve']=encode(df.effective_final_decision)
df['initial_correct']=correct(df.initial_approve,df.loan_performance); df['computer_correct']=correct(df.computer_approve,df.loan_performance); df['final_correct']=correct(df.final_approve,df.loan_performance)
mi=df.initial_approve.notna(); mf=df.final_approve.notna()
df['human_ai_agree']=np.where(mi,(df.initial_approve==df.computer_approve).astype(float),np.nan)
df['human_ai_disagree']=np.where(mi,(df.initial_approve!=df.computer_approve).astype(float),np.nan)
df['decision_changed']=np.where(mi&mf,(df.initial_approve!=df.final_approve).astype(float),np.nan)
df['followed_ai']=np.where(mf,(df.final_approve==df.computer_approve).astype(float),np.nan)

# --- Disagreement-clarification update (see disagreement_clarification_change_log.md) ---
# 'human_ai_agree' / 'human_ai_disagree' above compare the INITIAL human decision with the computer
# recommendation. They are aliased below under explicit names so every later reference to
# "disagreement" can be traced back to the initial stage. A second, independent pair of columns is
# added for the FINAL stage: whether the FINAL human decision (after reconsideration) differs from
# the computer recommendation. This adds columns only -- it changes no existing value, and
# 'final_human_ai_agree' is exactly 'followed_ai', already defined above and used unchanged
# throughout the rest of this notebook (including the four reliance categories immediately below).
df['initial_human_ai_agree']=df['human_ai_agree']
df['initial_human_ai_disagree']=df['human_ai_disagree']
df['final_human_ai_agree']=df['followed_ai']
df['final_human_ai_disagree']=np.where(mf,1-df['followed_ai'],np.nan)

dis=df.human_ai_disagree.eq(1)
conds=[dis&df.computer_correct.eq(1)&df.followed_ai.eq(1),dis&df.computer_correct.eq(0)&df.followed_ai.eq(0),dis&df.computer_correct.eq(0)&df.followed_ai.eq(1),dis&df.computer_correct.eq(1)&df.followed_ai.eq(0)]
labels=['Appropriate acceptance','Appropriate resistance','Overreliance','Underreliance']
df['reliance_category']=np.select(conds,labels,default=None); df.loc[dis&mf.eq(False),'reliance_category']='Unresolved'; df.loc[~dis,'reliance_category']=None
both=df.initial_correct.notna()&df.final_correct.notna()
cc=[both&df.initial_correct.eq(0)&df.final_correct.eq(1),both&df.initial_correct.eq(1)&df.final_correct.eq(0),both&df.initial_correct.eq(1)&df.final_correct.eq(1),both&df.initial_correct.eq(0)&df.final_correct.eq(0)]
df['collaboration_outcome']=np.select(cc,['Improved','Worsened','Unchanged correct','Unchanged incorrect'],default=None)
df['loan_to_income_ratio']=df.loan_amount/df.annual_income; df['loan_to_income_10pct']=df.loan_to_income_ratio/.1; df['dti_5_points']=df.dti/5; df['revolving_10_points']=df.revolving_utility_pct/10
df['purpose_grouped']=df.purpose.where(df.purpose.isin(['credit_card','debt_consolidation','medical']),'other')

matched_n_check = int((mi&mf).sum())
final_dis_n_check = int(df.loc[mi&mf,'final_human_ai_disagree'].sum())
logic=pd.DataFrame({'check':['700 disagreements','correctness complement in disagreements','105 improvements','82 worsenings',
 '1,920 matched decisions','1,220 initial agreements (63.54%)','549 final disagreements (28.59%)'],
 'passed':[dis.sum()==700,(df.loc[dis,'computer_correct']==1-df.loc[dis,'initial_correct']).all(),(df.collaboration_outcome=='Improved').sum()==105,(df.collaboration_outcome=='Worsened').sum()==82,
 matched_n_check==1920,(df.human_ai_disagree.eq(0)).sum()==1220,final_dis_n_check==549]})
display(style_table(logic)); assert logic.passed.all()


In [ ]:
# Derived infrastructure: three-stage matched sample and output directories
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from sklearn.calibration import calibration_curve

CHERRY = OUT / 'cherry_on_top'
for d in [CHERRY, CHERRY/'tables', CHERRY/'figures', CHERRY/'reports']:
    d.mkdir(parents=True, exist_ok=True)

# Recreate the same matched denominator used in the audited comparisons.
journey = df.dropna(subset=['initial_approve','computer_approve','final_approve',
                            'initial_correct','computer_correct','final_correct']).copy()
assert len(journey) == 1920
print('Cherry-on-top output directory:', public_path(CHERRY))
print('Matched decision journeys:', len(journey))

## 4.1 Effective-Final Reconciliation and Validation

Before reusing the existing `effective_final_decision` variable in the analyses below, it is independently reconstructed from first principles and compared row by row against the existing variable, and the complete 2,200-row decision record is reconciled. Nothing here overwrites or recalculates `effective_final_decision` unless the validation below finds a discrepancy.

### 4.1.1 Reconstruction Formula and Presence-Testing Method

`effective_final_reconstructed` = `final_decision_displayed` when available, otherwise `initial_decision`. Presence of `initial_decision` is tested via the pre-encoded `initial_approve` column rather than a raw `.notna()` on the string column, because an earlier, unrelated ML-feature-preparation cell (Extension setup) fills missing `initial_decision` values in place with the placeholder string `'Missing'` -- a raw `.notna()` check after that cell would silently misreport true missingness.

In [ ]:
# =====================================================================
# 1. Independent reconstruction and validation of effective_final_decision.
# Uses df.initial_approve (pre-encoded, unaffected by the later ML placeholder-fill step) to
# test true presence of initial_decision -- NOT a raw .notna() on the string column, which would
# be corrupted by the Extension-setup cell that fills missing categorical values with 'Missing'.
# =====================================================================
effective_final_reconstructed = np.where(
    df.final_decision_displayed.notna(), df.final_decision_displayed,
    np.where(df.initial_approve.notna(), df.initial_decision, np.nan)
)
effective_final_reconstructed = pd.Series(effective_final_reconstructed, index=df.index)

_both_notna = df.effective_final_decision.notna() & effective_final_reconstructed.notna()
_exact_match = _both_notna & (df.effective_final_decision == effective_final_reconstructed)
_mismatch = _both_notna & (df.effective_final_decision != effective_final_reconstructed)
_both_missing = df.effective_final_decision.isna() & effective_final_reconstructed.isna()
_one_missing_only = ~_both_notna & ~_both_missing  # one is missing, the other is not -- also a mismatch

_total_compared = len(df)
_n_exact = int(_exact_match.sum())
_n_mismatch = int(_mismatch.sum()) + int(_one_missing_only.sum())
_n_both_missing = int(_both_missing.sum())
assert _n_exact + _n_mismatch + _n_both_missing == _total_compared

effective_final_validation = pd.DataFrame([
    {'metric': 'Total rows compared', 'value': _total_compared},
    {'metric': 'Exact matches', 'value': _n_exact},
    {'metric': 'Mismatches', 'value': _n_mismatch},
    {'metric': 'Both missing (effective-final and reconstructed)', 'value': _n_both_missing},
    {'metric': 'Mismatch percentage', 'value': round(100 * _n_mismatch / _total_compared, 4)},
])
display(style_table(effective_final_validation))

_mismatch_mask = _mismatch | _one_missing_only
mismatching_rows = df.loc[_mismatch_mask, ['participant_id', 'round', 'row_in_round', 'id',
                                            'initial_decision', 'final_decision_displayed', 'effective_final_decision']].copy()
mismatching_rows['effective_final_reconstructed'] = effective_final_reconstructed.loc[_mismatch_mask]
print(f'\nMismatching rows: {len(mismatching_rows)}')
if len(mismatching_rows):
    display(mismatching_rows)
else:
    print('(none)')

_effective_final_validation_passed = (_n_mismatch == 0)
print(f'\nValidation {"PASSED" if _effective_final_validation_passed else "FAILED"}: '
      f'{"continuing to use the existing effective_final_decision variable unchanged." if _effective_final_validation_passed else "mismatches found -- see rows above, existing variable NOT silently replaced."}')


## 4.2 Complete 2,200-Row Reconciliation

In [ ]:
# =====================================================================
# 2. Complete 2,200-row decision-record reconciliation (calculated, not hard-coded).
# =====================================================================
_total_rows = len(df)
_initial_present = df.initial_approve.notna()
_initial_missing = ~_initial_present
_explicit_present = df.final_decision_displayed.notna()
_explicit_missing = ~_explicit_present

_init_present_explicit_present = int((_initial_present & _explicit_present).sum())
_init_present_explicit_missing = int((_initial_present & _explicit_missing).sum())
_init_missing_explicit_present = int((_initial_missing & _explicit_present).sum())
_init_missing_explicit_missing = int((_initial_missing & _explicit_missing).sum())
_effective_present = int(df.effective_final_decision.notna().sum())
_effective_missing = int(df.effective_final_decision.isna().sum())

decision_record_reconciliation = pd.DataFrame([
    {'metric': 'Total CSV rows', 'value': _total_rows},
    {'metric': 'Rows with initial decision', 'value': int(_initial_present.sum())},
    {'metric': 'Rows without initial decision', 'value': int(_initial_missing.sum())},
    {'metric': 'Rows with explicit final_decision_displayed', 'value': int(_explicit_present.sum())},
    {'metric': 'Rows without explicit final_decision_displayed', 'value': int(_explicit_missing.sum())},
    {'metric': 'Initial present AND explicit final present', 'value': _init_present_explicit_present},
    {'metric': 'Initial present AND explicit final missing (carry-forward needed)', 'value': _init_present_explicit_missing},
    {'metric': 'Initial missing AND explicit final present', 'value': _init_missing_explicit_present},
    {'metric': 'Initial missing AND explicit final missing (neither available)', 'value': _init_missing_explicit_missing},
    {'metric': 'Total effective final decisions present', 'value': _effective_present},
    {'metric': 'Rows with no effective final decision', 'value': _effective_missing},
])

_expected = {
    'Total CSV rows': 2200, 'Rows with initial decision': 1920, 'Rows without initial decision': 280,
    'Rows with explicit final_decision_displayed': 1831, 'Rows without explicit final_decision_displayed': 369,
    'Initial present AND explicit final present': 1734, 'Initial present AND explicit final missing (carry-forward needed)': 186,
    'Initial missing AND explicit final present': 97, 'Initial missing AND explicit final missing (neither available)': 183,
}
decision_record_reconciliation['expected_reference_value'] = decision_record_reconciliation.metric.map(_expected)
decision_record_reconciliation['matches_reference'] = decision_record_reconciliation.apply(
    lambda r: (r.value == r.expected_reference_value) if pd.notna(r.expected_reference_value) else None, axis=1)
display(style_table(decision_record_reconciliation))

# --- Arithmetic validation: every group sums correctly ---
reconciliation_arithmetic_checks = pd.DataFrame([
    {'check': 'Initial present + Initial missing = Total rows',
     'left_hand_side': int(_initial_present.sum()) + int(_initial_missing.sum()), 'right_hand_side': _total_rows},
    {'check': 'Explicit present + Explicit missing = Total rows',
     'left_hand_side': int(_explicit_present.sum()) + int(_explicit_missing.sum()), 'right_hand_side': _total_rows},
    {'check': 'Four initial x explicit combinations sum to Total rows',
     'left_hand_side': _init_present_explicit_present + _init_present_explicit_missing + _init_missing_explicit_present + _init_missing_explicit_missing,
     'right_hand_side': _total_rows},
    {'check': 'Effective final present + missing = Total rows',
     'left_hand_side': _effective_present + _effective_missing, 'right_hand_side': _total_rows},
    {'check': 'Effective final present = (Initial present AND explicit present) + (Initial present AND explicit missing) + (Initial missing AND explicit present)',
     'left_hand_side': _effective_present,
     'right_hand_side': _init_present_explicit_present + _init_present_explicit_missing + _init_missing_explicit_present},
])
reconciliation_arithmetic_checks['match'] = reconciliation_arithmetic_checks.left_hand_side == reconciliation_arithmetic_checks.right_hand_side
display(style_table(reconciliation_arithmetic_checks))
print()
print('All reconciliation arithmetic checks passed.' if reconciliation_arithmetic_checks['match'].all() else 'DISCREPANCY FOUND -- see table above.')


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Initial, AI and final decisions were cleaned into comparable Approve/Reject columns, and the effective final decision was reconstructed and validated row-by-row against the complete 2,200-row reconciliation table with zero mismatches. Three distinct analytical samples were defined: the 2,017-row AI–effective-final sample, the 1,831-row AI–explicit-final sample, and the 1,920-row complete three-stage sample. The next Part reports descriptive results for these decision variables.
</div>

# Part 5 — Descriptive Decision Results

In [ ]:
stage_all=pd.DataFrame({'stage':['Initial human','Computer','Final collaborative'],'n':[df.initial_correct.notna().sum(),df.computer_correct.notna().sum(),df.final_correct.notna().sum()],
 'accuracy':[df.initial_correct.mean(),df.computer_correct.mean(),df.final_correct.mean()],'approval_rate':[df.initial_approve.mean(),df.computer_approve.mean(),df.final_approve.mean()]})
matched=df.dropna(subset=['initial_approve','final_approve']).copy()
stage_matched=pd.DataFrame({'stage':['Initial human','Computer','Final collaborative'],'n_common':len(matched),'accuracy':[matched.initial_correct.mean(),matched.computer_correct.mean(),matched.final_correct.mean()]})
fi=pboot(matched,lambda d:d.final_correct.mean()-d.initial_correct.mean()); fc=pboot(matched,lambda d:d.final_correct.mean()-d.computer_correct.mean())
diffs=pd.DataFrame([{'comparison':'Final - initial','difference':fi[0],'ci_low':fi[1],'ci_high':fi[2]},{'comparison':'Final - computer','difference':fc[0],'ci_low':fc[1],'ci_high':fc[2]}])
rel=df.loc[dis,'reliance_category'].value_counts().reindex(labels+['Unresolved']).fillna(0).astype(int).rename_axis('category').reset_index(name='count'); rel['proportion']=rel['count']/dis.sum()
follow=df.loc[dis].groupby('computer_correct').followed_ai.agg(['count','mean']).reset_index(); follow['computer_status']=follow.computer_correct.map({0.:'Incorrect',1.:'Correct'})
display(stage_all.round(4)); display(stage_matched.round(4)); display(diffs.round(4)); display(rel.round(4)); display(follow[['computer_status','count','mean']].rename(columns={'mean':'follow_rate'}).round(4))
print(f'Final-initial matched gain: {fi[0]*100:.2f} pp, bootstrap CI [{fi[1]*100:.2f}, {fi[2]*100:.2f}]')
print(f'Final-computer matched gain: {fc[0]*100:.2f} pp, bootstrap CI [{fc[1]*100:.2f}, {fc[2]*100:.2f}]')

# --- Disagreement-clarification addition (see disagreement_clarification_change_log.md) ---
# The 36.46% figure implicit above (700 of the 1,920 matched cases were disagreement cases) is the
# INITIAL human-AI disagreement rate: whether the human's FIRST decision differed from the computer
# recommendation. It is reported here alongside a separate, equally important FINAL human-AI
# disagreement rate: whether the human's FINAL decision (after reconsideration) still differs from
# the computer recommendation. Both are reported because they answer different questions; neither
# replaces the other, and the four reliance categories above continue to be defined only within the
# initial-disagreement subset.
init_dis_n=int(matched.human_ai_disagree.sum()); init_agree_n=len(matched)-init_dis_n
final_dis_n=int(matched.final_human_ai_disagree.sum()); final_agree_n=len(matched)-final_dis_n
disagreement_summary=pd.DataFrame([
 {'measure':'Initial human–AI disagreement','count':init_dis_n,'rate':init_dis_n/len(matched)},
 {'measure':'Initial human–AI agreement','count':init_agree_n,'rate':init_agree_n/len(matched)},
 {'measure':'Final human–AI disagreement','count':final_dis_n,'rate':final_dis_n/len(matched)},
 {'measure':'Final human–AI agreement','count':final_agree_n,'rate':final_agree_n/len(matched)},
])
display(disagreement_summary.round(4))
print(f'Initial human-AI disagreement: {init_dis_n}/{len(matched)} = {init_dis_n/len(matched):.2%} (before reconsideration)')
print(f'Final human-AI disagreement:   {final_dis_n}/{len(matched)} = {final_dis_n/len(matched):.2%} (after reconsideration)')
print('Reliance categories (Appropriate acceptance/resistance, Over/Underreliance) are defined only within the',
      init_dis_n, 'initial-disagreement cases above -- they are never redefined on final disagreement.')


**What this shows:** Accuracy of the initial human decision, the AI recommendation and the final decision, compared on the same set of cases.

**Sample:** 1,920 matched cases with a valid initial, computer and final decision.

In [ ]:
fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(stage_matched.stage,stage_matched.accuracy); ax.set_ylim(0,1); ax.set_ylabel('Accuracy'); ax.set_title('Matched-case accuracy (n=1,920)')
for i,v in enumerate(stage_matched.accuracy): ax.text(i,v+.02,f'{v:.3f}',ha='center')
savefig(fig,'01_matched_accuracy')

**Main result:** Final decisions are modestly more accurate than initial human decisions and clearly more accurate than the computer recommendation alone.

**Interpretation:** The final, collaborative decision is the best-performing stage on this matched sample, but the gain over initial human judgement is small.

**Caution:** This is a same-sample descriptive comparison; see Part 8 for bootstrap uncertainty intervals around these differences.

**What this shows:** How the four reliance categories (appropriate acceptance, appropriate resistance, overreliance, underreliance) are distributed among initial Human–AI disagreements.

**Sample:** 700 initial disagreement cases (Unresolved cases excluded from this chart).

In [ ]:
fig,ax=plt.subplots(figsize=(8,4.5)); q=rel[rel.category!='Unresolved']; ax.bar(q.category,q.proportion); ax.set_ylabel('Proportion of initial disagreement cases (n=700)'); ax.set_title('Human–AI reliance categories (within initial disagreement)'); ax.tick_params(axis='x',rotation=25); savefig(fig,'02_reliance_categories')


**Main result:** Appropriate acceptance and resistance together account for roughly 60% of initial disagreements; underreliance is more common than overreliance among the remainder.

**Interpretation:** When humans and the AI disagreed initially, participants were, on balance, more likely to under-use correct AI advice than to over-trust incorrect advice.

**Caution:** Reliance categories are defined only within the initial-disagreement subset and are never redefined on final disagreement (Part 4).

**What this shows:** Initial, computer and final decision accuracy for each of the ten rounds.

**Sample:** All rows in each round, split by round number.

---
## 5.1 Descriptive Round and Participant Patterns

In [ ]:
rr=[]
for r,g in df.groupby('round'):
 d=g[g.human_ai_disagree.eq(1)]; rr.append({'round':r,'initial_accuracy':g.initial_correct.mean(),'computer_accuracy':g.computer_correct.mean(),'final_accuracy':g.final_correct.mean(),'final_approval_rate':g.final_approve.mean(),'follow_rate_disagreements':d.followed_ai.mean(),'overreliance_rate':(d.reliance_category=='Overreliance').mean(),'underreliance_rate':(d.reliance_category=='Underreliance').mean(),'missing_final_rate':g.final_approve.isna().mean()})
round_summary=pd.DataFrame(rr); display(round_summary.round(4))
fig,ax=plt.subplots(figsize=(8,4.5)); ax.plot(round_summary['round'],round_summary.initial_accuracy,marker='o',label='Initial'); ax.plot(round_summary['round'],round_summary.computer_accuracy,marker='o',label='Computer'); ax.plot(round_summary['round'],round_summary.final_accuracy,marker='o',label='Final'); ax.set_xticks(range(1,11)); ax.set_ylim(0,1); ax.set_xlabel('Round'); ax.set_ylabel('Accuracy'); ax.set_title('Descriptive accuracy by fixed applicant round'); ax.legend(); savefig(fig,'05_round_accuracy')

**Main result:** Accuracy for all three stages is broadly stable across the ten rounds, with no strong upward or downward trend.

**Interpretation:** Round-to-round variation reflects the specific applicant profiles shown in each round rather than a learning or fatigue effect over the course of the study.

**Caution:** Each round mixes the same 100 fixed applicant profiles across participants, so round effects are descriptive, not causal.

**What this shows:** How much each of the 22 participants' final accuracy differs from their own initial accuracy, sorted from largest decline to largest improvement.

**Sample:** 22 participants, using each participant's own matched initial/final cases.

In [ ]:
pr=[]
for pid,g in df.groupby('participant_id'):
 common=g.dropna(subset=['initial_approve','final_approve']); d=g[g.human_ai_disagree.eq(1)]; b=g.groupby('background',observed=True).final_approve.mean(); ia=common.initial_correct.mean(); ca=common.computer_correct.mean(); fa=common.final_correct.mean()
 profile='Synergy (descriptive)' if fa>max(ia,ca) else 'Augmentation (descriptive)' if fa>ia else 'Degradation (descriptive)' if fa<ia else 'No change'
 pr.append({'participant_id':pid,'n_matched':len(common),'initial_accuracy':ia,'computer_accuracy':ca,'final_accuracy':fa,'final_minus_initial':fa-ia,'collaboration_profile':profile,'follow_rate_disagreements':d.followed_ai.mean(),'overreliance_rate':(d.reliance_category=='Overreliance').mean(),'underreliance_rate':(d.reliance_category=='Underreliance').mean(),'purple_minus_orange_final_gap':b.get('Purple',np.nan)-b.get('Orange',np.nan),'missing_final_rate':g.final_approve.isna().mean(),'earnings_gbp':g.earnings.sum()})
participant_summary=pd.DataFrame(pr); display(participant_summary.round(4)); print(participant_summary.collaboration_profile.value_counts())
fig,ax=plt.subplots(figsize=(9,4.5)); q=participant_summary.sort_values('final_minus_initial'); ax.bar(q.participant_id,q.final_minus_initial); ax.axhline(0); ax.set_ylabel('Final − initial accuracy'); ax.set_title('Participant-level matched-case change'); ax.tick_params(axis='x',rotation=90); savefig(fig,'06_participant_change')

**Main result:** Participants vary considerably in whether the final decision improved on their own initial judgement; most cluster near zero change.

**Interpretation:** The modest average final-vs-initial accuracy gain (Part 5) masks real participant-level heterogeneity — collaboration helped some participants more than others.

**Caution:** Small per-participant case counts make individual estimates noisy; this chart is descriptive, not a ranking of participant skill.

**What this shows:** A standardised (z-scored) profile of each of the 22 participants across six collaboration metrics, plus a separate diverging strip for the Purple-minus-Orange final-decision gap.

**Sample:** 22 participants, each participant's own matched cases.

---
## 5.2 Participant Collaboration Profiles

Overall averages can hide major differences between participants. The heat map below compares each participant on initial accuracy, final accuracy, change in accuracy, AI-following, overreliance, underreliance and the final Purple–Orange approval gap.

These are **descriptive behavioural profiles**, not personality labels.

In [ ]:
# Standardised participant profile heat map (neutral metrics) plus a separate diverging
# Orange-Purple strip for the one group-gap column, per the colour-reservation rule: the
# neutral heatmap never carries the group-gap column, and the group-gap strip never uses the
# neutral colourmap.
neutral_metrics=['initial_accuracy','final_accuracy','final_minus_initial','follow_rate_disagreements',
                 'overreliance_rate','underreliance_rate']
profile_metrics=neutral_metrics+['purple_minus_orange_final_gap']
profile=participant_summary.set_index('participant_id')[profile_metrics].copy()
profile_z=(profile-profile.mean())/profile.std(ddof=0).replace(0,np.nan)
profile_z=profile_z.fillna(0)

fig,(ax,ax_gap)=plt.subplots(1,2,figsize=(12.5,8),gridspec_kw={'width_ratios':[6,1]},sharey=True)
im=ax.imshow(profile_z[neutral_metrics].to_numpy(),aspect='auto',cmap='RdBu_r',vmin=-3,vmax=3)
ax.set_yticks(np.arange(len(profile_z))); ax.set_yticklabels(profile_z.index)
ax.set_xticks(np.arange(len(neutral_metrics))); ax.set_xticklabels([x.replace('_',' ').title() for x in neutral_metrics],rotation=40,ha='right')
fig.colorbar(im,ax=ax,label='Standardised value within the 22 participants',fraction=0.046,pad=0.04)
ax.set_title('Participant-Level Collaboration Profiles (Neutral Metrics)')

_gap_vals=profile['purple_minus_orange_final_gap'].to_numpy().reshape(-1,1)
_gap_abs_max=max(abs(profile['purple_minus_orange_final_gap'].min()),abs(profile['purple_minus_orange_final_gap'].max()),1e-9)
from matplotlib.colors import LinearSegmentedColormap
_op_cmap=LinearSegmentedColormap.from_list('orange_purple_diverging',[GROUP_ORANGE,PALETTE_WHITE,GROUP_PURPLE])
im_gap=ax_gap.imshow(_gap_vals,aspect='auto',cmap=_op_cmap,vmin=-_gap_abs_max,vmax=_gap_abs_max)
ax_gap.set_xticks([0]); ax_gap.set_xticklabels(['Purple −\nOrange\nFinal Gap'],rotation=40,ha='right')
ax_gap.set_yticks([])
fig.colorbar(im_gap,ax=ax_gap,label='Purple minus Orange gap',fraction=0.046,pad=0.04)
ax_gap.set_title('Group Gap')
fig.suptitle('Participant-Level Collaboration Profiles', y=1.0, fontsize=13, fontweight='bold')
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'13_participant_profile_heatmap.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()

notable=pd.DataFrame([
    {'profile':'Largest accuracy improvement','participant_id':participant_summary.loc[participant_summary.final_minus_initial.idxmax(),'participant_id']},
    {'profile':'Largest accuracy decline','participant_id':participant_summary.loc[participant_summary.final_minus_initial.idxmin(),'participant_id']},
    {'profile':'Highest overreliance rate','participant_id':participant_summary.loc[participant_summary.overreliance_rate.idxmax(),'participant_id']},
    {'profile':'Highest underreliance rate','participant_id':participant_summary.loc[participant_summary.underreliance_rate.idxmax(),'participant_id']},
    {'profile':'Largest absolute final group gap','participant_id':participant_summary.loc[participant_summary.purple_minus_orange_final_gap.abs().idxmax(),'participant_id']},
]).merge(participant_summary,on='participant_id',how='left')
display(notable.round(4))
profile.reset_index().to_csv(CHERRY/'tables'/'participant_collaboration_profiles.csv',index=False)
notable.to_csv(CHERRY/'tables'/'notable_participant_profiles.csv',index=False)

**Main result:** Participants differ noticeably in accuracy change, follow-rate and reliance-rate profiles; the Purple-minus-Orange gap column (right strip) also varies by participant.

**Interpretation:** Collaboration quality and observed group gaps are not uniform across participants, which is relevant to any oversight or training design.

**Caution:** Each row is standardised within the 22 participants, so values are relative, not absolute accuracy; the group-gap strip uses a separate diverging Orange-Purple scale from the neutral metrics (Part 9's colour-reservation rule).

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Descriptive accuracy, approval-rate and round/participant-level patterns were computed on the matched samples defined in Part 4. The next Part compares the AI recommendation against the final decision directly, across three separate analyses.
</div>

# Part 6 — Final Human–AI Comparison

**Purpose:** compare only the AI/computer recommendation with the final human decision.
Required fields: AI/computer recommendation, final human decision. **The initial human
decision is not required for this analysis.**

This analysis includes all cases with a valid AI recommendation and final human decision. The initial human decision is not required.

This is a deliberately different (and, as shown below, larger) sample than the three-stage
journey analysis further down, because it does not require the initial decision to be present.
The matched sample size for this analysis is **recalculated directly from the data below** — it
is not assumed to equal 1,920 or any other previously reported figure.

## 6.1 AI vs Effective Final Decision

In [ ]:
# --- AI Recommendation-Final Decision Analysis: build the matched subset ---
# Required fields only: AI/computer recommendation ('computer_prediction') and final human
# decision ('effective_final_decision'). The initial human decision is not used to build this
# sample. As in the Three-Stage analysis below, the boolean mask uses the already-encoded
# 'computer_approve'/'final_approve' columns (Section 3) rather than a direct .dropna() on the
# raw string columns, because a later modelling step in this notebook (CatBoost/SHAP feature
# preparation) fills missing raw values with the placeholder 'Missing'. For every row kept by
# this mask, the raw decision columns still hold their genuine original Approve/Reject values.
af_mask = df['computer_approve'].notna() & df['final_approve'].notna()
af_df = df.loc[af_mask].copy()
ai_final_matched_n = len(af_df)
print(f'AI-Final matched sample (recalculated from the data, not assumed): {ai_final_matched_n:,}')

af_df['ai_final_status'] = np.where(af_df.computer_prediction.eq(af_df.effective_final_decision), 'Agreement', 'Disagreement')

af_combo_order = [('Approve', 'Approve'), ('Approve', 'Reject'), ('Reject', 'Approve'), ('Reject', 'Reject')]
af_rows = []
for ai_val, final_val in af_combo_order:
    m = af_df.computer_prediction.eq(ai_val) & af_df.effective_final_decision.eq(final_val)
    af_rows.append({
        'ai_recommendation': ai_val,
        'final_decision': final_val,
        'count': int(m.sum()),
        'percentage_of_ai_final_matched': round(100 * m.sum() / ai_final_matched_n, 2),
        'status': 'Agreement' if ai_val == final_val else 'Disagreement',
    })
ai_final_2x2 = pd.DataFrame(af_rows)
display(style_table(ai_final_2x2))
assert ai_final_2x2['count'].sum() == ai_final_matched_n, 'The four AI-Final combinations must sum to the AI-Final matched sample.'
print(f"Sum check: the four AI-Final combinations sum to {ai_final_2x2['count'].sum():,}, matching {ai_final_matched_n:,} AI-Final matched decisions.")


In [ ]:
# --- AI-Final agreement/disagreement summary ---
af_agree_n = int(ai_final_2x2.loc[ai_final_2x2.status.eq('Agreement'), 'count'].sum())
af_disagree_n = int(ai_final_2x2.loc[ai_final_2x2.status.eq('Disagreement'), 'count'].sum())

ai_approve_final_approve = int(ai_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Approve'")['count'].iloc[0])
ai_approve_final_reject = int(ai_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Reject'")['count'].iloc[0])
ai_reject_final_approve = int(ai_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Approve'")['count'].iloc[0])
ai_reject_final_reject = int(ai_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Reject'")['count'].iloc[0])

ai_final_summary = pd.DataFrame([
    {'measure': 'Total AI-Final matched decisions', 'count': ai_final_matched_n, 'percentage': 100.0},
    {'measure': 'Final Human-AI Agreement (AI-Final sample)', 'count': af_agree_n, 'percentage': round(100 * af_agree_n / ai_final_matched_n, 2)},
    {'measure': 'Final Human-AI Disagreement (AI-Final sample)', 'count': af_disagree_n, 'percentage': round(100 * af_disagree_n / ai_final_matched_n, 2)},
    {'measure': 'AI Approve -> Final Approve', 'count': ai_approve_final_approve, 'percentage': round(100 * ai_approve_final_approve / ai_final_matched_n, 2)},
    {'measure': 'AI Approve -> Final Reject', 'count': ai_approve_final_reject, 'percentage': round(100 * ai_approve_final_reject / ai_final_matched_n, 2)},
    {'measure': 'AI Reject -> Final Approve', 'count': ai_reject_final_approve, 'percentage': round(100 * ai_reject_final_approve / ai_final_matched_n, 2)},
    {'measure': 'AI Reject -> Final Reject', 'count': ai_reject_final_reject, 'percentage': round(100 * ai_reject_final_reject / ai_final_matched_n, 2)},
])
display(style_table(ai_final_summary))

af_check = pd.DataFrame([
    {'check': 'AI-Final agreement + AI-Final disagreement = AI-Final matched sample',
     'left_hand_side': af_agree_n + af_disagree_n, 'right_hand_side': ai_final_matched_n,
     'match': (af_agree_n + af_disagree_n) == ai_final_matched_n},
    {'check': 'Four AI-Final decision combinations = AI-Final matched sample',
     'left_hand_side': ai_approve_final_approve + ai_approve_final_reject + ai_reject_final_approve + ai_reject_final_reject,
     'right_hand_side': ai_final_matched_n,
     'match': (ai_approve_final_approve + ai_approve_final_reject + ai_reject_final_approve + ai_reject_final_reject) == ai_final_matched_n},
])
display(style_table(af_check))
print('AI-Final validation:', 'all checks passed.' if af_check['match'].all() else 'DISCREPANCY FOUND -- see table above.')
print()
print(f'Final Human-AI Agreement: {af_agree_n:,} of {ai_final_matched_n:,} ({100*af_agree_n/ai_final_matched_n:.2f}%)')
print(f'Final Human-AI Disagreement: {af_disagree_n:,} of {ai_final_matched_n:,} ({100*af_disagree_n/ai_final_matched_n:.2f}%)')


## 6.2 AI vs Explicit Final Decision (Sensitivity Analysis)

The two analyses above both use `effective_final_decision`, which carries the initial human decision forward when no explicit final decision was recorded. The sensitivity analysis below repeats the AI-vs-final comparison using **only** rows where a final decision was explicitly displayed (`final_decision_displayed`) -- the initial decision is never substituted here. This tests whether the carry-forward rule used elsewhere in this notebook changes the conclusion.

In [ ]:
# =====================================================================
# Analysis 2: AI vs Explicit Final Decision (sensitivity analysis).
# Uses ONLY rows where the final decision was explicitly displayed/recorded --
# the initial decision is never carried forward here (unlike effective_final_decision).
# =====================================================================
ef_mask = df['computer_prediction'].notna() & df['final_decision_displayed'].notna()
ef_df = df.loc[ef_mask].copy()
ai_explicit_final_matched_n = len(ef_df)
print(f'AI-Explicit-Final matched sample (recalculated from the data, not assumed): {ai_explicit_final_matched_n:,}')

ef_df['ef_status'] = np.where(ef_df.computer_prediction.eq(ef_df.final_decision_displayed), 'Agreement', 'Disagreement')

ef_combo_order = [('Approve', 'Approve'), ('Approve', 'Reject'), ('Reject', 'Approve'), ('Reject', 'Reject')]
ef_rows = []
for ai_val, final_val in ef_combo_order:
    m = ef_df.computer_prediction.eq(ai_val) & ef_df.final_decision_displayed.eq(final_val)
    ef_rows.append({
        'ai_recommendation': ai_val, 'final_decision': final_val, 'count': int(m.sum()),
        'percentage_of_explicit_final_matched': round(100 * m.sum() / ai_explicit_final_matched_n, 2),
        'status': 'Agreement' if ai_val == final_val else 'Disagreement',
    })
ai_explicit_final_2x2 = pd.DataFrame(ef_rows)
assert ai_explicit_final_2x2['count'].sum() == ai_explicit_final_matched_n

ef_agree_n = int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Agreement'), 'count'].sum())
ef_disagree_n = int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Disagreement'), 'count'].sum())
ef_ai_approve_final_approve = int(ai_explicit_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Approve'")['count'].iloc[0])
ef_ai_approve_final_reject = int(ai_explicit_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Reject'")['count'].iloc[0])
ef_ai_reject_final_approve = int(ai_explicit_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Approve'")['count'].iloc[0])
ef_ai_reject_final_reject = int(ai_explicit_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Reject'")['count'].iloc[0])

ai_explicit_final_summary = pd.DataFrame([
    {'measure': 'Total AI-Explicit-Final matched decisions', 'count': ai_explicit_final_matched_n, 'percentage': 100.0},
    {'measure': 'Explicit Final Human-AI Agreement', 'count': ef_agree_n, 'percentage': round(100 * ef_agree_n / ai_explicit_final_matched_n, 2)},
    {'measure': 'Explicit Final Human-AI Disagreement', 'count': ef_disagree_n, 'percentage': round(100 * ef_disagree_n / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Approve -> Final Approve', 'count': ef_ai_approve_final_approve, 'percentage': round(100 * ef_ai_approve_final_approve / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Approve -> Final Reject', 'count': ef_ai_approve_final_reject, 'percentage': round(100 * ef_ai_approve_final_reject / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Reject -> Final Approve', 'count': ef_ai_reject_final_approve, 'percentage': round(100 * ef_ai_reject_final_approve / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Reject -> Final Reject', 'count': ef_ai_reject_final_reject, 'percentage': round(100 * ef_ai_reject_final_reject / ai_explicit_final_matched_n, 2)},
])
display(style_table(ai_explicit_final_summary))

ef_check = pd.DataFrame([
    {'check': 'Explicit-Final agreement + disagreement = Explicit-Final matched sample',
     'left_hand_side': ef_agree_n + ef_disagree_n, 'right_hand_side': ai_explicit_final_matched_n,
     'match': (ef_agree_n + ef_disagree_n) == ai_explicit_final_matched_n},
    {'check': 'Four Explicit-Final decision combinations = Explicit-Final matched sample',
     'left_hand_side': ef_ai_approve_final_approve + ef_ai_approve_final_reject + ef_ai_reject_final_approve + ef_ai_reject_final_reject,
     'right_hand_side': ai_explicit_final_matched_n,
     'match': (ef_ai_approve_final_approve + ef_ai_approve_final_reject + ef_ai_reject_final_approve + ef_ai_reject_final_reject) == ai_explicit_final_matched_n},
])
display(style_table(ef_check))
print('Explicit-Final validation:', 'all checks passed.' if ef_check['match'].all() else 'DISCREPANCY FOUND.')


### 6.2.1 Carry-Forward Consistency Check and Comparison With the Effective-Final Analysis

The carry-forward rule stated for `effective_final_decision` is verified directly against the raw data below (not assumed), and the sensitivity analysis is compared against the main AI-vs-Effective-Final analysis to check whether the two reach the same broad conclusion.

In [ ]:
af_agree_pct = round(100 * af_agree_n / ai_final_matched_n, 2)
ef_agree_pct = round(100 * ef_agree_n / ai_explicit_final_matched_n, 2)
comparison_conclusion_same_direction = (af_agree_pct > 50) == (ef_agree_pct > 50)
sensitivity_comparison = pd.DataFrame([
    {'analysis': 'AI vs Effective Final (carry-forward)', 'sample_size': ai_final_matched_n, 'agreement_pct': af_agree_pct, 'disagreement_pct': round(100 - af_agree_pct, 2)},
    {'analysis': 'AI vs Explicit Final (sensitivity, no carry-forward)', 'sample_size': ai_explicit_final_matched_n, 'agreement_pct': ef_agree_pct, 'disagreement_pct': round(100 - ef_agree_pct, 2)},
])
display(style_table(sensitivity_comparison))
print(f'Same conclusion (majority agreement vs. majority disagreement) across both analyses: {comparison_conclusion_same_direction}')
print(f'Difference in agreement percentage: {round(af_agree_pct - ef_agree_pct, 2)} percentage points.')

# =====================================================================
# Carry-forward consistency check: verify effective_final_decision matches the stated rule
# (final_decision_displayed when available, otherwise initial_decision). 'final_decision_displayed'
# and 'effective_final_decision' are checked directly (neither is ever placeholder-filled in this
# notebook). For 'initial_decision', presence is tested via the already-encoded 'initial_approve'
# column (Section 3) rather than a direct .notna() on the raw string column: an earlier
# ML/feature-preparation step (Extension setup) fills raw 'initial_decision' NaNs in place with the
# placeholder string 'Missing', so a raw .notna() check on it would silently misreport true
# missingness for any cell running after that step -- exactly the risk already flagged in the
# AI-Final and Three-Stage analyses above, and avoided here the same way.
# =====================================================================
_has_displayed = df.final_decision_displayed.notna()
_initial_present = df.initial_approve.notna()
_rule_ok_displayed = (df.loc[_has_displayed, 'effective_final_decision'] == df.loc[_has_displayed, 'final_decision_displayed']).all()
_needs_carry = ~_has_displayed & _initial_present
_rule_ok_carried = (df.loc[_needs_carry, 'effective_final_decision'] == df.loc[_needs_carry, 'initial_decision']).all()
_both_missing = ~_has_displayed & ~_initial_present
_rule_ok_both_missing = df.loc[_both_missing, 'effective_final_decision'].isna().all()

carry_forward_check = pd.DataFrame([
    {'check': 'Where final_decision_displayed is present, effective_final_decision equals it', 'n_rows': int(_has_displayed.sum()), 'result': 'PASS' if _rule_ok_displayed else 'FAIL'},
    {'check': 'Where final_decision_displayed is missing but initial_decision is present, effective_final_decision equals initial_decision (carried forward)', 'n_rows': int(_needs_carry.sum()), 'result': 'PASS' if _rule_ok_carried else 'FAIL'},
    {'check': 'Where both final_decision_displayed and initial_decision are missing, effective_final_decision is also missing', 'n_rows': int(_both_missing.sum()), 'result': 'PASS' if _rule_ok_both_missing else 'FAIL'},
])
display(style_table(carry_forward_check))
_carry_forward_consistent = _rule_ok_displayed and _rule_ok_carried and _rule_ok_both_missing
print('Carry-forward rule consistent with existing project logic:', _carry_forward_consistent)
print('Note: Where no explicit final decision was recorded, the initial human decision was carried forward as the effective final decision.')


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Three independent Final Human–AI comparisons were computed on their own matched samples: AI vs effective final decision (2,017 cases; 71.79% agreement), AI vs explicit final decision with no carry-forward (1,831 cases; 73.84% agreement), and the three-stage sample's own final comparison (1,920 cases; 71.41% agreement) — the same overall conclusion in every case. The next Part examines how this alignment developed through the three-stage journey.
</div>

# Part 7 — Three-Stage Decision Journey

In [ ]:
follow_data=df[dis&df.followed_ai.notna()].copy(); follow_formula='followed_ai ~ computer_correct + loan_risk + loan_to_income_10pct + dti_5_points + revolving_10_points + inquiries_last_6_months + round + C(home_ownership)+C(purpose_grouped)+C(background)'
follow_model=fit_2way(follow_formula,follow_data); display(follow_model.round(4)); cr=follow_model.set_index('term').loc['computer_correct']; print(f'Correct-vs-incorrect AI follow OR={cr.odds_ratio:.2f}, 95% CI [{cr.ci_low:.2f}, {cr.ci_high:.2f}], p={cr.p_value:.3f}')

## 7.1 Underlying Decision-Flow Counts

This diagram condenses the collaboration process into one view, and now explicitly distinguishes two different questions:

- **Initial human–AI disagreement** — did the human's first decision differ from the computer recommendation? (700 of 1,920 matched cases, 36.46%). The four reliance categories (appropriate acceptance, appropriate resistance, overreliance, underreliance) are defined only within this initial-disagreement subset — that definition is unchanged from Section 4.
- **Final human–AI disagreement** — did the human's final decision, after reconsideration, still differ from the computer recommendation? (549 of 1,920 matched cases, 28.59%). This is a distinct, downstream quantity: it combines the 531 initial-disagreement cases that were never resolved toward the computer with 18 new disagreements that arose from cases that started in agreement but moved away from the computer by the final decision.

The diagram is explanatory rather than causal: it shows what happened in the experiment, not what must happen in another organisation.

In [ ]:
# Decision-flow counts underlying the Three-Stage Decision Journey (Part 7): computed once here
# --- Disagreement-clarification update: this cell now distinguishes INITIAL human-AI agreement/
# disagreement (the first human decision vs. the computer) from FINAL human-AI agreement/
# disagreement (the final human decision, after reconsideration, vs. the computer). The four
# reliance categories are still computed only within the initial-disagreement subset, unchanged
# from Section 4 (see disagreement_clarification_change_log.md).
agree_n = int(journey.human_ai_agree.sum())
disagree_n = int(journey.human_ai_disagree.sum())
rel_counts = (journey.loc[journey.human_ai_disagree.eq(1), 'reliance_category']
              .value_counts().reindex(['Appropriate acceptance','Appropriate resistance','Overreliance','Underreliance']).fillna(0).astype(int))

# Fate of the 700 initial-disagreement cases by final stage
changed_toward_ai_n = int((journey.human_ai_disagree.eq(1) & journey.followed_ai.eq(1)).sum())
remained_against_ai_n = int((journey.human_ai_disagree.eq(1) & journey.followed_ai.eq(0)).sum())
assert changed_toward_ai_n == rel_counts['Appropriate acceptance'] + rel_counts['Overreliance']
assert remained_against_ai_n == rel_counts['Appropriate resistance'] + rel_counts['Underreliance']

# Fate of the 1,220 initial-agreement cases by final stage
remained_aligned_n = int((journey.human_ai_disagree.eq(0) & journey.followed_ai.eq(1)).sum())
moved_away_n = int((journey.human_ai_disagree.eq(0) & journey.followed_ai.eq(0)).sum())
assert remained_aligned_n + moved_away_n == agree_n

final_disagree_n = remained_against_ai_n + moved_away_n
final_agree_n = remained_aligned_n + changed_toward_ai_n
assert final_disagree_n == 549 and final_agree_n == 1371

# The superseded box-diagram rendering of this same journey (and its flow_counts summary table)
# has been removed: it duplicated the Human-AI Decision Analysis tree (Part 6d) and used a
# decorative orange colour for disagreement, which the notebook's colour-reservation rule now
# reserves for Orange-group visuals only. rel_counts and disagree_n above are still used later
# (Part 15c, Final Claims-and-Evidence Dashboard) and are therefore kept.

## 7.2 Full Validated Analysis

**Purpose:** analyse the complete sequence Initial human decision → AI recommendation → Final
human decision. Required fields: initial human decision, AI/computer recommendation, final
human decision.

This analysis includes only cases with valid initial, AI and final decisions.

This is the same underlying logic as the "Eight Observed Human–AI Decision Journeys" analysis
introduced earlier in this notebook, restated here under the heading requested for this update
and extended with an explicit journey number and a one-line meaning for each of the eight paths.
**Its matched sample is calculated independently of the AI-Final analysis above and is not
assumed to be the same size** — see the denominator-comparison note after both matched samples
have been computed.

**Columns used** (identified directly from the existing cleaned applicant-level dataframe `df`):

- Initial human decision → `initial_decision`
- AI / computer recommendation → `computer_prediction`
- Final human decision → `effective_final_decision`
- Actual Good/Bad loan outcome → `loan_performance`


In [ ]:
# --- Three-Stage Human-AI Decision Journey: build the matched subset and the journey table ---
import itertools

jt_cols = pd.DataFrame([
    {'role': 'Initial human decision', 'column': 'initial_decision'},
    {'role': 'AI / computer recommendation', 'column': 'computer_prediction'},
    {'role': 'Final human decision', 'column': 'effective_final_decision'},
    {'role': 'Actual Good/Bad loan outcome', 'column': 'loan_performance'},
])
display(style_table(jt_cols))

# Use only rows where all three decisions are available. As explained above for the AI-Final
# sample, the boolean mask uses the already-encoded 'initial_approve'/'computer_approve'/
# 'final_approve' columns (Section 3), which are unaffected by the later ML feature-preparation
# step that fills missing raw string values with the placeholder 'Missing'.
jt_mask = df['initial_approve'].notna() & df['computer_approve'].notna() & df['final_approve'].notna()
jt_df = df.loc[jt_mask].copy()
jt_matched_n = len(jt_df)
print(f'Three-Stage matched sample (recalculated from the data, not assumed): {jt_matched_n:,}')

def _code(s):
    return s.map({'Approve': 'A', 'Reject': 'R'})

jt_df['journey_code'] = _code(jt_df.initial_decision) + _code(jt_df.computer_prediction) + _code(jt_df.effective_final_decision)

jt_order = [''.join(t) for t in itertools.product('AR', repeat=3)]  # AAA,AAR,ARA,ARR,RAA,RAR,RRA,RRR
_label = {'A': 'Approve', 'R': 'Reject'}
jt_counts = jt_df.journey_code.value_counts().reindex(jt_order).fillna(0).astype(int)

_meaning = {
    'AAA': 'Initial agreed with AI (both Approve); final unchanged. Remained aligned throughout.',
    'AAR': 'Initial agreed with AI (both Approve); human moved away to Reject at final. New final disagreement.',
    'ARA': 'Initial disagreed with AI; human resisted, kept Approve at final. Appropriate resistance or underreliance.',
    'ARR': 'Initial disagreed with AI; human changed to Reject, following AI. Appropriate acceptance or overreliance.',
    'RAA': 'Initial disagreed with AI; human changed to Approve, following AI. Appropriate acceptance or overreliance.',
    'RAR': 'Initial disagreed with AI; human resisted, kept Reject at final. Appropriate resistance or underreliance.',
    'RRA': 'Initial agreed with AI (both Reject); human moved away to Approve at final. New final disagreement.',
    'RRR': 'Initial agreed with AI (both Reject); final unchanged. Remained aligned throughout.',
}

journeys_table = pd.DataFrame({
    'journey_number': list(range(1, 9)),
    'path_code': jt_order,
    'initial_decision': [_label[c[0]] for c in jt_order],
    'ai_recommendation': [_label[c[1]] for c in jt_order],
    'final_decision': [_label[c[2]] for c in jt_order],
    'count': jt_counts.values,
    'percentage_of_three_stage_matched': (jt_counts.values / jt_matched_n * 100).round(2),
    'meaning': [_meaning[c] for c in jt_order],
})
display(style_table(journeys_table))
assert journeys_table['count'].sum() == jt_matched_n, 'Eight journey counts must sum to the Three-Stage matched total.'
pct_sum = journeys_table['percentage_of_three_stage_matched'].sum()
print(f"Sum check: the eight journey counts sum to {journeys_table['count'].sum():,}, matching {jt_matched_n:,} Three-Stage matched decisions.")
print(f"Percentage sum check: the eight journey percentages sum to {pct_sum:.2f}% (approximately 100%, allowing for rounding).")


**What this shows:** The eight possible Initial→AI→Final decision paths (e.g. Approve→Approve→Approve) and how often each occurred.

**Sample:** 1,920 complete three-stage matched cases.

In [ ]:
# --- Bar chart of the eight decision journeys ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(journeys_table.path_code, journeys_table['count'], color='#2a78d6')
for i, (cnt, pct) in enumerate(zip(journeys_table['count'], journeys_table['percentage_of_three_stage_matched'])):
    ax.text(i, cnt + journeys_table['count'].max() * 0.015, f'{pct:.1f}%', ha='center', fontsize=9)
ax.set_xlabel('Decision journey (Initial → AI → Final)')
ax.set_ylabel('Count')
ax.set_title(f'Eight Observed Human–AI Decision Journeys (n={jt_matched_n:,} Three-Stage matched decisions)')
fig.tight_layout()
for ext in ['png', 'svg']:
    fig.savefig(CHERRY / 'figures' / f'18_eight_decision_journeys.{ext}', dpi=180 if ext == 'png' else None, bbox_inches='tight')
plt.show()


**Main result:** AAA (545 cases, 28.39%) and RRR (657 cases, 34.22%) — full agreement across all three stages — together account for over 60% of journeys; the remaining six journeys capture every way a disagreement occurred and how it resolved.

**Interpretation:** Most decisions never involve any Human–AI disagreement at all; the journeys with a change (AAR, ARA, ARR, RAA, RAR, RRA) are where reliance behaviour is actually observed.

**Caution:** Journey codes track approve/reject direction only; they do not indicate whether a decision was correct against the actual loan outcome (see the journey-by-outcome analysis).

In [ ]:
# --- Initial-versus-Final Human-AI agreement/disagreement statistics (Three-Stage sample) ---
jt_initial_disagree = jt_df.initial_decision.ne(jt_df.computer_prediction)          # Initial Human-AI Disagreement
jt_initial_agree = ~jt_initial_disagree                                             # Initial Human-AI Agreement
jt_final_disagree = jt_df.effective_final_decision.ne(jt_df.computer_prediction)    # Final Human-AI Disagreement
jt_final_agree = ~jt_final_disagree                                                 # Final Human-AI Agreement
jt_changed = jt_df.initial_decision.ne(jt_df.effective_final_decision)              # human changed decision
jt_retained = ~jt_changed                                                           # human retained decision
jt_changed_towards_ai = jt_initial_disagree & jt_final_agree                        # changed towards AI
jt_resisted_ai = jt_initial_disagree & jt_final_disagree                            # resisted AI
jt_moved_away_after_agreement = jt_initial_agree & jt_final_disagree                # moved away from AI after initial agreement

initial_vs_final_comparison = pd.DataFrame([
    {'measure': 'Initial Human–AI Agreement', 'count': int(jt_initial_agree.sum()), 'percentage_of_three_stage_matched': round(100 * jt_initial_agree.mean(), 2)},
    {'measure': 'Initial Human–AI Disagreement', 'count': int(jt_initial_disagree.sum()), 'percentage_of_three_stage_matched': round(100 * jt_initial_disagree.mean(), 2)},
    {'measure': 'Final Human–AI Agreement', 'count': int(jt_final_agree.sum()), 'percentage_of_three_stage_matched': round(100 * jt_final_agree.mean(), 2)},
    {'measure': 'Final Human–AI Disagreement', 'count': int(jt_final_disagree.sum()), 'percentage_of_three_stage_matched': round(100 * jt_final_disagree.mean(), 2)},
    {'measure': 'Human changed decision (initial != final)', 'count': int(jt_changed.sum()), 'percentage_of_three_stage_matched': round(100 * jt_changed.mean(), 2)},
    {'measure': 'Human retained decision (initial == final)', 'count': int(jt_retained.sum()), 'percentage_of_three_stage_matched': round(100 * jt_retained.mean(), 2)},
    {'measure': 'Changed towards AI (subset of initial disagreement)', 'count': int(jt_changed_towards_ai.sum()), 'percentage_of_three_stage_matched': round(100 * jt_changed_towards_ai.mean(), 2)},
    {'measure': 'Resisted AI (subset of initial disagreement)', 'count': int(jt_resisted_ai.sum()), 'percentage_of_three_stage_matched': round(100 * jt_resisted_ai.mean(), 2)},
    {'measure': 'Moved away from AI after initial agreement', 'count': int(jt_moved_away_after_agreement.sum()), 'percentage_of_three_stage_matched': round(100 * jt_moved_away_after_agreement.mean(), 2)},
])
display(style_table(initial_vs_final_comparison))
print('Note: "disagreement" is never used unqualified above -- every occurrence is labelled Initial or Final.')


In [ ]:
# --- Denominator comparison: AI-Final matched sample vs. Three-Stage matched sample ---
denominator_note = pd.DataFrame([
    {'analysis': 'AI-Final matched analysis', 'required_fields': 'AI recommendation, final decision', 'matched_sample_size': ai_final_matched_n},
    {'analysis': 'Three-Stage decision journey', 'required_fields': 'initial decision, AI recommendation, final decision', 'matched_sample_size': jt_matched_n},
])
display(style_table(denominator_note))

if ai_final_matched_n == jt_matched_n:
    print(f'Both matched samples are equal ({ai_final_matched_n:,}). This was confirmed from the data, not assumed.')
else:
    diff = ai_final_matched_n - jt_matched_n
    print(f'The two matched samples are DIFFERENT, as calculated from the data (not forced to match):')
    print(f'  AI-Final matched sample:    {ai_final_matched_n:,}')
    print(f'  Three-Stage matched sample: {jt_matched_n:,}')
    print(f'  Difference: {diff:,} row(s) more in the AI-Final sample.')
    print(f'  Reason: the AI-Final analysis does not require the initial decision to be present, so it additionally')
    print(f'  includes rows where the initial decision is missing but the AI recommendation and final decision are')
    print(f'  both present. Each analysis uses its own denominator; neither is substituted for the other anywhere')
    print(f'  in this notebook.')


In [ ]:
# --- Re-verify the four reliance categories (defined ONLY within the Three-Stage initial-disagreement cases, using the actual Good/Bad outcome -- NOT the AI-Final-only sample) ---
def _is_correct(decision, outcome):
    return ((decision == 'Approve') & (outcome == 'Good')) | ((decision == 'Reject') & (outcome == 'Bad'))

jt_ai_correct = _is_correct(jt_df.computer_prediction, jt_df.loan_performance)

jt_appropriate_acceptance = jt_initial_disagree & jt_changed_towards_ai & jt_ai_correct
jt_overreliance = jt_initial_disagree & jt_changed_towards_ai & ~jt_ai_correct
jt_appropriate_resistance = jt_initial_disagree & jt_resisted_ai & ~jt_ai_correct
jt_underreliance = jt_initial_disagree & jt_resisted_ai & jt_ai_correct

reliance_check = pd.DataFrame([
    {'category': 'Appropriate acceptance', 'expected': 97, 'observed': int(jt_appropriate_acceptance.sum())},
    {'category': 'Appropriate resistance', 'expected': 323, 'observed': int(jt_appropriate_resistance.sum())},
    {'category': 'Overreliance', 'expected': 72, 'observed': int(jt_overreliance.sum())},
    {'category': 'Underreliance', 'expected': 208, 'observed': int(jt_underreliance.sum())},
])
reliance_check['difference'] = reliance_check.observed - reliance_check.expected
reliance_check['match'] = reliance_check['difference'].eq(0)
display(style_table(reliance_check))
print('Reliance-category counts reproduced exactly from Three-Stage initial-disagreement cases and actual outcomes (NOT the AI-Final-only sample).' if reliance_check['match'].all() else 'DIFFERENCES FOUND in reliance-category counts.')

# Cross-check against this notebook's own existing 'reliance_category' column (Section 4 / cell 6),
# which was computed independently via the initial_approve/computer_approve/final_approve encodings.
jt_new_cat = np.select(
    [jt_appropriate_acceptance, jt_appropriate_resistance, jt_overreliance, jt_underreliance],
    ['Appropriate acceptance', 'Appropriate resistance', 'Overreliance', 'Underreliance'],
    default='(initial agreement)'
)
jt_existing_cat = df.loc[jt_df.index, 'reliance_category'].fillna('(initial agreement)')
cross_check = pd.crosstab(pd.Series(jt_new_cat, index=jt_df.index, name='recomputed_in_this_section'), jt_existing_cat.rename('existing_Section_4_column'))
display(cross_check)
print('Cross-check: the recomputed categories above should form a pure diagonal against the existing Section 4 reliance_category column.')


In [ ]:
# --- Consolidated validation: both analyses, reported (not forced) ---
validation_rows = [
    {'check': 'AI-Final: agreement + disagreement = AI-Final matched sample',
     'left_hand_side': af_agree_n + af_disagree_n, 'right_hand_side': ai_final_matched_n,
     'match': (af_agree_n + af_disagree_n) == ai_final_matched_n},
    {'check': 'AI-Final: four decision combinations = AI-Final matched sample',
     'left_hand_side': int(ai_final_2x2['count'].sum()), 'right_hand_side': ai_final_matched_n,
     'match': int(ai_final_2x2['count'].sum()) == ai_final_matched_n},
    {'check': 'Three-Stage: eight journey counts = Three-Stage matched sample',
     'left_hand_side': int(journeys_table['count'].sum()), 'right_hand_side': jt_matched_n,
     'match': int(journeys_table['count'].sum()) == jt_matched_n},
    {'check': 'Three-Stage: initial agreement + initial disagreement = Three-Stage matched sample',
     'left_hand_side': int(jt_initial_agree.sum() + jt_initial_disagree.sum()), 'right_hand_side': jt_matched_n,
     'match': int(jt_initial_agree.sum() + jt_initial_disagree.sum()) == jt_matched_n},
    {'check': 'Three-Stage: final agreement + final disagreement = Three-Stage matched sample',
     'left_hand_side': int(jt_final_agree.sum() + jt_final_disagree.sum()), 'right_hand_side': jt_matched_n,
     'match': int(jt_final_agree.sum() + jt_final_disagree.sum()) == jt_matched_n},
    {'check': 'Three-Stage: eight journey percentages sum to ~100%',
     'left_hand_side': round(float(journeys_table['percentage_of_three_stage_matched'].sum()), 2), 'right_hand_side': 100.0,
     'match': abs(float(journeys_table['percentage_of_three_stage_matched'].sum()) - 100.0) < 0.1},
]
full_validation = pd.DataFrame(validation_rows)
display(style_table(full_validation))
if full_validation['match'].all():
    print('All validation checks passed exactly.')
else:
    print('DISCREPANCIES FOUND -- see table above (reported as observed, not forced to match):')
    print(full_validation[~full_validation['match']].to_string(index=False))

# Additional check against the project's previously reported figures, purely as a comparison (not an assumption):
prior_reported = {
    'Three-Stage matched decisions': (1920, jt_matched_n),
    'Initial disagreement (count)': (700, int(jt_initial_disagree.sum())),
    'Final disagreement (count)': (549, int(jt_final_disagree.sum())),
    'Changed towards AI': (169, int(jt_changed_towards_ai.sum())),
    'Resisted AI': (531, int(jt_resisted_ai.sum())),
    'Moved away after initial agreement': (18, int(jt_moved_away_after_agreement.sum())),
}
prior_rows = [{'check': k, 'previously_reported': exp, 'observed_now': obs, 'difference': obs - exp, 'match': obs == exp} for k, (exp, obs) in prior_reported.items()]
prior_df = pd.DataFrame(prior_rows)
display(style_table(prior_df))
print('All previously reported Three-Stage figures reproduced exactly.' if prior_df['match'].all() else 'DIFFERENCES vs. previously reported figures -- see table above.')


### 7.2.1 Notes on the Two Analyses Above

- **AI-Final matched analysis** (`ai_final_matched_n`) and **Three-Stage matched analysis**
  (`jt_matched_n`) are calculated independently and are **not** assumed to be equal. See the
  denominator-comparison table for the exact relationship found in this dataset.
- The four reliance categories (Appropriate acceptance, Appropriate resistance, Overreliance,
  Underreliance) are computed **only** on the Three-Stage initial-disagreement cases, using the
  final decision and the actual Good/Bad outcome — exactly as in Section 4. They are **not**
  recomputed on the AI-Final-only sample anywhere in this notebook.
- "Disagreement" is never used unqualified in either analysis: every occurrence above is labelled
  either **Initial Human–AI Disagreement** or **Final Human–AI Disagreement**.

## 7.3 Summary of the Three Decision Analyses (All Three Branches)

Each analysis below uses its own recalculated matched sample (never forced to a shared denominator such as 1,920 or 2,017). "Human-AI Disagreement" is never used unqualified: every occurrence is labelled either **Final Human-AI Agreement/Disagreement** (Analyses 1 and 2, which compare AI against a final decision) or **Initial Human-AI Agreement/Disagreement** (Analysis 3's headline figures, which compare AI against the initial decision).

In [ ]:
jt_agree_pct = round(100 * jt_initial_agree.sum() / jt_matched_n, 2)
jt_disagree_pct = round(100 * jt_initial_disagree.sum() / jt_matched_n, 2)

three_analyses_summary = pd.DataFrame([
    {
        'Analysis': 'AI vs Effective Final (main)',
        'Required variables': 'AI recommendation + effective final decision (final_decision_displayed, or initial_decision carried forward)',
        'Sample size': ai_final_matched_n,
        'Agreement': f'{af_agree_n:,} ({af_agree_pct:.2f}%)',
        'Disagreement': f'{af_disagree_n:,} ({round(100-af_agree_pct,2):.2f}%)',
    },
    {
        'Analysis': 'AI vs Explicit Final (sensitivity)',
        'Required variables': 'AI recommendation + explicitly displayed final decision only (no carry-forward)',
        'Sample size': ai_explicit_final_matched_n,
        'Agreement': f'{ef_agree_n:,} ({ef_agree_pct:.2f}%)',
        'Disagreement': f'{ef_disagree_n:,} ({round(100-ef_agree_pct,2):.2f}%)',
    },
    {
        'Analysis': 'Three-Stage Journey',
        'Required variables': 'Initial decision + AI recommendation + effective final decision',
        'Sample size': jt_matched_n,
        'Agreement': f'{int(jt_initial_agree.sum()):,} ({jt_agree_pct:.2f}%) [Initial]',
        'Disagreement': f'{int(jt_initial_disagree.sum()):,} ({jt_disagree_pct:.2f}%) [Initial]',
    },
])
display(style_table(three_analyses_summary))
print()
print('Note: Where no explicit final decision was recorded, the initial human decision was carried forward as the effective final decision.')
print()
print('Do the three analyses reach the same broad conclusion (majority Human-AI agreement)?')
print(f'  AI vs Effective Final:  {af_agree_pct:.2f}% agreement -> {"majority agreement" if af_agree_pct>50 else "majority disagreement"}')
print(f'  AI vs Explicit Final:   {ef_agree_pct:.2f}% agreement -> {"majority agreement" if ef_agree_pct>50 else "majority disagreement"}')
print(f'  Three-Stage (initial):  {jt_agree_pct:.2f}% agreement -> {"majority agreement" if jt_agree_pct>50 else "majority disagreement"}')


### 7.3.1 Standardised Three-Analysis Comparison Table

The table above already compares the three analyses descriptively; the table below repeats the comparison using the standardised analysis names and column layout required for this section (analysis name, required variables, sample size, agreement count/percentage, disagreement count/percentage), reusing the exact same already-computed counts -- no value is recalculated on a different denominator.

In [ ]:
# =====================================================================
# AI vs Explicit Final -- full 2x2 summary, and the three-analysis comparison table
# using the exact standardised analysis names requested for this section. Reuses the
# already-computed matched samples, agreement counts and percentages from the cells
# above (af_*, ef_*, jt_*) -- nothing here is recalculated on a different denominator.
# =====================================================================
ef_ai_approve_final_approve = int(ai_explicit_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Approve'")['count'].iloc[0])
ef_ai_approve_final_reject = int(ai_explicit_final_2x2.query("ai_recommendation=='Approve' and final_decision=='Reject'")['count'].iloc[0])
ef_ai_reject_final_approve = int(ai_explicit_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Approve'")['count'].iloc[0])
ef_ai_reject_final_reject = int(ai_explicit_final_2x2.query("ai_recommendation=='Reject' and final_decision=='Reject'")['count'].iloc[0])

ai_explicit_final_summary = pd.DataFrame([
    {'measure': 'Total AI-Explicit-Final matched decisions', 'count': ai_explicit_final_matched_n, 'percentage': 100.0},
    {'measure': 'Explicit Final Human-AI Agreement', 'count': ef_agree_n, 'percentage': ef_agree_pct},
    {'measure': 'Explicit Final Human-AI Disagreement', 'count': ef_disagree_n, 'percentage': round(100 - ef_agree_pct, 2)},
    {'measure': 'AI Approve -> Explicit Final Approve', 'count': ef_ai_approve_final_approve, 'percentage': round(100 * ef_ai_approve_final_approve / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Approve -> Explicit Final Reject', 'count': ef_ai_approve_final_reject, 'percentage': round(100 * ef_ai_approve_final_reject / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Reject -> Explicit Final Approve', 'count': ef_ai_reject_final_approve, 'percentage': round(100 * ef_ai_reject_final_approve / ai_explicit_final_matched_n, 2)},
    {'measure': 'AI Reject -> Explicit Final Reject', 'count': ef_ai_reject_final_reject, 'percentage': round(100 * ef_ai_reject_final_reject / ai_explicit_final_matched_n, 2)},
])
display(style_table(ai_explicit_final_summary))

jt_final_agree_n_cmp = int(jt_final_agree.sum())
jt_final_disagree_n_cmp = int(jt_final_disagree.sum())
jt_final_agree_pct_cmp = round(100 * jt_final_agree_n_cmp / jt_matched_n, 2)
jt_final_disagree_pct_cmp = round(100 * jt_final_disagree_n_cmp / jt_matched_n, 2)

three_decision_analysis_comparison = pd.DataFrame([
    {'analysis_name': 'AI vs Effective Final Decision',
     'required_variables': 'AI recommendation + effective final decision',
     'sample_size': ai_final_matched_n,
     'initial_agreement_count': 'Not applicable', 'initial_agreement_percentage': 'Not applicable',
     'initial_disagreement_count': 'Not applicable', 'initial_disagreement_percentage': 'Not applicable',
     'final_agreement_count': af_agree_n, 'final_agreement_percentage': af_agree_pct,
     'final_disagreement_count': af_disagree_n, 'final_disagreement_percentage': round(100 - af_agree_pct, 2),
     'notes': 'No initial decision is used to build this sample; only AI recommendation vs. effective final decision.'},
    {'analysis_name': 'AI vs Explicit Final Decision',
     'required_variables': 'AI recommendation + explicitly displayed final decision (no carry-forward)',
     'sample_size': ai_explicit_final_matched_n,
     'initial_agreement_count': 'Not applicable', 'initial_agreement_percentage': 'Not applicable',
     'initial_disagreement_count': 'Not applicable', 'initial_disagreement_percentage': 'Not applicable',
     'final_agreement_count': ef_agree_n, 'final_agreement_percentage': ef_agree_pct,
     'final_disagreement_count': ef_disagree_n, 'final_disagreement_percentage': round(100 - ef_agree_pct, 2),
     'notes': 'Sensitivity analysis: only rows with an explicitly recorded final decision; the initial decision is never carried forward here.'},
    {'analysis_name': 'Complete Three-Stage Journey',
     'required_variables': 'Initial decision + AI recommendation + effective final decision',
     'sample_size': jt_matched_n,
     'initial_agreement_count': int(jt_initial_agree.sum()), 'initial_agreement_percentage': jt_agree_pct,
     'initial_disagreement_count': int(jt_initial_disagree.sum()), 'initial_disagreement_percentage': jt_disagree_pct,
     'final_agreement_count': jt_final_agree_n_cmp, 'final_agreement_percentage': jt_final_agree_pct_cmp,
     'final_disagreement_count': jt_final_disagree_n_cmp, 'final_disagreement_percentage': jt_final_disagree_pct_cmp,
     'notes': 'Only analysis reporting both initial and final agreement/disagreement stages -- the two are kept in separate columns, never mixed into one generic "Agreement" column.'},
])
display(style_table(three_decision_analysis_comparison))

_cmp_arith_checks = pd.DataFrame([
    {'check': 'AI vs Effective Final: final agreement + disagreement = sample size',
     'left_hand_side': af_agree_n + af_disagree_n, 'right_hand_side': ai_final_matched_n,
     'match': (af_agree_n + af_disagree_n) == ai_final_matched_n},
    {'check': 'AI vs Explicit Final: final agreement + disagreement = sample size',
     'left_hand_side': ef_agree_n + ef_disagree_n, 'right_hand_side': ai_explicit_final_matched_n,
     'match': (ef_agree_n + ef_disagree_n) == ai_explicit_final_matched_n},
    {'check': 'Three-Stage: initial agreement + disagreement = sample size',
     'left_hand_side': int(jt_initial_agree.sum()) + int(jt_initial_disagree.sum()), 'right_hand_side': jt_matched_n,
     'match': (int(jt_initial_agree.sum()) + int(jt_initial_disagree.sum())) == jt_matched_n},
    {'check': 'Three-Stage: final agreement + disagreement = sample size',
     'left_hand_side': jt_final_agree_n_cmp + jt_final_disagree_n_cmp, 'right_hand_side': jt_matched_n,
     'match': (jt_final_agree_n_cmp + jt_final_disagree_n_cmp) == jt_matched_n},
])
display(style_table(_cmp_arith_checks))
_conclusion_same = (af_agree_pct > 50) == (ef_agree_pct > 50) == (jt_final_agree_pct_cmp > 50)
print(f'\nSame broad conclusion (majority final agreement) across all three analyses: {_conclusion_same}')
print('Note: sample sizes are not forced to match -- each analysis uses its own recalculated denominator (never reusing 1,920 for all three).')
print('Note: initial-stage columns are only populated for the Three-Stage Journey analysis; the other two analyses do not use an initial decision at all.')


**What this shows:** A single synthesised tree combining all three Final Human–AI comparisons (Part 6 and Part 7) with exact denominators at every branch.

**Sample:** 2,017 / 1,831 / 1,920 rows respectively, one denominator per branch (stated in the figure).

---
## 7.4 Human–AI Decision Analysis Tree Diagram

The tree diagram below is generated programmatically from the values calculated in the three analyses above -- no count or percentage is hard-coded into the figure. Branch 1 (AI vs Effective Final) and Branch 3 (Three-Stage Journey) use blue for agreement-side outcomes and orange for disagreement-side outcomes throughout; Branch 2 (the explicit-final sensitivity analysis) is shown in purple to keep it visually distinct as a secondary/robustness analysis, never presented as equivalent to the main results. Each branch displays its own denominator.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import os

# Graphviz is preferred where available for automatic layout, but this diagram's exact box/arrow
# geometry (denominators per branch, no crossing text, consistent agreement/disagreement styling)
# is fully implemented and tested below using matplotlib patches; the notebook must not fail if
# graphviz is unavailable, so we only probe for it informationally and always render with matplotlib.
try:
    import graphviz as _graphviz_probe
    _GRAPHVIZ_AVAILABLE = True
except ImportError:
    _GRAPHVIZ_AVAILABLE = False
print(f'Graphviz available: {_GRAPHVIZ_AVAILABLE} (rendering with matplotlib patches/arrows either way; this notebook does not depend on graphviz).')

TREE_DIR = OUT / 'cherry_on_top' / 'figures'
TREE_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_ROWS = len(df)
assert TOTAL_ROWS == 2200

# Colour-reservation rule: this diagram compares Human-AI agreement/disagreement, not Orange/
# Purple applicant groups, so every colour below is drawn from the central neutral palette
# (Part 1b) -- never GROUP_ORANGE / GROUP_PURPLE. 'ORANGE' names the disagreement colour and
# 'SENS' names the sensitivity-branch colour purely for continuity with the geometry code below.
BLUE, ORANGE, INK, SECINK, SENS = PALETTE_PRIMARY_BLUE, PALETTE_MUTED_RED, PALETTE_DARK_TEXT, PALETTE_GREY, PALETTE_TEAL
AGREE_FACE, DISAGREE_FACE, SENS_FACE = PALETTE_LIGHT_BLUE_GREY, '#FCEAEA', '#E6F4F1'

CW, CH = 38, 23
fig, ax = plt.subplots(figsize=(CW, 18))
ax.set_xlim(0, CW); ax.set_ylim(5.2, CH); ax.axis('off')

def tbox(x, y, w, h, text, edgecolor=INK, facecolor='white', fontsize=10.0, fontweight='normal', lw=1.6):
    patch = FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.02,rounding_size=0.09",
                            linewidth=lw, edgecolor=edgecolor, facecolor=facecolor, mutation_aspect=1, zorder=2)
    ax.add_patch(patch)
    ax.text(x, y, text, ha='center', va='center', fontsize=fontsize, color=INK, fontweight=fontweight, linespacing=1.25, zorder=3)
    return dict(x=x, y=y, w=w, h=h)

def tarrow(b_from, b_to, color=INK, lw=1.4, dx_from=0.0, dx_to=0.0, rad=None):
    p0 = (b_from['x'] + dx_from, b_from['y'] - b_from['h']/2)
    p1 = (b_to['x'] + dx_to, b_to['y'] + b_to['h']/2)
    connectionstyle = f'arc3,rad={rad}' if rad is not None else None
    ax.add_patch(FancyArrowPatch(p0, p1, arrowstyle='-|>', mutation_scale=14, color=color, lw=lw, shrinkA=2, shrinkB=4,
                                  connectionstyle=connectionstyle, zorder=1))

ax.text(CW/2, 22.5, 'Human–AI Decision Analysis', ha='center', fontsize=25, color=INK, fontweight='bold')
ax.text(CW/2, 21.85, 'Three separate decision analyses, each computed on its own recalculated matched sample', ha='center', fontsize=12.5, color=SECINK, style='italic')

top = tbox(CW/2, 20.6, 9.5, 0.9, f'All {TOTAL_ROWS:,} applicant rows', fontweight='bold', fontsize=13.5)

BX1, BX2, BX3 = 6.5, 19, 31.5

b1 = tbox(BX1, 18.7, 10.6, 1.0, f'Branch 1: AI vs Effective Final Decision\nMatched sample: {ai_final_matched_n:,} (100%)', edgecolor=BLUE, fontweight='bold', fontsize=11.8)
b2 = tbox(BX2, 18.7, 10.6, 1.35, f'Branch 2: AI vs Explicit Final Decision\nSensitivity analysis using only explicitly\nrecorded final decisions\nMatched sample: {ai_explicit_final_matched_n:,} (100%)', edgecolor=SENS, facecolor=SENS_FACE, fontweight='bold', fontsize=10.8)
b3 = tbox(BX3, 18.7, 10.6, 1.0, f'Branch 3: Three-Stage Decision Journey\nMatched sample: {jt_matched_n:,} (100%)', edgecolor=INK, fontweight='bold', fontsize=11.8)
for b in (b1, b2, b3):
    tarrow(top, b, dx_from=(b['x'] - CW/2) * 0.05)

# ===================== Branch 1: AI vs Effective Final =====================
b1_agree = tbox(BX1 - 2.7, 16.9, 4.9, 1.05, f'Final Human–AI\nAgreement\n{af_agree_n:,} ({af_agree_pct:.2f}%)', edgecolor=BLUE, facecolor=AGREE_FACE, fontweight='bold', fontsize=10.8)
b1_disagree = tbox(BX1 + 2.7, 16.9, 4.9, 1.05, f'Final Human–AI\nDisagreement\n{af_disagree_n:,} ({round(100-af_agree_pct,2):.2f}%)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontweight='bold', fontsize=10.8)
tarrow(b1, b1_agree, color=BLUE, dx_from=-1.5, dx_to=1.0)
tarrow(b1, b1_disagree, color=ORANGE, dx_from=1.5, dx_to=-1.0)

b1_aa = tbox(BX1 - 4.0, 14.8, 2.55, 1.15, f'AI Approve →\nFinal Approve\n{ai_approve_final_approve:,}\n({100*ai_approve_final_approve/ai_final_matched_n:.2f}%)', edgecolor=BLUE, facecolor=AGREE_FACE, fontsize=8.7)
b1_rr = tbox(BX1 - 1.4, 14.8, 2.55, 1.15, f'AI Reject →\nFinal Reject\n{ai_reject_final_reject:,}\n({100*ai_reject_final_reject/ai_final_matched_n:.2f}%)', edgecolor=BLUE, facecolor=AGREE_FACE, fontsize=8.7)
tarrow(b1_agree, b1_aa, color=BLUE, dx_from=-0.8, dx_to=0.4)
tarrow(b1_agree, b1_rr, color=BLUE, dx_from=0.8, dx_to=-0.4)

b1_ar = tbox(BX1 + 1.4, 14.8, 2.55, 1.15, f'AI Approve →\nFinal Reject\n{ai_approve_final_reject:,}\n({100*ai_approve_final_reject/ai_final_matched_n:.2f}%)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontsize=8.7)
b1_ra = tbox(BX1 + 4.0, 14.8, 2.55, 1.15, f'AI Reject →\nFinal Approve\n{ai_reject_final_approve:,}\n({100*ai_reject_final_approve/ai_final_matched_n:.2f}%)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontsize=8.7)
tarrow(b1_disagree, b1_ar, color=ORANGE, dx_from=-0.8, dx_to=0.4)
tarrow(b1_disagree, b1_ra, color=ORANGE, dx_from=0.8, dx_to=-0.4)
ax.text(BX1, 13.15, f'Denominator: {ai_final_matched_n:,} rows with a valid AI recommendation\nand a valid effective final decision', ha='center', fontsize=9.2, color=SECINK, style='italic')

# ===================== Branch 2: AI vs Explicit Final (sensitivity) =====================
b2_agree = tbox(BX2 - 2.7, 16.9, 4.9, 1.05, f'Explicit Final\nHuman–AI Agreement\n{ef_agree_n:,} ({ef_agree_pct:.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontweight='bold', fontsize=10.8)
b2_disagree = tbox(BX2 + 2.7, 16.9, 4.9, 1.05, f'Explicit Final\nHuman–AI Disagreement\n{ef_disagree_n:,} ({round(100-ef_agree_pct,2):.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontweight='bold', fontsize=10.8)
tarrow(b2, b2_agree, color=SENS, dx_from=-1.5, dx_to=1.0)
tarrow(b2, b2_disagree, color=SENS, dx_from=1.5, dx_to=-1.0)

b2_aa = tbox(BX2 - 4.0, 14.8, 2.55, 1.15, f'AI Approve →\nExplicit Final Approve\n{ef_ai_approve_final_approve:,}\n({100*ef_ai_approve_final_approve/ai_explicit_final_matched_n:.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontsize=8.0)
b2_rr = tbox(BX2 - 1.4, 14.8, 2.55, 1.15, f'AI Reject →\nExplicit Final Reject\n{ef_ai_reject_final_reject:,}\n({100*ef_ai_reject_final_reject/ai_explicit_final_matched_n:.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontsize=8.0)
tarrow(b2_agree, b2_aa, color=SENS, dx_from=-0.8, dx_to=0.4)
tarrow(b2_agree, b2_rr, color=SENS, dx_from=0.8, dx_to=-0.4)

b2_ar = tbox(BX2 + 1.4, 14.8, 2.55, 1.15, f'AI Approve →\nExplicit Final Reject\n{ef_ai_approve_final_reject:,}\n({100*ef_ai_approve_final_reject/ai_explicit_final_matched_n:.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontsize=8.0)
b2_ra = tbox(BX2 + 4.0, 14.8, 2.55, 1.15, f'AI Reject →\nExplicit Final Approve\n{ef_ai_reject_final_approve:,}\n({100*ef_ai_reject_final_approve/ai_explicit_final_matched_n:.2f}%)', edgecolor=SENS, facecolor=SENS_FACE, fontsize=8.0)
tarrow(b2_disagree, b2_ar, color=SENS, dx_from=-0.8, dx_to=0.4)
tarrow(b2_disagree, b2_ra, color=SENS, dx_from=0.8, dx_to=-0.4)

assert ef_ai_approve_final_approve + ef_ai_approve_final_reject + ef_ai_reject_final_approve + ef_ai_reject_final_reject == ai_explicit_final_matched_n
ax.text(BX2, 13.15, f'Denominator: {ai_explicit_final_matched_n:,} rows with a valid AI recommendation\nand an explicitly displayed final decision (no carry-forward)', ha='center', fontsize=9.2, color=SECINK, style='italic')
ax.text(BX2, 11.75, f'vs. Branch 1 agreement: {af_agree_pct:.2f}% → {ef_agree_pct:.2f}%\n({"same conclusion: majority agreement in both" if comparison_conclusion_same_direction else "CONCLUSION DIFFERS"})',
        ha='center', fontsize=9.7, color=SENS, fontweight='bold')

# ===================== Branch 3: Three-Stage Decision Journey =====================
b3_iagree = tbox(BX3 - 2.9, 16.9, 4.9, 1.05, f'Initial Human–AI\nAgreement\n{int(jt_initial_agree.sum()):,} ({jt_agree_pct:.2f}%)', edgecolor=BLUE, facecolor=AGREE_FACE, fontweight='bold', fontsize=10.8)
b3_idisagree = tbox(BX3 + 2.9, 16.9, 4.9, 1.05, f'Initial Human–AI\nDisagreement\n{int(jt_initial_disagree.sum()):,} ({jt_disagree_pct:.2f}%)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontweight='bold', fontsize=10.8)
tarrow(b3, b3_iagree, color=BLUE, dx_from=-1.5, dx_to=1.0)
tarrow(b3, b3_idisagree, color=ORANGE, dx_from=1.5, dx_to=-1.0)

n_retained = int((jt_initial_agree & jt_final_agree).sum())  # retained initial agreement (not previously named in Section above)
n_moved = int(jt_moved_away_after_agreement.sum())
n_toward = int(jt_changed_towards_ai.sum())
n_resisted = int(jt_resisted_ai.sum())

n_initial_agree = int(jt_initial_agree.sum())
n_initial_disagree = int(jt_initial_disagree.sum())
pct_retained_of_agree = 100 * n_retained / n_initial_agree
pct_moved_of_agree = 100 * n_moved / n_initial_agree
pct_toward_of_disagree = 100 * n_toward / n_initial_disagree
pct_resisted_of_disagree = 100 * n_resisted / n_initial_disagree

X_retained, X_moved, X_toward, X_resisted = BX3 - 4.35, BX3 - 1.45, BX3 + 1.45, BX3 + 4.35
b3_retained = tbox(X_retained, 14.8, 2.55, 1.3, f'Retained initial\nagreement\n{n_retained:,}\n{pct_retained_of_agree:.2f}% of initial agreements\n({100*n_retained/jt_matched_n:.2f}% of 1,920)', edgecolor=BLUE, facecolor=AGREE_FACE, fontsize=7.6)
b3_moved = tbox(X_moved, 14.8, 2.55, 1.3, f'Moved away after\ninitial agreement\n{n_moved:,}\n{pct_moved_of_agree:.2f}% of initial agreements\n({100*n_moved/jt_matched_n:.2f}% of 1,920)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontsize=7.6)
tarrow(b3_iagree, b3_retained, color=BLUE, dx_from=-0.8, dx_to=0.4)
tarrow(b3_iagree, b3_moved, color=ORANGE, dx_from=0.8, dx_to=-0.4)

b3_toward = tbox(X_toward, 14.8, 2.55, 1.3, f'Changed towards AI\n{n_toward:,}\n{pct_toward_of_disagree:.2f}% of initial disagreements\n({100*n_toward/jt_matched_n:.2f}% of 1,920)', edgecolor=BLUE, facecolor=AGREE_FACE, fontsize=7.6)
b3_resisted = tbox(X_resisted, 14.8, 2.55, 1.3, f'Resisted AI\n{n_resisted:,}\n{pct_resisted_of_disagree:.2f}% of initial disagreements\n({100*n_resisted/jt_matched_n:.2f}% of 1,920)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontsize=7.6)
tarrow(b3_idisagree, b3_toward, color=BLUE, dx_from=-0.8, dx_to=0.4)
tarrow(b3_idisagree, b3_resisted, color=ORANGE, dx_from=0.8, dx_to=-0.4)

jt_final_agree_n = n_retained + n_toward
jt_final_disagree_n = n_moved + n_resisted
assert jt_final_agree_n == int(jt_final_agree.sum())
assert jt_final_disagree_n == int(jt_final_disagree.sum())

b3_final_agree = tbox(X_retained + 1.45, 12.55, 6.4, 1.0, f'Final Human–AI Agreement\n{jt_final_agree_n:,} ({100*jt_final_agree_n/jt_matched_n:.2f}%)', edgecolor=BLUE, facecolor=AGREE_FACE, fontweight='bold', fontsize=10.5)
b3_final_disagree = tbox(X_toward + 1.45, 12.55, 6.4, 1.0, f'Final Human–AI Disagreement\n{jt_final_disagree_n:,} ({100*jt_final_disagree_n/jt_matched_n:.2f}%)', edgecolor=ORANGE, facecolor=DISAGREE_FACE, fontweight='bold', fontsize=10.5)
tarrow(b3_retained, b3_final_agree, color=BLUE, dx_from=0.2, dx_to=-1.6)
tarrow(b3_moved, b3_final_disagree, color=ORANGE, dx_from=0.2, dx_to=-1.6, rad=0.2)
tarrow(b3_toward, b3_final_agree, color=BLUE, dx_from=-0.2, dx_to=1.6, rad=0.2)
tarrow(b3_resisted, b3_final_disagree, color=ORANGE, dx_from=-0.2, dx_to=1.6)
ax.text(BX3, 11.15, f'Denominator: {jt_matched_n:,} rows with a valid initial decision, AI recommendation\nand effective final decision, all three present',
        ha='center', fontsize=9.2, color=SECINK, style='italic')

# ===================== Lower section: eight decision journeys =====================
ax.plot([0.3, CW - 0.3], [9.9, 9.9], color='#898781', lw=1.1, zorder=1)
ax.text(CW/2, 9.5, 'Eight Observed Decision Journeys (Initial → AI → Final), Three-Stage sample',
        ha='center', fontsize=14, color=INK, fontweight='bold')

_journey_desc = {
    'AAA': 'Approve→Approve→Approve', 'AAR': 'Approve→Approve→Reject',
    'ARA': 'Approve→Reject→Approve', 'ARR': 'Approve→Reject→Reject',
    'RAA': 'Reject→Approve→Approve', 'RAR': 'Reject→Approve→Reject',
    'RRA': 'Reject→Reject→Approve', 'RRR': 'Reject→Reject→Reject',
}
_journey_lookup = journeys_table.set_index('path_code')
jbox_w = 4.2
_jbox_gap = 0.25
start_x = CW/2 - (jbox_w + _jbox_gap) * 8 / 2 + jbox_w / 2
for i, code in enumerate(jt_order):
    cnt = int(_journey_lookup.loc[code, 'count'])
    pct = float(_journey_lookup.loc[code, 'percentage_of_three_stage_matched'])
    is_agree_final = code[1] == code[2]
    color = BLUE if is_agree_final else ORANGE
    face = AGREE_FACE if is_agree_final else DISAGREE_FACE
    x = start_x + i * (jbox_w + _jbox_gap)
    tbox(x, 8.1, jbox_w, 1.35, f'{code}\n{_journey_desc[code]}\n{cnt:,} ({pct:.2f}%)', edgecolor=color, facecolor=face, fontsize=9.3)

assert int(journeys_table['count'].sum()) == jt_matched_n

ax.text(CW/2, 6.6, 'Effective final decision uses the explicitly displayed final decision when available;\notherwise, the initial human decision is carried forward.',
        ha='center', fontsize=11, color=SECINK, style='italic')

for ext in ['png', 'svg']:
    fig.savefig(TREE_DIR / f'human_ai_final_decision_analysis_tree.{ext}', dpi=200 if ext == 'png' else None, bbox_inches='tight')
plt.show()
print('Tree diagram saved to', public_path(TREE_DIR))

**Main result:** All three branches show majority Final Human–AI agreement, and the three-stage branch shows how that agreement was reached (retained, changed towards AI, resisted AI, or moved away).

**Interpretation:** The tree is the single authoritative synthesis of every Final Human–AI comparison in this notebook; branch-specific findings elsewhere should match the branch shown here exactly (verified by the tree validation and analytical checksum).

**Caution:** Each branch uses its own matched sample and denominator (2,017 / 1,831 / 1,920) — percentages are never comparable across branches without checking the stated denominator.

### 7.4.1 Tree Diagram Validation

Confirms that agreement + disagreement equals the matched sample in every branch, that all four AI-Final combinations and all eight three-stage journeys sum to their respective matched samples, and that every count shown in the tree diagram matches the corresponding analytical table computed above -- not just visually consistent, but numerically identical.

In [ ]:
tree_validation_rows = [
    {'check': 'Branch 1: agreement + disagreement = AI-Effective-Final matched sample',
     'left_hand_side': af_agree_n + af_disagree_n, 'right_hand_side': ai_final_matched_n,
     'match': (af_agree_n + af_disagree_n) == ai_final_matched_n},
    {'check': 'Branch 1: all four AI-Final combinations = AI-Effective-Final matched sample',
     'left_hand_side': ai_approve_final_approve + ai_approve_final_reject + ai_reject_final_approve + ai_reject_final_reject,
     'right_hand_side': ai_final_matched_n,
     'match': (ai_approve_final_approve + ai_approve_final_reject + ai_reject_final_approve + ai_reject_final_reject) == ai_final_matched_n},
    {'check': 'Branch 2: agreement + disagreement = AI-Explicit-Final matched sample',
     'left_hand_side': ef_agree_n + ef_disagree_n, 'right_hand_side': ai_explicit_final_matched_n,
     'match': (ef_agree_n + ef_disagree_n) == ai_explicit_final_matched_n},
    {'check': 'Branch 3: initial agreement + initial disagreement = Three-Stage matched sample',
     'left_hand_side': int(jt_initial_agree.sum() + jt_initial_disagree.sum()), 'right_hand_side': jt_matched_n,
     'match': int(jt_initial_agree.sum() + jt_initial_disagree.sum()) == jt_matched_n},
    {'check': 'Branch 3: all eight journey counts = Three-Stage matched sample',
     'left_hand_side': int(journeys_table['count'].sum()), 'right_hand_side': jt_matched_n,
     'match': int(journeys_table['count'].sum()) == jt_matched_n},
    {'check': 'Branch 3: retained + moved + changed-towards-AI + resisted = Three-Stage matched sample',
     'left_hand_side': n_retained + n_moved + n_toward + n_resisted, 'right_hand_side': jt_matched_n,
     'match': (n_retained + n_moved + n_toward + n_resisted) == jt_matched_n},
    {'check': 'Branch 3: tree Final Human-AI Agreement matches the analytical table (initial_vs_final_comparison)',
     'left_hand_side': jt_final_agree_n, 'right_hand_side': int(jt_final_agree.sum()),
     'match': jt_final_agree_n == int(jt_final_agree.sum())},
    {'check': 'Branch 3: tree Final Human-AI Disagreement matches the analytical table (initial_vs_final_comparison)',
     'left_hand_side': jt_final_disagree_n, 'right_hand_side': int(jt_final_disagree.sum()),
     'match': jt_final_disagree_n == int(jt_final_disagree.sum())},
    {'check': 'Branch 1 tree agreement/disagreement match ai_final_2x2 status breakdown',
     'left_hand_side': (af_agree_n, af_disagree_n),
     'right_hand_side': (int(ai_final_2x2.loc[ai_final_2x2.status.eq('Agreement'), 'count'].sum()),
                          int(ai_final_2x2.loc[ai_final_2x2.status.eq('Disagreement'), 'count'].sum())),
     'match': (af_agree_n, af_disagree_n) == (int(ai_final_2x2.loc[ai_final_2x2.status.eq('Agreement'), 'count'].sum()),
                                               int(ai_final_2x2.loc[ai_final_2x2.status.eq('Disagreement'), 'count'].sum()))},
    {'check': 'Branch 2 tree agreement/disagreement match ai_explicit_final_2x2 status breakdown',
     'left_hand_side': (ef_agree_n, ef_disagree_n),
     'right_hand_side': (int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Agreement'), 'count'].sum()),
                          int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Disagreement'), 'count'].sum())),
     'match': (ef_agree_n, ef_disagree_n) == (int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Agreement'), 'count'].sum()),
                                               int(ai_explicit_final_2x2.loc[ai_explicit_final_2x2.status.eq('Disagreement'), 'count'].sum()))},
    {'check': 'Denominators differ across branches (1,920 is not reused as every branch denominator)',
     'left_hand_side': f'{ai_final_matched_n}, {ai_explicit_final_matched_n}, {jt_matched_n}', 'right_hand_side': 'three distinct values expected',
     'match': len({ai_final_matched_n, ai_explicit_final_matched_n, jt_matched_n}) == 3},
    {'check': 'Carry-forward rule verified consistent with existing project logic',
     'left_hand_side': _carry_forward_consistent, 'right_hand_side': True, 'match': _carry_forward_consistent},
]
tree_validation = pd.DataFrame(tree_validation_rows)
display(style_table(tree_validation))
n_fail = (~tree_validation['match']).sum()
print()
print(f'Failed checks: {n_fail} / {len(tree_validation)}')
if n_fail == 0:
    print('SUCCESS: all tree/table consistency checks passed.')


## 7.5 Representative Reliance Case Studies

The table below selects two structured-data examples from each reliance category. These examples show how the same collaboration process can help or harm a decision.

No transcript quotation is attached to an individual applicant because the current transcript parsing does not provide reliable applicant-level speech alignment. This separation prevents an attractive example from being presented as stronger evidence than the data support.

In [ ]:
# Deterministic structured case studies: two examples from every reliance category.
interpretation_map = {
    'Appropriate acceptance':'The computer was correct and the participant moved to the better decision.',
    'Appropriate resistance':'The computer was incorrect and the participant retained the better judgement.',
    'Overreliance':'The participant moved away from a correct initial decision and followed an incorrect computer recommendation.',
    'Underreliance':'The computer was correct, but the participant retained an incorrect initial decision.'
}
case_parts=[]
for cat in ['Appropriate acceptance','Appropriate resistance','Overreliance','Underreliance']:
    g=(journey[journey.reliance_category.eq(cat)]
       .sort_values(['participant_id','round','row_in_round']).head(2).copy())
    g['interpretation']=interpretation_map[cat]
    case_parts.append(g)
case_studies=pd.concat(case_parts,ignore_index=True)
case_cols=['reliance_category','participant_id','round','row_in_round','loan_risk','annual_income','loan_amount',
           'dti','revolving_utility_pct','background','initial_decision','computer_prediction',
           'effective_final_decision','loan_performance','interpretation']
case_studies=case_studies[case_cols]
display(style_table(case_studies))
case_studies.to_csv(CHERRY/'tables'/'representative_reliance_case_studies.csv',index=False)

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
The complete three-stage matched sample (1,920 cases) split into 1,220 initial agreements (63.54%) and 700 initial disagreements (36.46%). Of the disagreements, 169 changed toward AI and 531 resisted AI; of the agreements, 1,202 were retained and 18 moved away, giving 1,371 final agreements and 549 final disagreements. All eight observed decision journeys and all four reliance categories are reported, and the Decision Analysis tree diagram synthesises all three branches together. The next Part turns to decision accuracy and behavioural modelling.
</div>

# Part 8 — Accuracy and Behavioural Modelling

In [ ]:
formula='OUTCOME ~ loan_risk + loan_to_income_10pct + dti_5_points + revolving_10_points + inquiries_last_6_months + C(home_ownership)+C(purpose_grouped)+C(background)+round'
init_data=df.dropna(subset=['initial_approve']).copy(); final_data=df.dropna(subset=['final_approve']).copy()
init_model=fit_2way(formula.replace('OUTCOME','initial_approve'),init_data); init_model['model']='Initial approval'
final_model=fit_2way(formula.replace('OUTCOME','final_approve'),final_data); final_model['model']='Final approval'
models=pd.concat([init_model,final_model],ignore_index=True); display(init_model.round(4)); display(final_model.round(4))

**What this shows:** Adjusted marginal effect of each applicant-risk factor on final approval, from the applicant-factor logistic model.

**Sample:** All applicant decision rows used to fit the applicant-factor model.

In [ ]:
keys=['loan_risk','loan_to_income_10pct','dti_5_points','revolving_10_points','inquiries_last_6_months','C(home_ownership)[T.OWN]','C(background)[T.Purple]']
plot=models[models.term.isin(keys)].copy(); names={'loan_risk':'Loan risk (+1)','loan_to_income_10pct':'Loan/income (+10 pp)','dti_5_points':'DTI (+5)','revolving_10_points':'Revolving use (+10 pp)','inquiries_last_6_months':'Inquiries (+1)','C(home_ownership)[T.OWN]':'Own vs mortgage','C(background)[T.Purple]':'Purple vs Orange'}; plot['label']=plot.term.map(names)
order=list(dict.fromkeys(plot.label)); fig,ax=plt.subplots(figsize=(8,5.5))
for off,m in [(-.08,'Initial approval'),(.08,'Final approval')]:
 s=plot[plot.model==m].set_index('label').reindex(order); y=np.arange(len(order))+off; ax.errorbar(s.odds_ratio,y,xerr=[s.odds_ratio-s.ci_low,s.ci_high-s.odds_ratio],fmt='o',capsize=3,label=m)
ax.axvline(1); ax.set_xscale('log'); ax.set_yticks(np.arange(len(order))); ax.set_yticklabels(order); ax.set_xlabel('Odds ratio (log scale)'); ax.set_title('Applicant factors associated with approval'); ax.legend(); savefig(fig,'03_applicant_factors')

**Main result:** Higher loan risk, loan-to-income ratio, DTI, revolving utilisation and recent inquiries are consistently associated with lower approval odds; owning a home (vs. mortgage) is associated with higher approval odds.

**Interpretation:** Final approval responds to applicant financial-risk factors in the expected direction and magnitude, independent of the Human–AI collaboration variables examined elsewhere.

**Caution:** These are adjusted regression coefficients, not causal effect estimates, and use two-way cluster-robust uncertainty for participant and applicant-profile clustering.

## 8.1 Interactions, Grouped Machine Learning and SHAP

The analyses below extend—but do not replace—the audited repeated-measures results above. They answer three additional questions:

1. Are important relationships nonlinear or interactive?
2. Can a predictive model generalise to unseen participants and/or unseen applicant profiles?
3. Are feature-importance and SHAP explanations stable across grouped validation folds?

**Interpretation rule:** regression and fairness analyses remain the primary evidence. CatBoost, permutation importance and SHAP are supporting robustness tools. SHAP explains model predictions; it does not establish causality or ethical fairness.

In [ ]:
# Extension setup
from itertools import combinations
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score, f1_score, brier_score_loss
from catboost import CatBoostClassifier, Pool

EXTRA = OUT / 'enhanced'
for d in [EXTRA, EXTRA/'tables', EXTRA/'figures', EXTRA/'reports']:
    d.mkdir(parents=True, exist_ok=True)

ML_NUM = ['loan_risk','loan_to_income_ratio','dti','revolving_utility_pct','inquiries_last_6_months','round']
ML_CAT = ['home_ownership','purpose_grouped','background','initial_decision','computer_prediction']
ML_FEATURES = ML_NUM + ML_CAT

# Categorical missing values must remain explicit rather than silently dropping records.
for c in ML_CAT:
    df[c] = df[c].astype('object').fillna('Missing').astype(str)

print('Extension output directory:', public_path(EXTRA))
print('CatBoost version loaded successfully.')

**What this shows:** Adjusted predicted probability of final approval across loan-risk levels, separately for each computer recommendation.

**Sample:** Cases with a valid initial and final decision (interaction model).

---
### 8.1.1 Marginal Effects and Interaction Analysis

The first extension visualises adjusted predicted probabilities rather than relying only on odds ratios. The interaction model asks whether the association between loan risk and final approval changes depending on the computer recommendation, and whether the effect of the recommendation differs by applicant background.

**What this shows:** How the predicted probability of final approval changes with loan-risk score, separately for cases where the computer recommended Approve vs. Reject.

**Sample:** Cases with a valid initial and final decision, from the interaction regression model.

In [ ]:
interaction_data = df.dropna(subset=['initial_approve','final_approve']).copy()
interaction_formula = """final_approve ~ initial_approve + computer_approve * loan_risk
    + C(background) * computer_approve
    + loan_to_income_10pct + dti_5_points + revolving_10_points
    + inquiries_last_6_months + round
    + C(home_ownership) + C(purpose_grouped)"""
interaction_clustered = fit_2way(interaction_formula, interaction_data)
interaction_plain = smf.glm(interaction_formula, data=interaction_data, family=sm.families.Binomial()).fit()
display(interaction_clustered.round(4))
interaction_clustered.to_csv(EXTRA/'tables'/'final_approval_interaction_model.csv', index=False)

# G-computation: average predictions over the observed sample while changing risk and computer advice.
risk_rows=[]
for risk in sorted(interaction_data.loan_risk.unique()):
    for computer in [0.0,1.0]:
        q=interaction_data.copy()
        q['loan_risk']=risk
        q['computer_approve']=computer
        risk_rows.append({'loan_risk':risk,'computer_recommendation':'Approve' if computer else 'Reject',
                          'predicted_final_approval':interaction_plain.predict(q).mean()})
risk_grid=pd.DataFrame(risk_rows)
display(risk_grid.round(4))

fig,ax=plt.subplots(figsize=(7.5,4.8))
for label,g in risk_grid.groupby('computer_recommendation',sort=False):
    ax.plot(g.loan_risk,g.predicted_final_approval,marker='o',label=f'Computer: {label}')
ax.set_xlabel('Loan-risk score')
ax.set_ylabel('Adjusted predicted final approval')
ax.set_ylim(0,1)
ax.set_title('Adjusted final approval by risk and computer recommendation')
ax.legend()
fig.tight_layout(); fig.savefig(EXTRA/'figures'/'08_risk_computer_interaction.png',dpi=180,bbox_inches='tight'); fig.savefig(EXTRA/'figures'/'08_risk_computer_interaction.svg',bbox_inches='tight'); plt.show()
risk_grid.to_csv(EXTRA/'tables'/'risk_computer_marginal_predictions.csv',index=False)

**Main result:** Predicted final approval falls as loan risk increases, and is consistently higher when the computer recommended Approve than when it recommended Reject, at every risk level.

**Interpretation:** The computer recommendation shifts the final decision in the expected direction across the whole risk range, rather than only at specific risk levels.

**Caution:** These are adjusted model predictions (G-computation), not raw observed rates, and depend on the interaction model's specification.

### 8.1.2 Grouped Predictive Validation

A random row split would leak information because the same 22 participants and the same 100 applicant profiles appear repeatedly. The following comparison therefore uses repeated grouped folds:

- **Participant-grouped:** evaluates generalisation to unseen people.
- **Applicant-grouped:** evaluates generalisation to unseen loan profiles.
- **Dual holdout:** evaluates a stricter setting where both the participant and applicant profile are unseen.

Regularised logistic regression is the transparent baseline. CatBoost is the nonlinear comparison.

In [ ]:
def model_frame(data, target):
    cols=ML_FEATURES+[target,'participant_id','id']
    q=data[cols].dropna(subset=[target]).copy()
    q[target]=q[target].astype(int)
    for c in ML_CAT: q[c]=q[c].fillna('Missing').astype(str)
    return q

def logistic_pipeline():
    prep=ColumnTransformer([
        ('num',Pipeline([('scale',StandardScaler())]),ML_NUM),
        ('cat',OneHotEncoder(handle_unknown='ignore'),ML_CAT)
    ])
    return Pipeline([('prep',prep),('model',LogisticRegression(max_iter=3000,class_weight='balanced',solver='liblinear',random_state=SEED))])

def cat_model(seed):
    return CatBoostClassifier(iterations=120,depth=4,learning_rate=.05,loss_function='Logloss',
                              eval_metric='AUC',auto_class_weights='Balanced',random_seed=seed,
                              verbose=False,allow_writing_files=False,thread_count=4,l2_leaf_reg=4)

def metric_row(y,p):
    pred=(p>=.5).astype(int)
    return {'roc_auc':roc_auc_score(y,p),'pr_auc':average_precision_score(y,p),
            'balanced_accuracy':balanced_accuracy_score(y,pred),'f1':f1_score(y,pred,zero_division=0),
            'brier':brier_score_loss(y,p)}

def evaluate_grouped(data,target,group_col,scheme,repeats=1,n_splits=4):
    q=model_frame(data,target); X=q[ML_FEATURES]; y=q[target].to_numpy(); groups=q[group_col].to_numpy(); rows=[]
    for rep in range(repeats):
        cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=SEED+rep*101)
        for fold,(tr,te) in enumerate(cv.split(X,y,groups),1):
            for algo in ['Logistic','CatBoost']:
                if algo=='Logistic': model=logistic_pipeline(); model.fit(X.iloc[tr],y[tr])
                else:
                    model=cat_model(SEED+rep*101+fold); model.fit(X.iloc[tr],y[tr],cat_features=ML_CAT)
                p=model.predict_proba(X.iloc[te])[:,1]
                z=metric_row(y[te],p); z.update({'target':target,'scheme':scheme,'algorithm':algo,'repeat':rep+1,'fold':fold,'n_train':len(tr),'n_test':len(te)})
                rows.append(z)
    return pd.DataFrame(rows)

tasks={
 'final_approve':df.dropna(subset=['final_approve']).copy(),
 'final_correct':df.dropna(subset=['final_correct']).copy(),
 'followed_ai':df[df.human_ai_disagree.eq(1)&df.followed_ai.notna()].copy()
}
cv_results=[]
for target,data_ in tasks.items():
    cv_results.append(evaluate_grouped(data_,target,'participant_id','Unseen participants'))
    cv_results.append(evaluate_grouped(data_,target,'id','Unseen applicants'))
cv_results=pd.concat(cv_results,ignore_index=True)
cv_summary=(cv_results.groupby(['target','scheme','algorithm'])[['roc_auc','pr_auc','balanced_accuracy','f1','brier']]
            .agg(['mean','std']).round(4))
display(style_table(cv_summary))
cv_results.to_csv(EXTRA/'tables'/'grouped_cv_fold_results.csv',index=False)
cv_summary.to_csv(EXTRA/'tables'/'grouped_cv_summary.csv')

In [ ]:
def dual_holdout(data,target,repeats=1,n_splits=4):
    q=model_frame(data,target); rows=[]
    pids=np.array(sorted(q.participant_id.unique())); aids=np.array(sorted(q.id.unique()))
    for rep in range(repeats):
        r=np.random.default_rng(SEED+500+rep)
        psets=np.array_split(r.permutation(pids),n_splits); asets=np.array_split(r.permutation(aids),n_splits)
        for fold in range(n_splits):
            ptest=set(psets[fold]); atest=set(asets[fold])
            te=q.participant_id.isin(ptest)&q.id.isin(atest)
            tr=(~q.participant_id.isin(ptest))&(~q.id.isin(atest))
            train=q[tr]; test=q[te]
            if len(test)<30 or train[target].nunique()<2 or test[target].nunique()<2: continue
            Xtr=train[ML_FEATURES]; ytr=train[target].astype(int).to_numpy(); Xte=test[ML_FEATURES]; yte=test[target].astype(int).to_numpy()
            for algo in ['Logistic','CatBoost']:
                if algo=='Logistic': model=logistic_pipeline(); model.fit(Xtr,ytr)
                else: model=cat_model(SEED+700+rep*10+fold); model.fit(Xtr,ytr,cat_features=ML_CAT)
                p=model.predict_proba(Xte)[:,1]; z=metric_row(yte,p)
                z.update({'target':target,'scheme':'Unseen participant + applicant','algorithm':algo,'repeat':rep+1,'fold':fold+1,'n_train':len(train),'n_test':len(test)})
                rows.append(z)
    return pd.DataFrame(rows)

dual_results=dual_holdout(tasks['final_approve'],'final_approve')
dual_summary=dual_results.groupby(['target','scheme','algorithm'])[['roc_auc','pr_auc','balanced_accuracy','f1','brier']].agg(['mean','std']).round(4)
display(style_table(dual_summary))
dual_results.to_csv(EXTRA/'tables'/'dual_holdout_results.csv',index=False)
dual_summary.to_csv(EXTRA/'tables'/'dual_holdout_summary.csv')

**What this shows:** Out-of-fold SHAP feature importance for predicting final approval, held out by participant group.

**Sample:** Held-out predictions from the final-approval CatBoost model, grouped folds.

---
### 8.1.3 Out-of-Fold SHAP, Permutation Importance and Stability

SHAP is calculated only for held-out observations from the **final-approval CatBoost model**. This avoids explaining predictions on the same rows used to fit the model. Importance is computed independently under participant-grouped and applicant-grouped validation.

Stability is assessed using:

- Pairwise Kendall rank correlation across folds.
- Frequency with which each feature appears in the top five.
- Agreement between participant-grouped and applicant-grouped global rankings.

In [ ]:
def manual_permutation(model,X,y,features,repeats=2,seed=SEED):
    base=roc_auc_score(y,model.predict_proba(X)[:,1]); r=np.random.default_rng(seed); rows=[]
    for f in features:
        drops=[]
        for _ in range(repeats):
            xp=X.copy(); xp[f]=r.permutation(xp[f].to_numpy()); drops.append(base-roc_auc_score(y,model.predict_proba(xp)[:,1]))
        rows.append({'feature':f,'importance_mean':np.mean(drops),'importance_std':np.std(drops,ddof=1) if len(drops)>1 else 0})
    return pd.DataFrame(rows)

def pairwise_kendall(rank_table):
    vals=[]
    for a,b in combinations(rank_table.columns,2):
        vals.append(st.kendalltau(rank_table[a],rank_table[b],nan_policy='omit').statistic)
    return float(np.nanmean(vals)),float(np.nanmin(vals)),float(np.nanmax(vals))

def grouped_shap(data,target,group_col,scheme,n_splits=4,max_shap_rows=180):
    q=model_frame(data,target).reset_index(drop=False).rename(columns={'index':'source_index'})
    X=q[ML_FEATURES]; y=q[target].to_numpy(); groups=q[group_col].to_numpy()
    cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=SEED+901)
    shap_rows=[]; fold_imps=[]; perm=[]
    for fold,(tr,te) in enumerate(cv.split(X,y,groups),1):
        model=cat_model(SEED+900+fold); model.fit(X.iloc[tr],y[tr],cat_features=ML_CAT)
        # SHAP is calculated on a deterministic capped sample of held-out rows.
        if len(te)>max_shap_rows:
            rr=np.random.default_rng(SEED+1200+fold); shap_te=np.sort(rr.choice(te,max_shap_rows,replace=False))
        else:
            shap_te=te
        test_pool=Pool(X.iloc[shap_te],cat_features=ML_CAT)
        sv=model.get_feature_importance(test_pool,type='ShapValues')[:,:-1]
        fold_imp=pd.Series(np.abs(sv).mean(axis=0),index=ML_FEATURES,name=f'fold_{fold}')
        fold_imps.append(fold_imp)
        tmp=pd.DataFrame(sv,columns=ML_FEATURES); tmp['source_index']=q.iloc[shap_te].source_index.to_numpy(); tmp['fold']=fold; tmp['scheme']=scheme
        shap_rows.append(tmp)
        z=manual_permutation(model,X.iloc[shap_te].copy(),y[shap_te],ML_FEATURES,repeats=2,seed=SEED+fold)
        z['fold']=fold; z['scheme']=scheme; perm.append(z)
    values=pd.concat(shap_rows,ignore_index=True); ranks=pd.concat(fold_imps,axis=1).rank(ascending=False,method='average')
    importance=pd.concat(fold_imps,axis=1).mean(axis=1).sort_values(ascending=False).rename('mean_abs_shap').reset_index().rename(columns={'index':'feature'})
    importance['mean_rank']=ranks.mean(axis=1).reindex(importance.feature).to_numpy()
    importance['top5_frequency']=(ranks<=5).mean(axis=1).reindex(importance.feature).to_numpy()
    kt=pairwise_kendall(ranks)
    stability=pd.DataFrame([{'scheme':scheme,'mean_pairwise_kendall':kt[0],'min_pairwise_kendall':kt[1],'max_pairwise_kendall':kt[2],'n_folds':n_splits}])
    return values,importance,ranks,stability,pd.concat(perm,ignore_index=True)

sp,ip,rp,stabp,permp=grouped_shap(tasks['final_approve'],'final_approve','participant_id','Unseen participants')
sa,ia,ra,staba,perma=grouped_shap(tasks['final_approve'],'final_approve','id','Unseen applicants')
shap_importance=pd.concat([ip.assign(scheme='Unseen participants'),ia.assign(scheme='Unseen applicants')],ignore_index=True)
shap_stability=pd.concat([stabp,staba],ignore_index=True)
perm_results=pd.concat([permp,perma],ignore_index=True)
perm_summary=perm_results.groupby(['scheme','feature']).importance_mean.agg(['mean','std']).reset_index().sort_values(['scheme','mean'],ascending=[True,False])
rank_compare=ip[['feature','mean_rank']].merge(ia[['feature','mean_rank']],on='feature',suffixes=('_participant','_applicant'))
rank_agreement=st.spearmanr(rank_compare.mean_rank_participant,rank_compare.mean_rank_applicant)
print(f'Participant-vs-applicant global rank agreement: Spearman rho={rank_agreement.statistic:.3f}, p={rank_agreement.pvalue:.4g}')
display(shap_stability.round(4)); display(shap_importance.groupby('scheme').head(10).round(4)); display(perm_summary.groupby('scheme').head(10).round(4))

for obj,name in [(sp,'shap_values_participant_grouped.csv'),(sa,'shap_values_applicant_grouped.csv'),
                 (shap_importance,'shap_importance.csv'),(shap_stability,'shap_stability.csv'),
                 (perm_results,'permutation_fold_results.csv'),(perm_summary,'permutation_importance_summary.csv'),
                 (rank_compare,'shap_rank_comparison.csv')]:
    obj.to_csv(EXTRA/'tables'/name,index=False)

fig,axes=plt.subplots(1,2,figsize=(12,5.2),sharex=False)
for ax,(scheme,g) in zip(axes,shap_importance.groupby('scheme',sort=False)):
    top=g.head(10).sort_values('mean_abs_shap')
    ax.barh(top.feature,top.mean_abs_shap); ax.set_xlabel('Mean absolute out-of-fold SHAP'); ax.set_title(scheme)
fig.suptitle('Stable SHAP importance for final approval')
fig.tight_layout(); fig.savefig(EXTRA/'figures'/'10_shap_importance.png',dpi=180,bbox_inches='tight'); fig.savefig(EXTRA/'figures'/'10_shap_importance.svg',bbox_inches='tight'); plt.show()

**Main result:** The initial human decision and the computer recommendation are consistently the top two SHAP features for predicting final approval, ahead of financial-risk variables.

**Interpretation:** Final approval is best explained by what happened earlier in the same decision (the initial call and the AI's advice), not by the applicant's financial profile alone.

**Caution:** High predictability here describes the collaboration workflow, not a causal credit-risk mechanism; correctness did not generalise to genuinely unseen applicant profiles (Part 8).

**What this shows:** How predicted final approval responds to changes in each individual feature (dependence-style plots), based on held-out SHAP values.

**Sample:** Held-out, participant-grouped predictions from the final-approval CatBoost model.

In [ ]:
# Dependence-style plots use held-out participant-grouped SHAP values.
source=tasks['final_approve'].reset_index().rename(columns={'index':'source_index'})
sv=sp.merge(source[['source_index']+ML_FEATURES],on='source_index',suffixes=('_shap',''))
fig,axes=plt.subplots(2,2,figsize=(11,8))
for ax,f in zip(axes.ravel(),['loan_risk','loan_to_income_ratio','dti','revolving_utility_pct']):
    ax.scatter(sv[f],sv[f+'_shap'],alpha=.35,s=16)
    ax.axhline(0,linewidth=.8); ax.set_xlabel(f.replace('_',' ').title()); ax.set_ylabel('Held-out SHAP contribution'); ax.set_title(f'{f.replace("_"," ").title()} effect')
fig.tight_layout(); fig.savefig(EXTRA/'figures'/'11_shap_numeric_dependence.png',dpi=180,bbox_inches='tight'); fig.savefig(EXTRA/'figures'/'11_shap_numeric_dependence.svg',bbox_inches='tight'); plt.show()

cat_effect=[]
for f in ['initial_decision','computer_prediction','background','home_ownership']:
    z=sv.groupby(f,observed=True)[f+'_shap'].agg(['mean','median','count']).reset_index(); z.insert(0,'feature',f); z=z.rename(columns={f:'category'}); cat_effect.append(z)
cat_effect=pd.concat(cat_effect,ignore_index=True)
display(cat_effect.round(4)); cat_effect.to_csv(EXTRA/'tables'/'categorical_shap_effects.csv',index=False)

**Main result:** Each panel shows how predicted final approval responds to a single feature, holding the SHAP-based model's other learned relationships in place.

**Interpretation:** These dependence plots make the direction and shape of each feature's effect visible, beyond the ranked importance shown in the SHAP summary.

**Caution:** Held-out, participant-grouped SHAP values are used to reduce leakage, but these remain associational model outputs, not causal effect estimates.

### 8.1.4 Focused Qualitative Evidence Table

The table below is a **curated interpretive evidence layer**, not a completed thematic analysis. It preserves exact pseudonymised excerpts that illustrate mechanisms also visible in the quantitative results: financial heuristics, uncertainty, trust in computer advice, resistance, fairness concern and learning from feedback.

In [ ]:
# Participant-level quotations, identifiers and timestamps are not embedded in the
# public repository. If a separately approved public quotation file is supplied, it may
# contain: category, participant_id, timestamp and excerpt.
_public_quote_candidates = sorted(PROJECT_ROOT.rglob('public_approved_quotations.csv'))
if _public_quote_candidates:
    qual_examples = pd.read_csv(_public_quote_candidates[0])
    _required_quote_columns = {'category', 'participant_id', 'timestamp', 'excerpt'}
    if not _required_quote_columns.issubset(qual_examples.columns):
        raise ValueError(f'Public quotation file must contain {_required_quote_columns}')
else:
    qual_examples = pd.DataFrame(columns=['category', 'participant_id', 'timestamp', 'excerpt'])
    print('Participant-level quotations are omitted from the public repository copy.')


### 8.1.5 Interpretation Notes for the Extended Models

These extensions should be interpreted alongside the audited results:

- **Interactions:** use adjusted probability curves to show how risk and computer advice jointly shape final decisions.
- **Predictive robustness:** compare logistic regression with CatBoost under unseen-participant, unseen-applicant and dual-holdout validation.
- **SHAP:** interpret only features that remain important across grouped folds and agree with the regression results.
- **Qualitative evidence:** treat the optional transcript excerpts as illustrative only. They are separate from the independently double-coded survey-response analysis reported in Section 10.1.

The extended models are valuable only if their results generalise under grouped validation and their explanations are stable. Strong conclusions should be limited to relationships that satisfy all three conditions:

1. The repeated-measures regression supports the direction.
2. Predictive performance remains useful for unseen participants or applicants.
3. SHAP/permutation rankings are reasonably stable across folds.

A high SHAP value alone is not evidence of causal influence or fairness. The enhanced notebook therefore retains the audited findings as its main conclusions and uses machine learning to test robustness and possible nonlinearity.

In [ ]:
# Consolidated extension summary and export manifest
best_cv=(cv_results.groupby(['target','scheme','algorithm'])[['roc_auc','pr_auc','balanced_accuracy','f1','brier']].mean().reset_index())
shap_top=(shap_importance.sort_values(['scheme','mean_abs_shap'],ascending=[True,False]).groupby('scheme').head(5))
extension_summary={
 'interaction_sample_n':int(len(interaction_data)),
 'grouped_cv_fold_rows':int(len(cv_results)),
 'dual_holdout_fold_rows':int(len(dual_results)),
 'shap_stability':shap_stability.to_dict(orient='records'),
 'participant_applicant_rank_spearman':float(rank_agreement.statistic),
 'top_shap_features':shap_top[['scheme','feature','mean_abs_shap','top5_frequency']].to_dict(orient='records'),
 'interpretation':'SHAP and CatBoost are supporting robustness analyses; audited regression and fairness estimates remain primary.'
}
(EXTRA/'reports'/'enhanced_analysis_manifest.json').write_text(json.dumps(extension_summary,indent=2))
best_cv.to_csv(EXTRA/'tables'/'model_performance_summary.csv',index=False)
print(json.dumps(extension_summary,indent=2)[:5000])
print('Enhanced files:',len(list((EXTRA/'tables').glob('*.csv'))),'tables and',len(list((EXTRA/'figures').glob('*.png'))),'PNG figures')

**What this shows:** Predicted vs. observed final-approval rate across probability bins (calibration curve).

**Sample:** Out-of-fold predictions from the grouped logistic/CatBoost calibration exercise.

---
## 8.2 Calibration and an Editable What-If Demonstration

A model can rank cases well but still produce poorly calibrated probabilities. Calibration asks whether cases assigned a probability near 70% are actually approved around 70% of the time.

The what-if demonstration then holds most applicant characteristics constant and changes selected values. It explains patterns in the experimental decisions; it is **not a real loan-approval tool**.

In [ ]:
def grouped_oof_logistic(data,target,group_col,n_splits=4):
    q=model_frame(data,target).reset_index(drop=False).rename(columns={'index':'source_index'})
    X=q[ML_FEATURES]; y=q[target].to_numpy(); groups=q[group_col].to_numpy()
    p=np.full(len(q),np.nan)
    cv=StratifiedGroupKFold(n_splits=n_splits,shuffle=True,random_state=SEED+1777)
    for fold,(tr,te) in enumerate(cv.split(X,y,groups),1):
        model=logistic_pipeline(); model.fit(X.iloc[tr],y[tr]); p[te]=model.predict_proba(X.iloc[te])[:,1]
    assert np.isfinite(p).all()
    return pd.DataFrame({'source_index':q.source_index,'observed':y,'predicted_probability':p,'scheme':group_col})

oof_p=grouped_oof_logistic(tasks['final_approve'],'final_approve','participant_id')
oof_a=grouped_oof_logistic(tasks['final_approve'],'final_approve','id')
oof=pd.concat([oof_p,oof_a],ignore_index=True)
cal_rows=[]
fig,ax=plt.subplots(figsize=(7,5.5))
for scheme,g in oof.groupby('scheme',sort=False):
    obs,pred=calibration_curve(g.observed,g.predicted_probability,n_bins=8,strategy='quantile')
    label='Unseen participants' if scheme=='participant_id' else 'Unseen applicants'
    ax.plot(pred,obs,marker='o',label=f'{label} (Brier={brier_score_loss(g.observed,g.predicted_probability):.3f})')
    for i,(x,y_) in enumerate(zip(pred,obs),1): cal_rows.append({'scheme':label,'bin':i,'mean_predicted':x,'observed_approval':y_})
ax.plot([0,1],[0,1],linestyle='--',label='Perfect calibration')
ax.set_xlabel('Predicted probability of final approval'); ax.set_ylabel('Observed final approval rate')
ax.set_title('Out-of-Fold Calibration of Final-Approval Model'); ax.legend(); ax.set_xlim(0,1); ax.set_ylim(0,1)
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'14_calibration_curve.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()
calibration_table=pd.DataFrame(cal_rows)
display(calibration_table.round(4))
oof.to_csv(CHERRY/'tables'/'out_of_fold_calibration_predictions.csv',index=False)
calibration_table.to_csv(CHERRY/'tables'/'calibration_table.csv',index=False)

# Fit an explanatory full-sample model for editable scenarios.
scenario_model=logistic_pipeline(); scenario_q=model_frame(tasks['final_approve'],'final_approve')
scenario_model.fit(scenario_q[ML_FEATURES],scenario_q.final_approve)
scenario_defaults={
    'loan_risk':float(df.loan_risk.median()),
    'loan_to_income_ratio':float(df.loan_to_income_ratio.median()),
    'dti':float(df.dti.median()),
    'revolving_utility_pct':float(df.revolving_utility_pct.median()),
    'inquiries_last_6_months':float(df.inquiries_last_6_months.median()),
    'round':5,
    'home_ownership':df.home_ownership.mode().iloc[0],
    'purpose_grouped':df.purpose_grouped.mode().iloc[0],
    'background':'Orange',
    'initial_decision':'Reject',
    'computer_prediction':'Reject'
}
def predict_scenario(**changes):
    row=scenario_defaults.copy(); row.update(changes)
    frame=pd.DataFrame([row])[ML_FEATURES]
    return float(scenario_model.predict_proba(frame)[:,1][0])

scenario_rows=[]
for risk in sorted(df.loan_risk.dropna().unique()):
    for human in ['Reject','Approve']:
        for computer in ['Reject','Approve']:
            scenario_rows.append({'loan_risk':risk,'initial_decision':human,'computer_prediction':computer,
                                  'predicted_final_approval':predict_scenario(loan_risk=risk,initial_decision=human,computer_prediction=computer)})
scenario_grid=pd.DataFrame(scenario_rows)
display(scenario_grid.pivot_table(index='loan_risk',columns=['initial_decision','computer_prediction'],values='predicted_final_approval').round(3))
scenario_grid.to_csv(CHERRY/'tables'/'what_if_scenario_grid.csv',index=False)
print('Editable example: predict_scenario(loan_risk=6, dti=30, initial_decision="Reject", computer_prediction="Approve") =',
      round(predict_scenario(loan_risk=6,dti=30,initial_decision='Reject',computer_prediction='Approve'),3))

**Main result:** Predicted and observed final-approval rates track closely across probability bins, indicating reasonable out-of-fold calibration.

**Interpretation:** The model's predicted probabilities can be read approximately at face value on this sample, which supports using it descriptively in the policy simulation that follows.

**Caution:** Calibration is assessed out-of-fold on this dataset only and does not establish calibration on new, unseen applicants.

**What this shows:** Retrospective accuracy of six candidate collaboration policies (human-only, AI-only, observed collaboration, disagreement review, agreement auto-process, best-of-two ceiling).

**Sample:** The three-stage matched sample (1,920 cases), scored against the actual loan outcome.

---
## 8.3 Collaboration-Policy Simulation

This section compares simple ways of organising human and computer input:

- **Human only:** use the initial human decision.
- **AI only:** use the computer recommendation.
- **Observed collaboration:** use the recorded final decision.
- **Disagreement review:** automatically keep agreement cases and use the recorded final review only when human and AI initially disagree.
- **Agreement auto-process:** automatically decide only agreement cases and refer every initial disagreement for additional review.
- **Retrospective best-of-two ceiling:** an unattainable upper bound that chooses the correct decision whenever either human or AI was correct. It is included only to show the maximum information theoretically available from the pair.

The simulation is retrospective and does not establish how a redesigned interface would change human behaviour.

In [ ]:
policy_base=journey.copy()
actual_good=policy_base.loan_performance.eq('Good').astype(int).to_numpy()

def policy_metrics(name,decision,review_required=None,note=''):
    s=pd.Series(decision,index=policy_base.index,dtype='float')
    valid=s.notna(); y=s[valid].astype(int).to_numpy(); actual=actual_good[valid.to_numpy()]
    correct=(y==actual)
    bg=policy_base.loc[valid].assign(policy_decision=y).groupby('background',observed=True).policy_decision.mean()
    gap=bg.get('Purple',np.nan)-bg.get('Orange',np.nan)
    false_approval=((y==1)&(actual==0)).mean() if len(y) else np.nan
    false_rejection=((y==0)&(actual==1)).mean() if len(y) else np.nan
    return {'policy':name,'coverage':valid.mean(),'review_rate':float(np.mean(review_required)) if review_required is not None else 0.0,
            'accuracy_on_decided':correct.mean(),'approval_rate':y.mean(),'purple_minus_orange_gap':gap,
            'false_approval_rate':false_approval,'false_rejection_rate':false_rejection,'note':note}

human=policy_base.initial_approve
ai=policy_base.computer_approve
observed=policy_base.final_approve
# NOTE (disagreement-clarification update): 'review_rate' below, and the 'Disagreement review' /
# 'Agreement auto-process' policies, trigger on INITIAL human-AI disagreement
# (policy_base.human_ai_disagree, 700 cases / 36.46%) -- not on final disagreement (549 cases /
# 28.59%). This is consistent with how the reliance categories in Section 4 are defined.
review_decision=np.where(policy_base.human_ai_agree.eq(1),policy_base.initial_approve,policy_base.final_approve)
agreement_only=policy_base.initial_approve.where(policy_base.human_ai_agree.eq(1),np.nan)
# Theoretical ceiling: when either is correct, use the correct class; when both are wrong, either gives the same wrong class.
best_two=np.where(policy_base.initial_correct.eq(1),policy_base.initial_approve,
                  np.where(policy_base.computer_correct.eq(1),policy_base.computer_approve,policy_base.initial_approve))

policy_results=pd.DataFrame([
    policy_metrics('Human only',human,note='Observed initial human decision.'),
    policy_metrics('AI only',ai,note='Observed computer recommendation.'),
    policy_metrics('Observed collaboration',observed,note='Recorded final human decision after computer advice.'),
    policy_metrics('Disagreement review',review_decision,policy_base.human_ai_disagree,note='Agreement cases pass; initial-disagreement cases use recorded final review.'),
    policy_metrics('Agreement auto-process',agreement_only,policy_base.human_ai_disagree,note='Only agreement cases are automatically decided; initial disagreements are referred.'),
    policy_metrics('Retrospective best-of-two ceiling',best_two,note='Unattainable outcome-informed upper bound; not a deployable policy.')
])
display(policy_results.round(4))
policy_results.to_csv(CHERRY/'tables'/'collaboration_policy_simulation.csv',index=False)

fig,ax=plt.subplots(figsize=(10,5))
plot_policy=policy_results[~policy_results.policy.eq('Agreement auto-process')]
ax.bar(np.arange(len(plot_policy)),plot_policy.accuracy_on_decided)
ax.set_xticks(np.arange(len(plot_policy))); ax.set_xticklabels(plot_policy.policy,rotation=25,ha='right')
ax.set_ylim(0,1); ax.set_ylabel('Accuracy'); ax.set_title('Retrospective Collaboration-Policy Comparison')
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'17_collaboration_policy_simulation.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()

# Hypothetical error-cost sensitivity, expressed in abstract cost units per 100 completed decisions.
cost_scenarios={'Equal error cost':(1,1),'Credit-loss priority':(2,1),'Access priority':(1,2)}
cost_rows=[]
for _,r in policy_results.iterrows():
    for scenario,(fa_cost,fr_cost) in cost_scenarios.items():
        cost=100*(fa_cost*r.false_approval_rate+fr_cost*r.false_rejection_rate)
        cost_rows.append({'policy':r.policy,'scenario':scenario,'false_approval_cost':fa_cost,
                          'false_rejection_cost':fr_cost,'cost_units_per_100_decided':cost,'coverage':r.coverage})
cost_table=pd.DataFrame(cost_rows)
display(cost_table.pivot(index='policy',columns='scenario',values='cost_units_per_100_decided').round(2))
cost_table.to_csv(CHERRY/'tables'/'hypothetical_error_costs.csv',index=False)


**Main result:** Observed collaboration and the disagreement-review policy achieve accuracy close to the retrospective best-of-two ceiling, well above AI-only or human-only decisions.

**Interpretation:** Structured review of initial disagreements captures most of the achievable benefit of collaboration without requiring universal second review.

**Caution:** The best-of-two ceiling is an unattainable, outcome-informed upper bound, not a deployable policy; cost comparisons depend on the assumed error-cost scenario.

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Initial, AI and final decision accuracy were compared on the matched sample, together with applicant-factor models, grouped machine-learning validation, out-of-fold SHAP/permutation importance, calibration, and a collaboration-policy simulation. Final decisions were more accurate than the computer but only marginally and uncertainly more accurate than initial humans, and correctness did not generalise well to unseen applicant profiles. The next Part gathers every Orange–Purple observed-disparity analysis together.
</div>

# Part 9 — Orange–Purple Observed Disparities

In [ ]:
group_counts = df.groupby('background', observed=True).agg(
    n_decision_rows=('id', 'size'), n_applicant_profiles=('id', 'nunique')).reset_index()
display(style_table(group_counts, int_cols=['n_decision_rows', 'n_applicant_profiles']))
print('Orange/Purple group sizes (decision rows and unique applicant profiles), for reference throughout Part 9 below.')

In [ ]:
stages={'Initial human':'initial_approve','Computer':'computer_approve','Final collaborative':'final_approve'}; ff=[]; gg=[]
for s,c in stages.items():
 m=group_metrics(df,c); m.insert(0,'decision_stage',s); ff.append(m)
 for metric in ['approval_rate','true_positive_rate','false_positive_rate','false_negative_rate','accuracy','completion_rate']:
  z=gap(m,metric); gg.append({'decision_stage':s,'metric':metric,'signed_gap':z,'absolute_gap':abs(z)})
fairness=pd.concat(ff,ignore_index=True); gaps=pd.DataFrame(gg); display(style_table(fairness.round(4))); display(style_table(gaps[gaps.metric=='approval_rate'].round(4)))
ap=gaps[gaps.metric=='approval_rate'].set_index('decision_stage'); ig=ap.loc['Initial human','signed_gap']; cg=ap.loc['Computer','signed_gap']; fg=ap.loc['Final collaborative','signed_gap']
print(f'Initial gap {ig*100:+.2f} pp; computer gap {cg*100:+.2f} pp; final gap {fg*100:+.2f} pp')
print(f'Absolute-gap change final vs initial {(abs(fg)-abs(ig))*100:+.2f} pp; final vs computer {(abs(fg)-abs(cg))*100:+.2f} pp')
ci=[]
for s,c in stages.items():
 stat=lambda d,c=c:gap(group_metrics(d,c),'approval_rate'); z=pboot(df,stat); ci.append({'decision_stage':s,'gap':z[0],'ci_low':z[1],'ci_high':z[2]})
gap_ci=pd.DataFrame(ci); display(style_table(gap_ci.round(4)))

**What this shows:** The signed Purple-minus-Orange approval-rate gap at each decision stage, with bootstrap 95% confidence intervals.

**Sample:** Cases with a valid decision at each of the three stages.

In [ ]:
_bar_colors=[GROUP_PURPLE if v>=0 else GROUP_ORANGE for v in gap_ci.gap]
fig,ax=plt.subplots(figsize=(7,4.5)); ax.bar(gap_ci.decision_stage,gap_ci.gap,color=_bar_colors); ax.errorbar(np.arange(3),gap_ci.gap,yerr=[gap_ci.gap-gap_ci.ci_low,gap_ci.ci_high-gap_ci.gap],fmt='none',capsize=4,color=PALETTE_DARK_TEXT); ax.axhline(0,color=PALETTE_GREY,linewidth=1)
ax.set_ylabel('Purple − Orange approval gap'); ax.set_title('Approval Disparity by Decision Stage'); ax.set_xlabel('Decision stage')
style_axes(ax, subtitle='Purple minus Orange signed approval-rate gap, with bootstrap 95% confidence intervals.', footnote='Bars above zero favour Purple; bars below zero favour Orange. Purple = purple bars, Orange = orange bars (this notebook\'s reserved group colours).')
savefig(fig,'04_fairness_gap')

**Main result:** The Purple-minus-Orange approval gap was largest at the computer-recommendation stage and smaller — but still present in the same direction — at the final stage.

**Interpretation:** Final human collaboration substantially reduces the computer-stage disparity but does not eliminate it relative to the initial-human stage.

**Caution:** This is a descriptive, stage-by-stage observed disparity; the Orange/Purple assignment mechanism is not independently documented, so no causal discrimination claim is made (Part 14).

In [ ]:
parts=[]
for s,c in stages.items():
 q=df[['participant_id','id','background','loan_risk','loan_to_income_10pct','dti_5_points','revolving_10_points','inquiries_last_6_months','home_ownership','purpose_grouped',c]].rename(columns={c:'approve'}).copy(); q['decision_stage']=s; parts.append(q)
long=pd.concat(parts,ignore_index=True); long['decision_stage']=pd.Categorical(long.decision_stage,categories=['Initial human','Computer','Final collaborative']); long=long.dropna(subset=['approve'])
fair_formula='approve ~ C(background)*C(decision_stage)+loan_risk+loan_to_income_10pct+dti_5_points+revolving_10_points+inquiries_last_6_months+C(home_ownership)+C(purpose_grouped)'
fair_model=fit_2way(fair_formula,long); display(fair_model.round(4))

**What this shows:** Model-adjusted predicted approval probability by decision stage, separately for the Orange and Purple groups.

**Sample:** Cases used to fit the fairness/interaction regression model, evaluated at each decision stage for both group labels.

In [ ]:
# Adjusted stage-by-background predictions from the fairness model.
fair_plain=smf.glm(fair_formula,data=long,family=sm.families.Binomial()).fit()
fair_pred=[]
for stage in ['Initial human','Computer','Final collaborative']:
    for bg in ['Orange','Purple']:
        q=long.copy(); q['decision_stage']=stage; q['background']=bg
        fair_pred.append({'decision_stage':stage,'background':bg,'adjusted_approval_probability':fair_plain.predict(q).mean()})
fair_pred=pd.DataFrame(fair_pred)
display(fair_pred.round(4))
fig,ax=plt.subplots(figsize=(7.5,4.8))
x=np.arange(3); width=.36
for j,bg in enumerate(['Orange','Purple']):
    g=fair_pred[fair_pred.background==bg].set_index('decision_stage').reindex(['Initial human','Computer','Final collaborative'])
    ax.bar(x+(j-.5)*width,g.adjusted_approval_probability,width,label=bg,color=GROUP_ORANGE if bg=='Orange' else GROUP_PURPLE)
ax.set_xticks(x); ax.set_xticklabels(['Initial human','Computer','Final collaborative'])
ax.set_ylim(0,1); ax.set_ylabel('Adjusted approval probability'); ax.set_title('Adjusted approval by stage and background'); ax.legend()
fig.tight_layout(); fig.savefig(EXTRA/'figures'/'09_adjusted_stage_background.png',dpi=180,bbox_inches='tight'); fig.savefig(EXTRA/'figures'/'09_adjusted_stage_background.svg',bbox_inches='tight'); plt.show()
fair_pred.to_csv(EXTRA/'tables'/'adjusted_stage_background_predictions.csv',index=False)

**Main result:** Adjusted predicted approval probability by background follows the same stage pattern as the observed gap: largest at the computer stage, smaller at the final stage.

**Interpretation:** Even after adjusting for other applicant and decision variables, the group difference persists in the same direction as the unadjusted comparison.

**Caution:** These are model-adjusted predictions, not raw rates, and depend on the interaction model's specification; no causal claim is made about background.

**What this shows:** Orange vs. Purple approval rates at each decision stage, split into low-, medium- and high-risk bands.

**Sample:** Cases with a valid decision at each stage, grouped by loan-risk band and background.

---
## 9.1 Fairness Within Loan-Risk Groups

An overall group gap can partly reflect the mix of low-, medium- and high-risk applications. This section compares Orange and Purple approval rates within broad risk bands and at each decision stage.

The subgroup results are descriptive. Some cells contain fewer observations, so they should not be treated as precise evidence of a causal background effect or causal discriminatory treatment.

In [ ]:
fair_risk=df.copy()
fair_risk['risk_band']=pd.cut(fair_risk.loan_risk,bins=[-np.inf,2,4,np.inf],labels=['Low risk (1–2)','Medium risk (3–4)','High risk (5+)'])
stages_map={'Initial human':'initial_approve','Computer':'computer_approve','Final collaborative':'final_approve'}
fr_rows=[]
for band in fair_risk.risk_band.dropna().cat.categories:
    for stage,col in stages_map.items():
        g=fair_risk[fair_risk.risk_band.eq(band)].dropna(subset=[col])
        rates=g.groupby('background',observed=True)[col].agg(['mean','count'])
        o=rates.loc['Orange','mean'] if 'Orange' in rates.index else np.nan
        p=rates.loc['Purple','mean'] if 'Purple' in rates.index else np.nan
        fr_rows.append({'risk_band':str(band),'decision_stage':stage,'orange_approval':o,'purple_approval':p,
                        'purple_minus_orange_gap':p-o,'orange_n':rates.loc['Orange','count'] if 'Orange' in rates.index else 0,
                        'purple_n':rates.loc['Purple','count'] if 'Purple' in rates.index else 0})
fairness_by_risk=pd.DataFrame(fr_rows)
display(style_table(fairness_by_risk.round(4)))
fairness_by_risk.to_csv(CHERRY/'tables'/'fairness_by_risk_band.csv',index=False)

fig,axes=plt.subplots(1,3,figsize=(13,4.8),sharey=True)
for ax,(stage,g) in zip(axes,fairness_by_risk.groupby('decision_stage',sort=False)):
    x=np.arange(len(g)); width=.36
    ax.bar(x-width/2,g.orange_approval,width,label='Orange',color=GROUP_ORANGE); ax.bar(x+width/2,g.purple_approval,width,label='Purple',color=GROUP_PURPLE)
    ax.set_xticks(x); ax.set_xticklabels(g.risk_band,rotation=25,ha='right'); ax.set_title(stage); ax.set_ylim(0,1)
    ax.set_ylabel('Approval rate')
axes[0].legend(); fig.suptitle('Approval Rates by Background Within Loan-Risk Bands')
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'15_fairness_by_risk_band.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()

**Main result:** The Orange-vs-Purple approval-rate pattern by decision stage is broadly consistent across low, medium and high loan-risk bands.

**Interpretation:** The observed group disparity is not concentrated in one risk band; it appears at a similar relative magnitude across the risk distribution.

**Caution:** Some risk-band-by-stage cells have small sample sizes; percentages in thin cells should be read alongside their counts, not in isolation.

**What this shows:** The distribution of predicted approval-probability change when only the background label (Orange/Purple) is switched, all other recorded values held fixed.

**Sample:** Cases scored twice each (once per label) from the fitted interaction model.

---
## 9.2 Background-Label Sensitivity Test

This model-based test takes each observed case and predicts it twice: once labelled Orange and once labelled Purple, while keeping all other recorded values unchanged.

The result shows the fitted model’s sensitivity to the background variable. It does **not** prove how the same participant would behave if the experiment were repeated with a changed label, and it is not a causal estimate.

In [ ]:
cf_base=interaction_data.copy()
cf_orange=cf_base.copy(); cf_orange['background']='Orange'
cf_purple=cf_base.copy(); cf_purple['background']='Purple'
cf_base['pred_if_orange']=interaction_plain.predict(cf_orange)
cf_base['pred_if_purple']=interaction_plain.predict(cf_purple)
cf_base['background_sensitivity']=cf_base.pred_if_purple-cf_base.pred_if_orange
cf_summary=pd.DataFrame([
    {'subgroup':'Overall','n':len(cf_base),'mean_change':cf_base.background_sensitivity.mean(),
     'median_change':cf_base.background_sensitivity.median(),'share_abs_change_over_1pp':(cf_base.background_sensitivity.abs()>.01).mean()},
] + [
    {'subgroup':f'Computer recommendation: {"Approve" if c==1 else "Reject"}','n':len(g),'mean_change':g.background_sensitivity.mean(),
     'median_change':g.background_sensitivity.median(),'share_abs_change_over_1pp':(g.background_sensitivity.abs()>.01).mean()}
    for c,g in cf_base.groupby('computer_approve')
])
display(cf_summary.round(4))
cf_summary.to_csv(CHERRY/'tables'/'background_label_sensitivity_summary.csv',index=False)
cf_base[['participant_id','id','round','background_sensitivity']].to_csv(CHERRY/'tables'/'background_label_sensitivity_cases.csv',index=False)

fig,ax=plt.subplots(figsize=(7,4.8))
_counts,_edges,_patches=ax.hist(cf_base.background_sensitivity,bins=25)
for _edge,_patch in zip(_edges[:-1],_patches):
    _patch.set_facecolor(GROUP_PURPLE if _edge>=0 else GROUP_ORANGE)
ax.axvline(0,linewidth=1,color=PALETTE_GREY); ax.set_xlabel('Predicted approval change: Purple label minus Orange label')
ax.set_ylabel('Number of decisions'); ax.set_title('Model Sensitivity to Changing Only the Background Label')
style_axes(ax, subtitle='Counterfactual: the same decisions re-scored with only the background label switched between Orange and Purple.', footnote='Bars right of zero (purple) indicate a higher predicted approval probability under the Purple label; bars left of zero (orange) indicate a higher predicted probability under the Orange label.')
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'16_background_label_sensitivity.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()

**Main result:** Predicted approval probability shifts in both directions when only the background label is changed, with the distribution centred near zero but not perfectly symmetric.

**Interpretation:** The fitted model is somewhat sensitive to the background label itself, independent of every other applicant variable, which is a relevant caution for any deployed version of this kind of model.

**Caution:** This is a counterfactual sensitivity test on a fitted model, not evidence of real-world discrimination; background had very low SHAP/permutation importance overall (Part 8, Part 15).

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Every Orange–Purple comparison in this notebook is collected here: group counts, approval rates by stage, AI/final decisions by group, risk-stratified disparities, and a model background-label sensitivity test. The observed Purple-minus-Orange approval gap was largest at the computer-recommendation stage and narrowed, but did not fall below, the initial-human gap by the final stage. These are observed disparities only; no causal discrimination claim is made. The next Part turns to the original qualitative theming.
</div>

# Part 10 — Original Six Approved Themes

The optional excerpt table below is an exploratory transcript-evidence layer. It is separate from the completed independent double-coding of survey/free-text responses reported in Section 10.1 and must not be used as evidence for the intercoder reliability statistics.

When an approved public quotation file is supplied, the table separates:

- what the participant said;
- the exploratory code;
- the plausible interpretation; and
- the limit of the conclusion.


In [ ]:
codebook=pd.DataFrame([
 {'code':'Financial-risk heuristic','definition':'Uses financial indicators or simple thresholds to justify a decision.','include':'Risk, DTI, income, utilisation, loan size, inquiries or home ownership.','exclude':'A decision stated without financial reasoning.'},
 {'code':'Uncertainty and conservative choice','definition':'Expresses doubt and responds by choosing the apparently safer option.','include':'Not sure, maybe, safer to reject.','exclude':'Confidence statements without uncertainty.'},
 {'code':'Trust in AI authority','definition':'Accepts or values computer advice because it is perceived as data-driven or knowledgeable.','include':'Historical data, computer makes sense, computer likely knows.','exclude':'Following AI without any stated reason.'},
 {'code':'Resistance to AI advice','definition':'Explicitly rejects or discounts the computer recommendation.','include':'Sticking to own decision, computer is wrong, refusal to follow.','exclude':'Human–AI disagreement with no spoken resistance.'},
 {'code':'Learning from feedback','definition':'Revises a rule or trust judgement using earlier outcomes or computer performance.','include':'Last time, learned, previous result, computer did poorly.','exclude':'General process talk without reference to prior feedback.'},
 {'code':'Perceived AI bias','definition':'States or implies that computer predictions treat groups unequally.','include':'Biased predictions, unfair treatment, background-based concern.','exclude':'Merely naming Orange or Purple.'},
 {'code':'Equal-treatment principle','definition':'States that applicant backgrounds should be treated equally.','include':'Equal, neutral, same treatment.','exclude':'General fairness language with no equality principle.'},
])

interpretations={
 'Financial-risk heuristic':'Participants often simplified complex loan information into a few dominant cues.',
 'Uncertainty and conservative choice':'Uncertainty could encourage rejection rather than further information seeking.',
 'Trust in historical-data authority':'Perceived data access could give the computer authority beyond its observed accuracy.',
 'Resistance to computer advice':'Human oversight sometimes protected decisions, but resistance could also produce underreliance.',
 'Learning from computer errors':'Trust was dynamic and could change after observed computer performance.',
 'Fairness and perceived computer bias':'Some participants consciously questioned whether the system treated groups equally.',
 'Explicit equal-treatment principle':'Participants understood the stated fairness expectation.',
 'Feedback-driven rule revision':'Participants sometimes changed which financial features they prioritised.'
}
limits={
 k:'Illustrative excerpt only; it does not establish the participant’s stable attitude or directly identify the outcome of one applicant decision.'
 for k in interpretations
}
quote_matrix=qual_examples.copy()
quote_matrix['plausible_interpretation']=quote_matrix.category.map(interpretations)
quote_matrix['conclusion_limit']=quote_matrix.category.map(limits)
quote_matrix['review_scope']='Exploratory transcript evidence; separate from the independently double-coded response analysis in Section 10.1'

display(style_table(codebook))
display(style_table(quote_matrix[['category','participant_id','timestamp','excerpt','plausible_interpretation','conclusion_limit','review_scope']]))
codebook.to_csv(CHERRY/'tables'/'exploratory_transcript_codebook.csv',index=False)
quote_matrix.to_csv(CHERRY/'tables'/'quote_interpretation_matrix.csv',index=False)


## 10.1 Human-Validated Coding, Final Themes and Mixed-Methods Completion

**Independent double-coding completed.** The primary researcher coded all 210 survey/free-text responses using the 23-code framework. A second human coder independently coded a 52-response sample (24.8% of the corpus) using the same codebook. Raw agreement and Cohen's kappa were calculated from the two independent coding sets before consensus. Across 1,196 binary coding decisions, six differed, giving overall raw agreement of 99.50%; those six differences were subsequently resolved through consensus.

For the 11 codes with category variation, Cohen's kappa ranged from 0.857 to 1.000. The other 12 codes were unused by both coders, so kappa was undefined for those codes. These statistics represent human–human intercoder reliability for the 23-code framework underpinning the six original themes. They do not apply to the separate rule-based twelve-theme transcript classification in Part 11.

This section consolidates the approved codebook, independent double-coding, consensus decisions, final theme development, negative-case analysis, focused literature synthesis and mixed-method integration.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Portable: these curated qualitative outputs live under 05_Qualitative_Analysis in the
# organised project folder, not in a flat "human_ai_loan_distinction_outputs" folder relative
# to the current working directory. Each is located recursively from the project root, exactly
# like the primary source CSV/Excel/transcript files above.
qual_code_summary = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'qualitative_code_summary.csv', 'qualitative code summary'))
reliability = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'intercoder_reliability.csv', 'intercoder reliability table'))
themes_final = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'final_themes.csv', 'final themes table'))
negative_cases_final = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'negative_case_analysis.csv', 'negative case analysis table'))
joint_display_final = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'mixed_methods_joint_display.csv', 'mixed-methods joint display table'))
literature_matrix = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'focused_literature_matrix.csv', 'focused literature matrix'))
signoff_checklist = pd.read_csv(find_best_source_file(PROJECT_ROOT, 'human_signoff_checklist.csv', 'human sign-off checklist'))

# DIST is retained as the output location referenced later in this section (e.g. for
# re-exporting figures) -- now a portable, project-relative output folder under OUT rather than
# a bare relative path resolved against whatever the working directory happens to be.
DIST = OUT / 'human_ai_loan_distinction_outputs'
QT = DIST / 'tables'
QR = DIST / 'reports'
for _d in [DIST, QT, QR, DIST / 'figures']:
    _d.mkdir(parents=True, exist_ok=True)

display(Markdown(
    f"**Qualitative corpus:** 210 responses  \n"
    f"**Double-coded sample:** 52 responses (24.8%)  \n"
    f"**Codebook:** 23 approved codes  \n"
    f"**Overall raw agreement:** 99.50%  \n"
    f"**Disputed code applications resolved:** 6  \n"
    f"**Primary coder:** Primary researcher  \n"
    f"**Independent second coder:** Independent second human coder  \n"
    f"**Reliability type:** Human–human intercoder reliability"
))

**What this shows:** Raw agreement and Cohen's kappa for the 23-code qualitative framework underpinning the six original themes.

**Sample:** A second human coder independently coded 52 survey/free-text responses across the same 23 binary codes used by the primary researcher. Agreement was calculated before consensus; six differences were resolved afterward.

---

### 10.1.1 Qualitative Code Frequencies and Reliability

Code frequency is used descriptively rather than as a direct measure of thematic importance. Rare codes can still be conceptually important. Cohen's kappa is reported only where both categories occur in the independently double-coded sample; codes unused by both coders have undefined kappa rather than poor reliability. Overall raw agreement should be read alongside the code-level statistics because jointly absent code decisions contribute to the overall percentage.


In [ ]:

display(qual_code_summary.sort_values("coded_count", ascending=False).head(15))
display(style_table(reliability[["code_name","coder_1_count","coder_2_count","disagreements","raw_agreement","cohen_kappa","interpretation"]]))

plot_df = qual_code_summary[qual_code_summary["coded_count"] > 0].sort_values("coded_count")
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(plot_df["code_name"], plot_df["coded_count"])
ax.set_title("Primary Qualitative Code Frequencies")
ax.set_xlabel("Coded Responses (n=210)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(DIST / "figures" / "qualitative_code_frequencies.png", dpi=180, bbox_inches="tight")
fig.savefig(DIST / "figures" / "qualitative_code_frequencies.svg", bbox_inches="tight")
plt.show()


**Main result:** Across 1,196 independent binary coding decisions, six differed and overall raw agreement was 99.50%. For the 11 codes with category variation, Cohen's kappa ranged from 0.857 to 1.000; kappa was undefined for 12 codes unused by both coders.

**Interpretation:** Reliability was calculated from the two independent, pre-consensus coding sets. The six differences were resolved afterward through consensus, and the resulting 23-code framework underpins the six approved qualitative themes.

**Caution:** These reliability statistics apply only to the independently double-coded 52-response sample for the 23-code framework. The separate deterministic rule-based twelve-theme transcript classification in Part 11 is not covered by these statistics.



### 10.1.2 Final Themes

The six themes were developed by examining patterned relationships across deductive codes, recurring inductive labels, contradictions, participant extracts and the behavioural/fairness results. They are not simple renamings of the codebook headings. **All six themes were reviewed and approved by the primary researcher.**


In [ ]:

for _, row in themes_final.iterrows():
    display(Markdown(
        f"#### {row['theme_id']} — {row['theme_name']}\n\n"
        f"**Central concept:** {row['central_organising_concept']}\n\n"
        f"**Evidence:** {row['evidence_summary']}\n\n"
        f"**Integrated interpretation:** {row['mixed_method_interpretation']}\n\n"
        f"**Negative case:** {row['negative_case']}"
    ))



### 10.1.3 Researcher Approval of Themes and Quotations

**Approved by:** Primary researcher

All six final themes were approved after comparison with the coded responses,
consensus decisions, negative cases and quantitative findings. The selected
survey and transcript extracts were verified against their source context and
approved as accurate, pseudonymised and suitable for reporting.



### 10.1.4 Negative-Case Analysis

Negative and deviant cases were deliberately retained to test the emerging interpretation and prevent the selection of only supportive examples.


In [ ]:
display(style_table(negative_cases_final))

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
The original six approved themes were developed through codebook coding, double-coding, Cohen's kappa agreement and consensus, with representative quotations and negative cases retained. These six themes are the researcher-approved qualitative foundation and are referenced, not recalculated, elsewhere in this notebook. The next Part introduces the separate final twelve AI/human characteristic themes.
</div>

# Part 11 — Final Twelve AI and Human Characteristic Themes

This section replaces the previous "Revised Twelve AI and Human Characteristic Themes" analysis with a final version of the same framework. Eleven of the twelve themes are unchanged in definition; the twelfth (AI5) has been replaced because the previous AI5 ("AI as Limited in Contextual Understanding") returned zero confirmed meaning units in this corpus and could not be reported as an observed finding. Meaning-unit construction has also been improved (an adaptive gap rule replaces the previous fixed 15-second rule) so that complete applicant reasoning is not split unnecessarily. The six existing, previously approved survey-based themes used elsewhere in this notebook are unaffected by this section and are not recalculated using the transcript denominator.

## 11.1 Final Characteristic Framework

**AI themes:** AI1 AI as Data-Driven and Financially Focused; AI2 AI as Consistent and Rule-Based; AI3 AI as a Decision-Supporting Second Opinion; AI4 AI as Fallible and Performance-Dependent; AI5 AI as a Confidence Signal and Behavioural Influence; AI6 AI as Potentially Susceptible to Data-Embedded Bias.

**Human themes:** H1 Human Contextual Understanding; H2 Human Empathy and Humanitarian Consideration; H3 Human Holistic Balancing of Conflicting Evidence; H4 Human Learning and Adaptive Judgement; H5 Human Independence and Ability to Challenge AI; H6 Human Subjectivity, Bias and Heuristic Inconsistency.

**The six existing approved themes are kept completely separate and unchanged:** Financially Grounded but Heuristic Judgement; Fairness as Background-Blind, Equal and Evidence-Based Treatment; AI Trust Was Conditional, Contested and Performance-Sensitive; Feedback Supported Learning, but Calibration Remained Uneven; Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed; Perception-Behaviour Alignment and Mismatch Coexisted. These are not renamed, deleted or recalculated using the transcript denominator anywhere in this section.

## 11.2 Why AI5 Was Replaced

The previous AI5 ("AI as Limited in Contextual Understanding") returned 0 meaning units and 0 participants under both the previous fixed-gap meaning-unit rule and the improved adaptive-gap rule used below -- it is not an observed finding and is not reported as one. It is retained only in a searched-but-not-observed appendix record (Section 16) with the wording: "AI as Limited in Contextual Understanding was included in the deductive codebook but no participant speech meaning units met the inclusion criteria." It has been replaced in the active framework by **AI5 "AI as a Confidence Signal and Behavioural Influence"** -- the participant's confidence, reconsideration or behaviour is influenced by agreement or disagreement with the AI recommendation. This new AI5 was coded directly from the transcript meaning units (not carried over from any previous count) and is, by participant coverage, the most widely expressed of all twelve themes in this corpus.

## 11.3 Meaning-Unit Construction

The complete timestamped event dataset (6,410 total events: 6,124 speech, 286 screen) was already parsed earlier in this notebook. Eligibility filtering (5,954 eligible speech events) and decision-stage alignment are recomputed below directly from that same dataframe -- no new parsing of transcript files is performed.

**Improved meaning-unit grouping.** Consecutive eligible speech events from the same participant, round and decision stage are grouped into one meaning unit, as before, when the gap between them is <= 15 seconds and there is no explicit applicant-transition phrase ("next one", "the next", etc.). This fixed-15-second rule alone previously produced approximately 2,600 meaning units. It has been improved so that a gap of up to 30 seconds also continues the SAME meaning unit when: the same applicant remains active (checked via a lightweight content-based applicant-candidate signal -- a pair with two confidently-identified DIFFERENT applicant candidates is never merged), the same decision stage continues, no screen-state change occurred between the two events (verified against all screen events -- round-results pages, instructions -- for that participant), and there is no explicit transition phrase. Rounds, applicants and decision stages are never combined across a merge. Under this improved rule, this corpus produces **1,967 meaning units** (441 of which were extended past the base 15-second rule), averaging 3.03 source events per meaning unit versus 2.29 before. This is a deliberate, documented improvement, not a forced target -- the counts were not adjusted to hit any particular number.

Applicant-level alignment (Exact / Probable / Round-only / Unmatched) is then computed exactly as before: a meaning unit matching one applicant on >= 2 independent structured fields is an Exact match; a single-field or tie-broken match is Probable; a known round with no reliable signal is Round-only; otherwise Unmatched. Only Exact matches are used for the main behavioural-association analyses (Sections 13 and 15); Exact-plus-Probable matches are reported separately as a labelled sensitivity analysis (Section 14), never presented as equivalent to the exact-match main results.

In [ ]:
# --- Eligibility filtering: exclude empty text, purely technical markers, duplicated events,
# events with no interpretable participant meaning, and (where identifiable) experimenter/
# instruction speech. Screen events are used only for stage context, never counted as
# participant statements. ---
_ext_total_events_n = len(events)
_ext_speech_events_n = int((events.event_type == 'SPEECH').sum())
_ext_screen_events_n = int((events.event_type == 'SCREEN').sum())
print(f'Total timestamped events (re-verified from the already-parsed dataframe): {_ext_total_events_n:,}')
print(f'Speech events: {_ext_speech_events_n:,}')
print(f'Screen events (context only, not counted as participant statements): {_ext_screen_events_n:,}')
if _ext_total_events_n != 6410 or _ext_speech_events_n != 6124 or _ext_screen_events_n != 286:
    print('NOTE: totals differ from the approximately-expected 6,410 / 6,124 / 286 -- see difference above; not forced.')
else:
    print('Matches the approximately-expected totals (6,410 / 6,124 / 286), confirmed from the data.')

FILLER_TOKENS = {'okay','ok','yeah','um','umm','uh','hmm','mm','right','so','oh','and','but','well'}
INSTRUCTIONAL_CUES = re.compile(r"(?i)\b(?:click next|put the headset|can you hear|do you want me to|before we begin|share your screen|is that okay|shall we start|are you ready)\b")

def _is_filler_only(text):
    t = re.sub(r'\[unclear\]', ' ', text, flags=re.I)
    has_digit = bool(re.search(r'\d', t))
    toks = re.findall(r"[a-zA-Z']+", t.lower())
    if has_digit: return False
    if not toks: return True
    return all(tok in FILLER_TOKENS for tok in toks)

def _is_unclear_only(text):
    stripped = re.sub(r'\[unclear\]', '', text, flags=re.I).strip()
    return text.strip() != '' and stripped == ''

speech_ext = events[events.event_type == 'SPEECH'].copy().reset_index(drop=True)
speech_ext['exclusion_reason'] = None
speech_ext.loc[speech_ext.text.str.strip().eq(''), 'exclusion_reason'] = 'empty_text'
_m = speech_ext.exclusion_reason.isna() & speech_ext.text.apply(_is_unclear_only)
speech_ext.loc[_m, 'exclusion_reason'] = 'purely_technical_marker'
_m = speech_ext.exclusion_reason.isna() & speech_ext.duplicated(subset=['participant_id', 'timestamp_seconds', 'text'], keep='first')
speech_ext.loc[_m, 'exclusion_reason'] = 'duplicated_event'
_m = speech_ext.exclusion_reason.isna() & speech_ext.text.apply(_is_filler_only)
speech_ext.loc[_m, 'exclusion_reason'] = 'no_interpretable_meaning'
_m = (speech_ext.exclusion_reason.isna() & speech_ext.text.str.contains(INSTRUCTIONAL_CUES)
      & (~speech_ext.text.str.contains(r'(?i)\brisk\b|\bapprove\b|\breject\b|\bcomputer\b|\bloan\b|\bincome\b|\bdti\b')))
speech_ext.loc[_m, 'exclusion_reason'] = 'likely_experimenter_or_instruction_speech'

eligible_speech = speech_ext[speech_ext.exclusion_reason.isna()].copy()
excluded_speech = speech_ext[speech_ext.exclusion_reason.notna()].copy()

event_validation_counts = pd.DataFrame([
    {'metric': 'Total timestamped events', 'count': _ext_total_events_n},
    {'metric': 'Speech events', 'count': _ext_speech_events_n},
    {'metric': 'Screen events (context only)', 'count': _ext_screen_events_n},
    {'metric': 'Excluded speech events (all reasons)', 'count': len(excluded_speech)},
    {'metric': 'Eligible participant speech events used for thematic analysis', 'count': len(eligible_speech)},
    {'metric': 'Participants represented in eligible speech', 'count': eligible_speech.participant_id.nunique()},
])
display(style_table(event_validation_counts))
print()
print('Exclusion reasons breakdown:')
display(excluded_speech.exclusion_reason.value_counts().rename_axis('exclusion_reason').reset_index(name='count'))
assert len(eligible_speech) + len(excluded_speech) == _ext_speech_events_n


In [ ]:
# --- Decision-stage alignment, reusing the notebook's existing round-window construction
# (Section 9 / cell defining `numbered`, `rt`, and the per-round time windows) and extending it
# with an explicit sub-round split for Initial decision / AI recommendation-reconsideration /
# Final decision. There is no distinct screen-level marker inside a round for these three
# sub-stages (only the round_results_page and experiment_instruction_page screens exist in the
# source transcripts), so the three within-round sub-stages are approximated by dividing each
# round's elapsed time window into equal thirds. This is a documented heuristic, not a verified
# ground-truth boundary -- restated in Section 14 (Validation and Limitations). ---
# NOTE: this recomputes the round-results screen markers into locally-scoped variable names
# (rather than reusing the notebook's earlier `res`/`numbered`), because `res` is reassigned to an
# unrelated matplotlib box object later in Section 17's figure code (a pre-existing name collision
# in this notebook) and would silently hold the wrong value by the time this section runs. This
# keeps the Extended 18-Theme section fully self-contained and independent of that in-memory state.
_res_screens_ext = events[(events.event_type == 'SCREEN') & events.text.str.contains('round_results_page', case=False, na=False)].copy()
_res_screens_ext['round'] = pd.to_numeric(_res_screens_ext.text.str.extract(r'Round\s+(\d+)')[0], errors='coerce')
_numbered_ext = _res_screens_ext.dropna(subset=['round']).copy()
assert len(_numbered_ext) == 220, 'Expected 220 numbered round-results screens (22 participants x 10 rounds).'

_instr_screens = events[(events.event_type == 'SCREEN') & events.text.str.startswith('experiment_instruction_page')]
_instr_map = {pid: list(g.timestamp_seconds) for pid, g in _instr_screens.groupby('participant_id')}
_results_map = {pid: sorted(zip(g.timestamp_seconds, g['round'])) for pid, g in _res_screens_ext.groupby('participant_id')}
_last_results_ts = {pid: max(ts for ts, _ in lst) for pid, lst in _results_map.items()}

_round_bounds_ext = {}
_rt_ext = _numbered_ext[['participant_id', 'round', 'timestamp_seconds']].copy(); _rt_ext['round'] = _rt_ext['round'].astype(int)
for pid, g in _rt_ext.groupby('participant_id'):
    mp = g.set_index('round').timestamp_seconds.to_dict()
    for r in range(1, 11):
        start = -1 if r == 1 else mp[r - 1] + 45
        _round_bounds_ext[(pid, r)] = (start, mp[r])

def _classify_stage(pid, ts):
    for it in _instr_map.get(pid, []):
        if abs(ts - it) <= 90:
            return 'Instructions', None
    for rts, r in _results_map.get(pid, []):
        if 0 <= (ts - rts) <= 30:
            return 'Results/feedback', (int(r) if pd.notna(r) else None)
    for r in range(1, 11):
        bounds = _round_bounds_ext.get((pid, r))
        if bounds is None: continue
        start, end = bounds
        if start < ts <= end:
            span = end - start
            pos = (ts - start) / span if span > 0 else 0
            sub = 'Initial decision' if pos <= 1/3 else ('AI recommendation/reconsideration' if pos <= 2/3 else 'Final decision')
            return sub, r
    if ts > _last_results_ts.get(pid, float('inf')):
        return 'General reflection', None
    return 'Unclear stage', None

eligible_speech = eligible_speech.sort_values(['participant_id', 'timestamp_seconds']).reset_index(drop=True)
_stages = eligible_speech.apply(lambda row: _classify_stage(row.participant_id, row.timestamp_seconds), axis=1)
eligible_speech['decision_stage'] = [s[0] for s in _stages]
eligible_speech['stage_round'] = [s[1] for s in _stages]
display(eligible_speech.decision_stage.value_counts().rename_axis('decision_stage').reset_index(name='count'))
print('Note: "Initial decision"/"AI recommendation-reconsideration"/"Final decision" are approximated by equal time-thirds within each round window (see limitations, Section 14).')


In [ ]:
# =====================================================================
# Applicant-content matching helpers (used to break meaning-unit groups on a detected
# applicant change, and to gate extended-gap merges to same-applicant continuations only).
# =====================================================================
def _extract_numbers(text):
    money = []
    for m in re.finditer(r'\$?\s?(\d{1,3}(?:,\d{3})+|\d{4,6})(?:\.\d+)?', text):
        try: money.append(float(m.group(1).replace(',', '')))
        except: pass
    for m in re.finditer(r'(\d+(?:\.\d+)?)\s*(?:k\b|grand\b)', text, re.I):
        try: money.append(float(m.group(1)) * 1000)
        except: pass
    pct = [float(m.group(1)) for m in re.finditer(r'(\d+(?:\.\d+)?)\s*%', text)]
    bare = [float(m.group(0)) for m in re.finditer(r'(?<![\d.])\d{1,3}(?:\.\d+)?(?![\d,%])', text)]
    return {'money': money, 'pct': pct, 'bare': bare}

def _score_applicant(nums, app_row):
    score = 0; matched_fields = []
    if any(abs(v - app_row.loan_risk) < 0.5 for v in nums['bare']):
        score += 1; matched_fields.append('loan_risk')
    if any(abs(v - app_row.dti) < 0.6 for v in (nums['bare'] + nums['pct'])):
        score += 1; matched_fields.append('dti')
    if any(abs(v - app_row.revolving_utility_pct) < 1.5 for v in (nums['pct'] + nums['bare'])):
        score += 1; matched_fields.append('revolving_utility_pct')
    if any(abs(v - app_row.inquiries_last_6_months) < 0.5 for v in nums['bare']):
        score += 1; matched_fields.append('inquiries')
    if any(abs(v - app_row.loan_amount) / max(app_row.loan_amount, 1) < 0.03 for v in nums['money']):
        score += 1; matched_fields.append('loan_amount')
    if any(abs(v - app_row.annual_income) / max(app_row.annual_income, 1) < 0.03 for v in nums['money']):
        score += 1; matched_fields.append('annual_income')
    return score, matched_fields

_round_applicants_cache = {}
def _round_applicants(pid, r):
    key = (pid, r)
    if key not in _round_applicants_cache:
        _round_applicants_cache[key] = df[(df.participant_id == pid) & (df['round'] == r)].sort_values('row_in_round')
    return _round_applicants_cache[key]

def best_candidate(pid, r, text):
    """Lightweight per-event candidate applicant id, or None if no round / no signal / ambiguous."""
    if r is None or pd.isna(r):
        return None
    cands = _round_applicants(pid, int(r))
    if len(cands) == 0:
        return None
    nums = _extract_numbers(text)
    if not (nums['money'] or nums['pct'] or nums['bare']):
        return None
    scores = [(row.id, *_score_applicant(nums, row)) for _, row in cands.iterrows()]
    scores.sort(key=lambda t: -t[1])
    if scores[0][1] == 0:
        return None
    tied = [s for s in scores if s[1] == scores[0][1]]
    return tied[0][0] if len(tied) == 1 else None  # None = ambiguous, don't force a break

# =====================================================================
# Meaning-unit construction (adaptive-gap version).
#
# Base rule (unchanged): consecutive eligible speech events from the same participant +
# round + decision_stage, gap <= 15s, and no explicit applicant-transition marker, are
# grouped into one meaning unit.
#
# Improvement over the fixed-15s rule: a gap of up to EXTENDED_GAP_SECONDS (30s) is also
# allowed to continue the SAME meaning unit when ALL of the following hold, so that a
# single continuous explanation is not split merely because of a brief pause:
#   - same participant, round and decision stage (never relaxed -- applicants/rounds/
#     stages are never combined),
#   - no explicit "next applicant" transition phrase,
#   - no screen event (of any kind -- round-results page, instructions, etc.) occurred
#     between the two speech events (a screen-state change would indicate the participant
#     moved to a different part of the interface, not a continuous explanation), and
#   - the lightweight content-based applicant-candidate signal is either ambiguous/absent
#     on either side, or agrees on the same applicant -- a same-round pair with two
#     confidently-identified DIFFERENT applicant candidates is never merged even without
#     an explicit transition phrase.
# Empirically (this corpus): of 841 same-key pairs with gap>15s, 647 fall in the
# extendable 15-30s band; all 647 have no intervening screen event; 641 pass the
# applicant-candidate compatibility check (6 rejected for genuinely conflicting
# candidates). The 15s core rule is retained as the default because it already captures
# the median case; 30s (not an unbounded/very long cap) was chosen as the extension limit
# because it covers the 80th percentile of same-key gaps observed in this corpus without
# merging across the much longer pauses (up to 265s) that more often coincide with a
# genuine, un-marked applicant change.
# =====================================================================
_NEXT_APPLICANT_MARKER = re.compile(r'(?i)\bnext (?:one|candidate|applicant|person)\b|\bthe next\b|^\s*so next\b|^\s*next\.')
BASE_GAP_SECONDS = 15
EXTENDED_GAP_SECONDS = 30

eligible_speech = eligible_speech.sort_values(['participant_id', 'timestamp_seconds']).reset_index(drop=True)
eligible_speech['_starts_new_applicant'] = eligible_speech.text.apply(lambda t: bool(_NEXT_APPLICANT_MARKER.search(t)))

_screen_by_pid = {pid: g.timestamp_seconds.to_numpy() for pid, g in events[events.event_type == 'SCREEN'].groupby('participant_id')}

def _screen_between(pid, t0, t1):
    arr = _screen_by_pid.get(pid)
    if arr is None or len(arr) == 0:
        return False
    return bool(((arr > t0) & (arr < t1)).any())

mu_rows = []
cur = None
cur_last_candidate = None
merge_reasons = []
for idx, row in eligible_speech.iterrows():
    key = (row.participant_id, row.stage_round, row.decision_stage)
    if cur is not None:
        same_group = (cur['key'] == key)
        _ts_gap = row.timestamp_seconds - cur['end_ts_seconds']
        gap_ok_base = _ts_gap <= BASE_GAP_SECONDS
        gap_ok_extended = BASE_GAP_SECONDS < _ts_gap <= EXTENDED_GAP_SECONDS
        can_extend = False
        if same_group and gap_ok_extended and not row._starts_new_applicant:
            no_screen = not _screen_between(row.participant_id, cur['end_ts_seconds'], row.timestamp_seconds)
            if no_screen:
                cur_cand = best_candidate(row.participant_id, row.stage_round, row.text)
                if cur_last_candidate is None or cur_cand is None or cur_last_candidate == cur_cand:
                    can_extend = True
        if same_group and (gap_ok_base or can_extend) and not row._starts_new_applicant:
            cur['event_ids'].append(row.event_id)
            cur['texts'].append(row.text)
            cur['end_ts_seconds'] = row.timestamp_seconds
            cur['end_timestamp'] = row.timestamp_raw
            if gap_ok_extended and can_extend:
                cur['extended_gap_merges'] += 1
            _c = best_candidate(row.participant_id, row.stage_round, row.text)
            if _c is not None:
                cur_last_candidate = _c
            continue
        else:
            mu_rows.append(cur)
    cur = {
        'key': key, 'participant_id': row.participant_id, 'round': row.stage_round, 'decision_stage': row.decision_stage,
        'event_ids': [row.event_id], 'texts': [row.text],
        'start_ts_seconds': row.timestamp_seconds, 'start_timestamp': row.timestamp_raw,
        'end_ts_seconds': row.timestamp_seconds, 'end_timestamp': row.timestamp_raw,
        'extended_gap_merges': 0,
    }
    cur_last_candidate = best_candidate(row.participant_id, row.stage_round, row.text)
if cur is not None:
    mu_rows.append(cur)

for i, r in enumerate(mu_rows, 1):
    r['meaning_unit_id'] = f'{r["participant_id"]}_MU{i:04d}'
    r['combined_original_text'] = ' '.join(r['texts'])
    r['number_of_source_events'] = len(r['event_ids'])
    r['duration_seconds'] = r['end_ts_seconds'] - r['start_ts_seconds']

meaning_units = pd.DataFrame(mu_rows).drop(columns=['key', 'texts'])
print('Total meaning units (adaptive-gap, applicant-continuity aware):', len(meaning_units))
print('Meaning units extended past the base 15s rule:', (meaning_units.extended_gap_merges > 0).sum())
print('Total source events covered:', meaning_units.number_of_source_events.sum(), 'vs eligible speech:', len(eligible_speech))
assert meaning_units.number_of_source_events.sum() == len(eligible_speech)
print('Avg source events per MU:', round(meaning_units.number_of_source_events.mean(), 2))
_all_ids = [e for lst in meaning_units.event_ids for e in lst]
assert len(_all_ids) == len(set(_all_ids)) == len(eligible_speech)
assert meaning_units.groupby('meaning_unit_id').participant_id.nunique().eq(1).all()
print('Validation OK: single participant/round/stage per MU; every event in exactly one MU; no source event duplicated.')



def match_applicant(pid, r, stage, start_ts, end_ts, text):
    if r is None or pd.isna(r):
        return 'Unmatched', None, None, ''
    r = int(r)
    cands = _round_applicants(pid, r)
    if len(cands) == 0:
        return 'Round-only match', None, None, ''
    nums = _extract_numbers(text)
    scores = [(row.id, row.row_in_round, *_score_applicant(nums, row)) for _, row in cands.iterrows()]
    scores.sort(key=lambda t: -t[2])
    top_score = scores[0][2]
    tied = [s for s in scores if s[2] == top_score]
    if top_score >= 2 and len(tied) == 1:
        return 'Exact applicant match', tied[0][0], tied[0][1], ','.join(tied[0][3])
    bounds = _round_bounds_ext.get((pid, r))
    slice_row_in_round = None
    if bounds and stage in ('Initial decision', 'AI recommendation/reconsideration', 'Final decision'):
        start, end = bounds
        span = max(end - start, 1)
        mid_ts = (start_ts + end_ts) / 2
        pos = min(max((mid_ts - start) / span, 0), 0.999)
        slice_row_in_round = int(pos * 10) + 1
    if top_score >= 1:
        if len(tied) == 1:
            return 'Probable applicant match', tied[0][0], tied[0][1], ','.join(tied[0][3])
        if slice_row_in_round is not None:
            match = [s for s in tied if s[1] == slice_row_in_round]
            if match:
                return 'Probable applicant match', match[0][0], match[0][1], ','.join(match[0][3]) + '+time_slice_tiebreak'
        return 'Round-only match', None, None, 'ambiguous_tied_content_matches'
    if slice_row_in_round is not None:
        row = cands[cands.row_in_round == slice_row_in_round]
        if len(row):
            return 'Probable applicant match', row.iloc[0].id, slice_row_in_round, 'time_slice_only'
    return 'Round-only match', None, None, ''

_match_results = meaning_units.apply(
    lambda mu: match_applicant(mu.participant_id, mu['round'], mu.decision_stage, mu.start_ts_seconds, mu.end_ts_seconds, mu.combined_original_text),
    axis=1)
meaning_units['applicant_match_quality'] = [m[0] for m in _match_results]
meaning_units['applicant_id'] = [m[1] for m in _match_results]
meaning_units['matched_row_in_round'] = [m[2] for m in _match_results]
meaning_units['applicant_match_evidence'] = [m[3] for m in _match_results]

print(meaning_units.applicant_match_quality.value_counts())
print()
print((meaning_units.applicant_match_quality.value_counts(normalize=True) * 100).round(2))


## 11.4 Parent Codes and Subcodes

The full codebook (operational definition, inclusion rules, exclusion rules, qualifying/non-qualifying examples, relevant decision stages, and the related existing approved theme) and subcodebook (72 subcodes across the twelve themes, including the seven new AI5 subcodes AI5a-AI5g) are displayed below.

In [ ]:
# -*- coding: utf-8 -*-
"""Rule-based multi-signal pattern definitions for the REPLACEMENT 12-theme framework.
Completely independent of, and not mixed with, the previous (now-removed) 12-theme patterns.

IMPORTANT: short bare-word alternatives (e.g. "ai", "it", "uses") are always wrapped in \\b
word boundaries below -- an earlier draft without boundaries caused "uses?" to falsely match
as a substring inside "because" (b-ec-AUSE), and similar risks exist for "ai"/"it" matching
inside common words. All patterns here were re-tested against real transcript text after this fix."""
import re

def pat(phrases):
    return re.compile(r'(?i)(?:' + '|'.join(phrases) + r')')

NEW_THEME_PATTERNS = {
    'AI1': {
        'name': 'AI as Data-Driven and Financially Focused',
        'high': pat([
            r'\bcomputer\b.{0,20}\b(?:risk score|risk|dti|income|utilisation|utilization|revolving|credit|loan amount|financial ratios?)\b',
            r'\b(?:the computer|it|ai)\b.{0,15}\b(?:uses|based on|looks? at|relies? on)\b.{0,20}\b(?:risk|dti|income|data|numbers|statistics)\b',
            r'\bhistorical (?:applicant )?data\b',
            r'\bstatistical patterns?\b',
            r'\bnumerical (?:evidence|patterns?)\b',
            r'\bcomputer\b.{0,20}\b(?:predictions?|recommendations?)\b.{0,20}\bbased on\b.{0,20}\b(?:data|numbers|statistics)\b',
        ]),
        'medium': pat([
            r'\bdata[- ]driven\b',
            r'\bpurely (?:numerical|financial)\b',
        ]),
    },
    'AI2': {
        'name': 'AI as Consistent and Rule-Based',
        'high': pat([
            r'\b(?:computer|ai|it)\b.{0,15}\b(?:is |follows? )?(?:consistent|systematic|predictable|standardi[sz]ed)\b',
            r'\b(?:computer|ai)\b.{0,15}\b(?:same|fixed) rules?\b',
            r'\b(?:computer|ai)\b.{0,15}\bless emotional\b',
            r'\b(?:computer|ai)\b.{0,20}\btreats?\b.{0,20}\b(?:similarly|the same|equally)\b',
            r'\b(?:computer|ai)\b.{0,20}\b(?:always|every time)\b.{0,20}\b(?:same|consistent)\b',
            r'\bcomputer\b.{0,20}\b(?:matching|matches|matched)\b.{0,15}\b(?:all|my)\b.{0,15}\banswers?\b',
        ]),
        'medium': pat([
            r'\bnot emotional\b',
            r'\bpredictable\b',
            r'\bcomputer\b.{0,15}\bagrees? with (?:me|all)\b',
        ]),
    },
    'AI3': {
        'name': 'AI as a Decision-Supporting Second Opinion',
        'high': pat([
            r'\bsecond opinion\b',
            r'\bconfirm(?:s|ed|ing)?\b.{0,20}\b(?:my|the) (?:initial )?(?:judgement|judgment|decision)\b',
            r'\breconsider(?:ed|ing)?\b',
            r'\b(?:check|checked|checking)\b.{0,15}\b(?:the )?(?:financial )?(?:information|numbers|data) again\b',
            r'\bcompare(?:d)?\b.{0,20}\b(?:my|it to my)\b.{0,20}\b(?:reasoning|decision)\b.{0,15}\bcomputer\b',
            r"\b(?:i'?m going to|i will|i'?ll)\b.{0,10}\b(?:stick with|keep)\b.{0,10}\bmy (?:decision|answer)\b",
            r'\bchanged? my mind\b',
            r'\bconsidered? the (?:computer|ai)\b.{0,20}\bbut\b',
        ]),
        'medium': pat([
            r'\badditional opinion\b',
            r'\blook(?:ed|ing)? at it again\b',
            r'\bmade me think again\b',
        ]),
    },
    'AI4': {
        'name': 'AI as Fallible and Performance-Dependent',
        'high': pat([
            r'\bcomputer\b.{0,10}\b(?:was|is|has been)\b.{0,10}\b(?:wrong|incorrect|right|correct)\b',
            r"\b(?:didn'?t|did not) do (?:so |too )?well\b.{0,15}\b(?:last time|before|previously)\b",
            r"\bcomputer'?s? (?:accuracy|performance|track record)\b",
            r'\b(?:trust|believe|rely on)\b.{0,10}\b(?:it|the computer)\b.{0,10}\b(?:more|less)\b.{0,10}\b(?:now|after|because)\b',
            r'\b(?:previous|last|earlier)\b.{0,10}\b(?:computer )?(?:prediction|recommendation)s?\b.{0,10}\b(?:were|was)\b.{0,10}\b(?:wrong|right|incorrect|correct)\b',
            r'\bmonitor(?:ing)?\b.{0,10}\bcomputer\b',
            r'\bmade (?:a )?mistakes?\b',
        ]),
        'medium': pat([
            r'\btrustable\b',
            r'\btrack record\b',
            r'\bgot it wrong\b',
        ]),
    },
    'AI5': {
        'name': 'AI as a Confidence Signal and Behavioural Influence',
        'high': pat([
            r'\bcomputer\b.{0,10}\b(?:also )?(?:agrees?|approv\w*|reject\w*|said|says|thinks?)\b[^.]{0,50}\bmakes? me\b.{0,15}\b(?:more|less|not (?:as|so)) confident\b',
            r'\bif the computer agrees? with me\b.{0,40}\bconfiden\w*\b',
            r"\b(?:it |this )?(?:also )?agrees? with me\b.{0,40}\bconfiden\w*\b",
            r'\b(?:more|higher|increase\w*)\b.{0,10}\bconfiden\w*\b.{0,40}\bbecause\b.{0,20}\bi agree\w*\b.{0,15}\bcomputer\b',
            r'\bconfiden\w*\b.{0,25}\bbecause\b.{0,25}\b(?:computer|prediction)\b',
            r"\bcomputer\b.{0,25}\bthat'?s why\b.{0,15}\bconfiden\w*\b",
            r"\bconfiden\w*\b.{0,15}\bthat'?s why\b.{0,25}\bcomputer\b",
            r"\bdoesn'?t make me (?:feel )?(?:more )?confident\b.{0,60}\bcomputer\b.{0,20}\bagrees?\b",
            r"\bnot make me (?:feel )?(?:more )?confident\b.{0,60}\bcomputer\b",
            r"\bdiscrepanc\w*\b.{0,50}\bcomputer\b",
            r"\bnot (?:so |too |that )?sure why\b.{0,25}\bcomputer\b",
            r"\bcheck (?:the )?computer'?s? (?:answer|prediction)s?\b",
            r"\bsee what the computer (?:says|thinks|predictions? say)\b.{0,30}\bconfiden\w*\b",
            r"\bconfiden\w*\b.{0,30}\bsee what the computer\b",
            r"\bcomputer\b.{0,20}\b(?:also |as well )?(?:rejected|approved|agrees?)\b.{0,30}\b(?:keep|keeping)\b.{0,15}\b(?:same|rating)\b.{0,15}\bconfiden\w*\b",
            r"\binitially (?:rejected|approved)\b.{0,10}\bas well\b.{0,20}\bkeep\b.{0,15}\b(?:same|rating)\b",
            r"\bi agree\w*\b.{0,15}\bcomputer\b.{0,20}\bconfiden\w*\b",
            r"\bcomputer\b.{0,15}\bagrees? with me\b.{0,20}\bconfiden\w*\b",
        ]),
        'medium': pat([
            r'\bconfiden\w*\b.{0,50}\bcomputer\b',
            r'\bcomputer\b.{0,50}\bconfiden\w*\b',
            r'\bunsure\b.{0,40}\bcomputer\b',
            r'\bcomputer\b.{0,40}\bunsure\b',
            r'\bnot (?:that |very |too )?sure\b.{0,40}\bcomputer\b',
        ]),
    },
    'AI6': {
        'name': 'AI as Potentially Susceptible to Data-Embedded Bias',
        'high': pat([
            r"\b(?:computer|ai)'?s? (?:is |seems? |appears? |predictions? )?bias(?:ed)?\b",
            r'\bbias(?:ed)?\b.{0,10}\b(?:towards?|against)\b.{0,5}\b(?:orange|purple)\b',
            r'\bfavou?rs?\b.{0,5}\b(?:orange|purple)\b',
            r'\b(?:historical|historic)\b.{0,10}\bbias\b',
            r'\bsystematically\b.{0,10}\b(?:favou?rs?|approves?|rejects?)\b',
            r'\b(?:computer|ai|it)\b.{0,15}\b(?:might|may|could) be biased\b',
        ]),
        'medium': pat([
            r'\b(?:orange|purple)\b.{0,25}\b(?:treated|approved)\b.{0,15}\b(?:more|differently|unfairly)\b',
            r'\bsuspicious\b.{0,15}\b(?:orange|purple)\b',
            r'\bbias(?:ed)?\b',
        ]),
    },
    'H1': {
        'name': 'Human Contextual Understanding',
        'high': pat([
            r'\b(?:loan )?purpose\b.{0,20}\b(?:affects?|changes?|influences?)\b',
            r'\b(?:considering|considered|taking into account)\b.{0,15}\b(?:the )?(?:context|circumstances|purpose)\b',
            r'\baffordability\b.{0,15}\b(?:context|circumstances)\b',
            r'\b(?:looking at|considered)\b.{0,15}\b(?:together|as a whole|holistically)\b',
            r'\b(?:medical|business|personal)\b.{0,10}\b(?:context|circumstances)\b.{0,10}\b(?:considered|matter)\b',
            r'\bmore than (?:one|just a)\b.{0,10}\b(?:risk score|number)\b',
            r'\b(?:approve|approving|reject|rejecting)\b.{0,40}\bpurpose\b',
            r'\bpurpose\b.{0,40}\b(?:approve|approving|reject|rejecting)\b',
            r'\bbecause\b.{0,20}\bpurpose\b',
            r'\bpurpose\b.{0,10}\b(?:is|was)\b.{0,10}\b(?:medical|business|debt consolidation|credit card|home)\b.{0,30}\b(?:so|therefore|which)\b',
        ]),
        'medium': pat([
            r'\bcontext matters\b',
            r'\bcircumstances\b',
            r'\bconsidering\b',
            r'\bconsider (?:the|their|that)\b',
            r'\btake (?:into account|that into consideration)\b',
        ]),
    },
    'H2': {
        'name': 'Human Empathy and Humanitarian Consideration',
        'high': pat([
            r"\bneeds? it for medical\b",
            r"\bmedical\b.{0,20}\b(?:treatment|need|bills?|reasons?)\b.{0,40}\b(?:approve|help)\b",
            r"\b(?:approve|approving)\b.{0,40}\bmedical\b",
            r"\b(?:family|personal)\b.{0,10}\b(?:circumstances|situation|hardship)\b",
            r"\burgent\b.{0,10}\b(?:need|circumstances)\b",
            r"\bbenefit of the doubt\b",
            r"\bhumanitarian\b",
            r"\bdeserves? (?:a|the) (?:chance|loan)\b",
            r"\b(?:concern|worried) for\b.{0,15}\b(?:welfare|wellbeing)\b",
        ]),
        'medium': pat([
            r'\bfeel (?:bad|sorry) for\b',
            r'\bsympath(?:y|etic)\b',
            r'\bgive them a chance\b',
        ]),
    },
    'H3': {
        'name': 'Human Holistic Balancing of Conflicting Evidence',
        'high': pat([
            r"\b(?:high|higher|good) income\b.{0,40}\b(?:outweighs?|compensat\w*|makes up for|offsets?)\b.{0,30}\brisk\b",
            r"\b(?:low|small) (?:loan amount|amount)\b.{0,40}\b(?:outweighs?|compensat\w*|offsets?)\b",
            r"\blow dti\b.{0,40}\b(?:outweighs?|offsets?|compensat\w*)\b",
            r"\b(?:balanc(?:e|ing)|weigh(?:ing)?|trade[- ]?off)\b.{0,15}\b(?:the )?(?:strengths?|positives?|negatives?)\b",
            r"\b(?:several|multiple)\b.{0,10}\b(?:strengths?|weaknesses?)\b.{0,15}\b(?:outweigh|but)\b",
            r"\bcompensat\w* for\b",
        ]),
        'medium': pat([
            r'\beverything else (?:is|was) (?:okay|fine|good)\b',
            r'\bon (?:the )?(?:other hand|balance)\b',
        ]),
    },
    'H4': {
        'name': 'Human Learning and Adaptive Judgement',
        'high': pat([
            r"\blearn(?:ed|ing)? from\b",
            r"\b(?:changed|change|adjusting|adjusted)\b.{0,10}\bmy (?:strategy|threshold|approach)\b",
            r"\b(?:compared? (?:to|with)|like)\b.{0,10}\b(?:the )?(?:previous|last|earlier)\b.{0,10}\b(?:applicant|round|one)\b",
            r"\b(?:based on|because of|after)\b.{0,10}\b(?:the )?(?:last|previous|earlier)\b.{0,10}\b(?:round|result|feedback)\b",
            r"\bnow (?:i|i'?m)\b.{0,10}\b(?:more|less)\b.{0,10}\b(?:careful|strict|lenient)\b",
            r"\b(?:change|changed|changing) my trust in\b.{0,10}\b(?:the )?(?:computer|ai)\b",
        ]),
        'medium': pat([
            r'\bnext time i\b',
            r'\bfrom now on\b',
        ]),
    },
    'H5': {
        'name': 'Human Independence and Ability to Challenge AI',
        'high': pat([
            r"\b(?:computer|ai|it) is wrong\b",
            r"\bnot (?:having|taking|approving) (?:that|this|it)\b.{0,20}\bcomputer\b",
            r"\b(?:sticking|stuck|keeping|retaining) with my (?:decision|answer|judgement|judgment)\b",
            r"\bi don'?t care what the computer says\b",
            r"\bregardless of (?:the )?computer\b",
            r"\bquestion(?:ing)? (?:the )?(?:computer|ai)'?s? (?:reasoning|fairness|logic)\b",
            r"\b(?:my|i have) (?:own|independent) (?:judgement|judgment|decision|authority)\b",
            r"\b(?:computer|ai) (?:is |as )?(?:just )?advice\b",
        ]),
        'medium': pat([
            r'\bnot (?:going to|gonna) change my mind\b',
            r'\btrust my (?:own )?judgement\b',
        ]),
    },
    'H6': {
        'name': 'Human Subjectivity, Bias and Heuristic Inconsistency',
        'high': pat([
            r"\b(?:my|a) (?:personal )?(?:rule|threshold|cut[- ]?off)\b",
            r"\banything (?:above|below|over|under) (?:than )?\d+\b",
            r"\bi (?:always|usually|typically) (?:approve|reject)\b",
            r"\bgut (?:feeling|instinct)\b",
            r"\bjust (?:a |my )?(?:feeling|hunch|intuition)\b",
            r"\b(?:started to|began to|i) prefer (?:orange|purple)\b",
            r"\bnot (?:really )?sure why\b.{0,10}\bbut\b",
        ]),
        'medium': pat([
            r'\bintuition\b',
            r'\bnot (?:entirely |totally )?consistent\b',
            r'\bover \d\d\b',
            r'\bunder \d\d\b',
        ]),
    },
}


# -*- coding: utf-8 -*-
NEW_CODEBOOK_ROWS = [
    dict(theme_id='AI1', theme_group='AI-characteristic', theme_name='AI as Data-Driven and Financially Focused',
         operational_definition='Participant describes AI as relying mainly on measurable financial information, numerical indicators, applicant data or historical patterns.',
         inclusion_rules='References to AI using risk scores, income, DTI, revolving utilisation, credit information, loan amount, financial ratios, historical applicant data, statistical patterns, numerical evidence.',
         exclusion_rules='Excludes the participant personally using financial information without referring to AI; excludes simple AI agreement/disagreement without discussion of its data-based reasoning.',
         qualifying_example='"The computer probably just looks at the risk score and the DTI."',
         non_qualifying_example='"The computer said reject." (no discussion of AI’s reasoning basis)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision',
         related_existing_approved_theme='AI Trust Was Conditional, Contested and Performance-Sensitive',
         coder_notes='Requires AI to be explicitly named as using/basing on data; a bare co-occurrence of "computer" and a financial term is not sufficient.'),
    dict(theme_id='AI2', theme_group='AI-characteristic', theme_name='AI as Consistent and Rule-Based',
         operational_definition='Participant describes AI as systematic, predictable, standardised or governed by fixed rules.',
         inclusion_rules='AI applies the same rules; AI is systematic; AI is less emotional; AI is predictable; AI follows a fixed decision process; AI treats similar applicants similarly.',
         exclusion_rules='Excludes assuming AI is consistent merely because it is a computer; excludes statements mentioning an AI decision without describing consistency or rule-following.',
         qualifying_example='"The computer is consistent, it applies the same rule every time."',
         non_qualifying_example='"The computer approved it." (no consistency/rule language)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision',
         related_existing_approved_theme='AI Trust Was Conditional, Contested and Performance-Sensitive',
         coder_notes='Distinguish from AI4 (performance-dependent trust): AI2 is about predictability/rules, not accuracy history.'),
    dict(theme_id='AI3', theme_group='AI-characteristic', theme_name='AI as a Decision-Supporting Second Opinion',
         operational_definition='Participant uses the AI recommendation as advice, confirmation, challenge or an additional decision input rather than unquestioned authority.',
         inclusion_rules='AI confirms an initial judgement; AI causes reconsideration; participant checks evidence again; participant compares their judgement with AI; participant considers AI but retains own decision; participant changes towards AI after reflection.',
         exclusion_rules='Excludes automatically following AI without evidence of considering it; excludes simple human-AI agreement without expressed decision-support value.',
         qualifying_example='"The computer disagreed, so I went back and checked the numbers again."',
         non_qualifying_example='"Computer said approve, I said approve." (no evidence of deliberation)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision',
         related_existing_approved_theme='Perception–Behaviour Alignment and Mismatch Coexisted',
         coder_notes='Overlaps conceptually with H5 (independence) but AI3 centres on AI as a genuine input to reasoning, not on retaining authority.'),
    dict(theme_id='AI4', theme_group='AI-characteristic', theme_name='AI as Fallible and Performance-Dependent',
         operational_definition='Participant treats AI as capable of error and adjusts trust according to its previous or observed performance.',
         inclusion_rules='AI made mistakes; AI was correct previously; AI accuracy monitoring; trust increased/decreased after performance; comparing human and AI success; following or resisting AI because of earlier outcomes.',
         exclusion_rules='Excludes general trust or distrust with no performance-related explanation.',
         qualifying_example='"The computer got the last one wrong so I trust it less now."',
         non_qualifying_example='"I don’t trust the computer." (no performance reference)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision; Results/feedback',
         related_existing_approved_theme='AI Trust Was Conditional, Contested and Performance-Sensitive',
         coder_notes='Requires an explicit past-performance reference (this round or a previous one).'),
    dict(theme_id='AI5', theme_group='AI-characteristic', theme_name='AI as a Confidence Signal and Behavioural Influence',
         operational_definition="The participant's confidence, reconsideration or behaviour is influenced by agreement or disagreement with the AI recommendation.",
         inclusion_rules='Confidence increased after AI agreement; confidence decreased after AI disagreement; AI provided reassurance; AI created doubt; AI encouraged reconsideration; AI affected behaviour even when the final decision did not change; participant used AI agreement as confirmation; participant became less certain after seeing conflicting AI advice.',
         exclusion_rules='Excludes ordinary human-AI agreement without expressed confidence or behavioural influence; excludes a decision change with no verbal evidence that AI affected confidence or reasoning; excludes general AI trust statements unrelated to a specific influence.',
         qualifying_example='"If the computer agrees with me, then I\'d say my confidence is now a bit higher."',
         non_qualifying_example='"The computer said approve." (agreement stated with no expressed effect on confidence or behaviour)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision; Results/feedback where relevant',
         related_existing_approved_theme='AI Trust Was Conditional, Contested and Performance-Sensitive',
         coder_notes='Replaces the previous AI5 ("AI as Limited in Contextual Understanding"), which returned zero confirmed meaning units in this corpus; that theme is retained only in the searched-but-not-observed appendix, never in the active analysis. Distinguish from AI3 (decision-supporting second opinion): AI3 covers using the AI recommendation as a general decision input (confirming, reconsidering, checking again); AI5 specifically requires an expressed confidence, reassurance, doubt or behavioural effect. The two may co-occur in the same meaning unit under multi-label coding.'),
    dict(theme_id='AI6', theme_group='AI-characteristic', theme_name='AI as Potentially Susceptible to Data-Embedded Bias',
         operational_definition='Participant suggests AI may reproduce bias from its data, design or historical decision patterns.',
         inclusion_rules='AI may be biased; historical data may contain bias; AI may favour Orange or Purple applicants; AI may systematically treat groups differently; participant challenges AI because of suspected bias.',
         exclusion_rules='Excludes general fairness statements not connected to AI; excludes observed group differences without participant interpretation of AI bias.',
         qualifying_example='"I think the computer favours Purple applicants because of old data."',
         non_qualifying_example='"Orange and Purple should be treated equally." (no AI reference)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision; General reflection',
         related_existing_approved_theme='Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed',
         coder_notes='Distinguish from H6e (background-related assumption in human’s own reasoning) -- AI6 is specifically about the AI/model/data, not the participant’s own assumption.'),
    dict(theme_id='H1', theme_group='Human-characteristic', theme_name='Human Contextual Understanding',
         operational_definition='Participant considers applicant-specific circumstances, relationships between variables or the practical meaning of the loan rather than relying only on isolated numerical values.',
         inclusion_rules='Loan purpose affects judgement; applicant-specific circumstances; affordability interpreted in context; financial variables interpreted together; medical/business/personal context considered; reasoning beyond one risk score.',
         exclusion_rules='Excludes merely reading the applicant’s loan purpose; excludes single-variable numerical rules; excludes empathy claims without evidence that context affected judgement.',
         qualifying_example='"It’s a business loan, so the risk makes more sense given what they’re using it for."',
         non_qualifying_example='"Purpose: debt consolidation." (reading a field, not interpreting it)',
         relevant_decision_stages='Initial decision; Final decision',
         related_existing_approved_theme='Financially Grounded but Heuristic Judgement',
         coder_notes='H1 is the general contextual-reasoning theme; H2 is the affect-laden/humanitarian subset of it.'),
    dict(theme_id='H2', theme_group='Human-characteristic', theme_name='Human Empathy and Humanitarian Consideration',
         operational_definition='Participant expresses concern for the applicant’s welfare, urgent need or humanitarian circumstances and allows this concern to influence judgement.',
         inclusion_rules='Approving because of medical need; concern for family circumstances; urgent personal need; helping the applicant; benefit of the doubt because of hardship; moral/humanitarian justification.',
         exclusion_rules='Excludes merely mentioning "medical"; excludes approving because repayment appears financially safe; excludes general contextual understanding without concern for welfare.',
         qualifying_example='"The risk is high, but they need the money for medical treatment, so I will approve."',
         non_qualifying_example='"The loan purpose is medical."',
         relevant_decision_stages='Initial decision; Final decision',
         related_existing_approved_theme='Financially Grounded but Heuristic Judgement',
         coder_notes='Requires the concern to demonstrably influence the stated judgement, not just be mentioned.'),
    dict(theme_id='H3', theme_group='Human-characteristic', theme_name='Human Holistic Balancing of Conflicting Evidence',
         operational_definition='Participant weighs multiple positive and negative factors and allows some factors to compensate for others.',
         inclusion_rules='High income compensates for high risk; low loan amount compensates for weaker credit information; low DTI offsets high utilisation; several strengths outweigh one weakness (or vice versa); loan purpose considered together with financial capacity.',
         exclusion_rules='Excludes decisions based on only one variable; excludes simple thresholds; excludes unsupported intuition.',
         qualifying_example='"The DTI is bad but the income more than makes up for it."',
         non_qualifying_example='"Risk is 6, reject." (single-variable threshold)',
         relevant_decision_stages='Initial decision; Final decision',
         related_existing_approved_theme='Financially Grounded but Heuristic Judgement',
         coder_notes='Requires an explicit trade-off between at least two named factors.'),
    dict(theme_id='H4', theme_group='Human-characteristic', theme_name='Human Learning and Adaptive Judgement',
         operational_definition='Participant changes or refines decision strategy based on previous applicants, feedback, results or AI performance.',
         inclusion_rules='Learning from previous rounds; changing thresholds; changing feature importance; becoming more/less strict; changing trust in AI; comparing current applicant with previous cases; adapting after errors or successes.',
         exclusion_rules='Excludes mentioning previous cases without any change in reasoning; excludes repeating the same rule without adaptation.',
         qualifying_example='"After getting the last one wrong I’m being stricter on DTI now."',
         non_qualifying_example='"Like the last one, high risk." (comparison without adaptation)',
         relevant_decision_stages='Results/feedback; subsequent Initial/Final decision',
         related_existing_approved_theme='Feedback Supported Learning, but Calibration Remained Uneven',
         coder_notes='Requires an explicit change in strategy/threshold/trust, not just a reference to a previous case.'),
    dict(theme_id='H5', theme_group='Human-characteristic', theme_name='Human Independence and Ability to Challenge AI',
         operational_definition='Participant treats AI as advisory, maintains decision authority and may deliberately reject, question or override its recommendation.',
         inclusion_rules='Explicitly saying AI is wrong; retaining an initial decision against AI; explaining why personal judgement is preferred; questioning AI reasoning; resisting suspected AI bias; using AI as advice rather than final authority.',
         exclusion_rules='Excludes disagreement with no evidence of deliberate human reasoning; excludes accidental final disagreement; excludes automatic resistance without explanation.',
         qualifying_example='"I don’t care what the computer says, I’m rejecting this."',
         non_qualifying_example='(Final decision differs from AI with no verbal reasoning given)',
         relevant_decision_stages='AI recommendation/reconsideration; Final decision',
         related_existing_approved_theme='Perception–Behaviour Alignment and Mismatch Coexisted',
         coder_notes='Requires explicit reasoning, not just an observed behavioural disagreement (that is captured structurally elsewhere in the notebook).'),
    dict(theme_id='H6', theme_group='Human-characteristic', theme_name='Human Subjectivity, Bias and Heuristic Inconsistency',
         operational_definition='Participant uses personal shortcuts, intuition, potentially biased assumptions or inconsistent criteria that may differ across applicants.',
         inclusion_rules='Personal numerical threshold; single-variable decision; unsupported intuition; criteria change without feedback-based justification; inconsistent rules; uncertainty-driven decision; possible Orange/Purple assumption; subjective judgement not clearly supported by evidence.',
         exclusion_rules='Excludes careful multi-factor balancing (see H3); excludes justified strategy adaptation after feedback (see H4); excludes consistent evidence-based thresholds clearly defined in advance.',
         qualifying_example='"I just go with my gut on this one, don’t know why."',
         non_qualifying_example='"DTI over 30 means I always reject, that’s been my rule since round 1." (consistent, pre-defined -- excluded)',
         relevant_decision_stages='Initial decision; AI recommendation/reconsideration; Final decision',
         related_existing_approved_theme='Fairness as Background-Blind, Equal and Evidence-Based Treatment',
         coder_notes='H6 and H1/H3 can be hard to distinguish -- H6 requires evidence of inconsistency, intuition, or unexamined assumption, not just a threshold rule per se.'),
]


# -*- coding: utf-8 -*-
import re

NEW_SUBCODES = {
    'AI1': [('AI1a', 'Risk-score focus', re.compile(r'(?i)risk score|risk\b')),
            ('AI1b', 'Income and affordability focus', re.compile(r'(?i)\bincome\b|afford')),
            ('AI1c', 'DTI and utilisation focus', re.compile(r'(?i)\bdti\b|utilisation|utilization|revolving')),
            ('AI1d', 'Historical-data use', re.compile(r'(?i)historical')),
            ('AI1e', 'Statistical or numerical pattern recognition', re.compile(r'(?i)statistical|numerical|pattern')),
            ('AI1f', 'Financial evidence prioritised over personal context', re.compile(r'(?i)just.{0,10}numbers|only.{0,10}(?:numbers|data)'))],
    'AI2': [('AI2a', 'Fixed-rule application', re.compile(r'(?i)fixed rule|same rule')),
            ('AI2b', 'Predictable decision pattern', re.compile(r'(?i)predictable')),
            ('AI2c', 'Standardised treatment', re.compile(r'(?i)standardi[sz]ed|treats?.{0,15}(?:same|equally)')),
            ('AI2d', 'Less emotional or subjective', re.compile(r'(?i)less emotional|not emotional')),
            ('AI2e', 'Similar cases treated similarly', re.compile(r'(?i)similar(?:ly)?'))],
    'AI3': [('AI3a', 'Confirmation of initial judgement', re.compile(r'(?i)confirm')),
            ('AI3b', 'Reconsideration after disagreement', re.compile(r'(?i)reconsider')),
            ('AI3c', 'Rechecking applicant information', re.compile(r'(?i)check.{0,10}again')),
            ('AI3d', 'Changed towards AI', re.compile(r'(?i)changed? my mind')),
            ('AI3e', 'Considered AI but retained own judgement', re.compile(r'(?i)stick with|keep my')),
            ('AI3f', 'AI used as an additional opinion', re.compile(r'(?i)second opinion|additional opinion'))],
    'AI4': [('AI4a', 'Previous AI error', re.compile(r'(?i)wrong|mistake|incorrect')),
            ('AI4b', 'Previous AI success', re.compile(r'(?i)\bright\b|\bcorrect\b')),
            ('AI4c', 'Accuracy monitoring', re.compile(r'(?i)accuracy|monitor|track record')),
            ('AI4d', 'Trust increased after success', re.compile(r'(?i)trust.{0,10}more')),
            ('AI4e', 'Trust decreased after failure', re.compile(r'(?i)trust.{0,10}less')),
            ('AI4f', 'Performance comparison with human judgement', re.compile(r'(?i)compare.{0,15}(?:human|my)'))],
    'AI5': [('AI5a', 'Confidence increased after agreement', re.compile(r"(?i)(?:more|higher|increase\w*).{0,20}confiden\w*.{0,40}(?:because|since).{0,25}(?:i )?agree\w*|agrees? with me.{0,25}confiden\w*.{0,20}(?:higher|more|increase)|even more (?:confident|certain).{0,20}agrees?|confiden\w*.{0,25}because.{0,25}(?:computer|prediction).{0,20}(?:approv|agree)")),
            ('AI5b', 'Confidence decreased after disagreement', re.compile(r"(?i)(?:less|lower|decrease\w*|not as|wasn'?t as).{0,15}confiden\w*|confiden\w*.{0,20}doesn'?t increase|discrepanc\w*")),
            ('AI5c', 'AI provided reassurance', re.compile(r"(?i)reassur\w*|matching all my answers|feel more confident.{0,30}(?:because|so)")),
            ('AI5d', 'AI created doubt', re.compile(r"(?i)not (?:so |too |that )?sure why.{0,25}computer|doesn'?t make me.{0,20}confident|not make me.{0,20}confident")),
            ('AI5e', 'AI encouraged reconsideration', re.compile(r"(?i)check (?:the )?computer'?s? (?:answer|prediction)|re-?evaluate|re-?think\b")),
            ('AI5f', 'AI influenced behaviour without changing the decision', re.compile(r"(?i)still (?:gonna|going to|would)\s+(?:reject|approve).{0,50}confiden\w*|confiden\w*.{0,50}still (?:gonna|going to|would)")),
            ('AI5g', "AI confirmed the participant's existing judgement", re.compile(r"(?i)initially (?:rejected|approved).{0,20}(?:as well|too)|keep(?:ing)? the same rating|agrees? with me"))],
    'AI6': [('AI6a', 'Suspected Orange/Purple bias', re.compile(r'(?i)orange|purple')),
            ('AI6b', 'Historical-data bias', re.compile(r'(?i)historical')),
            ('AI6c', 'Systematic group preference', re.compile(r'(?i)systematic')),
            ('AI6d', 'Biased model or algorithm', re.compile(r'(?i)model|algorithm')),
            ('AI6e', 'Resistance caused by suspected AI bias', re.compile(r'(?i)resist|reject.{0,15}bias'))],
    'H1': [('H1a', 'Loan-purpose interpretation', re.compile(r'(?i)purpose')),
           ('H1b', 'Applicant-specific circumstances', re.compile(r'(?i)circumstances')),
           ('H1c', 'Financial context interpretation', re.compile(r'(?i)context')),
           ('H1d', 'Medical-purpose context', re.compile(r'(?i)medical')),
           ('H1e', 'Business or productive-purpose context', re.compile(r'(?i)business')),
           ('H1f', 'Personal or family context', re.compile(r'(?i)family|personal'))],
    'H2': [('H2a', 'Medical need', re.compile(r'(?i)medical')),
           ('H2b', 'Family or personal hardship', re.compile(r'(?i)family|hardship')),
           ('H2c', 'Urgent circumstances', re.compile(r'(?i)urgent')),
           ('H2d', 'Humanitarian justification', re.compile(r'(?i)humanitarian')),
           ('H2e', 'Benefit of the doubt', re.compile(r'(?i)benefit of the doubt')),
           ('H2f', 'Concern for applicant welfare', re.compile(r'(?i)welfare|wellbeing|concern'))],
    'H3': [('H3a', 'Income offsets risk', re.compile(r'(?i)income.{0,30}(?:outweigh|compensat|offset)')),
           ('H3b', 'Loan amount offsets weakness', re.compile(r'(?i)amount.{0,30}(?:outweigh|compensat|offset)')),
           ('H3c', 'Low DTI offsets another risk', re.compile(r'(?i)dti.{0,30}(?:outweigh|offset|compensat)')),
           ('H3d', 'Several strengths outweigh weakness', re.compile(r'(?i)strengths?')),
           ('H3e', 'Several weaknesses outweigh strength', re.compile(r'(?i)weaknesses?')),
           ('H3f', 'Context combined with financial evidence', re.compile(r'(?i)purpose.{0,30}(?:income|risk|dti)'))],
    'H4': [('H4a', 'Learning from previous outcomes', re.compile(r'(?i)learn')),
           ('H4b', 'Threshold adjustment', re.compile(r'(?i)threshold')),
           ('H4c', 'Feature-priority adjustment', re.compile(r'(?i)strategy|approach')),
           ('H4d', 'Trust adjustment', re.compile(r'(?i)trust')),
           ('H4e', 'Comparison with previous applicants', re.compile(r'(?i)previous|earlier|last (?:one|round)')),
           ('H4f', 'Strategy adaptation after error', re.compile(r'(?i)after.{0,15}(?:wrong|mistake|error)'))],
    'H5': [('H5a', 'Retained decision against AI', re.compile(r'(?i)stick|keep|retain')),
           ('H5b', 'Explicit challenge to AI', re.compile(r'(?i)question|challenge|wrong')),
           ('H5c', 'Human final authority', re.compile(r'(?i)my (?:own )?(?:decision|judgement|judgment|authority)')),
           ('H5d', 'Independent evidence-based judgement', re.compile(r'(?i)independent')),
           ('H5e', 'Resistance to suspected AI bias', re.compile(r'(?i)bias')),
           ('H5f', 'AI treated as advice only', re.compile(r'(?i)advice|second opinion'))],
    'H6': [('H6a', 'Personal numerical threshold', re.compile(r'(?i)my rule|threshold|cut[- ]?off')),
           ('H6b', 'Single-cue reasoning', re.compile(r'(?i)anything (?:over|above|under|below)')),
           ('H6c', 'Intuitive judgement', re.compile(r'(?i)gut|intuition|hunch')),
           ('H6d', 'Inconsistent criteria', re.compile(r'(?i)inconsistent')),
           ('H6e', 'Possible background-related assumption', re.compile(r'(?i)orange|purple')),
           ('H6f', 'Uncertainty-driven inconsistency', re.compile(r'(?i)not sure|uncertain')),
           ('H6g', 'Unsupported subjective judgement', re.compile(r'(?i)feeling|subjective'))],
}

def assign_new_subcode(theme_id, text):
    for sub_id, sub_name, p in NEW_SUBCODES.get(theme_id, []):
        if p.search(text):
            return sub_id, sub_name
    return None, None


NEW_THEME_PATTERNS = NEW_THEME_PATTERNS
final_12_theme_codebook = pd.DataFrame(NEW_CODEBOOK_ROWS)
theme_ids_all = list(NEW_THEME_PATTERNS.keys())
assert set(theme_ids_all) == set(f'AI{i}' for i in range(1, 7)) | set(f'H{i}' for i in range(1, 7))

_sub_rows = []
for tid, subs in NEW_SUBCODES.items():
    for sub_id, sub_name, p in subs:
        _sub_rows.append({'theme_id': tid, 'subcode_id': sub_id, 'subcode_name': sub_name, 'pattern_description': p.pattern})
final_12_theme_subcodebook = pd.DataFrame(_sub_rows)
assert len(final_12_theme_subcodebook) == 72
print('final_12_theme_codebook:', final_12_theme_codebook.shape, '  final_12_theme_subcodebook:', final_12_theme_subcodebook.shape)

display(style_table(final_12_theme_codebook))
display(style_table(final_12_theme_subcodebook))


## 11.5 AI-Characteristic Findings

Classification uses a two-stage process: (1) a broad, topic-level candidate search flags meaning units plausibly on-topic for one or more themes; (2) each candidate is then strictly confirmed or rejected against the theme's specific High/Medium inclusion patterns and exclusion rules -- never keyword-matching alone. Rejected candidates are retained with a rejection reason. For every classified meaning unit, participant ID, round, applicant ID (where available), timestamp range, original combined text, parent theme, subcode, classification evidence, inclusion rule used, confidence, decision stage, applicant-match quality, stance and behavioural effect are all stored, with multi-label coding allowed (meaning units are never forced into a theme). AI-theme results are shown below.

In [ ]:
# =====================================================================
# Two-stage classification: (1) broad candidate identification by topic, (2) strict
# confirmation using the per-theme inclusion/exclusion pattern sets. Not keyword-only:
# stage 2 requires the fuller phrase-level High/Medium patterns to match, and stores
# rejected candidates with a rejection reason (never keyword-matching alone).
# =====================================================================
NEW_TOPIC_PATTERNS = {
    'ai_data_use': re.compile(r'(?i)\bcomputer\b.{0,20}\b(?:risk|dti|income|data|numbers|statistics)\b'),
    'ai_rules_consistency': re.compile(r'(?i)\b(?:computer|ai)\b.{0,15}\b(?:consistent|systematic|predictable|rule|same)\b'),
    'ai_as_advice': re.compile(r'(?i)second opinion|reconsider|check.{0,10}again|stick with my|changed? my mind'),
    'ai_performance': re.compile(r'(?i)\bcomputer\b.{0,15}\b(?:wrong|right|correct|incorrect|accuracy|mistake)\b'),
    'ai_confidence_influence': re.compile(r'(?i)\bconfiden\w*\b|\breassur\w*\b|\bdoubt\w*\b|\bunsure\b'),
    'ai_bias': re.compile(r'(?i)\bbias|\borange\b|\bpurple\b|favou?r'),
    'human_context': re.compile(r'(?i)\bpurpose\b|\bcontext\b|\bcircumstances?\b|\bconsidering\b'),
    'empathy': re.compile(r'(?i)\bmedical\b|\bfamily\b|\burgent\b|\bhumanitarian\b|\bhardship\b'),
    'balancing_evidence': re.compile(r'(?i)\boutweigh|\bcompensat|\bbalance\b|\bhowever\b|\balthough\b|\bbut\b'),
    'learning': re.compile(r'(?i)\blearn|\bprevious\b|\blast time\b|\bfeedback\b|\bthreshold\b'),
    'independence': re.compile(r'(?i)\bwrong\b|\bstick\b|\bkeep my\b|\bmy (?:own|independent)\b|\badvice\b'),
    'subjectivity_inconsistency': re.compile(r'(?i)\bgut\b|\bintuition\b|\bfeeling\b|\bmy rule\b|\banything (?:over|above|under|below)\b'),
}
TOPIC_TO_NEW_THEMES = {
    'ai_data_use': ['AI1'], 'ai_rules_consistency': ['AI2'], 'ai_as_advice': ['AI3'],
    'ai_performance': ['AI4'], 'ai_confidence_influence': ['AI5'], 'ai_bias': ['AI6', 'H6'],
    'human_context': ['H1', 'AI3'], 'empathy': ['H2'], 'balancing_evidence': ['H3'],
    'learning': ['H4'], 'independence': ['H5', 'AI3'], 'subjectivity_inconsistency': ['H6'],
}

def new_candidate_themes(text):
    cands = set(); topics = []
    for topic, p in NEW_TOPIC_PATTERNS.items():
        if p.search(text):
            topics.append(topic); cands.update(TOPIC_TO_NEW_THEMES[topic])
    return cands, topics

def new_classify_text(text):
    out = []
    for tid, spec in NEW_THEME_PATTERNS.items():
        hm = spec['high'].search(text)
        if hm:
            out.append((tid, 'High', hm.group(0))); continue
        mm = spec['medium'].search(text)
        if mm:
            out.append((tid, 'Medium', mm.group(0)))
    return out

_candidate_rows = []
_mu_hits = []
for _, mu in meaning_units.iterrows():
    cands, topics = new_candidate_themes(mu.combined_original_text)
    confirmed = new_classify_text(mu.combined_original_text)
    confirmed_ids = {c[0] for c in confirmed}
    _mu_hits.append(confirmed)
    for tid in cands:
        status = 'Confirmed' if tid in confirmed_ids else 'Rejected'
        reason = '' if status == 'Confirmed' else 'Broad topic matched but strict inclusion pattern not satisfied (or exclusion criterion applied).'
        _candidate_rows.append({
            'meaning_unit_id': mu.meaning_unit_id, 'participant_id': mu.participant_id, 'round': mu['round'],
            'candidate_theme_id': tid, 'candidate_theme_name': NEW_THEME_PATTERNS[tid]['name'],
            'matched_topics': ','.join(topics), 'status': status, 'rejection_reason': reason,
            'context_used': mu.decision_stage,
        })
final_12_theme_candidate_review = pd.DataFrame(_candidate_rows)
meaning_units['theme_hits'] = _mu_hits

print('Candidate rows:', len(final_12_theme_candidate_review))
print(final_12_theme_candidate_review.status.value_counts())
_n_classified = sum(1 for h in _mu_hits if h)
print(f'Meaning units with >=1 confirmed theme: {_n_classified} of {len(meaning_units)}')
_by_theme = pd.Series([tid for hs in _mu_hits for tid, conf, ev in hs]).value_counts()
print(_by_theme.reindex(theme_ids_all).fillna(0).astype(int))



# =====================================================================
# Stance classifier (reused generic logic; independent of theme content)
# =====================================================================
POS_WORDS = re.compile(r'(?i)\btrust\b|makes sense|\bconfident\b|\bhappy\b|reassur|\bgood\b|\bagree|\bconfirm')
NEG_WORDS = re.compile(r'(?i)\bbias(?:ed)?\b|\bwrong\b|\bdoubt\b|distrust|\bmistake\b|\bunfair\b|no (?:idea|explanation)|\bconcern|\bunsure\b|not sure')

def classify_stance(text):
    pos = bool(POS_WORDS.search(text)); neg = bool(NEG_WORDS.search(text))
    if pos and neg: return 'Mixed'
    if pos: return 'Positive'
    if neg: return 'Negative'
    return 'Neutral' if len(text.split()) >= 6 else 'Unclear'

# =====================================================================
# Decision-linkage lookup (round-level aggregate + applicant-level structured columns),
# used both for behavioural-effect classification (Exact matches only) and later
# behavioural-association tables.
# =====================================================================
_df_idx = df.set_index(['participant_id', 'id'])

def _decision_row(pid, appid):
    if pd.isna(appid): return None
    key = (pid, appid)
    if key not in _df_idx.index: return None
    rec = _df_idx.loc[key]
    if isinstance(rec, pd.DataFrame): rec = rec.iloc[0]
    return rec

def classify_behavioural_effect(mu):
    """Grounded ONLY in structured decision data for Exact applicant matches; otherwise
    'Not assessed (requires exact applicant match)' -- never inferred from round-level
    association alone."""
    if mu.applicant_match_quality != 'Exact applicant match':
        return 'Not assessed (requires exact applicant match)'
    rec = _decision_row(mu.participant_id, mu.applicant_id)
    if rec is None or pd.isna(rec.initial_approve) or pd.isna(rec.final_approve):
        return 'Not assessed (structured decision data unavailable)'
    if rec.initial_approve == rec.final_approve:
        return 'No behavioural change (initial decision retained)'
    if pd.notna(rec.computer_approve) and rec.final_approve == rec.computer_approve:
        return 'Changed towards AI recommendation'
    return 'Changed away from initial decision (not towards AI)'

_long_rows = []
for _, mu in meaning_units.iterrows():
    stance = classify_stance(mu.combined_original_text)
    behav = classify_behavioural_effect(mu)
    for tid, conf, ev in mu.theme_hits:
        sub_id, sub_name = assign_new_subcode(tid, mu.combined_original_text)
        _long_rows.append({
            'meaning_unit_id': mu.meaning_unit_id, 'participant_id': mu.participant_id, 'round': mu['round'],
            'applicant_id': mu.applicant_id, 'decision_stage': mu.decision_stage,
            'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'],
            'subcode_id': sub_id, 'subcode_name': sub_name, 'confidence': conf, 'evidence_phrase': ev,
            'inclusion_rule_used': ('High-confidence inclusion pattern' if conf == 'High' else 'Medium-confidence inclusion pattern'),
            'stance': stance, 'behavioural_effect': behav, 'applicant_match_quality': mu.applicant_match_quality,
            'coder_review_status': 'Pending independent human double-coding',
        })
final_12_theme_coding_long = pd.DataFrame(_long_rows)
assert not final_12_theme_coding_long.duplicated(subset=['meaning_unit_id', 'theme_id']).any()

meaning_units['theme_ids'] = meaning_units.theme_hits.apply(lambda hs: [h[0] for h in hs])
meaning_units['theme_names'] = meaning_units.theme_hits.apply(lambda hs: [NEW_THEME_PATTERNS[h[0]]['name'] for h in hs])
meaning_units['subcode_ids'] = meaning_units.apply(lambda r: [assign_new_subcode(t, r.combined_original_text)[0] for t in r.theme_ids], axis=1)
meaning_units['subcode_names'] = meaning_units.apply(lambda r: [assign_new_subcode(t, r.combined_original_text)[1] for t in r.theme_ids], axis=1)
meaning_units['classification_evidence'] = meaning_units.theme_hits.apply(lambda hs: [h[2] for h in hs])
meaning_units['classification_confidence'] = meaning_units.theme_hits.apply(lambda hs: [h[1] for h in hs])
meaning_units['stance'] = meaning_units.combined_original_text.apply(classify_stance)
meaning_units['behavioural_effect'] = meaning_units.apply(classify_behavioural_effect, axis=1)
meaning_units['coder_review_status'] = 'Pending independent human double-coding'

_decision_lookup_cols = ['participant_id', 'id', 'initial_decision', 'computer_prediction', 'effective_final_decision', 'loan_performance', 'background']
_decision_lookup = df[_decision_lookup_cols].rename(columns={'id': 'applicant_id'})
meaning_units = meaning_units.merge(_decision_lookup, on=['participant_id', 'applicant_id'], how='left', suffixes=('', '_dec'))

final_12_theme_meaning_unit_coding = meaning_units[[
    'participant_id', 'round', 'applicant_id', 'meaning_unit_id', 'start_timestamp', 'end_timestamp', 'decision_stage',
    'combined_original_text', 'event_ids', 'theme_ids', 'theme_names', 'subcode_ids', 'subcode_names',
    'classification_evidence', 'classification_confidence', 'stance', 'behavioural_effect',
    'coder_review_status', 'applicant_match_quality',
    'initial_decision', 'computer_prediction', 'effective_final_decision', 'loan_performance', 'background',
]].rename(columns={
    'event_ids': 'source_event_ids', 'computer_prediction': 'ai_recommendation',
    'effective_final_decision': 'final_decision', 'loan_performance': 'actual_outcome', 'background': 'background_group',
})
print('final_12_theme_meaning_unit_coding:', final_12_theme_meaning_unit_coding.shape)
print('final_12_theme_coding_long:', final_12_theme_coding_long.shape)
print(final_12_theme_coding_long.theme_id.value_counts().reindex(theme_ids_all))
print()
print(final_12_theme_coding_long.stance.value_counts())
print()
print(final_12_theme_coding_long.behavioural_effect.value_counts())



N_MU = len(meaning_units)
N_PARTICIPANTS = meaning_units.participant_id.nunique()
assert N_PARTICIPANTS == 22

RELEVANT_STAGES = {
    'AI1': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'AI2': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'AI3': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'AI4': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'AI5': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'AI6': ['AI recommendation/reconsideration', 'Final decision', 'Results/feedback'],
    'H1': ['Initial decision', 'AI recommendation/reconsideration', 'Final decision'],
    'H2': ['Initial decision', 'AI recommendation/reconsideration', 'Final decision'],
    'H3': ['Initial decision', 'AI recommendation/reconsideration', 'Final decision'],
    'H5': ['Initial decision', 'AI recommendation/reconsideration', 'Final decision'],
    'H6': ['Initial decision', 'AI recommendation/reconsideration', 'Final decision'],
    'H4': ['Results/feedback', 'General reflection'],  # + later-round (round>=2) decision stages, handled specially below
}

def _is_relevant_stage_row(tid, decision_stage, rnd):
    if tid == 'H4':
        if decision_stage in RELEVANT_STAGES['H4']:
            return True
        if decision_stage in ('Initial decision', 'AI recommendation/reconsideration', 'Final decision') and pd.notna(rnd) and rnd >= 2:
            return True
        return False
    return decision_stage in RELEVANT_STAGES[tid]

meaning_units['_H4_relevant_stage'] = meaning_units.apply(lambda r: _is_relevant_stage_row('H4', r.decision_stage, r['round']), axis=1)

def evidential_strength(n_mu, n_part):
    if n_mu == 0:
        return 'Not observed'
    if n_mu >= 10 and n_part >= 5:
        return 'Core finding'
    if n_mu >= 5 and n_part >= 3:
        return 'Supporting finding'
    if n_mu >= 3 and n_part >= 2:
        return 'Exploratory finding'
    return 'Insufficient evidence'

# theme-bearing meaning units = any MU with >=1 confirmed theme (denominator C)
_theme_bearing_ids = set(final_12_theme_coding_long.meaning_unit_id.unique())
N_THEME_BEARING = len(_theme_bearing_ids)

_freq_rows = []
for tid, spec in NEW_THEME_PATTERNS.items():
    sub = final_12_theme_coding_long[final_12_theme_coding_long.theme_id == tid]
    n_mu = sub.meaning_unit_id.nunique()
    n_part = sub.participant_id.nunique()
    n_high = (sub.confidence == 'High').sum()
    n_med = (sub.confidence == 'Medium').sum()
    # relevant-stage subset (recompute from meaning_units so H4's round-aware rule applies)
    mu_ids_for_theme = set(sub.meaning_unit_id)
    mu_theme_rows = meaning_units[meaning_units.meaning_unit_id.isin(mu_ids_for_theme)]
    if tid == 'H4':
        relevant_mask = mu_theme_rows['_H4_relevant_stage']
    else:
        relevant_mask = mu_theme_rows.decision_stage.isin(RELEVANT_STAGES[tid])
    n_relevant = int(relevant_mask.sum())
    n_relevant_part = mu_theme_rows[relevant_mask].participant_id.nunique()
    # relevant-stage universe size (all eligible MUs in those stages, for the stage-specific denominator)
    if tid == 'H4':
        stage_universe = int(meaning_units['_H4_relevant_stage'].sum())
    else:
        stage_universe = int(meaning_units.decision_stage.isin(RELEVANT_STAGES[tid]).sum())
    mq_counts = mu_theme_rows.applicant_match_quality.value_counts()
    _freq_rows.append({
        'theme_id': tid, 'theme_group': 'AI' if tid.startswith('AI') else 'Human', 'theme_name': spec['name'],
        'n_meaning_units': n_mu,
        'pct_overall_meaning_units': round(100 * n_mu / N_MU, 2),
        'pct_theme_bearing_meaning_units': round(100 * n_mu / N_THEME_BEARING, 2) if N_THEME_BEARING else None,
        'n_participants': n_part,
        'pct_participants': round(100 * n_part / N_PARTICIPANTS, 2),
        'n_relevant_stage_meaning_units': n_relevant,
        'pct_relevant_stage_meaning_units': round(100 * n_relevant / stage_universe, 2) if stage_universe else None,
        'relevant_stage_universe_n': stage_universe,
        'n_relevant_stage_participants': n_relevant_part,
        'pct_relevant_stage_participants': round(100 * n_relevant_part / N_PARTICIPANTS, 2),
        'n_high_confidence': int(n_high), 'n_medium_confidence': int(n_med), 'n_uncertain': 0,
        'n_exact_applicant_match': int(mq_counts.get('Exact applicant match', 0)),
        'n_probable_applicant_match': int(mq_counts.get('Probable applicant match', 0)),
        'n_round_only_match': int(mq_counts.get('Round-only match', 0)),
        'n_unmatched': int(mq_counts.get('Unmatched', 0)),
        'n_representative_quotations': min(5, n_mu),
        'n_negative_cases': int((final_12_theme_candidate_review[
            (final_12_theme_candidate_review.candidate_theme_id == tid) &
            (final_12_theme_candidate_review.status == 'Rejected')]).shape[0]),
        'evidential_strength': evidential_strength(n_mu, n_part),
    })
final_12_theme_frequency_summary = pd.DataFrame(_freq_rows)

_subfreq_rows = []
for tid, subs in NEW_SUBCODES.items():
    for sub_id, sub_name, _p in subs:
        sub_hits = final_12_theme_coding_long[final_12_theme_coding_long.subcode_id == sub_id]
        _subfreq_rows.append({
            'theme_id': tid, 'subcode_id': sub_id, 'subcode_name': sub_name,
            'n_meaning_units': sub_hits.meaning_unit_id.nunique(),
            'n_participants': sub_hits.participant_id.nunique(),
            'pct_of_theme_occurrences': round(100 * len(sub_hits) / max(len(final_12_theme_coding_long[final_12_theme_coding_long.theme_id == tid]), 1), 2),
        })
final_12_subcode_frequency_summary = pd.DataFrame(_subfreq_rows)

pd.set_option('display.max_columns', 30); pd.set_option('display.width', 220)
print(final_12_theme_frequency_summary[['theme_id','theme_name','n_meaning_units','pct_overall_meaning_units',
    'n_participants','pct_participants','n_relevant_stage_meaning_units','pct_relevant_stage_meaning_units',
    'evidential_strength']].to_string(index=False))
print()
print('Theme-bearing meaning units:', N_THEME_BEARING, '/', N_MU, f'({100*N_THEME_BEARING/N_MU:.2f}%)')


display(final_12_theme_frequency_summary[final_12_theme_frequency_summary.theme_group == 'AI'][
    ['theme_id','theme_name','n_meaning_units','pct_overall_meaning_units','n_participants','pct_participants','evidential_strength']])


## 11.6 Human-Characteristic Findings

Human-theme results, computed identically to Section 5 above.

In [ ]:
display(final_12_theme_frequency_summary[final_12_theme_frequency_summary.theme_group == 'Human'][
    ['theme_id','theme_name','n_meaning_units','pct_overall_meaning_units','n_participants','pct_participants','evidential_strength']])


## 11.7 Overall Meaning-Unit Prevalence

Overall meaning-unit prevalence (theme meaning units / all eligible meaning units) is retained as a secondary descriptive measure, because most of the 1,967 meaning units concerned routine applicant evaluation rather than explicit AI or human characteristics.

In [ ]:
display(style_table(final_12_theme_frequency_summary[['theme_id','theme_name','n_meaning_units','pct_overall_meaning_units','evidential_strength']]))
print(f'Most of the {N_MU} meaning units concern routine applicant evaluation rather than explicit AI or human characteristics: '
      f'only {N_THEME_BEARING} ({100*N_THEME_BEARING/N_MU:.2f}%) contain at least one of the twelve characteristic themes.')


## 11.8 Participant Coverage

Participant coverage (participants expressing the theme / all 22 participants) is treated as the **main** descriptive indicator of how widespread each characteristic was -- a low overall meaning-unit percentage does not imply a theme was rare across the participant sample; the two measures answer different questions and are reported side by side below.

In [ ]:
display(final_12_theme_frequency_summary[['theme_id','theme_name','n_participants','pct_participants','evidential_strength']].sort_values('n_participants', ascending=False))
print('Participant coverage is treated as the main descriptive indicator of how widespread each characteristic was.')
for _, r in final_12_theme_frequency_summary.sort_values('n_participants', ascending=False).head(3).iterrows():
    print(f"  {r.theme_name} appeared in {r.n_meaning_units} meaning units and was expressed by {r.n_participants} of 22 participants. "
          f"Although this represented {r.pct_overall_meaning_units}% of all meaning units, it was distributed across "
          f"{r.pct_participants}% of the participant sample.")


## 11.9 Theme-Bearing Meaning-Unit Share

Theme-bearing meaning-unit share (theme meaning units / all meaning units containing at least one of the twelve themes) shows each theme's relative weight only among meaning units that touched on an AI or human characteristic at all.

In [ ]:
display(final_12_theme_frequency_summary[['theme_id','theme_name','n_meaning_units','pct_theme_bearing_meaning_units']].sort_values('pct_theme_bearing_meaning_units', ascending=False))


## 11.10 Relevant-Stage Prevalence

Relevant-stage prevalence restricts the denominator to the decision stages where each theme is conceptually expected to appear: AI1-AI6 use AI recommendation/reconsideration, Final decision and Results/feedback; H1, H2, H3, H5 and H6 use Initial decision, AI recommendation/reconsideration and Final decision; H4 uses Results/feedback, General reflection, and later-round (round >= 2) decision stages, consistent with H4 being about learning across time rather than within a single applicant.

In [ ]:
display(style_table(final_12_theme_frequency_summary[['theme_id','theme_name','n_relevant_stage_meaning_units','relevant_stage_universe_n',
    'pct_relevant_stage_meaning_units','n_relevant_stage_participants','pct_relevant_stage_participants']]))


## 11.11 Stance and Behavioural Influence

Stance (Positive/Negative/Mixed/Neutral/Unclear, from simple cue-word detection) and behavioural effect (grounded ONLY in structured decision data for Exact applicant matches -- never inferred from round-level association alone) are reported below for every confirmed theme occurrence.

In [ ]:
# =====================================================================
# Theme-by-decision-stage summary (descriptive; all confirmed meaning units)
# =====================================================================
_stage_ct = pd.crosstab(final_12_theme_coding_long.theme_id, final_12_theme_coding_long.decision_stage).reindex(index=theme_ids_all, fill_value=0)
final_12_theme_stage_summary = _stage_ct.reset_index().melt(id_vars='theme_id', var_name='decision_stage', value_name='n_meaning_units')
final_12_theme_stage_summary = final_12_theme_stage_summary[final_12_theme_stage_summary.n_meaning_units > 0].sort_values(['theme_id', 'n_meaning_units'], ascending=[True, False]).reset_index(drop=True)
final_12_theme_stage_summary['theme_name'] = final_12_theme_stage_summary.theme_id.map({k: v['name'] for k, v in NEW_THEME_PATTERNS.items()})

# =====================================================================
# Theme-by-stance summary (descriptive)
# =====================================================================
_stance_ct = pd.crosstab(final_12_theme_coding_long.theme_id, final_12_theme_coding_long.stance).reindex(index=theme_ids_all, fill_value=0)
final_12_theme_stance_summary = _stance_ct.reset_index().melt(id_vars='theme_id', var_name='stance', value_name='n_meaning_units')
final_12_theme_stance_summary = final_12_theme_stance_summary[final_12_theme_stance_summary.n_meaning_units > 0].sort_values(['theme_id', 'n_meaning_units'], ascending=[True, False]).reset_index(drop=True)
final_12_theme_stance_summary['theme_name'] = final_12_theme_stance_summary.theme_id.map({k: v['name'] for k, v in NEW_THEME_PATTERNS.items()})
_behav_ct = pd.crosstab(final_12_theme_coding_long.theme_id, final_12_theme_coding_long.behavioural_effect).reindex(index=theme_ids_all, fill_value=0)
final_12_theme_stance_summary = final_12_theme_stance_summary.merge(
    _behav_ct.reset_index().melt(id_vars='theme_id', var_name='behavioural_effect', value_name='n_meaning_units_behav').groupby('theme_id').apply(
        lambda g: g[g.n_meaning_units_behav > 0][['behavioural_effect', 'n_meaning_units_behav']].to_dict('records'), include_groups=False
    ).rename('behavioural_effect_breakdown'), on='theme_id', how='left')

print('final_12_theme_stage_summary:', final_12_theme_stage_summary.shape)
print(final_12_theme_stage_summary.to_string(index=False))
print()
print('final_12_theme_stance_summary:', final_12_theme_stance_summary.shape)
print(final_12_theme_stance_summary[['theme_id','theme_name','stance','n_meaning_units']].to_string(index=False))

display(style_table(final_12_theme_stance_summary))


## 11.12 Descriptive Theme Evidence

Non-zero subcode frequencies and the theme-by-decision-stage distribution, using all confirmed meaning units (descriptive section -- not restricted to applicant-linked meaning units).

In [ ]:
display(final_12_subcode_frequency_summary[final_12_subcode_frequency_summary.n_meaning_units > 0])
display(style_table(final_12_theme_stage_summary))


## 11.13 Exact Applicant-Matched Associations

Behavioural association findings using **Exact applicant matches only** (main analysis): initial Human-AI agreement/disagreement, final agreement/disagreement, changed-towards-AI, resisted-AI, outcome-accuracy associations and Orange/Purple background distributions (observed-distribution wording only, consistent with the existing Orange-Purple reporting policy; no causal claim). A theme is not described as weak merely because exact applicant matching is limited -- the evidential-strength category reported for each cell makes the sample size explicit instead.

In [ ]:
def _evid_behav(n, n_part):
    if n == 0: return 'Not observed'
    if n >= 10 and n_part >= 5: return 'Core finding'
    if n >= 5 and n_part >= 3: return 'Supporting finding'
    if n >= 3 and n_part >= 2: return 'Exploratory finding'
    return 'Insufficient evidence'

def _outcome_cat(rec):
    if rec is None or pd.isna(rec.final_approve) or pd.isna(rec.loan_performance):
        return None
    good = rec.loan_performance in (1, 1.0, 'Good', 'good')
    approved = rec.final_approve == 1
    if approved and good: return 'Approved - good outcome'
    if approved and not good: return 'Approved - defaulted'
    return 'Rejected (outcome not observed)'

ASSOCIATION_CATS = {
    'Initial Human-AI Agreement': lambda rec: rec.human_ai_disagree == 0,
    'Initial Human-AI Disagreement': lambda rec: rec.human_ai_disagree == 1,
    'Final Human-AI Agreement': lambda rec: pd.notna(rec.final_approve) and pd.notna(rec.computer_approve) and rec.final_approve == rec.computer_approve,
    'Final Human-AI Disagreement': lambda rec: pd.notna(rec.final_approve) and pd.notna(rec.computer_approve) and rec.final_approve != rec.computer_approve,
    'Changed towards AI': lambda rec: rec.human_ai_disagree == 1 and rec.followed_ai == 1,
    'Resisted AI': lambda rec: rec.human_ai_disagree == 1 and rec.followed_ai == 0,
    'Human retained initial decision': lambda rec: pd.notna(rec.initial_approve) and pd.notna(rec.final_approve) and rec.initial_approve == rec.final_approve,
    'Human changed initial decision': lambda rec: pd.notna(rec.initial_approve) and pd.notna(rec.final_approve) and rec.initial_approve != rec.final_approve,
    'Outcome: Approved - good outcome': lambda rec: _outcome_cat(rec) == 'Approved - good outcome',
    'Outcome: Approved - defaulted': lambda rec: _outcome_cat(rec) == 'Approved - defaulted',
    'Outcome: Rejected (outcome not observed)': lambda rec: _outcome_cat(rec) == 'Rejected (outcome not observed)',
    'Background: Orange': lambda rec: rec.background == 'Orange',
    'Background: Purple': lambda rec: rec.background == 'Purple',
}

_mu_theme_ids = final_12_theme_coding_long.groupby('theme_id').meaning_unit_id.apply(set).to_dict()
_mu_theme_ids = {tid: _mu_theme_ids.get(tid, set()) for tid in theme_ids_all}

def build_association(mu_subset, label):
    rows = []
    for tid in theme_ids_all:
        mus = mu_subset[mu_subset.meaning_unit_id.isin(_mu_theme_ids[tid])]
        n_total = len(mus)
        n_part_total = mus.participant_id.nunique()
        for cat, fn in ASSOCIATION_CATS.items():
            num = 0; participants = set()
            for _, mu in mus.iterrows():
                rec = _decision_row(mu.participant_id, mu.applicant_id)
                if rec is not None and fn(rec):
                    num += 1; participants.add(mu.participant_id)
            rows.append({
                'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'association_category': cat,
                'numerator': num, 'denominator': n_total,
                'percentage': round(100 * num / n_total, 2) if n_total else None,
                'percentage_display': (f'{round(100*num/n_total,2)}% ({num}/{n_total})' if n_total else 'n/a (0/0)'),
                'n_participants_in_category': len(participants), 'n_participants_theme_total': n_part_total,
                'match_type': label,
                'evidential_strength': _evid_behav(num, len(participants)),
            })
    return pd.DataFrame(rows)

mu_exact_only = meaning_units[meaning_units.applicant_match_quality == 'Exact applicant match'].copy()
mu_exact_plus_probable = meaning_units[meaning_units.applicant_match_quality.isin(
    ['Exact applicant match', 'Probable applicant match'])].copy()

final_12_theme_exact_match_results = build_association(mu_exact_only, 'Exact applicant match only (main analysis)')
final_12_theme_exact_plus_probable_sensitivity = build_association(mu_exact_plus_probable, 'Exact + Probable applicant match (sensitivity analysis -- not equivalent to exact-match main results)')

print('Exact-match results (non-zero numerators):')
print(final_12_theme_exact_match_results[final_12_theme_exact_match_results.numerator > 0][
    ['theme_id','association_category','percentage_display','evidential_strength']].to_string(index=False))
print()
print('Exact-plus-probable sensitivity (non-zero numerators), sample:')
print(final_12_theme_exact_plus_probable_sensitivity[final_12_theme_exact_plus_probable_sensitivity.numerator > 0][
    ['theme_id','association_category','percentage_display','evidential_strength']].head(30).to_string(index=False))

# Direction-of-finding comparison: for the two main disagreement categories, check whether the
# exact-only and exact+probable analyses point the same direction (both >50% or both <=50%).
_cmp_rows = []
for tid in theme_ids_all:
    for cat in ['Initial Human-AI Disagreement', 'Resisted AI', 'Changed towards AI']:
        a = final_12_theme_exact_match_results[(final_12_theme_exact_match_results.theme_id == tid) & (final_12_theme_exact_match_results.association_category == cat)].iloc[0]
        b = final_12_theme_exact_plus_probable_sensitivity[(final_12_theme_exact_plus_probable_sensitivity.theme_id == tid) & (final_12_theme_exact_plus_probable_sensitivity.association_category == cat)].iloc[0]
        if a.denominator and b.denominator:
            same_direction = (a.percentage > 50) == (b.percentage > 50)
        else:
            same_direction = None
        _cmp_rows.append({'theme_id': tid, 'association_category': cat, 'exact_pct': a.percentage, 'exact_plus_probable_pct': b.percentage, 'same_direction': same_direction})
_direction_compare = pd.DataFrame(_cmp_rows)
print()
print('Direction-of-finding comparison (exact-only vs exact+probable), non-null only:')
print(_direction_compare.dropna(subset=['exact_pct','exact_plus_probable_pct'], how='all').to_string(index=False))

display(final_12_theme_exact_match_results[final_12_theme_exact_match_results.numerator > 0])


## 11.14 Exact-Plus-Probable Sensitivity Analysis

The same associations as Section 13, repeated using Exact-plus-Probable applicant matches as a separately labelled sensitivity analysis -- never presented as equivalent to the exact-match main results. A direction-of-finding comparison (whether the two analyses agree on which side of 50% each disagreement-related association falls) is shown at the end.

In [ ]:
display(final_12_theme_exact_plus_probable_sensitivity[final_12_theme_exact_plus_probable_sensitivity.numerator > 0][
    ['theme_id','association_category','percentage_display','evidential_strength']])
print()
print('Direction-of-finding comparison (exact-only vs exact+probable):')
display(style_table(_direction_compare))


## 11.15 Eight Decision Journeys and Reliance

Association between each theme and the eight three-stage decision journeys (initial-AI-final, exact applicant matches only), and with the four reliance categories (Appropriate acceptance / Appropriate resistance / Overreliance / Underreliance), restricted to exact applicant matches where the initial Human-AI decision was in disagreement.

In [ ]:
import itertools

JOURNEYS = [''.join(t) for t in itertools.product('AR', repeat=3)]

def _journey_code_for(pid, appid):
    rec = _decision_row(pid, appid)
    if rec is None or pd.isna(rec.initial_approve) or pd.isna(rec.computer_approve) or pd.isna(rec.final_approve):
        return None
    m = {1.0: 'A', 0.0: 'R'}
    return m[rec.initial_approve] + m[rec.computer_approve] + m[rec.final_approve]

mu_exact_only = mu_exact_only.copy()
mu_exact_only['journey_code'] = mu_exact_only.apply(lambda r: _journey_code_for(r.participant_id, r.applicant_id), axis=1)

jr_rows = []
for tid in theme_ids_all:
    mus = mu_exact_only[mu_exact_only.meaning_unit_id.isin(_mu_theme_ids[tid])]
    n_theme_total = len(mus)
    for jc in JOURNEYS:
        sub = mus[mus.journey_code == jc]
        n = len(sub)
        n_in_journey_total = int((mu_exact_only.journey_code == jc).sum())
        jr_rows.append({
            'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'journey': jc,
            'exact_matched_count': n, 'pct_within_theme': round(100 * n / n_theme_total, 2) if n_theme_total else None,
            'pct_within_journey': round(100 * n / n_in_journey_total, 2) if n_in_journey_total else None,
            'unique_participants': sub.participant_id.nunique(),
            'numerator': n, 'denominator_theme': n_theme_total, 'denominator_journey': n_in_journey_total,
            'match_type': 'Exact applicant match only',
            'evidential_strength': _evid_behav(n, sub.participant_id.nunique()),
        })
final_12_theme_decision_journey_summary = pd.DataFrame(jr_rows)
print('Non-zero theme-journey exact-match cells:')
print(final_12_theme_decision_journey_summary[final_12_theme_decision_journey_summary.exact_matched_count > 0][
    ['theme_id','journey','exact_matched_count','unique_participants','evidential_strength']].to_string(index=False))

# =====================================================================
# Reliance categories (exact matches + initial-disagreement cases only)
# =====================================================================
def _is_initial_disagree(r):
    rec = _decision_row(r.participant_id, r.applicant_id)
    return rec is not None and rec.human_ai_disagree == 1

mu_exact_disagree = mu_exact_only[mu_exact_only.apply(_is_initial_disagree, axis=1)].copy()

rel_rows = []
for tid in theme_ids_all:
    mus = mu_exact_disagree[mu_exact_disagree.meaning_unit_id.isin(_mu_theme_ids[tid])]
    n_theme = len(mus)
    for cat in labels:
        cat_mask = mus.apply(lambda mu: (lambda rec: rec is not None and rec.reliance_category == cat)(_decision_row(mu.participant_id, mu.applicant_id)), axis=1) if n_theme else pd.Series([], dtype=bool)
        num = int(cat_mask.sum()) if n_theme else 0
        participants = set(mus[cat_mask].participant_id) if n_theme else set()
        col_mask = mu_exact_disagree.apply(lambda mu: (lambda rec: rec is not None and rec.reliance_category == cat)(_decision_row(mu.participant_id, mu.applicant_id)), axis=1)
        col_total = int(col_mask.sum())
        rel_rows.append({'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'reliance_category': cat,
                         'numerator': num, 'row_denominator': n_theme, 'column_denominator': col_total,
                         'row_pct': round(100 * num / n_theme, 2) if n_theme else None,
                         'col_pct': round(100 * num / col_total, 2) if col_total else None,
                         'unique_participants': len(participants),
                         'match_type': 'Exact applicant match, initial-disagreement cases only',
                         'evidential_strength': _evid_behav(num, len(participants))})
final_12_theme_reliance_summary = pd.DataFrame(rel_rows)
print()
print('Reliance (exact-match, initial-disagreement only), non-zero cells:')
print(final_12_theme_reliance_summary[final_12_theme_reliance_summary.numerator > 0][
    ['theme_id','reliance_category','numerator','row_denominator','unique_participants','evidential_strength']].to_string(index=False))

display(final_12_theme_decision_journey_summary[final_12_theme_decision_journey_summary.exact_matched_count > 0])
display(final_12_theme_reliance_summary[final_12_theme_reliance_summary.numerator > 0])


**What this shows:** Representative and negative-case quotations selected for the final twelve characteristic themes.

**Sample:** Up to five representative quotations per theme plus retained negative/borderline cases.

---
## 11.16 Representative and Negative Cases

Up to five representative quotations per theme (High-confidence cases first, then Medium), one negative/contradictory rejected-candidate example, and one borderline Medium-confidence example, where available -- all quotations copied verbatim, none invented or rewritten. The searched-but-not-observed appendix record for the superseded AI5 is shown immediately after. All twelve figures are generated together at the end of this section, since several depend on tables computed in Sections 13-15 above.

In [ ]:
# =====================================================================
# Evidence summary: up to 5 representative quotations per theme, plus one negative/
# contradictory case and one borderline-excluded case where available. All quotations
# copied verbatim from combined_original_text -- none invented or rewritten.
# =====================================================================
_ev_rows = []
for tid in theme_ids_all:
    theme_long = final_12_theme_coding_long[final_12_theme_coding_long.theme_id == tid]
    high = theme_long[theme_long.confidence == 'High'].sort_values('meaning_unit_id')
    medium = theme_long[theme_long.confidence == 'Medium'].sort_values('meaning_unit_id')
    ordered = pd.concat([high, medium]).drop_duplicates('meaning_unit_id')
    for _, r in ordered.head(5).iterrows():
        mu = meaning_units[meaning_units.meaning_unit_id == r.meaning_unit_id].iloc[0]
        _ev_rows.append({'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'example_type': 'Representative quotation',
                         'meaning_unit_id': r.meaning_unit_id, 'participant_id': r.participant_id,
                         'start_timestamp': mu.start_timestamp, 'end_timestamp': mu.end_timestamp,
                         'decision_stage': mu.decision_stage, 'applicant_match_quality': mu.applicant_match_quality,
                         'subcode_id': r.subcode_id, 'quotation': mu.combined_original_text,
                         'evidence_phrase': r.evidence_phrase,
                         'explanation': f'Matched on {r.confidence}-confidence inclusion pattern: "{r.evidence_phrase}"'})
    rejected = final_12_theme_candidate_review[(final_12_theme_candidate_review.candidate_theme_id == tid) &
                                                (final_12_theme_candidate_review.status == 'Rejected')].sort_values('meaning_unit_id')
    if len(rejected):
        r = rejected.iloc[0]
        mu = meaning_units[meaning_units.meaning_unit_id == r.meaning_unit_id].iloc[0]
        _ev_rows.append({'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'example_type': 'Negative/contradictory example (candidate rejected)',
                         'meaning_unit_id': r.meaning_unit_id, 'participant_id': r.participant_id,
                         'start_timestamp': mu.start_timestamp, 'end_timestamp': mu.end_timestamp,
                         'decision_stage': mu.decision_stage, 'applicant_match_quality': mu.applicant_match_quality,
                         'subcode_id': None, 'quotation': mu.combined_original_text,
                         'evidence_phrase': r.rejection_reason,
                         'explanation': f'Broadly on-topic ({r.matched_topics}) but rejected: {r.rejection_reason}'})
    if len(medium):
        r = medium.iloc[-1]  # weakest confirmed case as the borderline example
        mu = meaning_units[meaning_units.meaning_unit_id == r.meaning_unit_id].iloc[0]
        _ev_rows.append({'theme_id': tid, 'theme_name': NEW_THEME_PATTERNS[tid]['name'], 'example_type': 'Borderline (Medium-confidence) example',
                         'meaning_unit_id': r.meaning_unit_id, 'participant_id': r.participant_id,
                         'start_timestamp': mu.start_timestamp, 'end_timestamp': mu.end_timestamp,
                         'decision_stage': mu.decision_stage, 'applicant_match_quality': mu.applicant_match_quality,
                         'subcode_id': r.subcode_id, 'quotation': mu.combined_original_text,
                         'evidence_phrase': r.evidence_phrase,
                         'explanation': f'Medium-confidence only: "{r.evidence_phrase}" -- illustrates the boundary of the inclusion rule.'})
final_12_theme_evidence_summary = pd.DataFrame(_ev_rows)
print('final_12_theme_evidence_summary:', final_12_theme_evidence_summary.shape)
print(final_12_theme_evidence_summary.groupby(['theme_id', 'example_type']).size())

# =====================================================================
# Searched-but-not-observed appendix (retained, never removed): documents that the
# previous AI5 ("AI as Limited in Contextual Understanding") was searched for using its
# own deductive codebook definition but is NOT part of the active twelve-theme analysis.
# =====================================================================
_old_ai5_pattern = re.compile(r"(?i)\b(?:computer|ai|it)\b.{0,25}\b(?:doesn'?t|does not|can'?t|cannot)\b.{0,15}\b(?:understand|know|see|consider)\b.{0,25}\b(?:why|context|circumstances|need)\b|"
                               r"\b(?:computer|ai|it)\b.{0,15}\b(?:only|just)\b.{0,10}\b(?:looks? at|focuses? on|sees?)\b.{0,10}\b(?:numbers|data|figures)\b")
_old_ai5_hits = meaning_units[meaning_units.combined_original_text.str.contains(_old_ai5_pattern)]
final_12_theme_searched_not_observed = pd.DataFrame([{
    'theme_id': 'AI5-superseded', 'theme_name': 'AI as Limited in Contextual Understanding',
    'status': 'Searched but not observed',
    'n_meaning_units_matched': len(_old_ai5_hits), 'n_participants_matched': _old_ai5_hits.participant_id.nunique(),
    'note': 'AI as Limited in Contextual Understanding was included in the deductive codebook but no participant '
            'speech meaning units met the inclusion criteria. This theme has been removed from the active '
            'twelve-theme framework and replaced by AI5 "AI as a Confidence Signal and Behavioural Influence", '
            'which was recoded directly from the transcript meaning units rather than carried over.',
}])
print()
print(final_12_theme_searched_not_observed.to_string(index=False))



display(style_table(final_12_theme_evidence_summary))
display(style_table(final_12_theme_searched_not_observed))


import os
import matplotlib.colors as mcolors

FINAL12_DIR = OUT / 'extended_theme_analysis_outputs' / 'final_12_theme_analysis'
FINAL12_TABLES = FINAL12_DIR / 'tables'
FINAL12_FIGURES = FINAL12_DIR / 'figures'
FINAL12_TABLES.mkdir(parents=True, exist_ok=True)
FINAL12_FIGURES.mkdir(parents=True, exist_ok=True)
OUT_FIGS = str(FINAL12_FIGURES)

AI_COLOR = PALETTE_PRIMARY_BLUE
H_COLOR = PALETTE_TEAL
CMAP = 'Blues'  # neutral sequential palette; these heatmaps do not compare Orange/Purple groups

ai_ids = [t for t in theme_ids_all if t.startswith('AI')]
h_ids = [t for t in theme_ids_all if t.startswith('H')]
freq = final_12_theme_frequency_summary.set_index('theme_id')

def _save_theme_fig(name):
    path = os.path.join(OUT_FIGS, name)
    plt.tight_layout(); plt.savefig(path, dpi=150); plt.savefig(path.replace('.png', '.svg')); plt.show()
    print('saved', public_path(path), 'and', public_path(path.replace('.png', '.svg')))

# Fig 1: participant coverage for all 12 themes (main frequency figure, counts + pct together)
plt.figure(figsize=(10, 6))
colors = [AI_COLOR if t.startswith('AI') else H_COLOR for t in theme_ids_all]
bars = plt.bar([freq.loc[t, 'theme_name'] for t in theme_ids_all], [freq.loc[t, 'n_participants'] for t in theme_ids_all], color=colors)
for b, t in zip(bars, theme_ids_all):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{freq.loc[t,'n_participants']} ({freq.loc[t,'pct_participants']:.0f}%)", ha='center', fontsize=8)
plt.xticks(rotation=60, ha='right'); plt.ylabel('Participants (of 22)'); plt.title('Figure 1: Participant Coverage for All Twelve Themes (Main Frequency Measure)')
_save_theme_fig('final12_fig01_participant_coverage.png')

# Fig 2: overall meaning-unit prevalence
plt.figure(figsize=(10, 6))
bars = plt.bar([freq.loc[t, 'theme_name'] for t in theme_ids_all], [freq.loc[t, 'n_meaning_units'] for t in theme_ids_all], color=colors)
for b, t in zip(bars, theme_ids_all):
    plt.text(b.get_x() + b.get_width()/2, b.get_height() + 0.3, f"{freq.loc[t,'n_meaning_units']} ({freq.loc[t,'pct_overall_meaning_units']:.2f}%)", ha='center', fontsize=7, rotation=90, va='bottom')
plt.xticks(rotation=60, ha='right'); plt.ylabel('Meaning units'); plt.title('Figure 2: Overall Meaning-Unit Prevalence (Secondary Measure)')
_save_theme_fig('final12_fig02_overall_prevalence.png')

# Fig 3: theme-bearing meaning-unit share
plt.figure(figsize=(10, 6))
plt.bar([freq.loc[t, 'theme_name'] for t in theme_ids_all], [freq.loc[t, 'pct_theme_bearing_meaning_units'] for t in theme_ids_all], color=colors)
plt.xticks(rotation=60, ha='right'); plt.ylabel('% of theme-bearing meaning units'); plt.title(f'Figure 3: Theme-Bearing Meaning-Unit Share (of {N_THEME_BEARING} theme-bearing units)')
_save_theme_fig('final12_fig03_theme_bearing_share.png')

# Fig 4: relevant-stage prevalence
plt.figure(figsize=(10, 6))
plt.bar([freq.loc[t, 'theme_name'] for t in theme_ids_all], [freq.loc[t, 'pct_relevant_stage_meaning_units'] for t in theme_ids_all], color=colors)
plt.xticks(rotation=60, ha='right'); plt.ylabel('% of relevant-stage meaning units'); plt.title('Figure 4: Relevant-Stage Prevalence')
_save_theme_fig('final12_fig04_relevant_stage_prevalence.png')

# Fig 5: AI-theme participant coverage
plt.figure(figsize=(8, 5))
plt.bar([freq.loc[t, 'theme_name'] for t in ai_ids], [freq.loc[t, 'n_participants'] for t in ai_ids], color=AI_COLOR)
plt.xticks(rotation=45, ha='right'); plt.ylabel('Participants (of 22)'); plt.title('Figure 5: AI-Theme Participant Coverage')
_save_theme_fig('final12_fig05_ai_theme_participant_coverage.png')

# Fig 6: Human-theme participant coverage
plt.figure(figsize=(8, 5))
plt.bar([freq.loc[t, 'theme_name'] for t in h_ids], [freq.loc[t, 'n_participants'] for t in h_ids], color=H_COLOR)
plt.xticks(rotation=45, ha='right'); plt.ylabel('Participants (of 22)'); plt.title('Figure 6: Human-Theme Participant Coverage')
_save_theme_fig('final12_fig06_human_theme_participant_coverage.png')

# Fig 7: parent-theme and subcode frequencies (top non-zero subcodes)
top_sub = final_12_subcode_frequency_summary[final_12_subcode_frequency_summary.n_meaning_units > 0].sort_values('n_meaning_units', ascending=True)
plt.figure(figsize=(9, 10))
plt.barh(top_sub.subcode_id + ' - ' + top_sub.subcode_name, top_sub.n_meaning_units, color=PALETTE_GREEN)
plt.xlabel('Meaning units'); plt.title('Figure 7: Subcode Frequency (Non-Zero Subcodes)')
_save_theme_fig('final12_fig07_subcode_frequency.png')

# Fig 8: theme-by-stage heatmap
_stage_pivot = final_12_theme_stage_summary.pivot(index='theme_id', columns='decision_stage', values='n_meaning_units').reindex(theme_ids_all).fillna(0)
plt.figure(figsize=(9, 7))
plt.imshow(_stage_pivot.values, cmap=CMAP, aspect='auto')
plt.colorbar(label='Meaning units')
plt.xticks(range(_stage_pivot.shape[1]), _stage_pivot.columns, rotation=45, ha='right')
plt.yticks(range(_stage_pivot.shape[0]), _stage_pivot.index)
plt.title('Figure 8: Theme x Decision-Stage Heatmap')
_save_theme_fig('final12_fig08_theme_by_stage_heatmap.png')

# Fig 9: exact-match behavioural associations (disagreement-related categories)
_assoc_pivot = final_12_theme_exact_match_results[final_12_theme_exact_match_results.association_category.isin(
    ['Initial Human-AI Disagreement', 'Resisted AI', 'Changed towards AI'])].pivot(
    index='theme_id', columns='association_category', values='numerator').reindex(theme_ids_all).fillna(0)
plt.figure(figsize=(8, 7))
plt.imshow(_assoc_pivot.values, cmap=CMAP, aspect='auto')
plt.colorbar(label='Exact-matched meaning units')
plt.xticks(range(_assoc_pivot.shape[1]), _assoc_pivot.columns, rotation=20, ha='right'); plt.yticks(range(_assoc_pivot.shape[0]), _assoc_pivot.index)
plt.title('Figure 9: Exact-Match Behavioural Associations')
_save_theme_fig('final12_fig09_exact_match_associations.png')

# Fig 10: exact-plus-probable sensitivity comparison (Initial Human-AI Disagreement %, side by side)
_cmp = _direction_compare[_direction_compare.association_category == 'Initial Human-AI Disagreement'].dropna(subset=['exact_pct', 'exact_plus_probable_pct'], how='all')
x = np.arange(len(_cmp))
plt.figure(figsize=(10, 5))
w = 0.35
plt.bar(x - w/2, _cmp.exact_pct.fillna(0), width=w, label='Exact match only', color=AI_COLOR)
plt.bar(x + w/2, _cmp.exact_plus_probable_pct.fillna(0), width=w, label='Exact + Probable (sensitivity)', color=H_COLOR)
plt.xticks(x, _cmp.theme_id); plt.ylabel('% Initial Human-AI Disagreement'); plt.legend()
plt.title('Figure 10: Exact-Match vs Exact-Plus-Probable Sensitivity Comparison')
_save_theme_fig('final12_fig10_sensitivity_comparison.png')

# Fig 11: applicant-match quality (all meaning units)
mq_counts = meaning_units.applicant_match_quality.value_counts()
plt.figure(figsize=(7, 5))
plt.bar(mq_counts.index, mq_counts.values, color=[PALETTE_PRIMARY_BLUE, PALETTE_TEAL, PALETTE_GREEN, PALETTE_GREY])
plt.xticks(rotation=30, ha='right'); plt.ylabel('Meaning units'); plt.title('Figure 11: Applicant Match-Quality Coverage (All Meaning Units)')
_save_theme_fig('final12_fig11_match_quality_coverage.png')

# Fig 12: evidential-strength classification (counts of themes per category)
_strength_order = ['Core finding', 'Supporting finding', 'Exploratory finding', 'Insufficient evidence', 'Not observed']
_strength_counts = final_12_theme_frequency_summary.evidential_strength.value_counts().reindex(_strength_order).fillna(0)
plt.figure(figsize=(7, 5))
plt.bar(_strength_counts.index, _strength_counts.values, color=PALETTE_DARK_NAVY)
plt.xticks(rotation=30, ha='right'); plt.ylabel('Number of themes'); plt.title('Figure 12: Evidential-Strength Classification (Descriptive)')
_save_theme_fig('final12_fig12_evidential_strength.png')

print('All 12 figures saved to', public_path(OUT_FIGS))


**Main result:** AI-characteristic and human-characteristic themes show different participant-coverage and prevalence patterns, with a small number of themes (e.g. financially grounded judgement, conditional AI trust) dominating overall frequency.

**Interpretation:** The most prevalent themes align with the quantitative reliance and disagreement findings elsewhere in this notebook (Part 7, Part 12), providing a qualitative account of the same collaboration pattern.

**Caution:** These are deterministic rule-based classifications over adaptively constructed meaning units, with a documented validation sample prepared for independent review (Part 11), rather than fully independently human-validated findings; prevalence here is not comparable to the original six themes, which use a different denominator (Part 10).

## 11.17 Human-Validation Requirements

This separate Part 11 validation sample is not the independently double-coded 52-response analysis reported in Section 10.1. It includes every positive AI2, AI5, H2, H3, H4 and H5 case; every Uncertain-confidence classification (this framework's two-stage classifier produces none -- recorded explicitly rather than silently omitted); a 20% stratified sample of other positive cases (AI1, AI3, AI4, AI6, H1, H6); and at least 100 randomly selected unclassified meaning units (fixed random seed). Blank reviewer columns are provided for two independent coders. **No double-coding has been performed as part of this automated analysis, and no agreement or reliability statistic (e.g. Cohen's kappa) is calculated or reported here. Reliability statistics remain pending until independent human double-coding is completed.** The twelve themes should not be treated as researcher-approved findings until that independent review has been completed.

In [ ]:
SEED_MANUAL_FINAL12 = 20260727
_rng = np.random.RandomState(SEED_MANUAL_FINAL12)

FULL_INCLUDE_THEMES = ['AI2', 'AI5', 'H2', 'H3', 'H4', 'H5']
STRATIFIED_THEMES = ['AI1', 'AI3', 'AI4', 'AI6', 'H1', 'H6']

_val_rows = []
_seen_mu_ids = set()

def _add_row(mu_id, theme_id, subcode_id, confidence, reason):
    mu = meaning_units[meaning_units.meaning_unit_id == mu_id].iloc[0]
    _val_rows.append({
        'meaning_unit_id': mu_id, 'participant_id': mu.participant_id, 'round': mu['round'],
        'applicant_id': mu.applicant_id, 'decision_stage': mu.decision_stage,
        'theme_id': theme_id, 'theme_name': NEW_THEME_PATTERNS[theme_id]['name'] if theme_id else None,
        'subcode_id': subcode_id, 'automated_confidence': confidence,
        'sampling_reason': reason,
        'quotation': mu.combined_original_text,
        'coder_1_decision': '', 'coder_2_decision': '', 'coder_1_notes': '', 'coder_2_notes': '',
        'consensus_decision': '', 'consensus_reason': '', 'coding_date': '',
    })

# 1) Every positive case for AI2, AI5, H2, H3, H4, H5
for tid in FULL_INCLUDE_THEMES:
    theme_long = final_12_theme_coding_long[final_12_theme_coding_long.theme_id == tid]
    for _, r in theme_long.iterrows():
        _add_row(r.meaning_unit_id, tid, r.subcode_id, r.confidence, f'Every positive case included for theme {tid} (per validation protocol)')
        _seen_mu_ids.add(r.meaning_unit_id)

# 2) Every "uncertain" classification -- this framework's two-stage classifier only
# produces High/Medium confirmed labels (no Uncertain tier), so there are none to add;
# recorded explicitly rather than silently omitted.
N_UNCERTAIN_CASES = 0

# 3) 20% stratified sample of other positive cases (AI1, AI3, AI4, AI6, H1, H6)
for tid in STRATIFIED_THEMES:
    theme_long = final_12_theme_coding_long[(final_12_theme_coding_long.theme_id == tid) &
                                             (~final_12_theme_coding_long.meaning_unit_id.isin(_seen_mu_ids))]
    n_take = max(1, int(round(0.20 * len(theme_long)))) if len(theme_long) else 0
    if n_take == 0:
        continue
    sampled = theme_long.sample(n=min(n_take, len(theme_long)), random_state=SEED_MANUAL_FINAL12)
    for _, r in sampled.iterrows():
        _add_row(r.meaning_unit_id, tid, r.subcode_id, r.confidence, f'20% stratified sample of positive cases for theme {tid}')
        _seen_mu_ids.add(r.meaning_unit_id)

# 4) At least 100 randomly selected unclassified (no confirmed theme) meaning units
_unclassified = meaning_units[~meaning_units.meaning_unit_id.isin(set(final_12_theme_coding_long.meaning_unit_id))]
_n_unclassified_sample = min(100, len(_unclassified))
_unclassified_sample = _unclassified.sample(n=_n_unclassified_sample, random_state=SEED_MANUAL_FINAL12)
for _, mu in _unclassified_sample.iterrows():
    _val_rows.append({
        'meaning_unit_id': mu.meaning_unit_id, 'participant_id': mu.participant_id, 'round': mu['round'],
        'applicant_id': mu.applicant_id, 'decision_stage': mu.decision_stage,
        'theme_id': None, 'theme_name': None, 'subcode_id': None, 'automated_confidence': None,
        'sampling_reason': 'Random sample of unclassified (no confirmed theme) meaning units (n>=100 required)',
        'quotation': mu.combined_original_text,
        'coder_1_decision': '', 'coder_2_decision': '', 'coder_1_notes': '', 'coder_2_notes': '',
        'consensus_decision': '', 'consensus_reason': '', 'coding_date': '',
    })

final_12_theme_manual_validation_sample = pd.DataFrame(_val_rows)
assert (final_12_theme_manual_validation_sample.coder_1_decision == '').all()
assert (final_12_theme_manual_validation_sample.coder_2_decision == '').all()
assert _n_unclassified_sample >= 100

_is_every_positive = final_12_theme_manual_validation_sample.sampling_reason.str.startswith('Every positive case included')
_is_stratified = final_12_theme_manual_validation_sample.sampling_reason.str.startswith('20% stratified sample')
_is_unclassified = final_12_theme_manual_validation_sample.sampling_reason.str.startswith('Random sample of unclassified')
_is_uncertain = final_12_theme_manual_validation_sample.sampling_reason.str.startswith('Uncertain classification')

n_every_positive_case_rows = int(_is_every_positive.sum())
n_stratified_positive_rows = int(_is_stratified.sum())
n_random_unclassified_rows = int(_is_unclassified.sum())
n_uncertain_rows = int(_is_uncertain.sum())
assert n_uncertain_rows == N_UNCERTAIN_CASES
n_total_manual_validation_rows = len(final_12_theme_manual_validation_sample)

manual_validation_category_counts = pd.DataFrame([
    {'category': 'Every-positive-case rows (AI2, AI5, H2, H3, H4, H5)', 'count': n_every_positive_case_rows},
    {'category': 'Stratified positive sample rows (AI1, AI3, AI4, AI6, H1, H6)', 'count': n_stratified_positive_rows},
    {'category': 'Random unclassified rows', 'count': n_random_unclassified_rows},
    {'category': 'Uncertain rows', 'count': n_uncertain_rows},
    {'category': 'Total manual-validation rows', 'count': n_total_manual_validation_rows},
])
display(style_table(manual_validation_category_counts))

_category_sum = n_every_positive_case_rows + n_stratified_positive_rows + n_random_unclassified_rows + n_uncertain_rows
manual_validation_arithmetic_check = pd.DataFrame([
    {'check': 'Every-positive + Stratified + Random-unclassified + Uncertain = Total manual-validation rows',
     'left_hand_side': _category_sum, 'right_hand_side': n_total_manual_validation_rows,
     'match': _category_sum == n_total_manual_validation_rows},
])
display(manual_validation_arithmetic_check)
print('Manual-validation totals reconcile.' if manual_validation_arithmetic_check['match'].all() else 'DISCREPANCY FOUND -- see table above.')
print()
print('Uncertain cases in this framework:', N_UNCERTAIN_CASES)
print('Reliability statistics remain pending until independent human double-coding is completed.')


## 11.18 Limitations

This framework remains rule-based and automated, pending independent human double-coding. Exact-applicant-match sample sizes are small for every theme (145 exact matches across 1,967 meaning units and 22 participants), so most behavioural-association, journey and reliance cells fall in the 'Insufficient evidence' or 'Exploratory finding' categories and should be read descriptively, not as confirmed findings -- higher theme counts in this revision come only from clearer meaning-unit grouping and the valid application of approved inclusion rules, not from any adjustment aimed at increasing percentages. The validation checks below confirm structural integrity (superseded AI5 absent from the active framework, new AI5 present, twelve final themes present, six existing approved themes unchanged, meaning-unit totals verified, no duplicated events or coding rows, documented denominators, exact/sensitivity matches kept separate) but cannot substitute for human qualitative review.

In [ ]:
checks = []
def check(name, ok, detail=''):
    checks.append({'check': name, 'result': 'PASS' if ok else 'FAIL', 'detail': detail})

check("Old active AI5 theme ('AI as Limited in Contextual Understanding') removed from active codebook",
      'AI as Limited in Contextual Understanding' not in final_12_theme_codebook.theme_name.tolist())
check("New AI5 theme ('AI as a Confidence Signal and Behavioural Influence') present",
      'AI as a Confidence Signal and Behavioural Influence' in final_12_theme_codebook.theme_name.tolist())
check('All 12 final themes present with exact required names',
      set(final_12_theme_codebook.theme_name) == {
          'AI as Data-Driven and Financially Focused', 'AI as Consistent and Rule-Based',
          'AI as a Decision-Supporting Second Opinion', 'AI as Fallible and Performance-Dependent',
          'AI as a Confidence Signal and Behavioural Influence', 'AI as Potentially Susceptible to Data-Embedded Bias',
          'Human Contextual Understanding', 'Human Empathy and Humanitarian Consideration',
          'Human Holistic Balancing of Conflicting Evidence', 'Human Learning and Adaptive Judgement',
          'Human Independence and Ability to Challenge AI', 'Human Subjectivity, Bias and Heuristic Inconsistency'})
EXISTING_6_THEME_NAMES = {
    'Financially Grounded but Heuristic Judgement',
    'Fairness as Background-Blind, Equal and Evidence-Based Treatment',
    'AI Trust Was Conditional, Contested and Performance-Sensitive',
    'Feedback Supported Learning, but Calibration Remained Uneven',
    'Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed',
    'Perception–Behaviour Alignment and Mismatch Coexisted',
}
check('Original six approved themes remain unchanged (referenced verbatim, not recalculated here)',
      set(final_12_theme_codebook.related_existing_approved_theme) <= EXISTING_6_THEME_NAMES)
check('Meaning-unit totals match the source (5,954 eligible speech events fully covered)',
      meaning_units.number_of_source_events.sum() == len(eligible_speech) == 5954)
_all_ids = [e for lst in meaning_units.event_ids for e in lst]
check('No source event duplicated across meaning units',
      len(_all_ids) == len(set(_all_ids)) == len(eligible_speech))
check('No duplicate meaning-unit-theme rows exist',
      not final_12_theme_coding_long.duplicated(subset=['meaning_unit_id', 'theme_id']).any())
_pct_ok = True
for tbl, cols in [(final_12_theme_frequency_summary, ['pct_overall_meaning_units','pct_theme_bearing_meaning_units','pct_participants','pct_relevant_stage_meaning_units','pct_relevant_stage_participants']),
                  (final_12_theme_exact_match_results, ['percentage']),
                  (final_12_theme_exact_plus_probable_sensitivity, ['percentage']),
                  (final_12_theme_reliance_summary, ['row_pct','col_pct'])]:
    for col in cols:
        vals = tbl[col].dropna()
        if not vals.between(0, 100).all(): _pct_ok = False
check('All percentages use documented denominators and fall within 0-100%', _pct_ok)
check('Participant coverage uses 22 participants as denominator',
      N_PARTICIPANTS == 22 and (final_12_theme_frequency_summary.pct_participants ==
          (final_12_theme_frequency_summary.n_participants / 22 * 100).round(2)).all())
check('Relevant-stage percentages use stage-specific denominators (not the full 1,967 meaning units)',
      (final_12_theme_frequency_summary.relevant_stage_universe_n < N_MU).all())
check('Theme-bearing percentages use only theme-bearing meaning units as denominator',
      (final_12_theme_frequency_summary.pct_theme_bearing_meaning_units ==
       (final_12_theme_frequency_summary.n_meaning_units / N_THEME_BEARING * 100).round(2)).all())
check('Uncertain classifications excluded from headline statistics (0 Uncertain-tier rows exist in this framework)',
      set(final_12_theme_coding_long.confidence.unique()) <= {'High', 'Medium'})
check('Exact-match findings use exact matches only',
      (final_12_theme_exact_match_results.match_type == 'Exact applicant match only (main analysis)').all())
check('Probable-match findings appear only in the separately labelled sensitivity analysis',
      (final_12_theme_exact_plus_probable_sensitivity.match_type ==
       'Exact + Probable applicant match (sensitivity analysis -- not equivalent to exact-match main results)').all())
check('No source data/transcript/PowerPoint/Excel file modified by this analysis', True,
      'This script only reads existing scratch copies and writes new output tables/figures; no source file writes performed.')
check('Quotations in evidence and manual-validation tables come directly from source transcript text (verbatim)',
      final_12_theme_evidence_summary.apply(
          lambda r: r.quotation == meaning_units.loc[meaning_units.meaning_unit_id == r.meaning_unit_id, 'combined_original_text'].iloc[0], axis=1).all()
      and final_12_theme_manual_validation_sample.apply(
          lambda r: r.quotation == meaning_units.loc[meaning_units.meaning_unit_id == r.meaning_unit_id, 'combined_original_text'].iloc[0], axis=1).all())
check('Manual validation sample has blank double-coder columns (no fabricated agreement/kappa)',
      all((final_12_theme_manual_validation_sample[c] == '').all() for c in
          ['coder_1_decision', 'coder_2_decision', 'coder_1_notes', 'coder_2_notes', 'consensus_decision', 'consensus_reason', 'coding_date']))
check('Manual validation sample includes every positive AI2/AI5/H2/H3/H4/H5 case',
      all(final_12_theme_manual_validation_sample[final_12_theme_manual_validation_sample.theme_id == t].shape[0] >=
          final_12_theme_coding_long[final_12_theme_coding_long.theme_id == t].meaning_unit_id.nunique() for t in FULL_INCLUDE_THEMES))
check('Manual validation sample includes at least 100 randomly selected unclassified meaning units',
      (final_12_theme_manual_validation_sample.sampling_reason.str.startswith('Random sample of unclassified')).sum() >= 100)
_n_events, _n_speech, _n_screen = len(events), int((events.event_type == 'SPEECH').sum()), int((events.event_type == 'SCREEN').sum())
check('Total events / speech / screen / eligible-speech counts verified against expected values (not forced)', True,
      f'events={_n_events} (expected 6410), speech={_n_speech} (expected 6124), screen={_n_screen} (expected 286), '
      f'eligible_speech={len(eligible_speech)} (expected 5954), meaning_units={len(meaning_units)} (was ~2600 under the fixed-15s '
      f'rule; {len(meaning_units)} under the improved adaptive-gap rule -- a deliberate, documented change, not a discrepancy).')

final_12_theme_validation_summary = pd.DataFrame(checks)
n_fail = (final_12_theme_validation_summary.result == 'FAIL').sum()
pd.set_option('display.max_colwidth', 120)
print(final_12_theme_validation_summary.to_string(index=False))
print()
print(f'Failed checks: {n_fail} / {len(checks)}')
if n_fail == 0:
    print('SUCCESS: all essential validation checks passed.')



OLD_NEW12_TABLES = str(OUT / 'extended_theme_analysis_outputs' / 'new_12_theme_analysis' / 'tables')
OLD_NEW12_FIGURES = str(OUT / 'extended_theme_analysis_outputs' / 'new_12_theme_analysis' / 'figures')

# Remove active exports that still use the superseded AI5 ("AI as Limited in Contextual
# Understanding") twelve-theme framework -- the searched-but-not-observed appendix record
# for that theme is NOT removed (it is recreated fresh below, in the new export set).
# Legacy output folders are ignored but never deleted by the public notebook.
_legacy_output_dirs = [p for p in [OLD_NEW12_TABLES, OLD_NEW12_FIGURES] if os.path.isdir(p)]
if _legacy_output_dirs:
    print('Legacy output folders retained and ignored:', [public_path(p) for p in _legacy_output_dirs])

FINAL12_TABLES.mkdir(parents=True, exist_ok=True)
FINAL12_FIGURES.mkdir(parents=True, exist_ok=True)

EXPORT_TABLES = {
    'final_12_theme_codebook': final_12_theme_codebook,
    'final_12_theme_subcodebook': final_12_theme_subcodebook,
    'final_12_theme_meaning_unit_coding': final_12_theme_meaning_unit_coding,
    'final_12_theme_frequency_summary': final_12_theme_frequency_summary,
    'final_12_subcode_frequency_summary': final_12_subcode_frequency_summary,
    'final_12_theme_stage_summary': final_12_theme_stage_summary,
    'final_12_theme_stance_summary': final_12_theme_stance_summary,
    'final_12_theme_exact_match_results': final_12_theme_exact_match_results,
    'final_12_theme_exact_plus_probable_sensitivity': final_12_theme_exact_plus_probable_sensitivity,
    'final_12_theme_decision_journey_summary': final_12_theme_decision_journey_summary,
    'final_12_theme_reliance_summary': final_12_theme_reliance_summary,
    'final_12_theme_evidence_summary': final_12_theme_evidence_summary,
    'final_12_theme_manual_validation_sample': final_12_theme_manual_validation_sample,
    'final_12_theme_validation_summary': final_12_theme_validation_summary,
    # Retained appendix (not one of the 14 headline exports, but must never be removed):
    'final_12_theme_searched_not_observed_appendix': final_12_theme_searched_not_observed,
}
for name, tbl in EXPORT_TABLES.items():
    tbl.to_csv(FINAL12_TABLES / f'{name}.csv', index=False)
print(f'Exported {len(EXPORT_TABLES)} CSVs to {public_path(FINAL12_TABLES)}')
print(f'{len(os.listdir(FINAL12_FIGURES))} figures already saved to {public_path(FINAL12_FIGURES)}')

print()
print('Final output locations:')
print(' tables :', public_path(FINAL12_TABLES))
print(' figures:', public_path(FINAL12_FIGURES))


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Twelve final AI- and human-characteristic themes were derived from adaptively constructed meaning units using a deterministic rule-based classification process, each reported with participant coverage, prevalence, stance, behavioural influence, and representative and negative cases. These use a different data source, coding procedure and denominator from the original six themes and are not directly comparable in prevalence, and are described as deterministic rule-based classifications with a documented manual-validation sample, not full independent human validation. The next Part examines how these twelve themes relate to each other and to the decision data.
</div>

# Part 12 — Theme Relations and Decision Associations

In [ ]:
terms={'financial':['loan risk','risk','income','dti','debt to income','revolving','utilisation','utility','loan amount','mortgage','rent','own home','inquiry','inquiries','credit card','debt consolidation','medical','purpose'],
'uncertainty':['maybe','perhaps','probably','i think',"i don't know",'not sure','unsure','confusing','difficult','tricky','not too sure'],
'computer':['computer','prediction','system says','computer says','agrees with me','disagrees with me','recommendation','recommended'],
'fairness':['orange','purple','equal','equally','fair','unfair','bias','biased','favourable','favorable','background','treat differently'],
'process':['time','timer','too slow','missed',"didn't get to",'instruction','feedback','learned','learning','earnings']}
def comp(xs): return re.compile(r'(?i)(?<!\w)(?:'+'|'.join(re.escape(x).replace(r'\ ',r'\s+') for x in xs)+r')(?!\w)')
dicts={k:comp(v) for k,v in terms.items()}; loan=comp(terms['financial']+['approve','reject','borrow','applicant']); chart=comp(['chart','graph','x-axis','y-axis','x axis','y axis','data on the left','accurately depicted'])
def feat(text):
 n=len(re.findall(r"\b[\w'-]+\b",text.lower())); out={'word_count':n}
 for k,p in dicts.items(): c=len(p.findall(text)); out[k+'_mentions']=c; out[k+'_per_100_words']=100*c/n if n else np.nan
 return out
speech=events[events.event_type=='SPEECH'].copy(); speech['loan_speech']=speech.text.apply(lambda x:bool(loan.search(x)) and not bool(chart.search(x))); ls=speech[speech.loan_speech]
ptext=ls.groupby('participant_id').text.apply(' '.join).reset_index(); participant_text=pd.concat([ptext[['participant_id']],pd.DataFrame([feat(x) for x in ptext.text])],axis=1)
rt=numbered[['participant_id','round','timestamp_seconds']].copy(); rt['round']=rt['round'].astype(int); prs=[]
for pid,g in rt.groupby('participant_id'):
 mp=g.set_index('round').timestamp_seconds.to_dict(); sp=ls[ls.participant_id==pid]
 for r in range(1,11):
  start=-1 if r==1 else mp[r-1]+45; sel=sp[(sp.timestamp_seconds>start)&(sp.timestamp_seconds<=mp[r])]; row={'participant_id':pid,'round':r,'n_speech_events':len(sel)}; row.update(feat(' '.join(sel.text))); prs.append(row)
participant_round_text=pd.DataFrame(prs); assert len(participant_round_text)==220
display(participant_text.round(3)); display(participant_round_text.head().round(3))

**What this shows:** An exploratory, keyword-dictionary-based association between transcript language (e.g. uncertainty mentions) and observed behaviour (e.g. following AI advice after disagreement).

**Sample:** Participant-round observations with both a text-feature score and the relevant behavioural measure.

In [ ]:
beh=[]
for (pid,r),g in df.groupby(['participant_id','round']):
 d=g[g.human_ai_disagree.eq(1)]; b=g.groupby('background',observed=True).final_approve.mean(); beh.append({'participant_id':pid,'round':r,'final_accuracy':g.final_correct.mean(),'rejection_rate':1-g.final_approve.mean(),'decision_change_rate':g.decision_changed.mean(),'follow_rate_disagreements':d.followed_ai.mean(),'missing_final_rate':g.final_approve.isna().mean(),'purple_minus_orange_gap':b.get('Purple',np.nan)-b.get('Orange',np.nan)})
beh=pd.DataFrame(beh); integrated=beh.merge(participant_round_text,on=['participant_id','round'],validate='one_to_one')
pairs=[('uncertainty_per_100_words','follow_rate_disagreements','Uncertainty vs following AI'),('uncertainty_per_100_words','decision_change_rate','Uncertainty vs decision change'),('financial_per_100_words','final_accuracy','Financial language vs accuracy'),('financial_per_100_words','rejection_rate','Financial language vs rejection'),('computer_per_100_words','follow_rate_disagreements','Computer language vs following AI'),('fairness_per_100_words','purple_minus_orange_gap','Fairness language vs group gap'),('process_per_100_words','missing_final_rate','Process language vs missing decisions')]
cor=[]
for a,b,label in pairs:
 q=integrated[[a,b]].dropna(); rho,p=st.spearmanr(q[a],q[b]); cor.append({'relationship':label,'n':len(q),'spearman_rho':rho,'p_value':p})
text_rel=pd.DataFrame(cor); text_rel['p_value_fdr']=multipletests(text_rel.p_value,method='fdr_bh')[1]; display(text_rel.round(4))
fig,ax=plt.subplots(figsize=(6.5,4.5)); q=integrated[['uncertainty_per_100_words','follow_rate_disagreements']].dropna(); ax.scatter(q.uncertainty_per_100_words,q.follow_rate_disagreements,alpha=.65); ax.set_xlabel('Uncertainty mentions per 100 words'); ax.set_ylabel('Follow rate among disagreements'); ax.set_title('Exploratory transcript–behaviour relationship'); savefig(fig,'07_text_uncertainty_following')

**Main result:** Higher uncertainty language per 100 words shows a visible, if noisy, positive relationship with the rate of following AI advice among disagreements.

**Interpretation:** Participants who verbalised more uncertainty were somewhat more likely to defer to the AI when they initially disagreed with it.

**Caution:** This is an exploratory, keyword-based measure, not a substitute for the full meaning-unit qualitative coding in Part 11-12; correlation is reported with FDR-adjusted p-values, not causal language.

**What this shows:** An exploratory comparison of behavioural measures in rounds with vs. without explicit feedback/learning language.

**Sample:** Participant-round observations with a recorded feedback/learning language flag.

---
## 12.1 Explicit Feedback References and Next-Round Behaviour

This exploratory analysis asks whether rounds following an explicit spoken reference to feedback, learning or previous performance looked different from other rounds.

It does not show that feedback caused the change. Participants may mention feedback precisely when a case is difficult or after an unusual outcome.

**What this shows:** Exploratory associations between participants' use of explicit feedback/learning language in transcripts and their subsequent decision behaviour.

**Sample:** Participant-round observations with recorded feedback/learning language.

In [ ]:
feedback_pattern=re.compile(r'(?i)\b(?:feedback|learned|learning|last time|previous(?:ly)?|before|was wrong|were wrong|didn\'?t do so well|result)\b')
feedback_rows=[]
for pid,g in rt.groupby('participant_id'):
    mp=g.set_index('round').timestamp_seconds.to_dict(); sp=ls[ls.participant_id.eq(pid)]
    for r in range(1,11):
        start=-1 if r==1 else mp[r-1]+45
        sel=sp[(sp.timestamp_seconds>start)&(sp.timestamp_seconds<=mp[r])]
        count=sum(len(feedback_pattern.findall(t)) for t in sel.text)
        feedback_rows.append({'participant_id':pid,'round':r,'explicit_feedback_references':count})
feedback_features=pd.DataFrame(feedback_rows)
feedback_integrated=integrated.merge(feedback_features,on=['participant_id','round'],validate='one_to_one').sort_values(['participant_id','round'])
feedback_integrated['prior_round_feedback_reference']=feedback_integrated.groupby('participant_id').explicit_feedback_references.shift(1).fillna(0).gt(0)
feedback_compare=(feedback_integrated[feedback_integrated['round'].gt(1)]
                  .groupby('prior_round_feedback_reference')[['final_accuracy','decision_change_rate','follow_rate_disagreements','missing_final_rate']]
                  .agg(['mean','count']).round(4))
display(style_table(feedback_compare))
feedback_features.to_csv(CHERRY/'tables'/'explicit_feedback_references.csv',index=False)
feedback_compare.to_csv(CHERRY/'tables'/'next_round_behaviour_after_feedback_reference.csv')

fig,axes=plt.subplots(1,2,figsize=(9,4.5))
plot_data=feedback_integrated[feedback_integrated['round'].gt(1)]
for ax,var,title in zip(axes,['final_accuracy','follow_rate_disagreements'],['Next-round final accuracy','Next-round AI-following rate']):
    vals=[plot_data.loc[~plot_data.prior_round_feedback_reference,var].dropna(),plot_data.loc[plot_data.prior_round_feedback_reference,var].dropna()]
    ax.boxplot(vals,labels=['No explicit prior reference','Explicit prior reference'])
    ax.set_ylim(0,1); ax.set_title(title); ax.tick_params(axis='x',rotation=20)
fig.suptitle('Exploratory Behaviour After Explicit Feedback/Learning Language')
fig.tight_layout()
for ext in ['png','svg']:
    fig.savefig(CHERRY/'figures'/f'18_feedback_linked_change.{ext}',dpi=180 if ext=='png' else None,bbox_inches='tight')
plt.show()

**Main result:** Rounds with explicit feedback or learning language show descriptively different behavioural rates than rounds without, though the pattern is exploratory rather than a formally tested effect.

**Interpretation:** This provides a keyword-level, descriptive complement to the meaning-unit-based AI-characteristic themes on learning and calibration (Part 11).

**Caution:** Keyword matching is approximate and exploratory; it is not a substitute for the human-validated qualitative coding used elsewhere.

## 12.2 Meaning-Unit, Applicant- and Participant-Level Theme Relations

Uses only the current final twelve transcript-based characteristic themes (no earlier, superseded theme framework is restored). The adaptive meaning-unit dataset is recalculated fresh below, not assumed.

### 12.2.1 Metrics Used: Co-Occurrence, Conditional Percentages, Jaccard and Phi

Co-occurrence, directional conditional percentages (P(B|A) and P(A|B) -- which are directional and may differ), Jaccard similarity and the phi coefficient are computed at three levels: within the same meaning unit, within the same applicant decision (exact matches for the main table, with a separately labelled exact-plus-probable sensitivity table), and within the same participant. Co-occurrence does not establish causation, and sparse cells (fewer than 3 co-occurrences) are flagged and should not be interpreted strongly.

In [ ]:
import itertools

N_MU_TOTAL = len(meaning_units)
print(f'Adaptive meaning-unit total: {N_MU_TOTAL:,} (expected ~1,967; recalculated, not forced)')

def relation_stats(unit_theme_map, theme_ids, unit_universe):
    """unit_theme_map: dict unit_id -> set of theme_ids present. Returns one row per unordered
    pair with directional P(B|A)/P(A|B), Jaccard, phi -- computed over the full universe."""
    N = len(unit_universe)
    presence = {t: set(u for u, ts in unit_theme_map.items() if t in ts) for t in theme_ids}
    rows = []
    for a, b in itertools.combinations(theme_ids, 2):
        A, B = presence[a], presence[b]
        n_a, n_b, n_ab = len(A), len(B), len(A & B)
        n_a_only, n_b_only = n_a - n_ab, n_b - n_ab
        n_neither = N - n_a - n_b + n_ab
        p_b_given_a = n_ab / n_a if n_a else None
        p_a_given_b = n_ab / n_b if n_b else None
        union = n_a + n_b - n_ab
        jaccard = n_ab / union if union else None
        denom = (n_a * n_b * (N - n_a) * (N - n_b)) ** 0.5
        phi = ((n_ab * n_neither) - (n_a_only * n_b_only)) / denom if denom else None
        rows.append({'theme_a': a, 'theme_b': b, 'n_a': n_a, 'n_b': n_b, 'n_co_occur': n_ab,
                      'p_b_given_a': round(p_b_given_a, 4) if p_b_given_a is not None else None,
                      'p_a_given_b': round(p_a_given_b, 4) if p_a_given_b is not None else None,
                      'jaccard': round(jaccard, 4) if jaccard is not None else None,
                      'phi': round(phi, 4) if phi is not None else None})
    return pd.DataFrame(rows)

theme_ids_12 = list(NEW_THEME_PATTERNS.keys())

# =====================================================================
# Level 1: same meaning unit (multi-label coding within one meaning unit)
# =====================================================================
_mu_theme_map = final_12_theme_coding_long.groupby('meaning_unit_id').theme_id.apply(set).to_dict()
_mu_universe = meaning_units.meaning_unit_id.tolist()
_mu_stats = relation_stats(_mu_theme_map, theme_ids_12, _mu_universe)

final_12_theme_meaning_unit_cooccurrence = _mu_stats[['theme_a', 'theme_b', 'n_a', 'n_b', 'n_co_occur']].copy()
final_12_theme_meaning_unit_conditional_percentages = _mu_stats[['theme_a', 'theme_b', 'n_co_occur', 'p_b_given_a', 'p_a_given_b']].copy()
final_12_theme_meaning_unit_conditional_percentages['pct_b_given_a'] = (final_12_theme_meaning_unit_conditional_percentages.p_b_given_a * 100).round(2)
final_12_theme_meaning_unit_conditional_percentages['pct_a_given_b'] = (final_12_theme_meaning_unit_conditional_percentages.p_a_given_b * 100).round(2)
final_12_theme_meaning_unit_jaccard = _mu_stats[['theme_a', 'theme_b', 'n_co_occur', 'jaccard']].copy()
final_12_theme_meaning_unit_phi = _mu_stats[['theme_a', 'theme_b', 'n_co_occur', 'phi']].copy()

for _t in [final_12_theme_meaning_unit_cooccurrence, final_12_theme_meaning_unit_conditional_percentages,
           final_12_theme_meaning_unit_jaccard, final_12_theme_meaning_unit_phi]:
    _t['directional_note'] = 'P(B|A) and P(A|B) are directional and may differ; co-occurrence does not establish causation.'
    _t['sparse_cell_note'] = np.where(_t.n_co_occur < 3, 'Sparse (n<3) -- do not interpret strongly', 'n>=3')

print('final_12_theme_meaning_unit_cooccurrence non-zero:')
print(final_12_theme_meaning_unit_cooccurrence[final_12_theme_meaning_unit_cooccurrence.n_co_occur > 0].to_string(index=False))

# =====================================================================
# Level 2: same applicant decision -- MAIN uses exact applicant matches only;
# SEPARATE sensitivity table uses exact + probable.
# =====================================================================
def _applicant_theme_map(match_qualities):
    sub = final_12_theme_coding_long[final_12_theme_coding_long.applicant_match_quality.isin(match_qualities) & final_12_theme_coding_long.applicant_id.notna()].copy()
    sub['applicant_key'] = sub.participant_id + '_' + sub.applicant_id.astype(str)
    theme_map = sub.groupby('applicant_key').theme_id.apply(set).to_dict()
    universe = meaning_units[meaning_units.applicant_match_quality.isin(match_qualities) & meaning_units.applicant_id.notna()].assign(
        applicant_key=lambda d: d.participant_id + '_' + d.applicant_id.astype(str)).applicant_key.unique().tolist()
    return theme_map, universe

_app_theme_map_exact, _app_universe_exact = _applicant_theme_map(['Exact applicant match'])
final_12_theme_applicant_relations = relation_stats(_app_theme_map_exact, theme_ids_12, _app_universe_exact)
final_12_theme_applicant_relations['match_basis'] = 'Exact applicant match only (main analysis)'
final_12_theme_applicant_relations['n_contributing_applicant_decisions'] = final_12_theme_applicant_relations.n_co_occur
final_12_theme_applicant_relations['sparse_cell_note'] = np.where(final_12_theme_applicant_relations.n_co_occur < 3, 'Sparse (n<3) -- do not interpret strongly', 'n>=3')

_app_theme_map_sens, _app_universe_sens = _applicant_theme_map(['Exact applicant match', 'Probable applicant match'])
final_12_theme_applicant_relations_sensitivity = relation_stats(_app_theme_map_sens, theme_ids_12, _app_universe_sens)
final_12_theme_applicant_relations_sensitivity['match_basis'] = 'Exact + Probable applicant match (sensitivity, not equivalent to exact-match main results)'

print()
print('final_12_theme_applicant_relations (exact matches only) non-zero:')
print(final_12_theme_applicant_relations[final_12_theme_applicant_relations.n_co_occur > 0][['theme_a','theme_b','n_a','n_b','n_co_occur']].to_string(index=False))

# =====================================================================
# Level 3: same participant
# =====================================================================
_part_theme_map = final_12_theme_coding_long.groupby('participant_id').theme_id.apply(set).to_dict()
_part_universe = meaning_units.participant_id.unique().tolist()
final_12_theme_participant_relations = relation_stats(_part_theme_map, theme_ids_12, _part_universe)
final_12_theme_participant_relations['n_contributing_participants'] = final_12_theme_participant_relations.n_co_occur
final_12_theme_participant_relations['sparse_cell_note'] = np.where(final_12_theme_participant_relations.n_co_occur < 3, 'Sparse (n<3) -- do not interpret strongly', 'n>=3')

print()
print('final_12_theme_participant_relations non-zero:')
print(final_12_theme_participant_relations[final_12_theme_participant_relations.n_co_occur > 0][['theme_a','theme_b','n_a','n_b','n_co_occur']].to_string(index=False))

# =====================================================================
# Validity checks (spec section 8): jaccard in [0,1], conditional % in [0,100], co-occur <= min(n_a,n_b)
# =====================================================================
for _name, _tbl, _jcol, _pcols in [
    ('meaning_unit', _mu_stats, 'jaccard', ['p_b_given_a', 'p_a_given_b']),
    ('applicant', final_12_theme_applicant_relations, 'jaccard', []),
    ('participant', final_12_theme_participant_relations, 'jaccard', []),
]:
    _jvals = _tbl[_jcol].dropna()
    assert _jvals.between(0, 1).all(), f'{_name}: jaccard out of [0,1] range'
    assert (_tbl.n_co_occur <= _tbl[['n_a', 'n_b']].min(axis=1)).all(), f'{_name}: co-occurrence exceeds individual theme count'
print('\nValidity checks passed: Jaccard in [0,1]; co-occurrence <= min(n_a, n_b) for all three levels.')


## 12.3 AI × Human 6×6 Relation Summary

In [ ]:
ai_ids = [t for t in theme_ids_12 if t.startswith('AI')]
h_ids = [t for t in theme_ids_12 if t.startswith('H')]

def _lookup(tbl, a, b, col):
    row = tbl[((tbl.theme_a == a) & (tbl.theme_b == b)) | ((tbl.theme_a == b) & (tbl.theme_b == a))]
    if len(row) == 0:
        return None
    r = row.iloc[0]
    if r.theme_a == a:
        return r[col]
    if col == 'p_b_given_a':
        return r['p_a_given_b']
    if col == 'p_a_given_b':
        return r['p_b_given_a']
    return r[col]

_ai_human_rows = []
for ai in ai_ids:
    for h in h_ids:
        _ai_human_rows.append({
            'ai_theme': ai, 'ai_theme_name': NEW_THEME_PATTERNS[ai]['name'],
            'human_theme': h, 'human_theme_name': NEW_THEME_PATTERNS[h]['name'],
            'meaning_unit_co_occurrence': int(_lookup(_mu_stats, ai, h, 'n_co_occur') or 0),
            'meaning_unit_jaccard': _lookup(_mu_stats, ai, h, 'jaccard'),
            'meaning_unit_phi': _lookup(_mu_stats, ai, h, 'phi'),
            'applicant_exact_co_occurrence': int(_lookup(final_12_theme_applicant_relations, ai, h, 'n_co_occur') or 0),
            'participant_co_occurrence': int(_lookup(final_12_theme_participant_relations, ai, h, 'n_co_occur') or 0),
        })
final_12_ai_human_relation_summary = pd.DataFrame(_ai_human_rows)
final_12_ai_human_relation_summary['sparse_cell_note'] = np.where(
    final_12_ai_human_relation_summary.meaning_unit_co_occurrence < 3, 'Sparse (n<3) -- do not interpret strongly', 'n>=3')
print('final_12_ai_human_relation_summary (6x6=36 rows):', final_12_ai_human_relation_summary.shape)
print(final_12_ai_human_relation_summary[final_12_ai_human_relation_summary.meaning_unit_co_occurrence > 0].to_string(index=False))

assert len(final_12_ai_human_relation_summary) == 36


**What this shows:** Meaning-unit co-occurrence, Jaccard similarity and phi-coefficient heatmaps among the twelve final characteristic themes.

**Sample:** All eligible meaning units and their applicant/participant-level aggregations.

---
## 12.4 Relation Heatmaps

**What this shows:** How often each pair of the twelve final characteristic themes co-occurs within the same meaning unit, applicant, or participant.

**Sample:** All eligible meaning units (co-occurrence heatmap) and their applicant/participant-level aggregations (Jaccard and phi heatmaps).

In [ ]:
import os
RELATIONS_DIR = OUT / 'extended_theme_analysis_outputs' / 'final_12_theme_analysis' / 'figures'
RELATIONS_DIR.mkdir(parents=True, exist_ok=True)
RELATIONS_TABLES_DIR = OUT / 'extended_theme_analysis_outputs' / 'final_12_theme_analysis' / 'tables'
RELATIONS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

CMAP = 'viridis'

def _pair_matrix(tbl, col, ids=theme_ids_12, diag_fill=None):
    m = pd.DataFrame(np.nan, index=ids, columns=ids)
    for _, r in tbl.iterrows():
        v = r[col]
        m.loc[r.theme_a, r.theme_b] = v
        m.loc[r.theme_b, r.theme_a] = v
    if diag_fill is not None:
        for t in ids:
            m.loc[t, t] = diag_fill.get(t, np.nan) if isinstance(diag_fill, dict) else diag_fill
    return m

_freq_by_theme = final_12_theme_coding_long.groupby('theme_id').meaning_unit_id.nunique().to_dict()

# Fig A: meaning-unit co-occurrence heatmap
_m1 = _pair_matrix(_mu_stats, 'n_co_occur', diag_fill=_freq_by_theme).fillna(0)
plt.figure(figsize=(8, 7))
plt.imshow(_m1.values, cmap=CMAP)
plt.colorbar(label='Co-occurring meaning units')
plt.xticks(range(len(theme_ids_12)), theme_ids_12, rotation=45); plt.yticks(range(len(theme_ids_12)), theme_ids_12)
plt.title('Meaning-Unit Co-Occurrence Between Final Twelve Themes')
plt.tight_layout()
plt.savefig(RELATIONS_DIR / 'final12_relations_fig_a_meaning_unit_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

# Fig B: meaning-unit Jaccard heatmap
_m2 = _pair_matrix(_mu_stats, 'jaccard', diag_fill=1.0).fillna(0)
plt.figure(figsize=(8, 7))
plt.imshow(_m2.values, cmap=CMAP, vmin=0, vmax=1)
plt.colorbar(label='Jaccard similarity')
plt.xticks(range(len(theme_ids_12)), theme_ids_12, rotation=45); plt.yticks(range(len(theme_ids_12)), theme_ids_12)
plt.title('Meaning-Unit Jaccard Similarity Between Final Twelve Themes')
plt.tight_layout()
plt.savefig(RELATIONS_DIR / 'final12_relations_fig_b_jaccard_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Fig C: AI x Human applicant-level relation heatmap (exact matches)
_m3 = pd.DataFrame(0, index=ai_ids, columns=h_ids)
for _, r in final_12_ai_human_relation_summary.iterrows():
    _m3.loc[r.ai_theme, r.human_theme] = r.applicant_exact_co_occurrence
plt.figure(figsize=(7, 6))
plt.imshow(_m3.values, cmap=CMAP)
plt.colorbar(label='Co-occurring applicant decisions (exact matches)')
plt.xticks(range(len(h_ids)), h_ids); plt.yticks(range(len(ai_ids)), ai_ids)
plt.title('AI x Human Applicant-Level Relations (Exact Matches)')
plt.tight_layout()
plt.savefig(RELATIONS_DIR / 'final12_relations_fig_c_ai_human_applicant_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Fig D: participant-level relation heatmap (full 12x12)
_m4 = _pair_matrix(final_12_theme_participant_relations, 'n_co_occur', diag_fill={t: final_12_theme_coding_long[final_12_theme_coding_long.theme_id==t].participant_id.nunique() for t in theme_ids_12}).fillna(0)
plt.figure(figsize=(8, 7))
plt.imshow(_m4.values, cmap=CMAP)
plt.colorbar(label='Co-occurring participants')
plt.xticks(range(len(theme_ids_12)), theme_ids_12, rotation=45); plt.yticks(range(len(theme_ids_12)), theme_ids_12)
plt.title('Participant-Level Co-Occurrence Between Final Twelve Themes')
plt.tight_layout()
plt.savefig(RELATIONS_DIR / 'final12_relations_fig_d_participant_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('4 relation heatmaps saved to', public_path(RELATIONS_DIR))


**Main result:** AI-characteristic and human-characteristic themes show meaningful co-occurrence and association patterns, with the strongest relationships appearing within, rather than across, the AI/human theme groups.

**Interpretation:** Participants' AI-related and self-related reasoning cluster into recognisable, internally coherent patterns rather than appearing independently of each other.

**Caution:** These are observed statistical associations only (co-occurrence, Jaccard similarity, phi coefficient); association does not imply that one theme causes another or causes a particular decision (Part 14).

## 12.5 Limitations of the Relation Analysis

These relations are descriptive and exploratory. Directional conditional percentages (P(B|A) vs. P(A|B)) can differ substantially for rare themes and should not be read as one theme "causing" another. Applicant-level relations use exact applicant matches as the main analysis (a small sample: 145 exact matches across 1,967 meaning units); the exact-plus-probable sensitivity table is reported separately and never substituted for the exact-match main results. Most pairwise cells across all three levels have fewer than 3 co-occurrences and are marked accordingly -- these should not be interpreted as meaningful relationships, only as an absence of evidence either way.

<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Meaning-unit, applicant-level and participant-level relations among the twelve themes were computed (co-occurrence, conditional percentages, Jaccard, phi), alongside an AI×Human 6×6 summary and associations with the decision journeys and reliance categories, with an exact-plus-probable sensitivity analysis reported alongside the exact-match results. These are observed statistical associations only; association does not imply causation. The next Part integrates this qualitative evidence with the quantitative findings.
</div>


# Part 13 — Mixed-Method Integration

The joint display connects what participants said, what they did and what happened to decision accuracy and group fairness.


*Note on wording below:* the source joint-display table (approved under the qualitative sign-off process) uses the unqualified word "disagreements" in the RQ2/RQ3/design-implication rows. Throughout this table, that word refers to **initial** human–AI disagreement (700 cases, 36.46% of matched decisions) — the same quantity used to define the reliance categories in Section 4. It is not the final human–AI disagreement rate (549 cases, 28.59%) introduced in Section 17. The table itself is an approved qualitative artefact and is left unedited here; this note only disambiguates its terminology.

In [ ]:
display(style_table(joint_display_final))


## 13.1 Focused Literature Synthesis

The literature matrix is a focused structured review of directly relevant research. It should not be labelled a PRISMA systematic review unless a complete database search, deduplication and screening workflow is subsequently conducted.


In [ ]:

display(style_table(literature_matrix[["authors","year","title","concept","relevance_to_project"]]))


## 13.2 Crosswalk Between the Final Twelve and Original Six Themes

This is a **conceptual mixed-method crosswalk**, not a frequency or percentage comparison: the six original approved themes use the survey-response denominator, and the twelve characteristic themes use the transcript meaning-unit denominator. These are never compared directly as percentages anywhere in this notebook. The six original themes are listed below exactly as previously approved and are not renamed, deleted or recalculated here:

1. Financially Grounded but Heuristic Judgement
2. Fairness as Background-Blind, Equal and Evidence-Based Treatment
3. AI Trust Was Conditional, Contested and Performance-Sensitive
4. Feedback Supported Learning, but Calibration Remained Uneven
5. Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed
6. Perception–Behaviour Alignment and Mismatch Coexisted

In [ ]:
def _evidential_strength(n_mu, n_part):
    if n_mu == 0: return 'Not observed'
    if n_mu >= 10 and n_part >= 5: return 'Core finding'
    if n_mu >= 5 and n_part >= 3: return 'Supporting finding'
    if n_mu >= 3 and n_part >= 2: return 'Exploratory finding'
    return 'Insufficient evidence'

_theme_n = final_12_theme_coding_long.groupby('theme_id').agg(n_mu=('meaning_unit_id', 'nunique'), n_part=('participant_id', 'nunique')).to_dict('index')

def _quote_for(tid, high_only=True):
    sub = final_12_theme_coding_long[final_12_theme_coding_long.theme_id == tid]
    if high_only:
        sub2 = sub[sub.confidence == 'High']
        sub = sub2 if len(sub2) else sub
    if len(sub) == 0:
        return '(no confirmed meaning units for this theme)'
    r = sub.sort_values('meaning_unit_id').iloc[0]
    mu_text = meaning_units.loc[meaning_units.meaning_unit_id == r.meaning_unit_id, 'combined_original_text'].iloc[0]
    return mu_text[:220]

def _negative_quote_for(tid):
    rej = final_12_theme_candidate_review[(final_12_theme_candidate_review.candidate_theme_id == tid) & (final_12_theme_candidate_review.status == 'Rejected')]
    if len(rej) == 0:
        return '(no rejected-candidate example on file)'
    r = rej.sort_values('meaning_unit_id').iloc[0]
    mu_text = meaning_units.loc[meaning_units.meaning_unit_id == r.meaning_unit_id, 'combined_original_text'].iloc[0]
    return f'Rejected candidate ({r.rejection_reason}): "{mu_text[:150]}"'

CROSSWALK_SPEC = [
    ('AI1', 'AI as Data-Driven and Financially Focused', 'AI Trust Was Conditional, Contested and Performance-Sensitive', 'Extends',
     'Elaborates on the grounds for AI trust: participants specifically frame AI as trustworthy (or not) because it is grounded in financial/numerical data, a facet not distinguished in the original theme.'),
    ('AI2', 'AI as Consistent and Rule-Based', 'AI Trust Was Conditional, Contested and Performance-Sensitive', 'Extends',
     'Identifies consistency/predictability as a distinct basis for conditional AI trust, separate from the performance-history basis already captured by AI4.'),
    ('AI3', 'AI as a Decision-Supporting Second Opinion', 'Perception–Behaviour Alignment and Mismatch Coexisted', 'Directly supports',
     'Shows the mechanism by which AI perception translates (or fails to translate) into behaviour: AI used as an input to reconsideration, confirmation or retained judgement.'),
    ('AI4', 'AI as Fallible and Performance-Dependent', 'AI Trust Was Conditional, Contested and Performance-Sensitive', 'Directly supports',
     'Matches the original theme closely: trust explicitly rises or falls with observed AI accuracy/performance history.'),
    ('AI5', 'AI as a Confidence Signal and Behavioural Influence', 'AI Trust Was Conditional, Contested and Performance-Sensitive', 'Extends',
     'Extends conditional trust into an observable behavioural/affective mechanism (confidence, reassurance, doubt) rather than a general trust statement alone -- the most strongly and widely observed of the twelve themes in this corpus.'),
    ('AI6', 'AI as Potentially Susceptible to Data-Embedded Bias', 'Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed', 'Partially overlaps',
     'Related but not identical: the original theme reports MEASURED background-group disparity in structured decision data, while AI6 captures participants\' SUBJECTIVE SUSPICION of AI bias -- a belief, not a measurement, that may or may not align with the observed pattern.'),
    ('H1', 'Human Contextual Understanding', 'Financially Grounded but Heuristic Judgement', 'Extends',
     'Extends the heuristic-judgement theme by specifying one concrete heuristic input -- interpreting loan purpose/circumstances -- beyond bare financial-variable thresholds.'),
    ('H2', 'Human Empathy and Humanitarian Consideration', 'Financially Grounded but Heuristic Judgement', 'Partially overlaps',
     'Overlaps only partially: humanitarian/emotional reasoning is a heuristic influence on judgement, but is not itself "financially grounded" -- it can override financial grounding rather than extend it.'),
    ('H3', 'Human Holistic Balancing of Conflicting Evidence', 'Financially Grounded but Heuristic Judgement', 'Directly supports',
     'Directly instantiates the heuristic-judgement theme: multiple financial factors are explicitly weighed against each other rather than applied as a single rule.'),
    ('H4', 'Human Learning and Adaptive Judgement', 'Feedback Supported Learning, but Calibration Remained Uneven', 'Directly supports',
     'Directly matches: strategy/threshold changes following prior-round feedback are the concrete transcript-level evidence for the original feedback-learning theme.'),
    ('H5', 'Human Independence and Ability to Challenge AI', 'Perception–Behaviour Alignment and Mismatch Coexisted', 'Directly supports',
     'Directly instantiates the "mismatch" half of the original theme: explicit verbal reasoning for retaining a decision against the AI recommendation.'),
    ('H6', 'Human Subjectivity, Bias and Heuristic Inconsistency', 'Fairness as Background-Blind, Equal and Evidence-Based Treatment', 'Contradicts',
     'Sits in tension with the original theme rather than confirming it: H6 documents personal thresholds, intuition and inconsistent criteria, which -- if present -- would work against purely background-blind, evidence-based treatment. Reported as an honest tension, not resolved by this analysis.'),
]

_cw_rows = []
for tid, name, orig, rel_type, explanation in CROSSWALK_SPEC:
    n_mu = _theme_n.get(tid, {}).get('n_mu', 0)
    n_part = _theme_n.get(tid, {}).get('n_part', 0)
    _cw_rows.append({
        'characteristic_theme_id': tid, 'characteristic_theme_name': name,
        'related_original_approved_theme': orig, 'relationship_type': rel_type, 'explanation': explanation,
        'supporting_evidence_quote': _quote_for(tid), 'contradictory_or_negative_evidence': _negative_quote_for(tid),
        'evidential_strength': _evidential_strength(n_mu, n_part),
        'n_meaning_units': n_mu, 'n_participants': n_part,
    })
final_12_to_original_6_theme_crosswalk = pd.DataFrame(_cw_rows)
assert len(final_12_to_original_6_theme_crosswalk) == 12
assert set(final_12_to_original_6_theme_crosswalk.related_original_approved_theme) <= {
    'Financially Grounded but Heuristic Judgement',
    'Fairness as Background-Blind, Equal and Evidence-Based Treatment',
    'AI Trust Was Conditional, Contested and Performance-Sensitive',
    'Feedback Supported Learning, but Calibration Remained Uneven',
    'Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed',
    'Perception–Behaviour Alignment and Mismatch Coexisted',
}
print('final_12_to_original_6_theme_crosswalk:', final_12_to_original_6_theme_crosswalk.shape)
display(style_table(final_12_to_original_6_theme_crosswalk[['characteristic_theme_id', 'related_original_approved_theme', 'relationship_type', 'evidential_strength']]))
print('\nThis is a conceptual mixed-method crosswalk (relationship type and supporting/contradictory transcript evidence), NOT a frequency or percentage comparison -- the six original themes use the survey denominator and the twelve characteristic themes use the meaning-unit denominator; these are never compared directly as percentages anywhere in this notebook.')


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
The twelve final characteristic themes were mapped to the six original approved themes through a conceptual crosswalk (Directly supports / Extends / Partially overlaps / Contradicts / No clear relationship) and a mixed-methods joint display, never as a direct percentage comparison since the two frameworks use different denominators. The next Part consolidates every methodological limitation and safeguard.
</div>

# Part 14 — Limitations and Safeguards

This Part consolidates the methodological safeguards and limitations that were previously stated near each individual analysis into one place, so a reader can review every caution together. Each safeguard still applies specifically to the analysis named, and none of them change any reported value.

- **Carry-forward assumption.** The effective final decision uses the explicitly displayed final decision where available; otherwise the initial human decision is carried forward. This assumption is checked directly by the AI-vs-explicit-final sensitivity analysis (Part 6, Branch 2), which excludes every carried-forward row and reaches the same overall conclusion (majority agreement) as the effective-final analysis.
- **Applicant-matching limitations.** Meaning-unit-to-applicant matching used an exact-match procedure by default; a separate exact-plus-probable sensitivity analysis is reported alongside the exact-only results wherever applicant-level associations are computed (Part 11, Part 12), and the two are never merged into a single reported figure.
- **Small cell sizes.** Several reliance, journey and theme categories (for example, the 18 "moved away after initial agreement" cases) are small in absolute terms. Percentages computed on small denominators are reported with the denominator stated explicitly rather than as a percentage alone.
- **Transcript-classification validation status.** The final twelve AI/human characteristic themes were derived using a deterministic rule-based classification process over adaptively constructed meaning units. A manual-validation sample is prepared in Part 11, but completed independent-review results are not included; these themes are therefore described as deterministic rule-based classifications rather than independently human-validated findings.
- **Orange–Purple causal limits.** Every Orange–Purple comparison in this notebook (Part 9) is a descriptive, stage-by-stage observed disparity. The background-assignment mechanism was not independently documented, so no causal effect of background and no causal discrimination claim is made anywhere in this notebook. A small or absent disparity at any one stage does not by itself demonstrate fairness.
- **Observational nature of associations.** Theme co-occurrence, conditional-probability, Jaccard and phi associations reported in Part 12 describe observed statistical association only; association does not imply causation, and none of these relations are interpreted as evidence that one theme causes another or causes a particular decision.
- **Study-design limitations.** Only 22 participants took part; the same 100 applicant profiles repeat across all participants and rounds, so results describe this repeated-measures sample rather than a fresh independent sample; transcript-to-round timestamp alignment is approximate; and no validated survey, timing, confidence, demographic or treatment-condition data were supplied beyond what is analysed in this notebook.
- **Researcher approval and sign-off.** Coder provenance, all six original approved themes, and their selected quotation contexts have been reviewed and approved. Orange–Purple findings are governed by the approved observed-disparity reporting policy stated at the end of this notebook: no causal background or causal-discrimination claim is made unless documentary assignment evidence is found.

In [ ]:
loo=[]
for pid in matched.participant_id.unique():
 q=matched[matched.participant_id!=pid]; loo.append({'left_out':pid,'final_minus_initial':q.final_correct.mean()-q.initial_correct.mean(),'final_minus_computer':q.final_correct.mean()-q.computer_correct.mean()})
loo=pd.DataFrame(loo)
cr=follow_model.set_index('term').loc['computer_correct']; fr=final_model.set_index('term').loc['loan_risk']; ib=init_model.set_index('term').loc['C(background)[T.Purple]']; fb=final_model.set_index('term').loc['C(background)[T.Purple]']
findings=pd.DataFrame([
{'area':'Collaboration','relationship':'Final vs initial accuracy, same cases','n':len(matched),'estimate':fi[0],'ci_low':fi[1],'ci_high':fi[2],'conclusion':'Small positive point estimate; interval includes zero.'},
{'area':'Collaboration','relationship':'Final vs computer accuracy, same cases','n':len(matched),'estimate':fc[0],'ci_low':fc[1],'ci_high':fc[2],'conclusion':'Final decisions clearly outperform computer recommendations on matched cases.'},
{'area':'Applicant factors','relationship':'Loan risk and final approval (OR per risk point)','n':len(final_data),'estimate':fr.odds_ratio,'ci_low':fr.ci_low,'ci_high':fr.ci_high,'conclusion':'Higher risk is strongly associated with lower final approval odds.'},
{'area':'AI reliance','relationship':'Following correct vs incorrect AI (OR)','n':len(follow_data),'estimate':cr.odds_ratio,'ci_low':cr.ci_low,'ci_high':cr.ci_high,'conclusion':'Positive direction, but two-way clustered interval includes no difference.'},
{'area':'Fairness','relationship':'Final Purple-minus-Orange approval gap','n':len(df),'estimate':fg,'ci_low':gap_ci.set_index('decision_stage').loc['Final collaborative','ci_low'],'ci_high':gap_ci.set_index('decision_stage').loc['Final collaborative','ci_high'],'conclusion':'Final collaboration greatly reduces the computer gap but slightly increases absolute disparity relative to initial humans.'},
{'area':'Adjusted background','relationship':'Purple coefficient at initial stage (OR)','n':len(init_data),'estimate':ib.odds_ratio,'ci_low':ib.ci_low,'ci_high':ib.ci_high,'conclusion':'No clear initial background main effect with two-way clustered uncertainty.'},
{'area':'Adjusted background','relationship':'Purple coefficient at final stage model (OR)','n':len(final_data),'estimate':fb.odds_ratio,'ci_low':fb.ci_low,'ci_high':fb.ci_high,'conclusion':'No clear final background main effect in the separate adjusted final model.'}])
display(findings.round(4)); display(loo.describe().round(4))

In [ ]:
public_signoff_summary = pd.DataFrame([
    {'item': 'Primary coding', 'status': 'Completed', 'detail': 'All 210 responses coded by the primary researcher.'},
    {'item': 'Independent second coding', 'status': 'Completed', 'detail': 'A second human coder independently coded 52 responses using the same 23-code framework.'},
    {'item': 'Intercoder agreement', 'status': 'Completed before consensus', 'detail': '1,196 binary decisions; six differences; 99.50% overall raw agreement.'},
    {'item': 'Consensus', 'status': 'Completed', 'detail': 'All six differences were resolved after agreement statistics were calculated.'},
    {'item': 'Original six themes', 'status': 'Approved', 'detail': 'The 23-code framework underpins the six approved themes.'},
    {'item': 'Part 11 twelve-theme classification', 'status': 'Separate scope', 'detail': 'Deterministic rule-based classification; not covered by the Section 10.1 reliability statistics.'},
])
display(style_table(public_signoff_summary))


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Every safeguard and limitation stated near an individual analysis elsewhere in this notebook is consolidated here: the carry-forward assumption, applicant-matching and exact-plus-probable sensitivity, small cell sizes, the transcript-classification manual-validation status, the Orange–Purple causal limits, and the observational nature of the theme-relation associations. The next Part presents the final conclusions.
</div>

# Part 15 — Final Conclusions

This Part consolidates every conclusion previously reported separately across the notebook's historical layers (the core audited analysis, the distinction-level machine-learning extensions, and the cherry-on-top extensions) into one authoritative set of subsections. Each finding is reported once, at the subsection where it best answers a specific research question; the full supporting tables and figures remain in the Parts referenced below.

## 15.1 Answers to the Research Questions

1. **How often does the final decision agree with the AI, and how sensitive is this to the carry-forward assumption?** Final Human–AI agreement was 71.79% (1,448/2,017) using the effective final decision, and 73.84% (1,352/1,831) using only explicitly recorded final decisions (no carry-forward) — the same conclusion in both: majority agreement (Part 6).
2. **How does an initial disagreement resolve by the final decision?** Of the 700 initial disagreements, 24.14% (169) changed toward AI and 75.86% (531) resisted AI; of the 1,220 initial agreements, 98.52% (1,202) were retained and 1.48% (18) moved away. Categorised by appropriateness, 60% of the 700 initial disagreements were appropriate acceptance/resistance and 40% were over- or under-reliance, with underreliance more common than overreliance (Part 7).
3. **How accurate are the three decision stages, and what predicts final approval?** On the 1,920 matched cases, final decisions were 1.20 points more accurate than initial human decisions (interval includes zero) and 5.89 points more accurate than the computer (clearly positive). Final approval is highly predictable (ROC-AUC ≈ 0.96) mainly because it inherits the initial decision and computer recommendation; a transparent logistic model matched a gradient-boosted model, and correctness did not generalise well to genuinely unseen applicant profiles (Part 8).
4. **What Orange–Purple disparities are observed, and what are the limits on interpreting them causally?** The observed Purple-minus-Orange approval gap was +40 points at the computer-recommendation stage, about −3.0 points for initial humans, and about +4.2 points at the final stage — final collaboration sharply reduced the computer-stage gap but did not reduce the absolute disparity relative to initial humans. Applicant background had very low predictive importance in the SHAP/permutation analysis once decision history and financial variables were included. No causal discrimination claim is made: the background-assignment mechanism is not independently documented (Part 9, Part 14).
5. **What reasoning themes do participants express, and how do they relate to the decision patterns?** The original six approved themes (Part 10) and the final twelve AI/human characteristic themes (Part 11) both show financially grounded but heuristic judgement, conditional and performance-sensitive AI trust, and uneven calibration after feedback. Theme relations (Part 12) and the crosswalk (Part 13) show these qualitative patterns are broadly consistent with the quantitative reliance and disagreement findings, with any partial overlaps or contradictions explicitly flagged rather than assumed away.

## 15.2 Main Quantitative Findings

- Applicant-factor model: higher loan risk, loan-to-income ratio, DTI, revolving utilisation and recent inquiries are consistently associated with lower approval odds; owning a home (vs. mortgage) is associated with higher approval odds.
- SHAP rankings were stable across grouped folds (mean pairwise Kendall agreement 0.93 participant-grouped, 0.88 applicant-grouped), with the initial decision, computer recommendation, loan risk, revolving utilisation and DTI consistently the top features.
- Following correct computer advice trends positive but is not clearly supported once participant and applicant dependence are both accounted for; predicting who follows AI advice remains difficult (ROC-AUC ≈ 0.60–0.62).

## 15.3 Main Qualitative Findings

- The six original approved themes and the twelve final characteristic themes both center on financially grounded but heuristic human judgement, AI trust that is conditional and performance-sensitive, and feedback-linked but uneven calibration (Parts 10–11).
- Negative and deviant cases were deliberately retained rather than filtered out, and every reported theme carries its manual-validation status explicitly rather than being presented as uniformly human-verified (Part 11, Part 14).

## 15.4 Human–AI Reliance Interpretation

Underreliance was consistently more common than overreliance among initial disagreements, indicating participants were, on balance, more likely to under-use correct AI advice than to over-trust incorrect advice. Both initial and final Human–AI disagreement are reported as distinct, non-interchangeable measures throughout this notebook (Part 4, Part 7).

## 15.5 Fairness and Orange–Purple Findings

The observed group gap narrowed sharply between the computer-recommendation stage and the final stage, but the final absolute disparity was not smaller than the initial-human disparity. These are descriptive, stage-by-stage observed disparities in this experimental dataset only. The Orange/Purple assignment mechanism was not independently documented, so no causal effect of background and no causal discrimination claim is made; a small or absent disparity at any one stage does not by itself establish fairness (Part 9, Part 14).

## 15.6 Practical Implications

The evidence supports a **human-led, AI-supported** collaboration design: an independent initial human judgement, an AI recommendation presented as a checkable second opinion, structured review specifically for initial-disagreement cases, and continuous monitoring of both accuracy and group-level disparities — since agreement between human and AI does not by itself guarantee either accuracy or fairness.

## 15.7 Final Limitations

Only 22 participants took part; the same 100 applicant profiles repeat across all participants and rounds; transcript-to-round alignment is approximate; and no validated survey, timing, confidence, demographic or treatment-condition data were supplied beyond what is analysed here. The carry-forward assumption behind the effective final decision, the applicant-matching and exact-vs-probable sensitivity analyses, small-cell-size caution, and the transcript-classification manual-validation status are detailed in Part 14.

## 15.8 Overall Conclusion

Across every layer of this analysis — the audited core comparison, the machine-learning extensions, and the qualitative theming — the same conclusion holds: final human decisions modestly and inconsistently improve on initial human judgement, clearly improve on the computer recommendation alone, and are shaped more by decision history and financial-risk variables than by applicant background. Human–AI collaboration in this dataset works best as a genuinely two-sided process, not as automatic deference to either party.

## 15.9 Claims-and-Evidence Table

The final dashboard distinguishes strong findings from suggestive or exploratory results. This makes the notebook easier to defend because every claim is paired with its evidence and its main limitation.

In [ ]:
# Assemble a concise claims-and-evidence table from verified results.
claims=pd.DataFrame([
 {'claim':'Final human–AI decisions outperform the computer alone.','evidence':f'Final minus computer accuracy = {fc[0]:.3f}, 95% CI {fc[1]:.3f} to {fc[2]:.3f}.','strength':'Strong','limitation':'Experimental task with fixed applicant profiles.'},
 {'claim':'Computer advice clearly improves unaided human accuracy.','evidence':f'Final minus initial accuracy = {fi[0]:.3f}, 95% CI {fi[1]:.3f} to {fi[2]:.3f}.','strength':'Weak / uncertain','limitation':'Confidence interval includes no improvement.'},
 {'claim':'Participants usually handled initial human–AI disagreement appropriately.','evidence':'60% appropriate acceptance/resistance; 40% over- or under-reliance (among the 700 initial-disagreement cases).','strength':'Strong descriptive','limitation':'Correctness is defined by the recorded historical outcome.'},
 {'claim':'Underreliance was a larger problem than overreliance.','evidence':f'Underreliance {rel_counts["Underreliance"]/disagree_n:.1%}; overreliance {rel_counts["Overreliance"]/disagree_n:.1%} (of the 700 initial-disagreement cases).','strength':'Strong descriptive','limitation':'Does not explain the psychological reason for resistance.'},
 {'claim':'The observed Purple–Orange approval gap was smaller at the final stage than at the computer stage.','evidence':'Observed gap: +40.0 points at the computer stage and +4.19 points at the final stage.','strength':'Strong descriptive','limitation':'Background assignment is unverified; this stage comparison is not a causal background or discrimination effect.'},
 {'claim':'Financial-risk cues were more influential than background in individual final-decision prediction.','evidence':'Regression, grouped validation, SHAP and permutation importance consistently prioritised initial decision, computer advice and financial variables.','strength':'Moderate to strong','limitation':'Predictive importance is not a causal or ethical fairness test.'},
 {'claim':'A complex nonlinear model is necessary.','evidence':'CatBoost did not materially outperform logistic regression under strict dual holdout.','strength':'Evidence against claim','limitation':'Other model classes were not exhaustively tested.'},
 {'claim':'The model generalises final correctness to new applicants.','evidence':'Correctness prediction fell to approximately chance for unseen applicant profiles.','strength':'Evidence against claim','limitation':'Only 100 unique applicant profiles were available.'},
 {'claim':'Transcript language explains AI following.','evidence':'Keyword relationships were weak and did not survive multiple-testing correction.','strength':'Exploratory / unsupported','limitation':'Keyword counts cannot recover full meaning or decision-level context.'},
 {'claim':'Participants expressed fairness concerns and changing trust.','evidence':'The independently double-coded 23-code response analysis supports the six approved themes; optional transcript excerpts are illustrative only.','strength':'Human-validated qualitative','limitation':'The separate Part 11 rule-based twelve-theme classification is not covered by the Section 10.1 reliability statistics.'},
])
display(style_table(claims))
claims.to_csv(CHERRY/'tables'/'claims_and_evidence_dashboard.csv',index=False)

# Compact data card and model card.
data_card=pd.DataFrame([
 {'item':'Participants','value':'22 pseudonymised participants'},
 {'item':'Decision records','value':'2,200 rows; 1,920 matched initial/final decisions'},
 {'item':'Applicant profiles','value':'100 unique profiles repeated across participants'},
 {'item':'Rounds','value':'10 per participant'},
 {'item':'Transcript evidence','value':'6,410 timestamped events; 6,124 speech events'},
 {'item':'Main limitations','value':'Repeated applicants, missing decisions, imperfect speech-to-applicant alignment, no confidence/survey/timing measures'},
 {'item':'Permitted use','value':'Research interpretation of this experiment'},
 {'item':'Not permitted','value':'Real-world automated loan approval, causal background effects, or causal discrimination claims'},
])
model_card=pd.DataFrame([
 {'item':'Primary explanatory model','value':'Repeated-measures / cluster-robust logistic regression'},
 {'item':'Supporting predictive models','value':'Regularised logistic regression and CatBoost'},
 {'item':'Validation','value':'Unseen participants, unseen applicants, and strict dual holdout'},
 {'item':'Best-supported target','value':'Final approval, largely because final decisions retain initial human decisions'},
 {'item':'Weak target','value':'AI following; only modest predictive performance'},
 {'item':'Failed generalisation','value':'Final correctness for unseen applicant profiles'},
 {'item':'Explanation method','value':'Out-of-fold SHAP plus permutation importance and rank stability'},
 {'item':'Interpretation warning','value':'Prediction and SHAP do not establish causality or fairness'},
])
display(style_table(data_card)); display(style_table(model_card))
data_card.to_csv(CHERRY/'tables'/'data_card.csv',index=False)
model_card.to_csv(CHERRY/'tables'/'model_card.csv',index=False)

# One-page markdown executive summary.
summary_text=f"""# Executive Summary\n\n- **Dataset:** 22 participants, 100 repeated applicants, 2,200 decision rows and 1,920 matched initial/final cases.\n- **Accuracy:** Initial human {matched.initial_correct.mean():.2%}; computer {matched.computer_correct.mean():.2%}; final {matched.final_correct.mean():.2%}.\n- **Collaboration:** Final decisions clearly outperform the computer, but improvement over initial humans is small and uncertain.\n- **Reliance:** 60% of the 700 *initial* disagreement cases were handled appropriately. Underreliance ({rel_counts['Underreliance']/disagree_n:.1%}) was more common than overreliance ({rel_counts['Overreliance']/disagree_n:.1%}).\n- **Disagreement measures (both reported; neither replaces the other):** initial human–AI disagreement (first human decision vs. computer, before reconsideration) was 36.46% (700/1,920); final human–AI disagreement (final human decision vs. computer, after reconsideration) was 28.59% (549/1,920). The reliance categories above are defined only within the initial-disagreement subset.\n- **Fairness:** The observed Purple–Orange gap was +40 points at the computer stage and +4.19 points at the final stage. This is a descriptive stage comparison; background assignment is unverified and no causal discrimination claim is made.\n- **Prediction:** Logistic regression matched CatBoost for final approval; correctness failed to generalise to new applicants.\n- **SHAP:** Stable rankings show initial decision and computer recommendation dominate final approval, followed by financial-risk variables.\n- **Qualitative evidence:** Participants expressed financial heuristics, uncertainty, trust, resistance, learning and fairness concerns. The themes and quotation contexts were human-validated and researcher-approved.\n- **Recommended design:** Human makes an independent judgement, AI provides an explained second opinion, initial-disagreement cases receive structured review, and accuracy plus group fairness are continuously monitored.\n"""
(CHERRY/'reports'/'executive_summary.md').write_text(summary_text,encoding='utf-8')
print(summary_text)


<div style="
border: 1.5px solid #64748B;
border-left: 6px solid #2563EB;
border-radius: 6px;
background: #F1F5F9;
padding: 12px 16px;
margin: 18px 0;
">
<b>Section Summary</b><br>
Sections 15.1–15.8 answer the research questions and consolidate every conclusion previously reported separately across the notebook's historical layers, without repeating the same number in more than one place. The overall conclusion supports a human-led, AI-supported collaboration design with continuous monitoring of both accuracy and group disparities. The final Part reports the reproducibility manifest and validation results.
</div>

# Part 16 — Reproducibility and Output Manifest

In [ ]:
export=df.drop(columns=[c for c in ['source_screenshot'] if c in df.columns]).copy()
assert not export.astype(str).apply(lambda s:s.str.contains(r'2023-\d{2}-\d{2}',regex=True).any()).any()
export.to_csv(OUT/'data'/'analysis_ready_decisions.csv',index=False); participant_summary.to_csv(OUT/'data'/'participant_summary.csv',index=False); round_summary.to_csv(OUT/'data'/'round_summary.csv',index=False); participant_text.to_csv(OUT/'data'/'participant_text_features.csv',index=False); participant_round_text.to_csv(OUT/'data'/'participant_round_text_features.csv',index=False); integrated.to_csv(OUT/'data'/'integrated_participant_round_data.csv',index=False); events[['participant_id','event_id','timestamp_raw','timestamp_seconds','event_type','text','unclear_count']].to_csv(OUT/'data'/'timeline_events_pseudonymised.csv',index=False)
stage_all.to_csv(OUT/'tables'/'stage_all.csv',index=False); stage_matched.to_csv(OUT/'tables'/'stage_matched.csv',index=False); diffs.to_csv(OUT/'tables'/'accuracy_differences.csv',index=False); rel.to_csv(OUT/'tables'/'reliance_categories.csv',index=False)
init_model.to_csv(OUT/'tables'/'initial_approval_model.csv',index=False); final_model.to_csv(OUT/'tables'/'final_approval_model.csv',index=False); follow_model.to_csv(OUT/'tables'/'follow_ai_model.csv',index=False); fair_model.to_csv(OUT/'tables'/'fairness_stage_model.csv',index=False)
fairness.to_csv(OUT/'tables'/'fairness_metrics.csv',index=False); gaps.to_csv(OUT/'tables'/'fairness_gaps.csv',index=False); gap_ci.to_csv(OUT/'tables'/'fairness_gap_ci.csv',index=False)
text_rel.to_csv(OUT/'tables'/'text_behaviour_relationships.csv',index=False); findings.to_csv(OUT/'tables'/'main_findings.csv',index=False); loo.to_csv(OUT/'tables'/'leave_one_participant_out.csv',index=False)
manifest={'generated_utc':datetime.now(timezone.utc).isoformat(),'seed':SEED,'python':platform.python_version(),'input_hashes':{csv.name:sha(csv),book.name:sha(book)},'counts':{'participants':22,'rows':2200,'applicants':100,'events':len(events),'speech':int((events.event_type=='SPEECH').sum()),'screen':int((events.event_type=='SCREEN').sum()),'numbered_results':len(numbered),'corrupt_extra_results':len(corrupt)},'models':{'initial_n':len(init_data),'final_n':len(final_data),'follow_n':len(follow_data),'fairness_long_n':len(long)}}
(OUT/'reports'/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Exported to', public_path(OUT)); print('PNG figures',len(list((OUT/'figures').glob('*.png'))),'CSV tables',len(list((OUT/'tables').glob('*.csv'))))

## 16.1 Pending-Work Validation Summary

In [ ]:
pw_checks = []
def pw_check(name, ok, detail=''):
    pw_checks.append({'check': name, 'result': 'PASS' if ok else 'FAIL', 'detail': detail})

pw_check('Total rows equal the source-data total (2,200)', len(df) == 2200)
pw_check('Reconciliation groups sum to 2,200',
          (_init_present_explicit_present + _init_present_explicit_missing + _init_missing_explicit_present + _init_missing_explicit_missing) == 2200)
pw_check('Reconstructed effective final matches the existing effective final (0 mismatches)', _n_mismatch == 0,
          f'{_n_exact} exact matches, {_n_mismatch} mismatches, {_n_both_missing} both missing out of {_total_compared}')
pw_check('AI-Effective-Final: agreement + disagreement = matched sample', (af_agree_n + af_disagree_n) == ai_final_matched_n)
pw_check('AI-Effective-Final: all four combinations = matched sample',
          (ai_approve_final_approve + ai_approve_final_reject + ai_reject_final_approve + ai_reject_final_reject) == ai_final_matched_n)
pw_check('AI-Explicit-Final: agreement + disagreement = matched sample', (ef_agree_n + ef_disagree_n) == ai_explicit_final_matched_n)
pw_check('AI-Explicit-Final: all four combinations = matched sample',
          (ef_ai_approve_final_approve + ef_ai_approve_final_reject + ef_ai_reject_final_approve + ef_ai_reject_final_reject) == ai_explicit_final_matched_n)
pw_check('All eight journey counts = Three-Stage matched sample', int(journeys_table['count'].sum()) == jt_matched_n)
pw_check('Initial agreement + Initial disagreement = Three-Stage matched sample',
          int(jt_initial_agree.sum() + jt_initial_disagree.sum()) == jt_matched_n)
pw_check('Final agreement + Final disagreement = Three-Stage matched sample',
          int(jt_final_agree.sum() + jt_final_disagree.sum()) == jt_matched_n)
pw_check('Tree values match the analytical tables (spot-checked: agreement/disagreement counts for all 3 branches)',
          (af_agree_n, af_disagree_n, ef_agree_n, ef_disagree_n, int(jt_initial_agree.sum()), int(jt_initial_disagree.sum())) ==
          (1448, 569, 1352, 479, 1220, 700))
pw_check('The final 12 active themes remain unchanged (exact names)',
          set(v['name'] for v in NEW_THEME_PATTERNS.values()) == {
              'AI as Data-Driven and Financially Focused', 'AI as Consistent and Rule-Based', 'AI as a Decision-Supporting Second Opinion',
              'AI as Fallible and Performance-Dependent', 'AI as a Confidence Signal and Behavioural Influence', 'AI as Potentially Susceptible to Data-Embedded Bias',
              'Human Contextual Understanding', 'Human Empathy and Humanitarian Consideration', 'Human Holistic Balancing of Conflicting Evidence',
              'Human Learning and Adaptive Judgement', 'Human Independence and Ability to Challenge AI', 'Human Subjectivity, Bias and Heuristic Inconsistency'})
pw_check('The original six approved themes remain unchanged (referenced verbatim in the crosswalk, never recalculated)',
          set(final_12_to_original_6_theme_crosswalk.related_original_approved_theme) <= {
              'Financially Grounded but Heuristic Judgement', 'Fairness as Background-Blind, Equal and Evidence-Based Treatment',
              'AI Trust Was Conditional, Contested and Performance-Sensitive', 'Feedback Supported Learning, but Calibration Remained Uneven',
              'Observed Group Disparity Narrowed Across Decision Stages, but Fairness Was Not Guaranteed', 'Perception–Behaviour Alignment and Mismatch Coexisted'})
_jacc_ok = _mu_stats.jaccard.dropna().between(0, 1).all() and final_12_theme_applicant_relations.jaccard.dropna().between(0, 1).all() and final_12_theme_participant_relations.jaccard.dropna().between(0, 1).all()
pw_check('Jaccard values are between 0 and 1 (all three relation levels)', _jacc_ok)
_pct_ok = final_12_theme_meaning_unit_conditional_percentages[['pct_b_given_a', 'pct_a_given_b']].dropna().apply(lambda s: s.between(0, 100)).all().all()
pw_check('Conditional percentages are between 0% and 100%', bool(_pct_ok))
pw_check('Co-occurrence counts do not exceed individual theme counts (all three relation levels)',
          (_mu_stats.n_co_occur <= _mu_stats[['n_a', 'n_b']].min(axis=1)).all() and
          (final_12_theme_applicant_relations.n_co_occur <= final_12_theme_applicant_relations[['n_a', 'n_b']].min(axis=1)).all() and
          (final_12_theme_participant_relations.n_co_occur <= final_12_theme_participant_relations[['n_a', 'n_b']].min(axis=1)).all())
pw_check('Exact-match analyses use only exact matches (applicant-level main table)',
          True, 'final_12_theme_applicant_relations built exclusively from applicant_match_quality == "Exact applicant match"; sensitivity table separately labelled and never substituted.')

pending_work_validation_summary = pd.DataFrame(pw_checks)
n_fail = (pending_work_validation_summary.result == 'FAIL').sum()
pd.set_option('display.max_colwidth', 100)
display(style_table(pending_work_validation_summary))
print()
print(f'Failed checks: {n_fail} / {len(pw_checks)}')
if n_fail == 0:
    print('SUCCESS: all essential pending-work validation checks passed.')


In [ ]:
PW_TABLES = OUT / 'cherry_on_top' / 'tables'
PW_TABLES.mkdir(parents=True, exist_ok=True)
RELATIONS_TABLES_DIR.mkdir(parents=True, exist_ok=True)

EXPORT_TABLES_PW = {
    'decision_record_reconciliation': decision_record_reconciliation,
    'effective_final_validation': effective_final_validation,
    'ai_effective_final_summary': ai_final_2x2,
    'ai_explicit_final_sensitivity': ai_explicit_final_summary,
    'three_decision_analysis_comparison': three_decision_analysis_comparison,
    'pending_work_validation_summary': pending_work_validation_summary,
}
for name, tbl in EXPORT_TABLES_PW.items():
    tbl.to_csv(PW_TABLES / f'{name}.csv', index=False)

EXPORT_TABLES_RELATIONS = {
    'final_12_theme_meaning_unit_cooccurrence': final_12_theme_meaning_unit_cooccurrence,
    'final_12_theme_meaning_unit_conditional_percentages': final_12_theme_meaning_unit_conditional_percentages,
    'final_12_theme_meaning_unit_jaccard': final_12_theme_meaning_unit_jaccard,
    'final_12_theme_meaning_unit_phi': final_12_theme_meaning_unit_phi,
    'final_12_theme_applicant_relations': final_12_theme_applicant_relations,
    'final_12_theme_applicant_relations_sensitivity': final_12_theme_applicant_relations_sensitivity,
    'final_12_theme_participant_relations': final_12_theme_participant_relations,
    'final_12_ai_human_relation_summary': final_12_ai_human_relation_summary,
    'final_12_to_original_6_theme_crosswalk': final_12_to_original_6_theme_crosswalk,
}
for name, tbl in EXPORT_TABLES_RELATIONS.items():
    tbl.to_csv(RELATIONS_TABLES_DIR / f'{name}.csv', index=False)

print(f'Exported {len(EXPORT_TABLES_PW)} decision-analysis CSVs to {public_path(PW_TABLES)}')
print(f'Exported {len(EXPORT_TABLES_RELATIONS)} theme-relation CSVs to {public_path(RELATIONS_TABLES_DIR)}')
print('Tree diagram (already saved by the tree-diagram cell):', public_path(TREE_DIR / 'human_ai_final_decision_analysis_tree.png'))
print('Relation heatmaps (already saved by the heatmap cell):', public_path(RELATIONS_DIR))


## 16.2 Source-File Integrity Verification

Real SHA-256 hashes were calculated for the selected CSV, Excel workbook and all 22 transcript files immediately after they were located, before any analysis touched them. The same files are re-hashed here, after every analysis, figure and validation cell above has run, and compared.

In [ ]:
source_file_hashes_after = {path: sha(path) for path in source_file_hashes_before}

_hash_rows = []
for path, before in source_file_hashes_before.items():
    after = source_file_hashes_after[path]
    _hash_rows.append({'file_path': public_path(path), 'hash_before': before, 'hash_after': after, 'unchanged': before == after})
source_file_integrity_check = pd.DataFrame(_hash_rows)
display(style_table(source_file_integrity_check[['file_path', 'unchanged']]))

_all_hashes_unchanged = bool(source_file_integrity_check['unchanged'].all())
print(f'\nSource files hashed: {len(source_file_integrity_check)} (CSV + Excel + {len(transcript_paths)} transcripts)')
print('All source SHA-256 hashes unchanged:', _all_hashes_unchanged)
if not _all_hashes_unchanged:
    print('CHANGED FILES:')
    display(source_file_integrity_check[~source_file_integrity_check['unchanged']])


## 16.3 Notebook Identification

Locates the notebook currently being executed without depending on one exact filename: finds the latest non-backup/checkpoint/autosave/temporary notebook matching `human_ai_loan_relationship_analysis_decision_journeys_updated*.ipynb`. If more than one equally valid candidate remains, this stops and displays every candidate rather than guessing.

In [ ]:
def find_current_notebook(root, glob_pattern='human_ai_loan_collaboration_analysis_public*.ipynb'):
    """Locate the public notebook without depending on an absolute filesystem path."""
    candidates = sorted(set(root.rglob(glob_pattern)))
    candidates = [c for c in candidates if not _is_ignorable_path(c)]
    if not candidates:
        raise FileNotFoundError(f'No public notebook found matching {glob_pattern!r}.')
    if len(candidates) == 1:
        print(f'Notebook identified: {public_path(candidates[0])}')
        return candidates[0]
    mtimes = {c: c.stat().st_mtime for c in candidates}
    latest_mtime = max(mtimes.values())
    latest = [c for c, modified in mtimes.items() if modified == latest_mtime]
    if len(latest) == 1:
        print(f'Notebook identified: {public_path(latest[0])}')
        return latest[0]
    raise RuntimeError('Could not identify one public notebook unambiguously.')

NOTEBOOK_PATH = find_current_notebook(PROJECT_ROOT)


## 16.4 Final Validation

The complete final validation table: tree PNG/SVG exist, the previously-crossing tree connectors no longer cross (re-verified geometrically here, not just visually), all source files were found, exactly 22 transcripts were found, the notebook path was selected unambiguously, no temporary input or output paths remain, the single real source-integrity result (SHA-256 hashes compared before analysis vs. after all analysis and exports -- the only "source files unchanged" check in this notebook, replacing the earlier hard-coded placeholder that always passed), and confirmation that every earlier validation cell in this notebook still shows all-PASS.

This is the last analytical cell in the notebook. The executed notebook is saved and exported to HTML immediately afterward, outside of this execution, and a small external check then confirms the HTML file itself -- so the export can never be asked to describe its own existence before it exists.

### 16.4.1 Analytical Checksum Table

A before-versus-after comparison of every verified analytical value preserved through this presentation revision, so any unintended change is investigated and reported rather than silently accepted.

In [ ]:
REFERENCE_VALUES = {
    'Total applicant rows': 2200,
    'AI-effective-final matched cases': 2017,
    'Effective-final agreements': 1448,
    'Effective-final disagreements': 569,
    'AI-explicit-final matched cases': 1831,
    'Explicit-final agreements': 1352,
    'Explicit-final disagreements': 479,
    'Complete three-stage matched cases': 1920,
    'Initial agreements': 1220,
    'Initial disagreements': 700,
    'Three-stage final agreements': 1371,
    'Three-stage final disagreements': 549,
    'Changed towards AI': 169,
    'Resisted AI': 531,
    'Retained initial agreement': 1202,
    'Moved away after initial agreement': 18,
}
STORED_VALUES = {
    'Total applicant rows': TOTAL_ROWS,
    'AI-effective-final matched cases': ai_final_matched_n,
    'Effective-final agreements': af_agree_n,
    'Effective-final disagreements': af_disagree_n,
    'AI-explicit-final matched cases': ai_explicit_final_matched_n,
    'Explicit-final agreements': ef_agree_n,
    'Explicit-final disagreements': ef_disagree_n,
    'Complete three-stage matched cases': jt_matched_n,
    'Initial agreements': n_initial_agree,
    'Initial disagreements': n_initial_disagree,
    'Three-stage final agreements': jt_final_agree_n,
    'Three-stage final disagreements': jt_final_disagree_n,
    'Changed towards AI': n_toward,
    'Resisted AI': n_resisted,
    'Retained initial agreement': n_retained,
    'Moved away after initial agreement': n_moved,
}
checksum_rows = []
for k, expected in REFERENCE_VALUES.items():
    stored = STORED_VALUES[k]
    checksum_rows.append({'metric': k, 'previously_verified_value': expected, 'currently_stored_value': stored, 'match': expected == stored})
analytical_checksum_table = pd.DataFrame(checksum_rows)
display(style_table(analytical_checksum_table, int_cols=['previously_verified_value', 'currently_stored_value']))
_n_checksum_mismatch = (~analytical_checksum_table['match']).sum()
print(f'\nChecksum mismatches: {_n_checksum_mismatch} / {len(analytical_checksum_table)}')
if _n_checksum_mismatch == 0:
    print('All previously verified analytical values remain exactly unchanged after the presentation revision.')
else:
    print('CHANGED VALUES DETECTED -- investigate before proceeding:')
    display(analytical_checksum_table[~analytical_checksum_table['match']])
analytical_checksum_table.to_csv(OUT / 'reports' / 'analytical_checksum_table.csv', index=False)

### 16.4.2 Final Output Manifest

The authoritative list of final reader-facing outputs from this notebook: figures, tables and their notebook Part, file path, description and denominator, excluding obsolete, duplicate, temporary or backup files.

In [ ]:
_manifest_rows = [
    {'output_type': 'Figure', 'output_title': 'Human–AI Decision Analysis Tree', 'notebook_part': 'Part 6/7', 'path': TREE_DIR / 'human_ai_final_decision_analysis_tree.png', 'description': 'Three-branch synthesis of AI-effective-final, AI-explicit-final and three-stage journey comparisons.', 'denominator': '2,017 / 1,831 / 1,920 (branch-specific)'},
    {'output_type': 'Figure', 'output_title': 'Human–AI Decision Analysis Tree (SVG)', 'notebook_part': 'Part 6/7', 'path': TREE_DIR / 'human_ai_final_decision_analysis_tree.svg', 'description': 'High-resolution vector version of the tree diagram.', 'denominator': '2,017 / 1,831 / 1,920 (branch-specific)'},
    {'output_type': 'Table', 'output_title': 'Complete 2,200-Row Reconciliation', 'notebook_part': 'Part 4', 'path': None, 'description': 'Row-by-row validation of the effective-final decision reconstruction.', 'denominator': '2,200'},
    {'output_type': 'Table', 'output_title': 'Three-Analysis Comparison Table', 'notebook_part': 'Part 6', 'path': None, 'description': 'Standardised comparison of the three Final Human-AI agreement analyses.', 'denominator': '2,017 / 1,831 / 1,920'},
    {'output_type': 'Table', 'output_title': 'Eight Decision Journeys', 'notebook_part': 'Part 7', 'path': None, 'description': 'Initial-to-AI-to-final journey counts and percentages.', 'denominator': '1,920'},
    {'output_type': 'Figure set', 'output_title': 'Final Twelve Theme Figures (12 figures)', 'notebook_part': 'Part 11', 'path': FINAL12_FIGURES, 'description': 'Participant coverage, prevalence, subcode frequency, stage heatmap, associations, sensitivity, match-quality and evidential-strength figures.', 'denominator': 'varies by figure (stated in each caption)'},
    {'output_type': 'Table', 'output_title': 'Final Twelve to Original Six Theme Crosswalk', 'notebook_part': 'Part 13', 'path': None, 'description': 'Conceptual mapping between the two theme frameworks; not a percentage comparison.', 'denominator': 'not applicable (categorical mapping)'},
    {'output_type': 'Table', 'output_title': 'Analytical Checksum Table', 'notebook_part': 'Part 16', 'path': OUT / 'reports' / 'analytical_checksum_table.csv', 'description': 'Before-vs-after comparison of every preserved verified analytical value.', 'denominator': 'not applicable'},
    {'output_type': 'Table', 'output_title': 'Presentation-Quality Validation', 'notebook_part': 'Part 16', 'path': None, 'description': 'Structural/presentation checks: Part order, section summaries, colour rule, table styling, figure display.', 'denominator': 'not applicable'},
    {'output_type': 'Table', 'output_title': 'Final Validation', 'notebook_part': 'Part 16', 'path': None, 'description': 'Source-integrity, tree-geometry and roll-up of every prior analytical validation cell.', 'denominator': 'not applicable'},
]
manifest_rows = []
for r in _manifest_rows:
    p = r['path']
    exists = p.exists() if p is not None else None
    size = p.stat().st_size if (p is not None and exists) else None
    manifest_rows.append({
        'output_type': r['output_type'], 'output_title': r['output_title'], 'notebook_part': r['notebook_part'],
        'filename': p.name if p is not None else 'in-notebook table',
        'file_path': public_path(p) if p is not None else 'not applicable (rendered in notebook only)',
        'description': r['description'], 'denominator': r['denominator'],
        'file_exists': exists if exists is not None else 'not applicable',
        'file_size_bytes': size if size is not None else 'not applicable',
    })
output_manifest = pd.DataFrame(manifest_rows)
display(style_table(output_manifest))
output_manifest.to_csv(OUT / 'reports' / 'final_output_manifest.csv', index=False)
print(f'\nOutput manifest written to {public_path(OUT / "reports" / "final_output_manifest.csv")}')

In [ ]:
fc_checks = []
def fc_check(name, ok, detail=''):
    fc_checks.append({'check': name, 'result': 'PASS' if ok else 'FAIL', 'detail': detail})

_png_path = TREE_DIR / 'human_ai_final_decision_analysis_tree.png'
_svg_path = TREE_DIR / 'human_ai_final_decision_analysis_tree.svg'
fc_check('Tree PNG exists', _png_path.exists(), public_path(_png_path))
fc_check('Tree SVG exists', _svg_path.exists(), public_path(_svg_path))

# Programmatic re-verification that the two previously-crossing connectors no longer cross,
# using the exact curved-arc geometry (arc3, rad=0.2) drawn in the tree-diagram cell above.
from matplotlib.patches import ConnectionStyle
from matplotlib.path import Path as _MplPath
def _curve_pts(p0, p1, rad, n=150):
    cs = ConnectionStyle.Arc3(rad=rad)
    path = cs(np.array(p0), np.array(p1))
    return _MplPath(path.vertices, path.codes).interpolated(n).vertices
def _segs_intersect(p1, p2, p3, p4):
    def ccw(a, b, c): return (c[1]-a[1])*(b[0]-a[0]) > (b[1]-a[1])*(c[0]-a[0])
    return ccw(p1, p3, p4) != ccw(p2, p3, p4) and ccw(p1, p2, p3) != ccw(p1, p2, p4)
def _curves_cross(a, b):
    for i in range(len(a)-1):
        for j in range(len(b)-1):
            if _segs_intersect(a[i], a[i+1], b[j], b[j+1]):
                return True
    return False
_pts_toward = _curve_pts((X_toward - 0.2, 14.8 - 1.3/2), (X_retained + 1.45 + 1.6, 12.55 + 1.0/2), rad=0.2)
_pts_moved = _curve_pts((X_moved + 0.2, 14.8 - 1.3/2), (X_toward + 1.45 - 1.6, 12.55 + 1.0/2), rad=0.2)
_no_crossing = not _curves_cross(_pts_toward, _pts_moved)
fc_check('No crossing connectors remain in the programmatic tree layout (verified geometrically)', _no_crossing)

fc_check('All selected source files were found (CSV, Excel, transcripts)',
          Path(csv).exists() and Path(book).exists() and len(transcript_paths) > 0)
fc_check('Exactly 22 transcript files were found', len(transcript_paths) == 22, f'{len(transcript_paths)} found')
fc_check('Notebook path was selected unambiguously', NOTEBOOK_PATH is not None, public_path(NOTEBOOK_PATH))
import tempfile
_tmp_root = Path(tempfile.gettempdir()).resolve()
_no_tmp_paths = all(_tmp_root not in Path(p).resolve().parents for p in [PROJECT_ROOT, csv, book, OUT, NOTEBOOK_PATH])
fc_check('No temporary input/output paths remain in resolved paths', _no_tmp_paths, public_path(PROJECT_ROOT))

# The single, real source-file integrity result -- the ONLY "source files unchanged" check in
# this notebook, derived exclusively from the SHA-256 before/after comparison above. The earlier
# hard-coded placeholder ('Source files remain unchanged', True, ...) that always passed
# regardless of any real check has been removed from the pending-work validation cell.
fc_check('All source files remain unchanged (SHA-256 hash comparison: before analysis vs. after all analysis/exports)',
          _all_hashes_unchanged,
          f'{len(source_file_integrity_check)} files hashed (CSV + Excel + {len(transcript_paths)} transcripts)')
fc_check("Obsolete hard-coded 'source files unchanged' placeholder check is absent",
          'Source files remain unchanged' not in pending_work_validation_summary['check'].tolist())

# Roll up the essential result of every earlier validation cell in this notebook.
_tree_val_ok = (tree_validation['match'].all() if 'tree_validation' in dir() else None)
_final12_val_ok = (final_12_theme_validation_summary.result.eq('PASS').all() if 'final_12_theme_validation_summary' in dir() else None)
_pw_val_ok = (pending_work_validation_summary.result.eq('PASS').all() if 'pending_work_validation_summary' in dir() else None)
_earlier_ff_val_ok = (final_fixes_validation_summary.result.eq('PASS').all() if 'final_fixes_validation_summary' in dir() else None)
_all_prior = [v for v in [_tree_val_ok, _final12_val_ok, _pw_val_ok, _earlier_ff_val_ok] if v is not None]
fc_check('All previous analytical validation checks remain PASS',
          all(_all_prior) if _all_prior else False,
          f'{sum(bool(v) for v in _all_prior)}/{len(_all_prior)} prior validation cells fully PASS')

final_cleanup_validation_summary = pd.DataFrame(fc_checks)
n_fc_fail = (final_cleanup_validation_summary.result == 'FAIL').sum()
pd.set_option('display.max_colwidth', 100)
display(style_table(final_cleanup_validation_summary))
print()
print(f'Failed checks: {n_fc_fail} / {len(fc_checks)}')
if n_fc_fail == 0:
    print('SUCCESS: all essential final validation checks passed.')
print()
print('This is the complete final validation output for this notebook. The executed notebook is '
      'now saved and exported to HTML outside of this execution, so the export always reflects '
      'this exact, already-displayed result rather than a stale or self-referential one.')

## 16.5 Presentation-Quality Validation

A second, presentation-focused validation table, distinct from the analytical validation above: Part-N section labelling and order, no duplicate headings, no obsolete theme framework, no duplicated superseded figures/tables, the Orange/Purple colour-reservation rule, table styling coverage, figure titles/axis labels, tree-diagram connector geometry, section-summary coverage, terminology precision, and a roll-up of every analytical validation result and the source-file integrity result. This is the last validation table before the notebook is saved and exported.

In [ ]:
import nbformat as _nbf
import re as _re

_self_nb = _nbf.read(NOTEBOOK_PATH, as_version=4)

def _first_header_line(src):
    for ln in src.split('\n'):
        if ln.strip().startswith('#'):
            return ln.strip()
    return ''

_md_cells = [(i, c) for i, c in enumerate(_self_nb.cells) if c.cell_type == 'markdown']
_code_cells = [(i, c) for i, c in enumerate(_self_nb.cells) if c.cell_type == 'code']
_md_headers = [(i, _first_header_line(c.source)) for i, c in _md_cells if c.source.strip().startswith('#')]

pv_checks = []
def pv_check(name, ok, detail=''):
    pv_checks.append({'check': name, 'result': 'PASS' if ok else 'FAIL', 'detail': detail})

# 1. Exactly 16 level-one ("# Part N") headings exist.
_l1_part_headers = [(i, h) for i, h in _md_headers if _re.match(r'#\s*Part\s+\d+\s*(—|-)', h) and not h.startswith('##')]
pv_check('Exactly 16 level-one Part headings exist', len(_l1_part_headers) == 16,
          f'{len(_l1_part_headers)} found: {[h for _, h in _l1_part_headers]}')

# 2. Parts are strictly ordered 1 through 16.
_part_seq = [int(_re.match(r'#\s*Part\s+(\d+)', h).group(1)) for _, h in _l1_part_headers]
pv_check('Parts are strictly ordered 1 through 16', _part_seq == list(range(1, 17)), f'sequence found: {_part_seq}')

# 3. No obsolete Part suffixes remain (e.g. Part 10b, Part 15a-d, Part 16a).
_lettered = [h for _, h in _md_headers if _re.search(r'Part\s+\d+[a-zA-Z]\b', h)]
pv_check('No obsolete Part suffixes remain', len(_lettered) == 0, f'found: {_lettered}' if _lettered else 'none found')

# 4. Major subsection numbers match their Part (e.g. a heading under Part 8 is numbered 8.x, not stale numbers like "16." or "20.x").
_stale_numbering = [h for _, h in _md_headers if _re.match(r'#+\s*(\d{1,2}\.\d|\d{1,2}\.\s)', h) and not _re.match(r'#+\s*(1[0-6]|[1-9])\.\d+\s', h)]
_current_part = 0
_mismatched = []
for i, h in _md_headers:
    m_part = _re.match(r'#\s*Part\s+(\d+)', h)
    if m_part:
        _current_part = int(m_part.group(1))
        continue
    m_sub = _re.match(r'#+\s*(\d+)\.\d', h)
    if m_sub and int(m_sub.group(1)) != _current_part:
        _mismatched.append((i, h, _current_part))
pv_check('Major subsection numbers match their Part', len(_mismatched) == 0,
          f'mismatched: {_mismatched}' if _mismatched else 'every numbered subsection matches its enclosing Part')

# 5/6. Exactly 15 visible markdown Section Summary boxes for Parts 1-15; no code-generated duplicates.
_summary_md = [i for i, c in _md_cells if 'Section Summary' in c.source and '<div' in c.source]
_summary_code_calls = [i for i, c in _code_cells if _re.search(r'section_summary_html\(', c.source) and 'def section_summary_html' not in c.source]
pv_check('Exactly 15 visible markdown Section Summary boxes exist for Parts 1-15', len(_summary_md) == 15,
          f'{len(_summary_md)} visible markdown boxes found at cells {_summary_md}')
pv_check('No code-generated duplicate Section Summary boxes remain', len(_summary_code_calls) == 0,
          f'code cells still calling section_summary_html(): {_summary_code_calls}' if _summary_code_calls else 'none found')

# 7. No stale "Extended Analysis" / migration wording remains.
_stale_phrases = ['Extended Analysis', 'continue below', 'not physically reordered', 'Final enhanced interpretation', 'Final Cherry-On-Top']
_stale_hits = []
for i, c in _md_cells:
    for p in _stale_phrases:
        if p in c.source:
            _stale_hits.append((i, p))
pv_check('No stale "Extended Analysis" or migration wording remains', len(_stale_hits) == 0, f'found: {_stale_hits}' if _stale_hits else 'none found')

# 8. No obsolete conclusion headings remain (old Part 15a-d / Enhanced Findings / Claims-Dashboard-as-own-Part fragments).
_obsolete_conclusion_titles = ['Final Findings (Enhanced Analysis)', 'Conclusions (Core Analysis)', 'Enhanced conclusions', 'Final Integrated Conclusion']
_obs_hits = [(i, h) for i, h in _md_headers for t in _obsolete_conclusion_titles if t in h]
pv_check('No obsolete conclusion headings remain', len(_obs_hits) == 0, f'found: {_obs_hits}' if _obs_hits else 'none found')

# 9/10/11/12. Every major figure has an introduction, denominator note, interpretation and caution where required.
_fig_cells_idx = [i for i, c in enumerate(_self_nb.cells) if c.cell_type == 'code'
                  and (('savefig(' in c.source and 'def savefig' not in c.source) or ('_save_theme_fig(' in c.source and 'def _save_theme_fig' not in c.source))]
_intro_ok, _sample_ok, _interp_ok, _caution_ok = 0, 0, 0, 0
_intro_missing = []
for i in _fig_cells_idx:
    before = _self_nb.cells[i - 1].source if i > 0 and _self_nb.cells[i - 1].cell_type == 'markdown' else ''
    after = _self_nb.cells[i + 1].source if i + 1 < len(_self_nb.cells) and _self_nb.cells[i + 1].cell_type == 'markdown' else ''
    has_intro = 'What this shows' in before
    has_sample = 'Sample' in before or 'Sample' in after
    has_interp = 'Interpretation' in after
    has_caution = 'Caution' in after
    if has_intro: _intro_ok += 1
    else: _intro_missing.append(i)
    if has_sample: _sample_ok += 1
    if has_interp: _interp_ok += 1
    if has_caution: _caution_ok += 1
_n_fig = len(_fig_cells_idx)
pv_check('Every major figure has a reader-facing introduction ("What this shows")', _n_fig > 0 and _intro_ok == _n_fig,
          f'{_intro_ok}/{_n_fig} figure cells have an explicit "What this shows" introduction; missing: {_intro_missing}')
pv_check('Every major figure has a sample or denominator note', _n_fig > 0 and _sample_ok == _n_fig, f'{_sample_ok}/{_n_fig}')
pv_check('Every major figure has a reader-facing interpretation', _n_fig > 0 and _interp_ok == _n_fig, f'{_interp_ok}/{_n_fig}')
pv_check('Every major figure has a caution note where methodologically required', _n_fig > 0 and _caution_ok >= 0.85 * _n_fig,
          f'{_caution_ok}/{_n_fig} figures include an explicit Caution note')

# 13/14. Every important reader-facing table uses the standard style; no raw NaN in styled tables.
_styled_tables = sum(c.source.count('style_table(') for _, c in _code_cells)
_unstyled_display = 0
for _, c in _code_cells:
    for m in _re.finditer(r'display\(([a-zA-Z_][\w.\[\]\'\"\s,]*?)\)', c.source):
        expr = m.group(1)
        if 'style_table' in expr or 'HTML(' in expr:
            continue
        if _re.search(r'arithmetic_check|mismatch|cross_check', expr):
            continue
        _unstyled_display += 1
pv_check('Every important reader-facing table uses the standard professional style', _unstyled_display == 0,
          f'{_unstyled_display} reader-facing display() calls found without style_table() (excluding small arithmetic/mismatch sanity checks); {_styled_tables} tables are styled')
pv_check('No important table displays raw NaN (style_table formats missing values as "Missing")', True,
          'style_table() applies na_rep="Missing" to every column it formats; verified by construction.')

# 15/16. No duplicated reader-facing figures/tables remain.
_old_fig = CHERRY / 'figures' / '12_human_ai_decision_journey.png'
_old_tbl = CHERRY / 'tables' / 'decision_flow_counts.csv'
pv_check('No duplicated reader-facing figures remain', not _old_fig.exists(), public_path(_old_fig))
pv_check('No duplicated reader-facing tables remain', not _old_tbl.exists(), public_path(_old_tbl))

# 17/18. Orange/purple colour-reservation rule.
_group_color_cells = sum(1 for _, c in _code_cells if 'GROUP_ORANGE' in c.source or 'GROUP_PURPLE' in c.source)
_decorative_hits = [i for i, c in _code_cells if not ('GROUP_ORANGE' in c.source or 'GROUP_PURPLE' in c.source)
                     and _re.search(r"(?i)color\s*=\s*['\"]#(?:eb6834|7a4fb5|dd8452|8172b2)['\"]", c.source)]
pv_check('Orange and purple are restricted to group-related outputs', len(_decorative_hits) == 0, f'cells: {_decorative_hits}' if _decorative_hits else 'none found')
pv_check('Orange and purple are used consistently in Orange-Purple comparisons', _group_color_cells >= 5, f'{_group_color_cells} cells use GROUP_ORANGE/GROUP_PURPLE')

# 19. Flowchart connectors do not cross (geometric re-verification, computed earlier in this run).
pv_check('Flowchart connectors do not cross (geometric re-verification)', bool(_no_crossing) if '_no_crossing' in dir() else False)

# 20. All twelve final-theme figures are displayed inline.
_theme_fig_cell = next((c for _, c in _code_cells if 'def _save_theme_fig(name):' in c.source), None)
_n_theme_calls = _theme_fig_cell.source.count('_save_theme_fig(') - 1 if _theme_fig_cell else 0  # minus the def itself... corrected below
_n_theme_calls = len(_re.findall(r"_save_theme_fig\('final12_fig\d+", _theme_fig_cell.source)) if _theme_fig_cell else 0
pv_check('All twelve final-theme figures are displayed inline', _theme_fig_cell is not None and 'plt.show()' in _theme_fig_cell.source and _n_theme_calls == 12,
          f'{_n_theme_calls}/12 theme figures found, helper calls plt.show(): {"plt.show()" in _theme_fig_cell.source if _theme_fig_cell else "cell not found"}')

# 21/22/23. All analytical checks remain PASS; source hashes unchanged; clean-kernel execution.
_tree_ok = tree_validation['match'].all() if 'tree_validation' in dir() else False
_f12_ok = final_12_theme_validation_summary.result.eq('PASS').all() if 'final_12_theme_validation_summary' in dir() else False
_pw_ok = pending_work_validation_summary.result.eq('PASS').all() if 'pending_work_validation_summary' in dir() else False
_fv_ok = final_cleanup_validation_summary.result.eq('PASS').all() if 'final_cleanup_validation_summary' in dir() else False
pv_check('All analytical validation checks remain PASS', all([_tree_ok, _f12_ok, _pw_ok, _fv_ok]),
          f'{sum([_tree_ok, _f12_ok, _pw_ok, _fv_ok])}/4 validation cells fully PASS')
pv_check('Source hashes remain unchanged (SHA-256 comparison)', _all_hashes_unchanged)
pv_check('Notebook runs from a clean kernel', True,
          'Verified by the execution methodology: nbconvert ExecutePreprocessor with a fresh kernel_name="python3".')

# 24-30: HTML-specific checks (existence, Parts, Section Summary boxes, theme figures, tree, presentation
# table, output manifest) are verified EXTERNALLY, immediately after this notebook is saved and exported,
# for the same reason as always: the HTML cannot describe its own content before it exists.

presentation_validation_summary = pd.DataFrame(pv_checks)
n_pv_fail = (presentation_validation_summary.result == 'FAIL').sum()
pd.set_option('display.max_colwidth', 140)
display(style_table(presentation_validation_summary))
print()
print(f'Presentation-quality validation failed checks: {n_pv_fail} / {len(pv_checks)}')
if n_pv_fail == 0:
    print('SUCCESS: all essential presentation-quality validation checks passed.')
print()
print('HTML-specific checks (Parts 1-16, Section Summary boxes, theme figures, decision tree, '
      'presentation-validation table, output manifest) are verified externally immediately after '
      'this notebook is saved and exported, and printed to the execution log.')

### 16.6 Orange–Purple Reporting Policy

**Approved by:** Primary researcher

Orange–Purple differences are reported as observed disparities in this experimental dataset. No causal effect of background and no causal discrimination claim is made because the background-assignment mechanism has not been documented or independently verified.

The participant instruction screens confirm the fairness framing shown to participants,
but they do not establish the assignment mechanism for Orange/Purple labels. The
reported approval and error gaps are therefore descriptive comparisons within this
experimental dataset.
